In [ ]:
# [env] load .env (KEY=VALUE lines) into os.environ -- see .env.sample
# real env vars / Colab / Kaggle secrets take precedence; empty values are ignored
import os, pathlib
for _p in [pathlib.Path(".env"), pathlib.Path("../.env"),
           pathlib.Path("/content/drive/MyDrive/.env"),
           *sorted(pathlib.Path("/kaggle/input").glob("*/.env"))]:
    if _p.is_file():
        for _line in _p.read_text().splitlines():
            _k, _sep, _v = _line.partition("=")
            _k, _v = _k.strip(), _v.strip().strip("\"'")
            if _sep and _k and _v and not _k.startswith("#"):
                os.environ.setdefault(_k, _v)
        print("env loaded from", _p)
        break
else:
    print("no .env found -- copy .env.sample to .env and fill it in")

# Downloads

In [1]:
!pip install -q diffusers>=0.32.2 transformers>=4.49.0 accelerate>=1.4.0 peft>=0.14.0 \
    huggingface_hub torchmetrics lpips clean-fid kagglehub wandb scikit-learn pandas matplotlib


In [2]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.7 MB/s eta 0:00:00


# Libraries

In [3]:
import os, gc, random, warnings, glob
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, DDIMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
from accelerate import Accelerator
from accelerate.utils import set_seed

import kagglehub
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

@dataclass
class CFG:
    data_root:   str   = ""   # filled by kagglehub
    output_dir:  str   = "/content/checkpoints"
    img_size:    int   = 256
    seasons:     tuple = ("spring", "summer", "fall", "winter")
    val_fraction: float = 0.05
    max_pairs:   int   = None      # None = ALL 600k

    sd_model_id: str   = "runwayml/stable-diffusion-v1-5"
    lora_rank:   int   = 64        # ← was 16, now 64 for Blackwell
    lora_alpha:  int   = 64
    use_dora:    bool  = True

    num_train_steps: int = 200_000
    batch_size:      int   = 16    # ← fits in 96 GB
    grad_accum:      int   = 1     # effective batch = 16
    lr:              float = 1e-4  # slightly higher for big batch
    lr_warmup_steps: int   = 2_000
    mixed_precision: str   = "bf16" # ← Blackwell loves bf16
    gradient_checkpointing: bool = False # ← disable for speed; you have VRAM
    ema_decay:       float = 0.9999

    num_train_timesteps: int = 1000
    ddim_steps:          int = 50

    log_every:  int = 100
    save_every: int = 2_000
    vis_every:  int = 5_000
    seed:       int = 6
    patience:        int = 10_000

cfg = CFG()
set_seed(cfg.seed)
os.makedirs(cfg.output_dir, exist_ok=True)
# print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
# print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [4]:
import os
from google.colab import userdata; os.environ["HF_WRITE_TOKEN"] = userdata.get("HF_TOKEN")
!pip install -q lpips torchmetrics safetensors scipy torch-fidelity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 2.8 MB/s eta 0:00:00


Downloading Dataset

In [5]:
import kagglehub
path = kagglehub.dataset_download("shambac/augmented-sentinel-1-2")
print("Dataset path:", path)

cfg.data_root = path
root = Path(cfg.data_root)
print("Seasons:", [d.name for d in root.iterdir() if d.is_dir()])

100%|██████████| 48.2G/48.2G [07:48<00:00, 111MB/s]


Extracting files...
Dataset path: /root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1
Seasons: ['winter', 'summer', 'spring', 'fall']


Split

In [ ]:
def collect_pairs(root: Path, seasons: tuple, max_pairs: int = None):
    root_str = root.as_posix()
    season_buckets = {s: [] for s in seasons}

    for season in seasons:
        csv_files = list((root / season).glob("*.csv"))
        if not csv_files:
            print(f"[WARN] No CSV for {season}"); continue
        df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
        df["season"] = season
        df["region"] = df["region"].str.strip().str.lower()
        df["s1_fileName"] = (df["s1_fileName"]
                             .str.replace("\\", "/", regex=False)
                             .str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True))
        df["s2_fileName"] = df["s2_fileName"].str.replace("\\", "/", regex=False)
        df["s1"] = root_str + "/" + df["s1_fileName"]
        df["s2"] = root_str + "/" + df["s2_fileName"]
        season_buckets[season] = df[["s1","s2","season","region"]].to_dict("records")
        print(f"  [{season}] {len(season_buckets[season]):,} pairs")

    active = [s for s in seasons if season_buckets[s]]
    if max_pairs is None:
        pairs = [p for s in active for p in season_buckets[s]]
    else:
        per = max_pairs // len(active)
        pairs = []
        for s in active:
            b = season_buckets[s].copy(); random.shuffle(b)
            pairs.extend(b[:per])
    random.shuffle(pairs)
    return pairs


class SAROpticalDataset(Dataset):
    def __init__(self, pairs, img_size=256, augment=True):
        self.pairs, self.img_size, self.augment = pairs, img_size, augment
    def __len__(self): return len(self.pairs)
    def _load(self, path, mode):
        img = Image.open(path).convert(mode)
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        if mode == "L":
            arr = np.stack([arr, arr, arr], axis=2)
        t = torch.from_numpy(arr).permute(2,0,1)
        return t * 2.0 - 1.0
    def __getitem__(self, idx):
        p = self.pairs[idx]
        sar, opt = self._load(p["s1"], "L"), self._load(p["s2"], "RGB")
        if self.augment:
            if random.random() > 0.5: sar = TF.hflip(sar); opt = TF.hflip(opt)
            if random.random() > 0.5: sar = TF.vflip(sar); opt = TF.vflip(opt)
            k = random.randint(0,3)
            if k: sar = torch.rot90(sar, k, [1,2]); opt = torch.rot90(opt, k, [1,2])
        return {"sar": sar, "optical": opt, "season": p["season"], "region": p["region"]}


all_pairs = collect_pairs(root, cfg.seasons, cfg.max_pairs)
print(f"\nTotal: {len(all_pairs):,}")

strata = [f"{p['season']}_{p['region']}" for p in all_pairs]
train_p, val_p = train_test_split(all_pairs, test_size=cfg.val_fraction,
                                  stratify=strata, random_state=cfg.seed)

train_ds = SAROpticalDataset(train_p, cfg.img_size, augment=True)
val_ds   = SAROpticalDataset(val_p,   cfg.img_size, augment=False)

train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                      num_workers=4, pin_memory=True, drop_last=True,
                      prefetch_factor=2, persistent_workers=True)
val_dl   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                      num_workers=2, pin_memory=True)

print(f"Train {len(train_ds):,} | Val {len(val_ds):,}")

# Model

In [ ]:
vae = AutoencoderKL.from_pretrained(cfg.sd_model_id, subfolder="vae",
                                    torch_dtype=torch.bfloat16)
vae.requires_grad_(False)

unet = UNet2DConditionModel.from_pretrained(cfg.sd_model_id, subfolder="unet",
                                            torch_dtype=torch.float32)

# Expand conv_in 4 → 8
old_conv = unet.conv_in
new_conv = nn.Conv2d(8, old_conv.out_channels, old_conv.kernel_size,
                     old_conv.stride, old_conv.padding,
                     bias=(old_conv.bias is not None))
with torch.no_grad():
    new_conv.weight[:, :4].copy_(old_conv.weight)
    new_conv.weight[:, 4:].zero_()
    if old_conv.bias is not None: new_conv.bias.copy_(old_conv.bias)
unet.conv_in = new_conv
unet.config.in_channels = 8

text_embed_dim = unet.config.cross_attention_dim
null_text_embed = nn.Parameter(torch.randn(1, 77, text_embed_dim) * 0.01)

# DoRA targets: attention + FFN/MLP + block projections
target_modules = [
    "to_q", "to_k", "to_v", "to_out.0",
    "add_q_proj", "add_k_proj", "add_v_proj",
    "ff.net.0.proj", "ff.net.2",      # ← MLP/FFN (you already had these)
    "proj_in", "proj_out",            # ← extra linear projections (new)
]

lora_cfg = LoraConfig(
    r=cfg.lora_rank, lora_alpha=cfg.lora_alpha, use_dora=cfg.use_dora,
    init_lora_weights="gaussian", target_modules=target_modules,
    lora_dropout=0.0, bias="none",
)
unet = get_peft_model(unet, lora_cfg)
unet.conv_in.weight.requires_grad_(True)
if unet.conv_in.bias is not None: unet.conv_in.bias.requires_grad_(True)

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
total     = sum(p.numel() for p in unet.parameters())
print(f"DoRA rank={cfg.lora_rank} | Trainable {trainable/1e6:.2f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)")

noise_scheduler = DDPMScheduler(num_train_timesteps=cfg.num_train_timesteps,
                                beta_schedule="scaled_linear", prediction_type="epsilon")
ddim_scheduler  = DDIMScheduler.from_config(noise_scheduler.config)
ddim_scheduler.set_timesteps(cfg.ddim_steps)

ema_unet = EMAModel(unet.parameters(), decay=cfg.ema_decay)

Accelerator

In [ ]:
trainable_params = list(filter(lambda p: p.requires_grad, unet.parameters()))
trainable_params.append(null_text_embed)

optimizer = torch.optim.AdamW(trainable_params, lr=cfg.lr,
                              betas=(0.9,0.999), weight_decay=1e-2, eps=1e-8)

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=cfg.lr_warmup_steps * cfg.grad_accum,
    num_training_steps=cfg.num_train_steps * cfg.grad_accum,
)

accelerator = Accelerator(
    mixed_precision=cfg.mixed_precision,
    gradient_accumulation_steps=cfg.grad_accum,
    log_with="wandb",
    project_dir=cfg.output_dir,
)

set_seed(cfg.seed + accelerator.process_index)
unet, vae, optimizer, train_dl, lr_scheduler = accelerator.prepare(
    unet, vae, optimizer, train_dl, lr_scheduler
)

ema_unet.to(accelerator.device)
null_text_embed = null_text_embed.to(accelerator.device)

# --- RESUME LOGIC ---
RESUME_FROM = None
RESUME_STEP = 0
ckpts = sorted(glob.glob(os.path.join(cfg.output_dir, "step_*")))
if ckpts:
    RESUME_FROM = ckpts[-1]
    RESUME_STEP = int(os.path.basename(RESUME_FROM).split("_")[1])
    print(f"Resuming from {RESUME_FROM}")
    accelerator.load_state(RESUME_FROM)
    ema_path = os.path.join(RESUME_FROM, "ema.bin")
    if os.path.exists(ema_path):
        ema_unet.load_state_dict(torch.load(ema_path, map_location=accelerator.device))
    emb_path = os.path.join(RESUME_FROM, "null_embed.pt")
    if os.path.exists(emb_path):
        null_text_embed.data = torch.load(emb_path, map_location=accelerator.device).data
    for _ in range(RESUME_STEP * cfg.grad_accum):
        lr_scheduler.step()
    print(f"Fast-forwarded scheduler to step {RESUME_STEP}")
else:
    print("Starting from scratch.")

if accelerator.is_main_process:
    import wandb
    wandb.login()
    accelerator.init_trackers(
        project_name="sar-optical-blackwell",
        config={"lr": cfg.lr, "batch": cfg.batch_size, "rank": cfg.lora_rank,
                "steps": cfg.num_train_steps, "dataset": len(all_pairs)},
        init_kwargs={"wandb": {"name": f"dora-r{cfg.lora_rank}-blackwell"}}
    )

# Training Loop

In [ ]:
from IPython.display import display
import glob, shutil

# --- Early stopping / best-model tracking ---
BEST_LPIPS = float("inf")
BEST_STEP  = 0
PATIENCE   = 10_000   # stop if no improvement for 10k steps
LAST_IMPROVED = 0

@torch.no_grad()
def encode_to_latent(images):
    images = images.to(dtype=vae.dtype)
    vae_mod = vae.module if hasattr(vae, "module") else vae
    return vae_mod.encode(images).latent_dist.sample() * vae_mod.config.scaling_factor

@torch.no_grad()
def run_validation_and_plot(step, out_dir, n_show=8):
    """Returns average LPIPS on validation set + shows grid inline."""
    os.makedirs(out_dir, exist_ok=True)

    # Pick 2 per season for grid
    selected = []
    for s in cfg.seasons:
        pool = [p for p in val_p if p["season"] == s]
        if pool: selected.extend(random.sample(pool, min(2, len(pool))))
    selected = selected[:n_show]

    vae_mod = vae.module if hasattr(vae, "module") else vae
    unet_eval = accelerator.unwrap_model(unet)
    unet_eval.eval()

    # --- Compute LPIPS on a small val subset (fast) ---
    val_lpips_samples = []
    val_subset = random.sample(val_p, min(32, len(val_p)))
    for pair in val_subset:
        item = SAROpticalDataset([pair], cfg.img_size, False)[0]
        sar = item["sar"].unsqueeze(0).to(accelerator.device, dtype=vae.dtype)
        sar_lat = vae_mod.encode(sar).latent_dist.mean * vae_mod.config.scaling_factor
        lat = torch.randn_like(sar_lat) * ddim_scheduler.init_noise_sigma
        emb = null_text_embed.to(vae.dtype).expand(1,-1,-1)
        ddim_scheduler.set_timesteps(cfg.ddim_steps)
        for t in ddim_scheduler.timesteps:
            mi = torch.cat([lat, sar_lat], 1)
            npred = unet_eval(mi.float(), t.unsqueeze(0).to(accelerator.device),
                              encoder_hidden_states=emb.float()).sample
            lat = ddim_scheduler.step(npred.to(vae.dtype), t, lat).prev_sample
        pred = vae_mod.decode(lat / vae_mod.config.scaling_factor).sample.squeeze(0).clamp(-1,1)
        # quick LPIPS on single image
        p = pred.unsqueeze(0).float().to("cuda")
        g = item["optical"].unsqueeze(0).float().to("cuda")
        val_lpips_samples.append(lpips_fn(p, g).item())

    avg_lpips = np.mean(val_lpips_samples)

    # --- Build visual grid ---
    fig, axes = plt.subplots(len(selected), 3, figsize=(13, 3.8 * len(selected)))
    if len(selected) == 1: axes = axes.reshape(1, -1)

    for i, pair in enumerate(selected):
        item = SAROpticalDataset([pair], cfg.img_size, False)[0]
        sar = item["sar"].unsqueeze(0).to(accelerator.device, dtype=vae.dtype)
        sar_lat = vae_mod.encode(sar).latent_dist.mean * vae_mod.config.scaling_factor
        lat = torch.randn_like(sar_lat) * ddim_scheduler.init_noise_sigma
        emb = null_text_embed.to(vae.dtype).expand(1,-1,-1)
        ddim_scheduler.set_timesteps(cfg.ddim_steps)
        for t in ddim_scheduler.timesteps:
            mi = torch.cat([lat, sar_lat], 1)
            npred = unet_eval(mi.float(), t.unsqueeze(0).to(accelerator.device),
                              encoder_hidden_states=emb.float()).sample
            lat = ddim_scheduler.step(npred.to(vae.dtype), t, lat).prev_sample
        pred = vae_mod.decode(lat / vae_mod.config.scaling_factor).sample.squeeze(0).clamp(-1,1).float().cpu()

        def dn(t): return ((t.clamp(-1,1)+1)/2).permute(1,2,0).numpy()

        axes[i,0].imshow(dn(item["sar"])[:,:,0], cmap="gray")
        axes[i,0].set_title(f"SAR | {pair['season']} | {pair['region']}", fontsize=8)
        axes[i,0].axis("off")

        axes[i,1].imshow(dn(pred))
        axes[i,1].set_title(f"Predicted (step {step})", fontsize=8)
        axes[i,1].axis("off")

        axes[i,2].imshow(dn(item["optical"]))
        axes[i,2].set_title("Ground Truth", fontsize=8)
        axes[i,2].axis("off")

    plt.suptitle(f"SAR → Optical | Step {step} | Val LPIPS {avg_lpips:.4f} (↓ better)", fontsize=12, y=1.01)
    plt.tight_layout()

    # --- INLINE DISPLAY FOR COLAB ---
    display(fig)

    p = os.path.join(out_dir, f"val_step_{step:06d}.png")
    plt.savefig(p, dpi=120, bbox_inches="tight")
    plt.close(fig)

    unet_eval.train()
    return avg_lpips, p


# --- Training Loop ---
global_step = RESUME_STEP
unet.train()

# LPIPS for validation (declare here so validation can use it)
lpips_fn = lpips_lib.LPIPS(net="alex").to(accelerator.device)

pbar = tqdm(total=cfg.num_train_steps, initial=global_step,
            disable=not accelerator.is_main_process)

while global_step < cfg.num_train_steps:
    for batch in train_dl:
        with accelerator.accumulate(unet):
            sar = batch["sar"].to(accelerator.device)
            opt = batch["optical"].to(accelerator.device)

            opt_lat = encode_to_latent(opt)
            sar_lat = encode_to_latent(sar)
            noise   = torch.randn_like(opt_lat)
            B = opt_lat.shape[0]
            ts = torch.randint(0, cfg.num_train_timesteps, (B,),
                               device=accelerator.device, dtype=torch.long)

            noisy = noise_scheduler.add_noise(opt_lat, noise, ts)
            model_in = torch.cat([noisy, sar_lat], dim=1)
            enc = null_text_embed.to(opt_lat.dtype).expand(B, -1, -1)

            noise_pred = unet(model_in, ts, encoder_hidden_states=enc).sample
            loss = F.mse_loss(noise_pred.float(), noise.float())

            accelerator.backward(loss)
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(accelerator.unwrap_model(unet).parameters(), 1.0)

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        if accelerator.sync_gradients:
            ema_unet.step(accelerator.unwrap_model(unet).parameters())
            global_step += 1
            lr_now = lr_scheduler.get_last_lr()[0]
            pbar.update(1)
            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "lr": f"{lr_now:.1e}",
                "best": f"{BEST_STEP}",
                "best_l": f"{BEST_LPIPS:.4f}"
            })

            if global_step % cfg.log_every == 0 and accelerator.is_main_process:
                accelerator.log({
                    "train/loss": loss.item(),
                    "train/lr": lr_now,
                    "train/best_lpips": BEST_LPIPS,
                    "train/best_step": BEST_STEP,
                }, step=global_step)

            # --- VALIDATION + INLINE PLOT ---
            if global_step % cfg.vis_every == 0 and accelerator.is_main_process:
                tqdm.write(f"\n>>> Running validation at step {global_step}...")
                ema_unet.copy_to(accelerator.unwrap_model(unet).parameters())
                val_lpips, grid_path = run_validation_and_plot(
                    global_step,
                    os.path.join(cfg.output_dir, "val_grids")
                )
                tqdm.write(f">>> Val LPIPS: {val_lpips:.4f} | Grid: {grid_path}\n")

                # --- BEST MODEL SAVE ---
                if val_lpips < BEST_LPIPS:
                    BEST_LPIPS = val_lpips
                    BEST_STEP = global_step
                    LAST_IMPROVED = global_step

                    best_dir = os.path.join(cfg.output_dir, "best_model")
                    if os.path.exists(best_dir):
                        shutil.rmtree(best_dir)
                    os.makedirs(best_dir, exist_ok=True)

                    uw = accelerator.unwrap_model(unet)
                    uw.save_pretrained(os.path.join(best_dir, "unet_adapter"))
                    torch.save(null_text_embed, os.path.join(best_dir, "null_embed.pt"))
                    torch.save(ema_unet.state_dict(), os.path.join(best_dir, "ema.bin"))

                    # Save metadata
                    with open(os.path.join(best_dir, "info.txt"), "w") as f:
                        f.write(f"step: {global_step}\nlpips: {val_lpips:.6f}\n")

                    tqdm.write(f"*** NEW BEST MODEL *** LPIPS {val_lpips:.4f} -> {best_dir}")
                else:
                    # Early stopping check
                    if global_step - LAST_IMPROVED > PATIENCE:
                        tqdm.write(f"\n!!! EARLY STOPPING !!! No improvement for {PATIENCE} steps. Best was step {BEST_STEP}.")
                        global_step = cfg.num_train_steps  # break outer loop
                        break

            # --- REGULAR RESUME CHECKPOINT (full state) ---
            if global_step % cfg.save_every == 0 and accelerator.is_main_process:
                cdir = os.path.join(cfg.output_dir, f"step_{global_step}")
                os.makedirs(cdir, exist_ok=True)
                accelerator.save_state(cdir)
                torch.save(ema_unet.state_dict(), os.path.join(cdir, "ema.bin"))
                torch.save(null_text_embed, os.path.join(cdir, "null_embed.pt"))
                tqdm.write(f"Resume checkpoint -> {cdir}")

                # Cleanup old resume checkpoints (keep last 3)
                all_ckpts = sorted(glob.glob(os.path.join(cfg.output_dir, "step_*")))
                for old in all_ckpts[:-3]:
                    shutil.rmtree(old)
                    tqdm.write(f"Cleaned old checkpoint: {old}")

        if global_step >= cfg.num_train_steps:
            break

    gc.collect()
    torch.cuda.empty_cache()
    if global_step >= cfg.num_train_steps:
        break

pbar.close()

# --- FINAL SAVE ---
if accelerator.is_main_process:
    final_dir = os.path.join(cfg.output_dir, f"final_step_{global_step}")
    os.makedirs(final_dir, exist_ok=True)
    uw = accelerator.unwrap_model(unet)

    # Save final EMA state
    ema_unet.copy_to(uw.parameters())
    uw.save_pretrained(os.path.join(final_dir, "unet_adapter"))
    merged = uw.merge_and_unload()
    merged.save_pretrained(os.path.join(final_dir, "unet_full"))
    torch.save(null_text_embed, os.path.join(final_dir, "null_embed.pt"))
    torch.save(ema_unet.state_dict(), os.path.join(final_dir, "ema.bin"))

    print(f"\nTraining complete. Final -> {final_dir}")
    print(f"BEST MODEL was step {BEST_STEP} with LPIPS {BEST_LPIPS:.4f}")
    print(f"Located at: {os.path.join(cfg.output_dir, 'best_model')}")

accelerator.end_training()

# Evaluation

In [ ]:
import glob
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
import lpips as lpips_lib

# Find latest final checkpoint
final_dirs = sorted(glob.glob(os.path.join(cfg.output_dir, "final_step_*")))
if not final_dirs:
    raise FileNotFoundError("No final checkpoint found. Train first or set path manually.")

ckpt = final_dirs[-1]
print(f"Evaluating from: {ckpt}")

# Load base + adapter
eval_vae = AutoencoderKL.from_pretrained(cfg.sd_model_id, subfolder="vae",
                                         torch_dtype=torch.bfloat16).to("cuda").eval()
eval_vae.requires_grad_(False)

base_unet = UNet2DConditionModel.from_pretrained(cfg.sd_model_id, subfolder="unet",
                                                  torch_dtype=torch.bfloat16)
old = base_unet.conv_in
new = nn.Conv2d(8, old.out_channels, old.kernel_size, old.stride, old.padding,
                bias=(old.bias is not None)).to(torch.bfloat16)
with torch.no_grad():
    new.weight[:,:4].copy_(old.weight); new.weight[:,4:].zero_()
    if old.bias is not None: new.bias.copy_(old.bias)
base_unet.conv_in = new; base_unet.config.in_channels = 8

base_unet.load_adapter(os.path.join(ckpt, "unet_adapter"))
eval_unet = base_unet.to("cuda").eval()

eval_null = torch.load(os.path.join(ckpt, "null_embed.pt"), map_location="cuda")
if isinstance(eval_null, nn.Parameter): eval_null = eval_null.data
eval_null = eval_null.bfloat16().unsqueeze(0) if eval_null.dim()==2 else eval_null.bfloat16()

ddim = DDIMScheduler.from_config(noise_scheduler.config)
ddim.set_timesteps(50)

psnr_fn = PeakSignalNoiseRatio(data_range=2.0).to("cuda")
ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to("cuda")
lpips_fn = lpips_lib.LPIPS(net="alex").to("cuda")

@torch.no_grad()
def infer(sar_tensor):
    sar = sar_tensor.unsqueeze(0).to("cuda", dtype=torch.bfloat16)
    s_lat = eval_vae.encode(sar).latent_dist.mean * eval_vae.config.scaling_factor
    lat = torch.randn_like(s_lat) * ddim.init_noise_sigma
    emb = eval_null.expand(1,-1,-1)
    ddim.set_timesteps(50)
    for t in ddim.timesteps:
        mi = torch.cat([lat, s_lat], 1)
        npred = eval_unet(mi, t.unsqueeze(0).to("cuda"), encoder_hidden_states=emb).sample
        lat = ddim.step(npred, t, lat).prev_sample
    img = eval_vae.decode(lat/eval_vae.config.scaling_factor).sample
    return img.squeeze(0).clamp(-1,1).float()

# Run on 200 random val patches
n_eval = min(200, len(val_p))
subset = random.sample(val_p, n_eval)
results = {"psnr": [], "ssim": [], "lpips": []}

for pair in tqdm(subset, desc="Eval"):
    item = SAROpticalDataset([pair], cfg.img_size, False)[0]
    pred = infer(item["sar"])
    p = pred.unsqueeze(0).to("cuda")
    g = item["optical"].unsqueeze(0).to("cuda")
    results["psnr"].append(psnr_fn(p,g).item())
    results["ssim"].append(ssim_fn(p,g).item())
    results["lpips"].append(lpips_fn(p,g).item())

print(f"\nPSNR  : {np.mean(results['psnr']):.3f} ± {np.std(results['psnr']):.3f} dB")
print(f"SSIM  : {np.mean(results['ssim']):.4f} ± {np.std(results['ssim']):.4f}")
print(f"LPIPS : {np.mean(results['lpips']):.4f} ± {np.std(results['lpips']):.4f}  (↓ better)")

# Experiment # 7

In [ ]:
%%writefile /content/eval_resshift.py
import os, gc, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from diffusers import UNet2DConditionModel, AutoencoderKL
from diffusers.models.attention_processor import AttnProcessor2_0
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
import lpips as lpips_lib
import safetensors.torch as st
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split

# ── Config ────────────────────────────────────────────────────────────────────
HF_REPO_ID    = os.environ["HF_REPO_ID"]
SD_MODEL_ID   = "runwayml/stable-diffusion-v1-5"
DATA_ROOT     = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR    = "/content/eval_resshift"
IMG_SIZE      = 256
SEASONS       = ("spring", "summer", "fall", "winter")
TERRAIN_TYPES = ("tropical", "temperate", "arctic", "arid", "coastal", "urban")
VAL_FRACTION  = 0.05
SEED          = 42
N_EVAL        = 300
FIXED_SEED    = 999

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"Repo   : https://huggingface.co/{HF_REPO_ID}")


# ── ResShift Scheduler ────────────────────────────────────────────────────────
class ResShiftScheduler:
    def __init__(self, num_timesteps=15, kappa=2.0, eta=0.5,
                 schedule="exponential", device="cuda"):
        self.T      = num_timesteps
        self.device = device
        if schedule == "exponential":
            t     = torch.arange(1, num_timesteps + 1, dtype=torch.float32)
            eta_t = (torch.exp(t / num_timesteps * np.log(kappa + 1)) - 1) / kappa
        else:
            eta_t = torch.linspace(0, 1, num_timesteps + 1)[1:]
        self.eta_t     = eta_t.to(device)
        t_norm         = torch.arange(1, num_timesteps + 1, dtype=torch.float32) / num_timesteps
        self.sigma_t   = (eta * t_norm.sqrt()).to(device)
        self.timesteps = torch.arange(num_timesteps - 1, -1, -1, device=device)

    def get_x0_from_pred(self, x_t, y, t, x0_pred):
        if t.item() == 0:
            return x0_pred
        eta_prev = self.eta_t[t - 1].view(-1, 1, 1, 1)
        return x0_pred + eta_prev * (y - x0_pred)


# ── Helpers ───────────────────────────────────────────────────────────────────
def map_region_to_terrain(region):
    region = region.lower()
    for k, v in {
        "tropical": "tropical", "temperate": "temperate",
        "arctic":   "arctic",   "arid":      "arid",
        "desert":   "arid",     "coastal":   "coastal",
        "ocean":    "coastal",  "urban":     "urban",
        "city":     "urban",
    }.items():
        if k in region: return v
    return "temperate"


def denorm(t):
    return ((t.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()


# ── Dataset ───────────────────────────────────────────────────────────────────
def collect_pairs(root, seasons):
    root_str = Path(root).as_posix()
    buckets  = {s: [] for s in seasons}
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df["season"] = season
        df["region"] = df["region"].str.strip().str.lower()
        df["s1_fileName"] = (df["s1_fileName"]
                             .str.replace("\\", "/", regex=False)
                             .str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True))
        df["s2_fileName"] = df["s2_fileName"].str.replace("\\", "/", regex=False)
        df["s1"] = root_str + "/" + df["s1_fileName"]
        df["s2"] = root_str + "/" + df["s2_fileName"]
        buckets[season] = df[["s1","s2","season","region"]].to_dict("records")
    return [p for s in seasons if buckets[s] for p in buckets[s]]


class SARDataset(Dataset):
    def __init__(self, pairs, img_size=512):
        self.pairs = pairs; self.img_size = img_size

    def __len__(self): return len(self.pairs)

    def _load(self, path, mode):
        img = Image.open(path).convert(mode)
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        if mode == "L": arr = np.stack([arr, arr, arr], axis=2)
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0

    def __getitem__(self, idx):
        p = self.pairs[idx]
        return {"sar": self._load(p["s1"], "L"),
                "optical": self._load(p["s2"], "RGB"),
                "season": p["season"], "region": p["region"]}


# ── Step 1: Download from HF ──────────────────────────────────────────────────
print("\n" + "="*60)
print("Step 1: Downloading ResShift checkpoint from HuggingFace")
print("="*60)

def try_download(folder, filename):
    try:
        return hf_hub_download(repo_id=HF_REPO_ID, filename=f"{folder}/{filename}")
    except Exception:
        return None

# Search for weights across possible checkpoint tags
ckpt_tag   = None
model_path = None
for tag in [ "resshift_final"]:
    p = try_download(tag, "model/model.safetensors")
    if p:
        ckpt_tag = tag; model_path = p
        print(f"  Weights found : {tag}/model/model.safetensors")
        break
    p = try_download(tag, "model.safetensors")
    if p:
        ckpt_tag = tag; model_path = p
        print(f"  Weights found : {tag}/model.safetensors")
        break

if model_path is None:
    raise FileNotFoundError(
        "No ResShift weights found on HF. "
        "Expected: resshift_best/model/model.safetensors"
    )

embed_cache_path = hf_hub_download(repo_id=HF_REPO_ID,
                                    filename=f"{ckpt_tag}/embed_cache.pt")
rs_config_path   = hf_hub_download(repo_id=HF_REPO_ID,
                                    filename=f"{ckpt_tag}/resshift_config.pt")

try:
    step_path = hf_hub_download(repo_id=HF_REPO_ID,
                                 filename=f"{ckpt_tag}/step.txt")
    with open(step_path) as f:
        total_steps = f.read().strip()
except Exception:
    total_steps = "unknown"

print(f"  Checkpoint    : {ckpt_tag}")
print(f"  Total steps   : {total_steps}")


# ── Step 2: ResShift config & scheduler ───────────────────────────────────────
print("\n" + "="*60)
print("Step 2: Loading ResShift config")
print("="*60)

rs_cfg        = torch.load(rs_config_path, map_location="cpu", weights_only=False)
NUM_TIMESTEPS = rs_cfg.get("T",        15)
KAPPA         = rs_cfg.get("kappa",   2.0)
ETA           = rs_cfg.get("eta",     0.5)
SCHEDULE      = rs_cfg.get("schedule", "exponential")
print(f"  T={NUM_TIMESTEPS}  kappa={KAPPA}  eta={ETA}  schedule={SCHEDULE}")

rs_scheduler = ResShiftScheduler(
    num_timesteps=NUM_TIMESTEPS, kappa=KAPPA,
    eta=ETA, schedule=SCHEDULE, device=device,
)


# ── Step 3: Load UNet ─────────────────────────────────────────────────────────
print("\n" + "="*60)
print("Step 3: Loading ResShift UNet (8-channel conv_in)")
print("="*60)

unet     = UNet2DConditionModel.from_pretrained(
    SD_MODEL_ID, subfolder="unet", torch_dtype=torch.bfloat16)
old_conv = unet.conv_in
conv_8ch = nn.Conv2d(
    8, old_conv.out_channels, old_conv.kernel_size,
    old_conv.stride, old_conv.padding,
    bias=(old_conv.bias is not None),
).to(torch.bfloat16)
with torch.no_grad():
    conv_8ch.weight[:, :4].copy_(old_conv.weight)
    conv_8ch.weight[:, 4:].zero_()
    if old_conv.bias is not None: conv_8ch.bias.copy_(old_conv.bias)
unet.conv_in = conv_8ch
unet.config.in_channels = 8

state_dict = st.load_file(model_path, device="cpu")
missing, unexpected = unet.load_state_dict(state_dict, strict=False)
print(f"  Missing: {len(missing)}  Unexpected: {len(unexpected)}")

unet.set_attn_processor(AttnProcessor2_0())
unet.requires_grad_(False)
unet = unet.to(device).eval()
print(f"  UNet ready: {sum(p.numel() for p in unet.parameters())/1e6:.0f}M params")


# ── Step 4: VAE + embedding cache ─────────────────────────────────────────────
print("\n" + "="*60)
print("Step 4: Loading VAE + embedding cache")
print("="*60)

vae = AutoencoderKL.from_pretrained(
    SD_MODEL_ID, subfolder="vae", torch_dtype=torch.bfloat16).to(device).eval()
vae.requires_grad_(False)

embed_cache = {
    k: v.to(device)
    for k, v in torch.load(embed_cache_path, map_location=device,
                            weights_only=False).items()
}
print(f"  Embed cache: {len(embed_cache)} embeddings")


# ── Step 5: Validation split ──────────────────────────────────────────────────
print("\n" + "="*60)
print("Step 5: Building validation split")
print("="*60)

all_pairs = collect_pairs(DATA_ROOT, SEASONS)
_, val_pairs = train_test_split(
    all_pairs, test_size=VAL_FRACTION,
    stratify=[f"{p['season']}_{p['region']}" for p in all_pairs],
    random_state=SEED,
)
print(f"  Validation set: {len(val_pairs):,} pairs")


# ── Inference ─────────────────────────────────────────────────────────────────
def get_embed(season, region):
    t   = map_region_to_terrain(region)
    key = f"{season}_{t}"
    return embed_cache.get(key, embed_cache.get(f"season_{season}",
                                                 embed_cache["null"]))


@torch.no_grad()
def translate(sar_tensor, season, region, num_steps=None):
    sched   = rs_scheduler if num_steps is None else ResShiftScheduler(
        num_timesteps=num_steps, kappa=KAPPA, eta=ETA,
        schedule=SCHEDULE, device=device,
    )
    sar     = sar_tensor.unsqueeze(0).to(device, dtype=torch.bfloat16)
    sar_lat = vae.encode(sar).latent_dist.mean * vae.config.scaling_factor
    x_t     = sar_lat + sched.sigma_t[-1] * torch.randn_like(sar_lat)
    embed   = get_embed(season, region).to(torch.bfloat16).expand(1, -1, -1)

    for t_idx in sched.timesteps:
        t_tensor = t_idx.unsqueeze(0).to(device)
        model_in = torch.cat([x_t, sar_lat], dim=1).to(torch.bfloat16)
        x0_pred  = unet(model_in, t_tensor,
                        encoder_hidden_states=embed).sample.to(torch.bfloat16)
        x_t      = sched.get_x0_from_pred(x_t, sar_lat, t_idx, x0_pred)

    image = vae.decode(
        (x_t / vae.config.scaling_factor).to(vae.dtype)
    ).sample
    return image.squeeze(0).clamp(-1, 1).float().cpu()


# ── Step 6: Quantitative metrics ─────────────────────────────────────────────
print("\n" + "="*60)
print(f"Step 6: Quantitative metrics ({N_EVAL} patches)")
print("="*60)

psnr_fn  = PeakSignalNoiseRatio(data_range=2.0).to(device)
ssim_fn  = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
lpips_fn = lpips_lib.LPIPS(net="alex").to(device)

eval_subset     = random.sample(val_pairs, min(N_EVAL, len(val_pairs)))
season_metrics  = {s: {"psnr":[], "ssim":[], "lpips":[]} for s in SEASONS}
terrain_metrics = {t: {"psnr":[], "ssim":[], "lpips":[]} for t in TERRAIN_TYPES}
all_psnr, all_ssim, all_lpips = [], [], []

for pair in tqdm(eval_subset, desc="Computing metrics"):
    ds   = SARDataset([pair], img_size=IMG_SIZE)
    item = ds[0]
    pred = translate(item["sar"], pair["season"], pair["region"])
    p    = pred.unsqueeze(0).to(device).float()
    g    = item["optical"].unsqueeze(0).to(device).float()

    pv  = psnr_fn(p, g).item()
    sv  = ssim_fn(p, g).item()
    lv  = lpips_fn(p, g).item()
    all_psnr.append(pv); all_ssim.append(sv); all_lpips.append(lv)

    s = pair["season"]; t = map_region_to_terrain(pair["region"])
    for key, val in zip(["psnr","ssim","lpips"], [pv, sv, lv]):
        if s in season_metrics:  season_metrics[s][key].append(val)
        if t in terrain_metrics: terrain_metrics[t][key].append(val)

print(f"\n{'─'*58}")
print(f"OVERALL ({N_EVAL} patches) — ResShift {NUM_TIMESTEPS}-step | {total_steps} total steps")
print(f"  PSNR  : {np.mean(all_psnr):.3f} ± {np.std(all_psnr):.3f} dB")
print(f"  SSIM  : {np.mean(all_ssim):.4f} ± {np.std(all_ssim):.4f}")
print(f"  LPIPS : {np.mean(all_lpips):.4f} ± {np.std(all_lpips):.4f}  (↓ better)")
print(f"{'─'*58}")
print("BY SEASON:")
for s in SEASONS:
    m = season_metrics[s]
    if not m["psnr"]: continue
    print(f"  {s:<8}  PSNR {np.mean(m['psnr']):.3f}  "
          f"SSIM {np.mean(m['ssim']):.4f}  LPIPS {np.mean(m['lpips']):.4f}")
print(f"{'─'*58}")
print("BY TERRAIN:")
for t in TERRAIN_TYPES:
    m = terrain_metrics[t]
    if not m["psnr"]: continue
    print(f"  {t:<12}  PSNR {np.mean(m['psnr']):.3f}  "
          f"SSIM {np.mean(m['ssim']):.4f}  LPIPS {np.mean(m['lpips']):.4f}")


# ── Step 7: Visual comparison grid ───────────────────────────────────────────
print("\n" + "="*60)
print("Step 7: Visual comparison grid (2 per season, fixed seed)")
print("="*60)

rng      = random.Random(FIXED_SEED)
selected = []
for s in SEASONS:
    pool = [p for p in val_pairs if p["season"] == s]
    if pool: selected.extend(rng.sample(pool, min(2, len(pool))))

n   = len(selected)
fig = plt.figure(figsize=(15, 4 * n))
gs  = gridspec.GridSpec(n, 3, figure=fig, hspace=0.35, wspace=0.05)

for i, pair in enumerate(tqdm(selected, desc="Visual grid")):
    ds   = SARDataset([pair], img_size=IMG_SIZE)
    item = ds[0]
    pred = translate(item["sar"], pair["season"], pair["region"])

    p   = pred.unsqueeze(0).to(device).float()
    g   = item["optical"].unsqueeze(0).to(device).float()
    pv  = psnr_fn(p, g).item()
    sv  = ssim_fn(p, g).item()
    lv  = lpips_fn(p, g).item()
    ter = map_region_to_terrain(pair["region"])

    ax0 = fig.add_subplot(gs[i, 0])
    ax1 = fig.add_subplot(gs[i, 1])
    ax2 = fig.add_subplot(gs[i, 2])

    ax0.imshow(denorm(item["sar"]))
    ax0.set_title(f"SAR Input\n{pair['season']} | {ter}", fontsize=9, fontweight="bold")
    ax0.axis("off")

    ax1.imshow(denorm(pred))
    ax1.set_title(
        f"ResShift ({NUM_TIMESTEPS} steps)\n"
        f"PSNR {pv:.2f} dB  SSIM {sv:.3f}  LPIPS {lv:.3f}",
        fontsize=8, fontweight="bold",
    )
    ax1.axis("off")

    ax2.imshow(denorm(item["optical"]))
    ax2.set_title("Ground Truth (Sentinel-2)", fontsize=9, fontweight="bold")
    ax2.axis("off")

plt.suptitle(
    f"SAR-INTEL ResShift — {ckpt_tag} — {total_steps} steps",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
grid_path = os.path.join(OUTPUT_DIR, "visual_grid.png")
plt.savefig(grid_path, dpi=130, bbox_inches="tight")
plt.show(); plt.close()
print(f"  Saved -> {grid_path}")


# ── Step 8: Steps ablation ────────────────────────────────────────────────────
print("\n" + "="*60)
print("Step 8: Inference steps ablation (5 / 10 / 15 / 25)")
print("="*60)

ablation_subset = rng.sample(val_pairs, 20)
step_options    = [5, 10, 15, 25]
step_results    = {s: {"psnr":[], "ssim":[], "lpips":[]} for s in step_options}

for pair in tqdm(ablation_subset, desc="Steps ablation"):
    ds   = SARDataset([pair], img_size=IMG_SIZE)
    item = ds[0]
    for ns in step_options:
        pred = translate(item["sar"], pair["season"], pair["region"], num_steps=ns)
        p    = pred.unsqueeze(0).to(device).float()
        g    = item["optical"].unsqueeze(0).to(device).float()
        step_results[ns]["psnr"].append(psnr_fn(p, g).item())
        step_results[ns]["ssim"].append(ssim_fn(p, g).item())
        step_results[ns]["lpips"].append(lpips_fn(p, g).item())

best_ns = max(step_options, key=lambda s: np.mean(step_results[s]["psnr"]))
print(f"\n{'Steps':<8} {'PSNR':>8} {'SSIM':>8} {'LPIPS':>8}")
print("─" * 40)
for ns in step_options:
    m   = step_results[ns]
    tag = "  ← BEST" if ns == best_ns else ""
    print(f"{ns:<8} {np.mean(m['psnr']):>8.3f} {np.mean(m['ssim']):>8.4f} "
          f"{np.mean(m['lpips']):>8.4f}{tag}")


# ── Step 9: Distribution plots ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, vals, title, xlabel, color in zip(
    axes,
    [all_psnr, all_ssim, all_lpips],
    ["PSNR Distribution", "SSIM Distribution", "LPIPS Distribution (↓ better)"],
    ["dB", "SSIM", "LPIPS"],
    ["steelblue", "seagreen", "coral"],
):
    ax.hist(vals, bins=35, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(np.mean(vals), color="red", linestyle="--", lw=2,
               label=f"Mean {np.mean(vals):.3f}")
    ax.axvline(np.median(vals), color="orange", linestyle=":", lw=2,
               label=f"Median {np.median(vals):.3f}")
    ax.set_title(title, fontweight="bold"); ax.set_xlabel(xlabel); ax.legend()
plt.suptitle("ResShift Metric Distributions", fontsize=12, fontweight="bold")
dist_path = os.path.join(OUTPUT_DIR, "distributions.png")
plt.savefig(dist_path, dpi=130, bbox_inches="tight")
plt.show(); plt.close()


# ── Step 10: Per-season bars ──────────────────────────────────────────────────
seasons_ok = [s for s in SEASONS if season_metrics[s]["psnr"]]
x  = np.arange(len(seasons_ok))
cl = ["#4CAF50", "#FF9800", "#F44336", "#2196F3"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, key, title, yl in zip(
    axes,
    ["psnr", "ssim", "lpips"],
    ["PSNR by Season (↑ better)", "SSIM by Season (↑ better)", "LPIPS by Season (↓ better)"],
    ["dB", "SSIM", "LPIPS"],
):
    vals = [np.mean(season_metrics[s][key]) for s in seasons_ok]
    stds = [np.std(season_metrics[s][key])  for s in seasons_ok]
    bars = ax.bar(x, vals, color=cl[:len(seasons_ok)], edgecolor="white",
                  yerr=stds, capsize=4)
    ax.set_xticks(x); ax.set_xticklabels(seasons_ok)
    ax.set_title(title, fontweight="bold"); ax.set_ylabel(yl)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(stds) * 0.05,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
plt.suptitle("Per-Season Performance — ResShift", fontsize=12, fontweight="bold")
plt.tight_layout()
season_path = os.path.join(OUTPUT_DIR, "season_bars.png")
plt.savefig(season_path, dpi=130, bbox_inches="tight")
plt.show(); plt.close()


# ── Step 11: Per-terrain bars ─────────────────────────────────────────────────
terrains_ok = [t for t in TERRAIN_TYPES if terrain_metrics[t]["psnr"]]
x   = np.arange(len(terrains_ok))
clt = ["#9C27B0","#00BCD4","#8BC34A","#FF5722","#607D8B","#FFC107"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key, title, yl in zip(
    axes,
    ["psnr", "ssim", "lpips"],
    ["PSNR by Terrain (↑ better)", "SSIM by Terrain (↑ better)", "LPIPS by Terrain (↓ better)"],
    ["dB", "SSIM", "LPIPS"],
):
    vals = [np.mean(terrain_metrics[t][key]) for t in terrains_ok]
    stds = [np.std(terrain_metrics[t][key])  for t in terrains_ok]
    bars = ax.bar(x, vals, color=clt[:len(terrains_ok)], edgecolor="white",
                  yerr=stds, capsize=4)
    ax.set_xticks(x); ax.set_xticklabels(terrains_ok, rotation=30, ha="right")
    ax.set_title(title, fontweight="bold"); ax.set_ylabel(yl)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(stds) * 0.05,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
plt.suptitle("Per-Terrain Performance — ResShift", fontsize=12, fontweight="bold")
plt.tight_layout()
terrain_path = os.path.join(OUTPUT_DIR, "terrain_bars.png")
plt.savefig(terrain_path, dpi=130, bbox_inches="tight")
plt.show(); plt.close()


# ── Step 12: Verdict ──────────────────────────────────────────────────────────
mean_psnr  = np.mean(all_psnr)
mean_ssim  = np.mean(all_ssim)
mean_lpips = np.mean(all_lpips)

print("\n" + "="*60)
print("FINAL VERDICT")
print("="*60)
print(f"  Checkpoint  : {ckpt_tag}  ({total_steps} total steps)")
print(f"  Inference   : {NUM_TIMESTEPS} ResShift steps (deterministic)")
print(f"\n  PSNR  : {mean_psnr:.3f} dB")
print(f"  SSIM  : {mean_ssim:.4f}")
print(f"  LPIPS : {mean_lpips:.4f}")
print(f"\n  Baselines:")
print(f"    Phase A (156k, DDPM 50-step)    : ~10.6 dB / 0.055 / 0.772")
print(f"    ResShift 180k steps             : ~12.5 dB / 0.154 / 1.006")
print(f"    SOTA (color-supervised diffusion): ~19.7 dB / 0.312")
print(f"\n  Improvement vs Phase A : {mean_psnr - 10.6:+.2f} dB PSNR")
print(f"  Improvement vs 180k    : {mean_psnr - 12.5:+.2f} dB PSNR")

if mean_psnr >= 18.0:
    verdict = "EXCELLENT — Target reached."; action = "Deploy. Training complete."
elif mean_psnr >= 15.0:
    verdict = "GOOD — Strong improvement.";  action = "Run 50k more steps at LR=2e-6."
elif mean_psnr >= 13.0:
    verdict = "IMPROVING — Converging steadily."; action = "Run 50k more steps at LR=5e-6."
else:
    verdict = "EARLY — More training needed."; action = "Run 100k more steps."

print(f"\n  Verdict : {verdict}")
print(f"  Action  : {action}")
print(f"\n  Plots saved -> {OUTPUT_DIR}")
print("="*60)

gc.collect(); torch.cuda.empty_cache()

Writing /content/eval_resshift.py


In [ ]:
!pip install -q lpips torchmetrics
!python /content/eval_resshift.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Device : cuda
Repo   : https://huggingface.co/AliMusaRizvi/sar-to-optical-diffusion

Step 1: Downloading ResShift checkpoint from HuggingFace
resshift_final/model/model.safetensors: 100% 

In [ ]:
%%writefile /content/stage1_detail_vae.py
"""
SAR-INTEL Stage 1 — Detail-Aligned VAE (DA-VAE applied to Sentinel-2 optical)
Goal: lift the VAE reconstruction ceiling that caps the whole SAR->optical field.

Method (faithful to DA-VAE, CVPR 2026):
  - Base latent z (4ch): FROZEN SD-1.5 VAE encoder. This is the latent the
    286k-step bridge UNet already speaks — we must NOT disturb it.
  - Detail latent z_d (12ch): NEW trainable detail encoder. Total = 16 channels.
  - Decoder reconstructs from [z, z_d]; decoder.conv_in widened 4->16,
    extra channels ZERO-INIT (warm start: identical to vanilla at step 0).
  - Alignment loss: project z_d -> 4ch by grouped averaging, L2 to z.
    (DA-VAE ablation: removing this collapses quality.)

Stage 1 trains: detail_encoder + decoder. Frozen: encoder + quant_conv.
Output: a VAE whose 4 base channels are byte-identical in meaning to SD-1.5,
plus 12 detail channels that recover high-frequency optical detail.
"""
import os, gc, random, warnings, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from diffusers import AutoencoderKL
from accelerate import Accelerator
from accelerate.utils import set_seed
from tqdm.auto import tqdm
from huggingface_hub import HfApi
import lpips as lpips_lib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT       = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR      = "/content/drive/MyDrive/stage1_detail_vae"
SD_MODEL_ID     = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID      = os.environ["HF_REPO_ID"]
HF_WRITE_TOKEN  = os.environ.get("HF_WRITE_TOKEN", "")

IMG_SIZE        = 512
SEASONS         = ("spring", "summer", "fall", "winter")
VAL_FRACTION    = 0.05
SEED            = 42

BASE_CHANNELS   = 4          # SD-1.5 VAE latent channels (frozen, the UNet speaks this)
DETAIL_CHANNELS = 12         # added detail channels -> total 16
TOTAL_CHANNELS  = BASE_CHANNELS + DETAIL_CHANNELS

# Stage 1 is cheap & falsifiable — keep it short, eval often, stop early if ceiling lifts
NUM_TRAIN_STEPS = 40_000
BATCH_SIZE      = 4
GRAD_ACCUM      = 2          # effective batch 8
LR              = 1e-4
LR_WARMUP       = 500
ALIGN_WEIGHT    = 0.5        # DA-VAE default; ablated as the sweet spot
LPIPS_WEIGHT    = 1.0
L1_WEIGHT       = 1.0

CEILING_EVAL_EVERY = 5_000   # measure recon PSNR vs vanilla at these intervals
N_CEILING_EVAL     = 200     # patches for the ceiling measurement
VIS_EVERY          = 5_000
SAVE_EVERY         = 10_000
FIXED_VAL_SEED     = 999

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ── Data (optical only needed for reconstruction) ─────────────────────────────
def collect_pairs(root, seasons):
    root_str = Path(root).as_posix()
    buckets  = {s: [] for s in seasons}
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df["season"] = season
        df["region"] = df["region"].str.strip().str.lower()
        df["s2_fileName"] = df["s2_fileName"].str.replace("\\", "/", regex=False)
        df["s2"] = root_str + "/" + df["s2_fileName"]
        buckets[season] = df[["s2", "season", "region"]].to_dict("records")
        print(f"  [{season}] {len(buckets[season]):,} optical images")
    pairs = [p for s in seasons if buckets[s] for p in buckets[s]]
    random.shuffle(pairs)
    return pairs


class OpticalDataset(Dataset):
    def __init__(self, pairs, img_size=512, augment=True):
        self.pairs = pairs; self.img_size = img_size; self.augment = augment

    def __len__(self): return len(self.pairs)

    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0

    def __getitem__(self, idx):
        opt = self._load(self.pairs[idx]["s2"])
        if self.augment:
            if random.random() > 0.5: opt = TF.hflip(opt)
            if random.random() > 0.5: opt = TF.vflip(opt)
            k = random.randint(0, 3)
            if k: opt = torch.rot90(opt, k, [1, 2])
        return {"optical": opt}


def build_fixed_val(val_pairs, n=2):
    rng = random.Random(FIXED_VAL_SEED)
    sel = []
    for s in SEASONS:
        pool = [p for p in val_pairs if p["season"] == s]
        if pool: sel.extend(rng.sample(pool, min(n, len(pool))))
    return sel


# ── Detail encoder: image(3) -> z_d (12ch), 8x downsample to match VAE ────────
class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CHANNELS, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),       # 512 -> 256
            nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf * 2, 3, 2, 1),      # 256 -> 128
            nn.GroupNorm(8, nf * 2), nn.SiLU(),
            nn.Conv2d(nf * 2, nf * 4, 3, 2, 1),  # 128 -> 64
            nn.GroupNorm(8, nf * 4), nn.SiLU(),
            nn.Conv2d(nf * 4, nf * 4, 3, 1, 1),
            nn.GroupNorm(8, nf * 4), nn.SiLU(),
            nn.Conv2d(nf * 4, out_ch, 3, 1, 1),  # -> 12ch at 64x64
        )

    def forward(self, x): return self.net(x)


# ── Detail-Aligned VAE wrapper ────────────────────────────────────────────────
class DetailAlignedVAE(nn.Module):
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae   = base_vae          # SD-1.5 VAE (encoder frozen)
        self.detail_enc = DetailEncoder()

        # Widen decoder.conv_in: Conv2d(4,512) -> Conv2d(16,512), zero-init extra
        old = base_vae.decoder.conv_in
        new = nn.Conv2d(TOTAL_CHANNELS, old.out_channels,
                        old.kernel_size, old.stride, old.padding,
                        bias=(old.bias is not None))
        with torch.no_grad():
            new.weight.zero_()
            new.weight[:, :BASE_CHANNELS].copy_(old.weight)  # base path preserved
            # detail input weights stay ZERO -> at init, decoder == vanilla
            if old.bias is not None:
                new.bias.copy_(old.bias)
        base_vae.decoder.conv_in = new

        # Freeze encoder + quant_conv (base latent must stay stable for the UNet)
        for p in base_vae.encoder.parameters():    p.requires_grad_(False)
        for p in base_vae.quant_conv.parameters(): p.requires_grad_(False)
        # Trainable: detail_enc + decoder (incl. widened conv_in) + post_quant_conv

    @torch.no_grad()
    def encode_base(self, x):
        return self.base_vae.encode(x).latent_dist.mode()   # (B,4,64,64) deterministic

    def encode_detail(self, x):
        return self.detail_enc(x)                            # (B,12,64,64)

    def decode_full(self, z_base, z_detail):
        z_base_pq = self.base_vae.post_quant_conv(z_base)    # (B,4,64,64)
        combined  = torch.cat([z_base_pq, z_detail], dim=1)  # (B,16,64,64)
        return self.base_vae.decoder(combined)               # (B,3,512,512)

    def forward(self, x):
        z_base   = self.encode_base(x)
        z_detail = self.encode_detail(x)
        recon    = self.decode_full(z_base, z_detail)
        return recon, z_base, z_detail


# ── Alignment loss: project z_d to 4ch via grouped averaging, L2 to base ──────
def alignment_loss(z_base, z_detail):
    B, D, H, W = z_detail.shape
    r = D // BASE_CHANNELS                       # 12 // 4 = 3
    grouped = z_detail.view(B, BASE_CHANNELS, r, H, W).mean(dim=2)  # (B,4,64,64)
    return F.mse_loss(grouped, z_base)


# ── Ceiling test: vanilla 4ch VAE vs detail-aligned 16ch, on same images ──────
@torch.no_grad()
def measure_ceiling(model, vanilla_vae, val_pairs, lpips_fn, n=N_CEILING_EVAL):
    model.eval()
    subset = random.Random(123).sample(val_pairs, min(n, len(val_pairs)))
    v_psnr, v_lpips, d_psnr, d_lpips = [], [], [], []
    for pair in subset:
        img = OpticalDataset([pair], IMG_SIZE, augment=False)[0]["optical"]
        x   = img.unsqueeze(0).to(device)

        # vanilla SD-1.5 VAE reconstruction (4ch)
        zb_v   = vanilla_vae.encode(x).latent_dist.mode()
        rec_v  = vanilla_vae.decode(zb_v).sample.clamp(-1, 1)

        # detail-aligned reconstruction (16ch)
        rec_d, _, _ = model(x)
        rec_d = rec_d.clamp(-1, 1)

        mse_v = F.mse_loss(rec_v, x).item()
        mse_d = F.mse_loss(rec_d, x).item()
        v_psnr.append(10 * np.log10(4.0 / mse_v))   # data range 2 -> max^2=4
        d_psnr.append(10 * np.log10(4.0 / mse_d))
        v_lpips.append(lpips_fn(rec_v, x).item())
        d_lpips.append(lpips_fn(rec_d, x).item())
    model.train()
    return (np.mean(v_psnr), np.mean(v_lpips),
            np.mean(d_psnr), np.mean(d_lpips))


def denorm(t): return ((t.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()


def show_recon(model, vanilla_vae, fixed_val, step):
    model.eval()
    n = len(fixed_val)
    fig, axes = plt.subplots(n, 3, figsize=(12, 3.5 * n))
    if n == 1: axes = axes.reshape(1, -1)
    with torch.no_grad():
        for i, pair in enumerate(fixed_val):
            img = OpticalDataset([pair], IMG_SIZE, augment=False)[0]["optical"]
            x   = img.unsqueeze(0).to(device)
            rec_v = vanilla_vae.decode(vanilla_vae.encode(x).latent_dist.mode()).sample
            rec_d, _, _ = model(x)
            axes[i, 0].imshow(denorm(img));            axes[i, 0].set_title("GT optical", fontsize=8); axes[i, 0].axis("off")
            axes[i, 1].imshow(denorm(rec_v[0]));       axes[i, 1].set_title("Vanilla 4ch", fontsize=8); axes[i, 1].axis("off")
            axes[i, 2].imshow(denorm(rec_d[0]));       axes[i, 2].set_title(f"Detail 16ch (step {step})", fontsize=8); axes[i, 2].axis("off")
    plt.suptitle(f"Reconstruction — step {step}", fontsize=11, fontweight="bold")
    plt.tight_layout(); display(fig); plt.close(fig)
    model.train()


def push_to_hf(folder, tag):
    if not HF_WRITE_TOKEN: return
    try:
        HfApi().upload_folder(folder_path=folder, path_in_repo=tag,
                              repo_id=HF_REPO_ID, repo_type="model", token=HF_WRITE_TOKEN)
        print(f"  HF push -> {tag}")
    except Exception as e:
        print(f"  HF push failed: {e}")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    accelerator = Accelerator(mixed_precision="no",  # fp32 for precise ceiling measurement
                              gradient_accumulation_steps=GRAD_ACCUM,
                              project_dir=OUTPUT_DIR)
    set_seed(SEED)
    is_main = accelerator.is_main_process
    if is_main:
        os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Loading SD-1.5 VAE (base, frozen encoder)...")
    base_vae    = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    # A second, untouched copy for the vanilla baseline in the ceiling test
    vanilla_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae").to(device).eval()
    vanilla_vae.requires_grad_(False)

    model = DetailAlignedVAE(base_vae)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f"  Detail-Aligned VAE ready. Trainable: {n_train:.1f}M params "
          f"(detail encoder + decoder). Base encoder frozen.")

    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)

    # Data
    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    train_pairs, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['region']}" for p in all_pairs],
        random_state=SEED)
    fixed_val = build_fixed_val(val_pairs)
    print(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,}")

    train_ds = OpticalDataset(train_pairs, IMG_SIZE, augment=True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True,
                          persistent_workers=True, prefetch_factor=2)

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=LR, betas=(0.9, 0.999), weight_decay=1e-4)
    from diffusers.optimization import get_cosine_schedule_with_warmup
    lr_sched = get_cosine_schedule_with_warmup(optimizer, LR_WARMUP, NUM_TRAIN_STEPS)

    model, optimizer, train_dl, lr_sched = accelerator.prepare(
        model, optimizer, train_dl, lr_sched)

    # Baseline ceiling BEFORE any training (sanity: detail VAE should ~= vanilla at step 0)
    if is_main:
        vp, vl, dp, dl = measure_ceiling(accelerator.unwrap_model(model),
                                         vanilla_vae, val_pairs, lpips_fn, n=100)
        print("\n" + "="*58)
        print("CEILING @ step 0 (zero-init: detail VAE should match vanilla)")
        print(f"  Vanilla 4ch : PSNR {vp:.3f} dB | LPIPS {vl:.4f}")
        print(f"  Detail 16ch : PSNR {dp:.3f} dB | LPIPS {dl:.4f}")
        print("="*58 + "\n")

    global_step = 0
    ema_l1 = ema_lp = ema_al = 0.0; alpha = 0.98
    steps_per_epoch = len(train_dl) // GRAD_ACCUM
    total_epochs    = (NUM_TRAIN_STEPS + steps_per_epoch - 1) // steps_per_epoch

    bar = tqdm(total=NUM_TRAIN_STEPS, desc="Stage 1: Detail-Aligned VAE",
               disable=not is_main, dynamic_ncols=True)
    model.train()

    for epoch in range(total_epochs):
        for batch in train_dl:
            with accelerator.accumulate(model):
                x = batch["optical"].to(device)
                recon, z_base, z_detail = model(x)

                loss_l1  = F.l1_loss(recon, x) * L1_WEIGHT
                loss_lp  = lpips_fn(recon.clamp(-1, 1), x).mean() * LPIPS_WEIGHT
                loss_al  = alignment_loss(z_base, z_detail) * ALIGN_WEIGHT
                loss     = loss_l1 + loss_lp + loss_al

                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(trainable, 1.0)
                optimizer.step(); lr_sched.step(); optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                global_step += 1
                ema_l1 = alpha*ema_l1 + (1-alpha)*loss_l1.item()
                ema_lp = alpha*ema_lp + (1-alpha)*loss_lp.item()
                ema_al = alpha*ema_al + (1-alpha)*loss_al.item()
                if is_main:
                    bar.update(1)
                    bar.set_postfix({"l1": f"{ema_l1:.4f}", "lpips": f"{ema_lp:.4f}",
                                     "align": f"{ema_al:.4f}"})

                if global_step % CEILING_EVAL_EVERY == 0 and is_main:
                    vp, vl, dp, dl = measure_ceiling(accelerator.unwrap_model(model),
                                                     vanilla_vae, val_pairs, lpips_fn)
                    tqdm.write("\n" + "-"*58)
                    tqdm.write(f"CEILING @ step {global_step}")
                    tqdm.write(f"  Vanilla 4ch : PSNR {vp:.3f} dB | LPIPS {vl:.4f}")
                    tqdm.write(f"  Detail 16ch : PSNR {dp:.3f} dB | LPIPS {dl:.4f}")
                    tqdm.write(f"  >>> GAIN    : PSNR {dp-vp:+.3f} dB | LPIPS {dl-vl:+.4f}")
                    tqdm.write("-"*58 + "\n")

                if global_step % VIS_EVERY == 0 and is_main:
                    show_recon(accelerator.unwrap_model(model), vanilla_vae, fixed_val, global_step)

                if global_step % SAVE_EVERY == 0 and is_main:
                    ckpt = os.path.join(OUTPUT_DIR, f"step_{global_step}")
                    os.makedirs(ckpt, exist_ok=True)
                    m = accelerator.unwrap_model(model)
                    torch.save(m.detail_enc.state_dict(), os.path.join(ckpt, "detail_encoder.pth"))
                    torch.save(m.base_vae.decoder.state_dict(), os.path.join(ckpt, "decoder.pth"))
                    torch.save({"base_channels": BASE_CHANNELS,
                                "detail_channels": DETAIL_CHANNELS,
                                "total_channels": TOTAL_CHANNELS},
                               os.path.join(ckpt, "detail_vae_config.pt"))
                    latest = os.path.join(OUTPUT_DIR, "latest")
                    if os.path.exists(latest): shutil.rmtree(latest)
                    shutil.copytree(ckpt, latest)
                    tqdm.write(f"  saved step {global_step}")
                    push_to_hf(ckpt, f"detail_vae_step_{global_step}")

            if global_step >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if global_step >= NUM_TRAIN_STEPS: break

    bar.close()
    if is_main:
        vp, vl, dp, dl = measure_ceiling(accelerator.unwrap_model(model),
                                         vanilla_vae, val_pairs, lpips_fn, n=300)
        print("\n" + "="*58)
        print("FINAL CEILING TEST (300 patches)")
        print(f"  Vanilla SD-1.5 4ch : PSNR {vp:.3f} dB | LPIPS {vl:.4f}")
        print(f"  Detail-Aligned 16ch: PSNR {dp:.3f} dB | LPIPS {dl:.4f}")
        print(f"  CEILING LIFT       : PSNR {dp-vp:+.3f} dB | LPIPS {dl-vl:+.4f}")
        print("="*58)
        if dp - vp >= 2.0:
            print("  VERDICT: Ceiling lifted >=2 dB. Stage 2 (bridge on 16ch) is worth it.")
        elif dp - vp >= 0.5:
            print("  VERDICT: Modest lift. Stage 2 may help; consider more detail channels.")
        else:
            print("  VERDICT: No meaningful lift. STOP — do not spend on Stage 2. Rethink.")
        final = os.path.join(OUTPUT_DIR, "final")
        os.makedirs(final, exist_ok=True)
        m = accelerator.unwrap_model(model)
        torch.save(m.detail_enc.state_dict(), os.path.join(final, "detail_encoder.pth"))
        torch.save(m.base_vae.decoder.state_dict(), os.path.join(final, "decoder.pth"))
        torch.save({"base_channels": BASE_CHANNELS, "detail_channels": DETAIL_CHANNELS,
                    "total_channels": TOTAL_CHANNELS}, os.path.join(final, "detail_vae_config.pt"))
        push_to_hf(final, "detail_vae_final")
    accelerator.end_training()


if __name__ == "__main__":
    main()

Writing /content/stage1_detail_vae.py


In [ ]:
import os
from google.colab import userdata; os.environ["HF_WRITE_TOKEN"] = userdata.get("HF_TOKEN")
!pip install -q lpips
!python /content/stage1_detail_vae.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading SD-1.5 VAE (base, frozen encoder)...
  Detail-Aligned VAE ready. Trainable: 50.5M params (detail encoder + decoder). Base encoder frozen.
Setting up [LPIPS] perceptual loss: trunk

In [ ]:
%%writefile /content/check_dataset_structure.py
"""
Diagnostic: does the same physical location appear across seasons?
If yes -> a temporal/cross-season signal exists (changes the modeling options).
If no  -> seasons are just per-patch attribute labels (standard SEN1-2).

Checks three independent ways and prints evidence for each:
  (A) geographic coordinate columns (gold standard, if present)
  (B) location IDs parsed from s2 filenames with the season token removed
  (C) raw cross-season filename / ROI overlap
"""
import re
import pandas as pd
from pathlib import Path
from collections import defaultdict

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
SEASONS   = ("spring", "summer", "fall", "winter")
pd.set_option("display.max_columns", 50, "display.width", 200)


def load_season_csvs(root, seasons):
    dfs = {}
    for s in seasons:
        folder = Path(root) / s
        csvs = list(folder.glob("*.csv"))
        if not csvs:
            print(f"  [{s}] no CSV found in {folder}")
            continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        dfs[s] = df
        print(f"  [{s}] {len(df):,} rows from {len(csvs)} csv(s)")
    return dfs


def section(t): print("\n" + "=" * 70 + f"\n{t}\n" + "=" * 70)


def main():
    section("0. DISCOVER + LOAD CSVs")
    dfs = load_season_csvs(DATA_ROOT, SEASONS)
    if not dfs:
        print("No CSVs loaded — check DATA_ROOT path.")
        return
    any_df = next(iter(dfs.values()))

    section("1. CSV STRUCTURE (columns + 3 sample rows from first season)")
    print("Columns:", list(any_df.columns))
    print()
    print(any_df.head(3).to_string())

    # ---------------------------------------------------------------- (A) coords
    section("A. COORDINATE-COLUMN CHECK (gold standard for 'same location')")
    coord_candidates = [c for c in any_df.columns
                        if re.search(r"(lat|lon|lng|x|y|coord|geo|east|north)", c, re.I)]
    if coord_candidates:
        print("Possible coordinate columns:", coord_candidates)
        # build a rounded-coordinate key per season and check overlap
        # try common lat/lon pairings
        lat_col = next((c for c in coord_candidates if re.search(r"lat", c, re.I)), None)
        lon_col = next((c for c in coord_candidates if re.search(r"lon|lng", c, re.I)), None)
        if lat_col and lon_col:
            print(f"Using lat='{lat_col}', lon='{lon_col}' (rounded to 3 decimals ~100m)")
            season_coords = {}
            for s, df in dfs.items():
                if lat_col in df and lon_col in df:
                    keys = set(zip(df[lat_col].round(3), df[lon_col].round(3)))
                    season_coords[s] = keys
                    print(f"  [{s}] {len(keys):,} unique coordinate cells")
            seasons_with = list(season_coords)
            shared = None
            for s in seasons_with:
                shared = season_coords[s] if shared is None else (shared & season_coords[s])
            if shared is not None:
                print(f"\n  >>> coordinate cells present in ALL {len(seasons_with)} seasons: {len(shared):,}")
                # pairwise too
                for i in range(len(seasons_with)):
                    for j in range(i + 1, len(seasons_with)):
                        a, b = seasons_with[i], seasons_with[j]
                        ov = len(season_coords[a] & season_coords[b])
                        print(f"      {a} ∩ {b}: {ov:,} shared cells")
        else:
            print("Found coordinate-like columns but no clear lat/lon pair; inspect manually above.")
    else:
        print("No coordinate columns detected. Falling back to filename-based checks.")

    # ---------------------------------------------------------------- (B) IDs
    section("B. LOCATION-ID FROM FILENAME (season token removed)")
    fname_col = next((c for c in any_df.columns
                     if re.search(r"s2.*file|file.*s2|s2_?name|optical", c, re.I)), None)
    if fname_col is None:
        fname_col = next((c for c in any_df.columns if re.search(r"file|name|path", c, re.I)), None)
    print(f"Using filename column: '{fname_col}'")
    if fname_col:
        print("Sample raw filenames:")
        for v in any_df[fname_col].head(5):
            print("   ", v)

        def loc_key(fn):
            base = Path(str(fn).replace("\\", "/")).name.lower()
            # remove season words, s1/s2 tokens, extension -> keep ROI/scene/patch
            base = re.sub(r"\.(png|jpg|jpeg|tif|tiff)$", "", base)
            for s in SEASONS:
                base = base.replace(s, "")
            base = re.sub(r"s[12]", "", base)
            base = re.sub(r"[_]+", "_", base).strip("_")
            return base

        print("\nSample location keys (season stripped):")
        for v in any_df[fname_col].head(5):
            print(f"    {v}  ->  {loc_key(v)}")

        season_keys = {}
        for s, df in dfs.items():
            if fname_col in df:
                season_keys[s] = set(df[fname_col].map(loc_key))
                print(f"  [{s}] {len(season_keys[s]):,} unique location keys")
        ss = list(season_keys)
        shared = None
        for s in ss:
            shared = season_keys[s] if shared is None else (shared & season_keys[s])
        if shared is not None:
            print(f"\n  >>> location keys present in ALL {len(ss)} seasons: {len(shared):,}")
            for i in range(len(ss)):
                for j in range(i + 1, len(ss)):
                    a, b = ss[i], ss[j]
                    print(f"      {a} ∩ {b}: {len(season_keys[a] & season_keys[b]):,}")
            if shared:
                print("  Example shared keys:", list(shared)[:5])

    # ---------------------------------------------------------------- (C) ROI
    section("C. ROI / SCENE-NUMBER OVERLAP (coarser than patch-level)")
    if fname_col:
        def roi_key(fn):
            base = Path(str(fn).replace("\\", "/")).name.lower()
            m = re.search(r"roi[s]?\d+", base)
            return m.group(0) if m else None
        season_rois = {}
        for s, df in dfs.items():
            if fname_col in df:
                rois = set(r for r in df[fname_col].map(roi_key) if r)
                season_rois[s] = rois
                print(f"  [{s}] {len(rois):,} unique ROI/scene ids "
                      f"(sample: {list(rois)[:3]})")
        ss = list(season_rois)
        if ss and all(season_rois[s] for s in ss):
            for i in range(len(ss)):
                for j in range(i + 1, len(ss)):
                    a, b = ss[i], ss[j]
                    print(f"      {a} ∩ {b}: {len(season_rois[a] & season_rois[b]):,} shared ROIs")

    section("VERDICT GUIDE")
    print("""If A/B/C all show ~0 cross-season overlap -> seasons are independent
patches (standard SEN1-2). No temporal graph available; proceed VAE+bridge,
no GNN. If overlap is substantial -> the SAME locations recur across seasons,
which WOULD justify a cross-season / temporal modeling component. Read the
three sections above and tell me the overlap numbers.""")


if __name__ == "__main__":
    main()

Writing /content/check_dataset_structure.py


In [ ]:
!python /content/check_dataset_structure.py


0. DISCOVER + LOAD CSVs
  [spring] 72,019 rows from 69 csv(s)
  [summer] 59,764 rows from 49 csv(s)
  [fall] 94,202 rows from 74 csv(s)
  [winter] 81,895 rows from 64 csv(s)

1. CSV STRUCTURE (columns + 3 sample rows from first season)
Columns: ['s1_fileName', 's2_fileName', 'season', 'region']

                                   s1_fileName                                  s2_fileName  season     region
0    spring/s1_65/ROIs1158_spring1s2_65_p1.png    spring/s2_65/ROIs1158_spring_s2_65_p1.png  spring  Temperate
1   spring/s1_65/ROIs1158_spring1s2_65_p10.png   spring/s2_65/ROIs1158_spring_s2_65_p10.png  spring  Temperate
2  spring/s1_65/ROIs1158_spring1s2_65_p100.png  spring/s2_65/ROIs1158_spring_s2_65_p100.png  spring  Temperate

A. COORDINATE-COLUMN CHECK (gold standard for 'same location')
No coordinate columns detected. Falling back to filename-based checks.

B. LOCATION-ID FROM FILENAME (season token removed)
Using filename column: 's2_fileName'
Sample raw filenames:
    spring/

In [ ]:
%%writefile /content/stage2_adapted16_256.py
"""
SAR-INTEL Stage 2 (ADAPTED) — fully end-to-end 16-channel SD-1.5 VAE @ 256px.

Removes the frozen-base weakness: the ENTIRE SD-1.5 autoencoder is unfrozen and
co-adapted to Sentinel-2 optical as a single native 16-channel latent.
Initialized from the Stage-1 detail VAE (37.7 dB) so we keep the head start.

Recipe (verified from LDM / Diffusability / practitioner sources):
  L = L1 + LPIPS + kl*KL(annealed 1e-6) + 0.5*align + 0.25*SE + adv*hinge(D)
  - KL very low (1e-6) annealed  -> prevents posterior collapse
  - PatchGAN adversarial, delayed start, hinge, D updated on detached recon
  - scale-equivariance (0.25)    -> keeps the 16ch latent diffusable
  - DA-VAE alignment (0.5)        -> keeps 16 channels structured (base+detail)
256px so results are comparable to published SAR->optical benchmarks.
"""
import os, gc, random, warnings, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from diffusers import AutoencoderKL
from diffusers.optimization import get_cosine_schedule_with_warmup
from accelerate import Accelerator
from accelerate.utils import set_seed
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_download
import lpips as lpips_lib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.fftpack import dctn
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT       = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR      = "/content/drive/MyDrive/stage2_adapted16"
SD_MODEL_ID     = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID      = os.environ["HF_REPO_ID"]
HF_WRITE_TOKEN  = os.environ.get("HF_WRITE_TOKEN", "")
RESUME_TAG      = "detail_vae_final"   # Stage 1 output

IMG_SIZE        = 256
SEASONS         = ("spring", "summer", "fall", "winter")
VAL_FRACTION    = 0.05
SEED            = 42

BASE_CHANNELS   = 4
DETAIL_CHANNELS = 12
TOTAL_CHANNELS  = BASE_CHANNELS + DETAIL_CHANNELS

NUM_TRAIN_STEPS = 20_000
BATCH_SIZE      = 8
LR_AE           = 1e-5      # low: protect the 37.7 dB init, stable adaptation
LR_DISC         = 1e-5
LR_WARMUP       = 300

# loss weights
L1_WEIGHT       = 1.0
LPIPS_WEIGHT    = 1.0
ALIGN_WEIGHT    = 0.5
SE_WEIGHT       = 0.25
ADV_WEIGHT      = 0.1       # gentle (recipe: keep adversarial small)
KL_WEIGHT_MAX   = 1e-6      # very low (prevents posterior collapse)
KL_ANNEAL_STEPS = 2_000
DISC_START      = 1_000     # let reconstruction settle before adversarial
SE_FACTORS      = (2, 4)

CEILING_EVAL_EVERY = 3_000
N_CEILING_EVAL     = 200
DCT_SAMPLES        = 32
SAVE_EVERY         = 4_000
FIXED_VAL_SEED     = 999

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ── Data ──────────────────────────────────────────────────────────────────────
def collect_pairs(root, seasons):
    root_str = Path(root).as_posix()
    buckets  = {s: [] for s in seasons}
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df["season"] = season
        df["region"] = df["region"].str.strip().str.lower()
        df["s2_fileName"] = df["s2_fileName"].str.replace("\\", "/", regex=False)
        df["s2"] = root_str + "/" + df["s2_fileName"]
        buckets[season] = df[["s2", "season", "region"]].to_dict("records")
        print(f"  [{season}] {len(buckets[season]):,} optical")
    pairs = [p for s in seasons if buckets[s] for p in buckets[s]]
    random.shuffle(pairs)
    return pairs


class OpticalDataset(Dataset):
    def __init__(self, pairs, img_size=256, augment=True):
        self.pairs = pairs; self.img_size = img_size; self.augment = augment
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0
    def __getitem__(self, idx):
        opt = self._load(self.pairs[idx]["s2"])
        if self.augment:
            if random.random() > 0.5: opt = TF.hflip(opt)
            if random.random() > 0.5: opt = TF.vflip(opt)
            k = random.randint(0, 3)
            if k: opt = torch.rot90(opt, k, [1, 2])
        return {"optical": opt}


def build_fixed_val(val_pairs, n=2):
    rng = random.Random(FIXED_VAL_SEED)
    sel = []
    for s in SEASONS:
        pool = [p for p in val_pairs if p["season"] == s]
        if pool: sel.extend(rng.sample(pool, min(n, len(pool))))
    return sel


# ── Detail encoder + adapted VAE wrapper (EVERYTHING trainable) ───────────────
class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CHANNELS, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),       nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf * 2, 3, 2, 1),      nn.GroupNorm(8, nf * 2), nn.SiLU(),
            nn.Conv2d(nf * 2, nf * 4, 3, 2, 1),  nn.GroupNorm(8, nf * 4), nn.SiLU(),
            nn.Conv2d(nf * 4, nf * 4, 3, 1, 1),  nn.GroupNorm(8, nf * 4), nn.SiLU(),
            nn.Conv2d(nf * 4, out_ch, 3, 1, 1),
        )
    def forward(self, x): return self.net(x)


class AdaptedVAE(nn.Module):
    """Native 16ch latent. base (variational, 4ch) + detail (deterministic, 12ch).
       ENTIRE module trainable — no frozen base."""
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae   = base_vae
        self.detail_enc = DetailEncoder()
        old = base_vae.decoder.conv_in
        new = nn.Conv2d(TOTAL_CHANNELS, old.out_channels,
                        old.kernel_size, old.stride, old.padding,
                        bias=(old.bias is not None))
        with torch.no_grad():
            new.weight.zero_()
            new.weight[:, :BASE_CHANNELS].copy_(old.weight)
            if old.bias is not None: new.bias.copy_(old.bias)
        base_vae.decoder.conv_in = new
        # NOTHING frozen — full end-to-end adaptation.

    def encode(self, x):
        base_dist = self.base_vae.encode(x).latent_dist     # grad flows (unfrozen)
        z_base    = base_dist.sample()                       # 4ch, stochastic
        z_detail  = self.detail_enc(x)                       # 12ch, deterministic
        return z_base, z_detail, base_dist

    def decode(self, z_base, z_detail):
        z_base_pq = self.base_vae.post_quant_conv(z_base)
        combined  = torch.cat([z_base_pq, z_detail], dim=1)  # 16ch
        return self.base_vae.decoder(combined), combined

    @torch.no_grad()
    def reconstruct(self, x):
        base_dist = self.base_vae.encode(x).latent_dist
        z_base    = base_dist.mode()                         # deterministic for eval
        z_detail  = self.detail_enc(x)
        recon, combined = self.decode(z_base, z_detail)
        return recon, combined


# ── PatchGAN discriminator (taming-transformers style) ────────────────────────
class NLayerDiscriminator(nn.Module):
    def __init__(self, in_ch=3, nf=64, n_layers=3):
        super().__init__()
        layers = [nn.Conv2d(in_ch, nf, 4, 2, 1), nn.LeakyReLU(0.2, True)]
        mult = 1
        for n in range(1, n_layers):
            mult_prev, mult = mult, min(2 ** n, 8)
            layers += [nn.Conv2d(nf * mult_prev, nf * mult, 4, 2, 1, bias=False),
                       nn.BatchNorm2d(nf * mult), nn.LeakyReLU(0.2, True)]
        mult_prev, mult = mult, min(2 ** n_layers, 8)
        layers += [nn.Conv2d(nf * mult_prev, nf * mult, 4, 1, 1, bias=False),
                   nn.BatchNorm2d(nf * mult), nn.LeakyReLU(0.2, True),
                   nn.Conv2d(nf * mult, 1, 4, 1, 1)]
        self.main = nn.Sequential(*layers)
    def forward(self, x): return self.main(x)


# ── Losses ────────────────────────────────────────────────────────────────────
def alignment_loss(z_base, z_detail):
    B, D, H, W = z_detail.shape
    r = D // BASE_CHANNELS
    grouped = z_detail.view(B, BASE_CHANNELS, r, H, W).mean(dim=2)
    return F.mse_loss(grouped, z_base)


def scale_equivariance_loss(model, x, combined):
    factor = random.choice(SE_FACTORS)
    x_down = F.interpolate(x, scale_factor=1.0 / factor, mode="bilinear", align_corners=False)
    c_down = F.interpolate(combined, scale_factor=1.0 / factor, mode="bilinear", align_corners=False)
    recon_down = model.base_vae.decoder(c_down)
    return F.l1_loss(recon_down, x_down)


def hinge_d(real_logits, fake_logits):
    return 0.5 * (F.relu(1.0 - real_logits).mean() + F.relu(1.0 + fake_logits).mean())


# ── Diagnostics ───────────────────────────────────────────────────────────────
def freq_profile(latent_bchw):
    arr = latent_bchw.detach().float().cpu().numpy()
    B, C, H, W = arr.shape
    maxk = (H - 1) + (W - 1)
    acc = np.zeros(maxk + 1); cnt = np.zeros(maxk + 1)
    for b in range(B):
        for c in range(C):
            D = np.abs(dctn(arr[b, c], norm="ortho")); D = D / (D[0, 0] + 1e-8)
            for u in range(H):
                for v in range(W):
                    acc[u + v] += D[u, v]; cnt[u + v] += 1
    prof = acc / np.maximum(cnt, 1)
    return np.linspace(0, 1, len(prof)), prof


def denorm(t): return ((t.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()


@torch.no_grad()
def measure_ceiling(model, vanilla_vae, val_pairs, lpips_fn, n=N_CEILING_EVAL):
    model.eval()
    subset = random.Random(123).sample(val_pairs, min(n, len(val_pairs)))
    vp, vl, dp, dl = [], [], [], []
    for pair in subset:
        img = OpticalDataset([pair], IMG_SIZE, augment=False)[0]["optical"]
        x   = img.unsqueeze(0).to(device)
        rec_v = vanilla_vae.decode(vanilla_vae.encode(x).latent_dist.mode()).sample.clamp(-1, 1)
        rec_d, _ = model.reconstruct(x); rec_d = rec_d.clamp(-1, 1)
        vp.append(10 * np.log10(4.0 / F.mse_loss(rec_v, x).item()))
        dp.append(10 * np.log10(4.0 / F.mse_loss(rec_d, x).item()))
        vl.append(lpips_fn(rec_v, x).item()); dl.append(lpips_fn(rec_d, x).item())
    model.train()
    return np.mean(vp), np.mean(vl), np.mean(dp), np.mean(dl)


@torch.no_grad()
def collect_profiles(model, vanilla_vae, val_pairs, n=DCT_SAMPLES):
    model.eval()
    subset = random.Random(7).sample(val_pairs, min(n, len(val_pairs)))
    van, det, rgb = [], [], []
    for pair in subset:
        img = OpticalDataset([pair], IMG_SIZE, augment=False)[0]["optical"]
        x   = img.unsqueeze(0).to(device)
        van.append(vanilla_vae.encode(x).latent_dist.mode().cpu())
        _, combined = model.reconstruct(x); det.append(combined.cpu())
        rgb.append(F.interpolate(x, size=(32, 32), mode="bilinear", align_corners=False).cpu())
    model.train()
    fa, pv = freq_profile(torch.cat(van))
    _,  pd_ = freq_profile(torch.cat(det))
    _,  pr  = freq_profile(torch.cat(rgb))
    return fa, pv, pd_, pr


def push_to_hf(folder, tag):
    if not HF_WRITE_TOKEN: return
    try:
        HfApi().upload_folder(folder_path=folder, path_in_repo=tag,
                              repo_id=HF_REPO_ID, repo_type="model", token=HF_WRITE_TOKEN)
        print(f"  HF push -> {tag}")
    except Exception as e:
        print(f"  HF push failed: {e}")


def load_stage1(model):
    de = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{RESUME_TAG}/detail_encoder.pth")
    dc = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{RESUME_TAG}/decoder.pth")
    model.detail_enc.load_state_dict(torch.load(de, map_location="cpu"))
    model.base_vae.decoder.load_state_dict(torch.load(dc, map_location="cpu"))
    print("  Init from Stage 1: detail_encoder + decoder loaded; encoder = SD-1.5 default.")
    print("  All components now UNFROZEN for end-to-end adaptation.")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    accelerator = Accelerator(mixed_precision="no", gradient_accumulation_steps=1,
                              project_dir=OUTPUT_DIR)
    set_seed(SEED)
    is_main = accelerator.is_main_process
    if is_main: os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Loading SD-1.5 VAE + building adapted 16ch model...")
    base_vae    = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vanilla_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae").to(device).eval()
    vanilla_vae.requires_grad_(False)

    model = AdaptedVAE(base_vae)
    load_stage1(model)
    disc  = NLayerDiscriminator(in_ch=3)
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    print(f"  VAE trainable: {sum(p.numel() for p in model.parameters())/1e6:.1f}M | "
          f"Disc: {sum(p.numel() for p in disc.parameters())/1e6:.1f}M")

    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    train_pairs, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['region']}" for p in all_pairs], random_state=SEED)
    fixed_val = build_fixed_val(val_pairs)
    print(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,} | @ {IMG_SIZE}px")

    train_ds = OpticalDataset(train_pairs, IMG_SIZE, augment=True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                          pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=2)

    opt_ae   = torch.optim.AdamW(model.parameters(), lr=LR_AE, betas=(0.5, 0.9), weight_decay=1e-4)
    opt_disc = torch.optim.AdamW(disc.parameters(),  lr=LR_DISC, betas=(0.5, 0.9))
    lr_sched = get_cosine_schedule_with_warmup(opt_ae, LR_WARMUP, NUM_TRAIN_STEPS)

    model, disc, opt_ae, opt_disc, train_dl, lr_sched = accelerator.prepare(
        model, disc, opt_ae, opt_disc, train_dl, lr_sched)

    # BEFORE snapshot
    if is_main:
        vp, vl, dp, dl = measure_ceiling(accelerator.unwrap_model(model), vanilla_vae, val_pairs, lpips_fn, 150)
        print("\n" + "="*62)
        print("BEFORE (256px, Stage-1 init, nothing adapted yet)")
        print(f"  Vanilla 4ch  : PSNR {vp:.3f} dB | LPIPS {vl:.4f}")
        print(f"  Adapted 16ch : PSNR {dp:.3f} dB | LPIPS {dl:.4f}  (gain {dp-vp:+.3f})")
        print("="*62 + "\n")
        fa, pv, pd_before, pr = collect_profiles(accelerator.unwrap_model(model), vanilla_vae, val_pairs)

    global_step = 0
    e_l1 = e_lp = e_kl = e_al = e_se = e_g = e_d = 0.0; a = 0.98
    steps_per_epoch = len(train_dl)
    total_epochs    = (NUM_TRAIN_STEPS + steps_per_epoch - 1) // steps_per_epoch
    bar = tqdm(total=NUM_TRAIN_STEPS, desc="Stage 2 adapted-16ch", disable=not is_main, dynamic_ncols=True)
    model.train(); disc.train()

    for epoch in range(total_epochs):
        for batch in train_dl:
            x = batch["optical"].to(device)
            z_base, z_detail, base_dist = model.encode(x)
            recon, combined = model.decode(z_base, z_detail)

            # reconstruction + regularizers
            l1   = F.l1_loss(recon, x) * L1_WEIGHT
            lp   = lpips_fn(recon.clamp(-1, 1), x).mean() * LPIPS_WEIGHT
            kl_w = KL_WEIGHT_MAX * min(1.0, global_step / KL_ANNEAL_STEPS)
            kl   = base_dist.kl().mean() * kl_w
            al   = alignment_loss(z_base, z_detail) * ALIGN_WEIGHT
            se   = scale_equivariance_loss(accelerator.unwrap_model(model), x, combined) * SE_WEIGHT
            adv_on = global_step >= DISC_START
            g_adv  = (-disc(recon).mean() * ADV_WEIGHT) if adv_on else torch.tensor(0.0, device=device)
            ae_loss = l1 + lp + kl + al + se + g_adv

            opt_ae.zero_grad(set_to_none=True)
            if adv_on: opt_disc.zero_grad(set_to_none=True)
            accelerator.backward(ae_loss)
            opt_ae.step(); lr_sched.step()

            # discriminator step
            d_loss_val = 0.0
            if adv_on:
                opt_disc.zero_grad(set_to_none=True)
                d_real = disc(x); d_fake = disc(recon.detach())
                d_loss = hinge_d(d_real, d_fake)
                accelerator.backward(d_loss); opt_disc.step()
                d_loss_val = d_loss.item()

            global_step += 1
            e_l1 = a*e_l1+(1-a)*l1.item(); e_lp = a*e_lp+(1-a)*lp.item()
            e_kl = a*e_kl+(1-a)*kl.item(); e_al = a*e_al+(1-a)*al.item()
            e_se = a*e_se+(1-a)*se.item(); e_g = a*e_g+(1-a)*float(g_adv); e_d = a*e_d+(1-a)*d_loss_val
            if is_main:
                bar.update(1)
                bar.set_postfix({"l1": f"{e_l1:.3f}", "lpips": f"{e_lp:.3f}", "se": f"{e_se:.3f}",
                                 "G": f"{e_g:.3f}", "D": f"{e_d:.3f}"})

            if global_step % CEILING_EVAL_EVERY == 0 and is_main:
                vp, vl, dp, dl = measure_ceiling(accelerator.unwrap_model(model), vanilla_vae, val_pairs, lpips_fn)
                tqdm.write(f"\n[step {global_step}] Adapted 16ch: PSNR {dp:.3f} | LPIPS {dl:.4f} "
                           f"(vanilla {vp:.3f}, gain {dp-vp:+.3f})\n")

            if global_step % SAVE_EVERY == 0 and is_main:
                ckpt = os.path.join(OUTPUT_DIR, f"step_{global_step}")
                os.makedirs(ckpt, exist_ok=True)
                m = accelerator.unwrap_model(model)
                torch.save(m.detail_enc.state_dict(), os.path.join(ckpt, "detail_encoder.pth"))
                torch.save(m.base_vae.state_dict(), os.path.join(ckpt, "base_vae.pth"))
                torch.save({"base_channels": BASE_CHANNELS, "detail_channels": DETAIL_CHANNELS,
                            "total_channels": TOTAL_CHANNELS, "img_size": IMG_SIZE,
                            "adapted": True}, os.path.join(ckpt, "adapted_vae_config.pt"))
                latest = os.path.join(OUTPUT_DIR, "latest")
                if os.path.exists(latest): shutil.rmtree(latest)
                shutil.copytree(ckpt, latest)
                push_to_hf(ckpt, f"adapted16_step_{global_step}")

            if global_step >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if global_step >= NUM_TRAIN_STEPS: break
    bar.close()

    if is_main:
        vp, vl, dp, dl = measure_ceiling(accelerator.unwrap_model(model), vanilla_vae, val_pairs, lpips_fn, 300)
        print("\n" + "="*62)
        print("AFTER  (256px, fully adapted 16ch + SE + adversarial)")
        print(f"  Vanilla 4ch  : PSNR {vp:.3f} dB | LPIPS {vl:.4f}")
        print(f"  Adapted 16ch : PSNR {dp:.3f} dB | LPIPS {dl:.4f}  (gain {dp-vp:+.3f})")
        print("="*62)
        _, _, pd_after, _ = collect_profiles(accelerator.unwrap_model(model), vanilla_vae, val_pairs)

        plt.figure(figsize=(8, 5))
        plt.semilogy(fa, pv,        label="SD vanilla 4ch", lw=2)
        plt.semilogy(fa, pd_before, label="Adapted 16ch (before)", lw=2, ls="--")
        plt.semilogy(fa, pd_after,  label="Adapted 16ch (after)", lw=2)
        plt.semilogy(fa, pr,        label="RGB reference", lw=2, color="black", alpha=0.6)
        plt.xlabel("Normalized frequency (low → high)"); plt.ylabel("Norm. DCT amplitude (log)")
        plt.title("Latent diffusability after end-to-end adaptation + SE")
        plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
        fp = os.path.join(OUTPUT_DIR, "diffusability_dct.png")
        plt.savefig(fp, dpi=150, bbox_inches="tight"); display(plt.gcf()); plt.close()

        final = os.path.join(OUTPUT_DIR, "final")
        os.makedirs(final, exist_ok=True)
        m = accelerator.unwrap_model(model)
        torch.save(m.detail_enc.state_dict(), os.path.join(final, "detail_encoder.pth"))
        torch.save(m.base_vae.state_dict(), os.path.join(final, "base_vae.pth"))
        torch.save({"base_channels": BASE_CHANNELS, "detail_channels": DETAIL_CHANNELS,
                    "total_channels": TOTAL_CHANNELS, "img_size": IMG_SIZE, "adapted": True},
                   os.path.join(final, "adapted_vae_config.pt"))
        shutil.copy(fp, os.path.join(final, "diffusability_dct.png"))
        push_to_hf(final, "adapted16_final")
        print("\nStage 2 (adapted) complete. Native 16ch SD-1.5 latent @ 256px.")
    accelerator.end_training()


if __name__ == "__main__":
    main()

Writing /content/stage2_adapted16_256.py


In [ ]:
import os
from google.colab import userdata; os.environ["HF_WRITE_TOKEN"] = userdata.get("HF_TOKEN")
!pip install -q lpips scipy
!python /content/stage2_adapted16_256.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading SD-1.5 VAE + building adapted 16ch model...
detail_vae_final/detail_encoder.pth: 100% 3.97M/3.97M [00:00<00:00, 5.36MB/s]
detail_vae_final/decoder.pth: 100% 198M/198M [00:01<00:00

# Stage 3

In [ ]:
%%writefile /content/stage3_bridge_256.py
"""
SAR-INTEL Stage 3 — ResShift bridge on the adapted 16ch latent @ 256px.
RESUMES from the 80k full-state on HF ('bridge_resume'). No periodic HF pushes
(removed to kill upload overhead). Evaluates every 5k. Saves FINAL model ONCE
to HF in safetensors. A fast LOCAL-only checkpoint every 20k is the sole safety
net (disk write, no upload). Set LOCAL_SAVE_EVERY=0 to disable entirely.
"""
import os, gc, re, math, random, warnings, shutil, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from safetensors.torch import save_file
import lpips as lpips_lib
import pandas as pd
from torchmetrics.image import StructuralSimilarityIndexMeasure
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT    = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
DRIVE_LATEST = "/content/drive/MyDrive/stage3_bridge/latest"  # legacy fallback only
OUTPUT_DIR   = "/content/stage3_bridge"
SD_MODEL_ID  = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID   = os.environ["HF_REPO_ID"]
HF_TOKEN     = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG      = "adapted16_final"
RESUME_TAG   = "bridge_resume"        # 80k full state lives here on HF (read once)

IMG_SIZE     = 256
SEASONS      = ("spring", "summer", "fall", "winter")
VAL_FRACTION = 0.05
SEED         = 42

BASE_CH, DETAIL_CH = 4, 12
TOTAL_CH     = 16

T_STEPS      = 15
KAPPA        = 2.0
ETA_1, ETA_T = 0.04, 0.99

NUM_TRAIN_STEPS = 150_000
BATCH_SIZE   = 4
LR           = 5e-5
LR_WARMUP    = 1_000
COND_DROPOUT = 0.10
VAL_EVERY    = 5_000
LOCAL_SAVE_EVERY = 20_000     # LOCAL disk only (fast, no upload). 0 = disable.
N_VAL        = 100
LOG_EVERY    = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEASON_DESC = {"spring": "spring", "summer": "summer", "fall": "autumn", "winter": "winter"}
TERRAINS = ("tropical", "temperate", "arctic", "arid", "coastal", "urban")

def map_region_to_terrain(region):
    r = (region or "").lower()
    if any(k in r for k in ("trop", "rainforest", "amazon", "jungle")): return "tropical"
    if any(k in r for k in ("arctic", "tundra", "snow", "ice", "polar")): return "arctic"
    if any(k in r for k in ("desert", "arid", "sahara", "dry")): return "arid"
    if any(k in r for k in ("coast", "beach", "shore", "marine", "sea")): return "coastal"
    if any(k in r for k in ("urban", "city", "metro")): return "urban"
    return "temperate"


# ── Data ──────────────────────────────────────────────────────────────────────
def collect_pairs(root, seasons):
    root_str = Path(root).as_posix()
    pairs = []
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for col in ("s1_fileName", "s2_fileName"):
            df[col] = df[col].astype(str).str.replace("\\", "/", regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            pairs.append({"sar": f"{root_str}/{row['s1_fileName']}",
                          "opt": f"{root_str}/{row['s2_fileName']}",
                          "season": season,
                          "terrain": map_region_to_terrain(row["region"])})
    random.shuffle(pairs)
    return pairs


class PairDataset(Dataset):
    def __init__(self, pairs, img_size=256, augment=True):
        self.pairs = pairs; self.img_size = img_size; self.augment = augment
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0
    def __getitem__(self, idx):
        p = self.pairs[idx]
        sar = self._load(p["sar"]); opt = self._load(p["opt"])
        if self.augment:
            if random.random() > 0.5: sar, opt = torch.flip(sar, [2]), torch.flip(opt, [2])
            if random.random() > 0.5: sar, opt = torch.flip(sar, [1]), torch.flip(opt, [1])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}


# ── Adapted VAE (frozen) ──────────────────────────────────────────────────────
class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CH, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),      nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf*2, 3, 2, 1),       nn.GroupNorm(8, nf*2), nn.SiLU(),
            nn.Conv2d(nf*2, nf*4, 3, 2, 1),     nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, nf*4, 3, 1, 1),     nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, out_ch, 3, 1, 1))
    def forward(self, x): return self.net(x)


class AdaptedVAE(nn.Module):
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae = base_vae
        self.detail_enc = DetailEncoder()
        old = base_vae.decoder.conv_in
        base_vae.decoder.conv_in = nn.Conv2d(TOTAL_CH, old.out_channels, old.kernel_size,
                                             old.stride, old.padding, bias=(old.bias is not None))
    @torch.no_grad()
    def encode_latent(self, x):
        z_base = self.base_vae.encode(x).latent_dist.mode()
        z_base_pq = self.base_vae.post_quant_conv(z_base)
        z_detail = self.detail_enc(x)
        return torch.cat([z_base_pq, z_detail], dim=1)
    @torch.no_grad()
    def decode_latent(self, lat):
        return self.base_vae.decoder(lat)


def load_adapted_vae():
    base_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(base_vae)
    bp = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/base_vae.pth")
    de = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/detail_encoder.pth")
    vae.base_vae.load_state_dict(torch.load(bp, map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(de, map_location="cpu"))
    vae = vae.to(device).eval().float(); vae.requires_grad_(False)
    print(f"  Adapted VAE loaded from {VAE_TAG}.")
    return vae


def build_embed_cache():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    txt.requires_grad_(False)
    cache = {}
    @torch.no_grad()
    def embed(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s, ter)] = embed(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = embed("")
    del txt; gc.collect(); torch.cuda.empty_cache()
    print(f"  Built {len(cache)} CLIP embeddings.")
    return cache


def build_unet():
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    ni = nn.Conv2d(2 * TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    with torch.no_grad():
        ni.weight.zero_()
        for i in range(2 * TOTAL_CH): ni.weight[:, i] = oi.weight[:, i % oi.in_channels] / 8.0
        ni.bias.copy_(oi.bias)
    unet.conv_in = ni; unet.config.in_channels = 2 * TOTAL_CH
    oo = unet.conv_out
    no = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    with torch.no_grad():
        for i in range(TOTAL_CH):
            no.weight[i] = oo.weight[i % oo.out_channels]; no.bias[i] = oo.bias[i % oo.out_channels]
    unet.conv_out = no; unet.config.out_channels = TOTAL_CH
    print(f"  UNet: conv_in {2*TOTAL_CH}ch, conv_out {TOTAL_CH}ch.")
    return unet


def make_eta(T, e1, eT):
    idx = np.arange(T)
    return torch.tensor(e1 * (eT / e1) ** (idx / (T - 1)), dtype=torch.float32)


def bridge_forward(x0, y, t, eta, kappa):
    et = eta[t].view(-1, 1, 1, 1).to(x0.device)
    return x0 + et * (y - x0) + kappa * torch.sqrt(et) * torch.randn_like(x0)


@torch.no_grad()
def bridge_sample(unet, y, cond, eta, kappa, dtype):
    B = y.shape[0]; T = len(eta)
    x = y + kappa * torch.sqrt(eta[-1]).to(y.device) * torch.randn_like(y)
    for t in reversed(range(T)):
        tb = torch.full((B,), int(t * 1000 / T), device=y.device, dtype=torch.long)
        x0 = unet(torch.cat([x, y], 1).to(dtype), tb, encoder_hidden_states=cond.to(dtype)).sample.float()
        if t > 0:
            ep = eta[t - 1].to(y.device)
            x = x0 + ep * (y - x0) + kappa * torch.sqrt(ep) * torch.randn_like(x)
        else:
            x = x0
    return x


def get_cond(seasons, terrains, cache, drop=0.0):
    out = []
    for s, ter in zip(seasons, terrains):
        if drop > 0 and random.random() < drop: out.append(cache["null"])
        else: out.append(cache.get((s, ter), cache["null"]))
    return torch.stack(out)


def push_hf(folder, tag):
    if not HF_TOKEN: return
    try:
        HfApi().upload_folder(folder_path=folder, path_in_repo=tag,
                              repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)
        print(f"  HF push -> {tag}")
    except Exception as e:
        print(f"  HF push failed: {e}")


def download_resume_from_hf(dest):
    try:
        snapshot_download(repo_id=HF_REPO_ID, allow_patterns=f"{RESUME_TAG}/*",
                          local_dir="/content/_resume_dl", token=HF_TOKEN)
        src = os.path.join("/content/_resume_dl", RESUME_TAG)
        if os.path.exists(os.path.join(src, "meta.pt")):
            if os.path.exists(dest): shutil.rmtree(dest)
            shutil.copytree(src, dest); return True
    except Exception as e:
        print(f"  no HF resume state ({e})")
    return False


def main():
    accelerator = Accelerator(mixed_precision="bf16", project_dir=OUTPUT_DIR)
    set_seed(SEED)
    is_main = accelerator.is_main_process
    dtype = torch.bfloat16
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Loading components...")
    vae = load_adapted_vae()
    embed_cache = build_embed_cache()
    unet = build_unet()
    eta = make_eta(T_STEPS, ETA_1, ETA_T).to(device)
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    train_pairs, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in all_pairs], random_state=SEED)
    print(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,}")

    train_ds = PairDataset(train_pairs, IMG_SIZE, True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                          pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=2)

    optimizer = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9, 0.999), weight_decay=1e-2)
    lr_sched = get_cosine_schedule_with_warmup(optimizer, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, optimizer, train_dl, lr_sched = accelerator.prepare(unet, optimizer, train_dl, lr_sched)

    # ── resume: HF 'bridge_resume' (80k) first; Drive fallback; else fresh ──
    latest = os.path.join(OUTPUT_DIR, "latest")
    start_step = 0; latent_scale = None; resume_dir = None
    if is_main and download_resume_from_hf(latest):
        resume_dir = latest; print("  Resume source: HF (bridge_resume, 80k).")
    accelerator.wait_for_everyone()
    if not resume_dir and os.path.exists(os.path.join(latest, "meta.pt")):
        resume_dir = latest
    if not resume_dir and os.path.exists(os.path.join(DRIVE_LATEST, "meta.pt")):
        resume_dir = DRIVE_LATEST; print("  Resume source: DRIVE (fallback).")
    if resume_dir and os.path.exists(os.path.join(resume_dir, "meta.pt")):
        meta = torch.load(os.path.join(resume_dir, "meta.pt"), map_location="cpu")
        start_step = meta["step"]; latent_scale = meta["latent_scale"]
        accelerator.load_state(resume_dir)
        print(f"  RESUMED @ step {start_step} (latent_scale={latent_scale:.4f}).")
    else:
        if is_main:
            stds = []
            for b in DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2):
                with torch.autocast("cuda", enabled=False):
                    stds.append(vae.encode_latent(b["opt"].to(device).float()).std().item())
                if len(stds) >= 10: break
            latent_scale = 1.0 / float(np.mean(stds))
            print(f"  Computed latent_scale = {latent_scale:.4f}")
        ls = torch.tensor(latent_scale if latent_scale else 1.0, device=device)
        latent_scale = float(accelerator.gather(ls).mean().item()) if accelerator.num_processes > 1 else float(ls)

    if is_main:
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            vb = next(iter(DataLoader(PairDataset(val_pairs, IMG_SIZE, False), batch_size=8)))
            x = vb["opt"].to(device).float()
            rec = vae.decode_latent(vae.encode_latent(x)).clamp(-1, 1)
            psnr = 10 * np.log10(4.0 / F.mse_loss(rec, x).item())
        print(f"  SANITY: VAE recon PSNR = {psnr:.2f} dB ({'OK' if psnr > 22 else 'LOW!'})")

    def encode_scaled(imgs):
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            return vae.encode_latent(imgs.float()) * latent_scale

    @torch.no_grad()
    def validate(step):
        sub = random.Random(7).sample(val_pairs, min(N_VAL, len(val_pairs)))
        vdl = DataLoader(PairDataset(sub, IMG_SIZE, False), batch_size=8, num_workers=2)
        P, S, L = [], [], []
        m = accelerator.unwrap_model(unet); m.eval()
        for b in vdl:
            opt = b["opt"].to(device); sar = b["sar"].to(device)
            y = encode_scaled(sar)
            cond = get_cond(b["season"], b["terrain"], embed_cache).to(device)
            x0 = bridge_sample(m, y, cond, eta, KAPPA, dtype)
            with torch.autocast("cuda", enabled=False):
                rec = vae.decode_latent((x0 / latent_scale).float()).clamp(-1, 1)
            P.append(10 * np.log10(4.0 / F.mse_loss(rec, opt).item()))
            S.append(ssim_fn(rec, opt).item()); L.append(lpips_fn(rec, opt).mean().item())
        m.train()
        print(f"\n[VAL step {step}] PSNR {np.mean(P):.3f} | SSIM {np.mean(S):.4f} | LPIPS {np.mean(L):.4f}\n")

    def local_save(step):
        if LOCAL_SAVE_EVERY == 0: return
        accelerator.wait_for_everyone()
        if not is_main: return
        os.makedirs(latest, exist_ok=True)
        accelerator.save_state(latest)                      # LOCAL disk, fast, no upload
        torch.save({"step": step, "latent_scale": latent_scale}, os.path.join(latest, "meta.pt"))
        print(f"  local safety checkpoint @ step {step}")

    def final_save(step):
        accelerator.wait_for_everyone()
        if not is_main: return
        tmp = os.path.join(OUTPUT_DIR, "_final"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k, v in accelerator.unwrap_model(unet).state_dict().items()}
        save_file(sd, os.path.join(tmp, "unet.safetensors"))
        with open(os.path.join(tmp, "bridge_config.json"), "w") as f:
            json.dump({"latent_scale": latent_scale, "T": T_STEPS, "kappa": KAPPA,
                       "eta_1": ETA_1, "eta_T": ETA_T, "img_size": IMG_SIZE,
                       "final_step": step}, f, indent=2)
        push_hf(tmp, "bridge_final")
        shutil.rmtree(tmp, ignore_errors=True)
        print(f"  FINAL model (safetensors) -> HF/bridge_final @ step {step}")

    global_step = start_step
    ema = 0.0; a = 0.98
    spe = len(train_dl); epochs = (NUM_TRAIN_STEPS + spe - 1) // spe
    bar = tqdm(total=NUM_TRAIN_STEPS, initial=start_step, desc="Stage 3 bridge",
               disable=not is_main, dynamic_ncols=True)
    unet.train()
    print(f"\n>>> Resuming @ {start_step}. No periodic HF push. Final safetensors save at the end.\n")

    for epoch in range(epochs):
        for batch in train_dl:
            x0 = encode_scaled(batch["opt"].to(device))
            y  = encode_scaled(batch["sar"].to(device))
            cond = get_cond(batch["season"], batch["terrain"], embed_cache, COND_DROPOUT).to(device)
            t = torch.randint(0, T_STEPS, (x0.shape[0],), device=device)
            x_t = bridge_forward(x0, y, t, eta, KAPPA)
            tb = (t * 1000 // T_STEPS).long()
            with accelerator.accumulate(unet):
                pred = unet(torch.cat([x_t, y], 1).to(dtype), tb,
                            encoder_hidden_states=cond.to(dtype)).sample
                loss = F.mse_loss(pred.float(), x0.float())
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), 1.0)
                optimizer.step(); lr_sched.step(); optimizer.zero_grad(set_to_none=True)
            if accelerator.sync_gradients:
                global_step += 1
                ema = a * ema + (1 - a) * loss.item()
                if is_main:
                    bar.update(1)
                    if global_step % LOG_EVERY == 0: bar.set_postfix({"mse": f"{ema:.4f}"})
                if global_step % VAL_EVERY == 0 and is_main: validate(global_step)
                if LOCAL_SAVE_EVERY and global_step % LOCAL_SAVE_EVERY == 0: local_save(global_step)
            if global_step >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if global_step >= NUM_TRAIN_STEPS: break

    bar.close()
    if is_main: validate(global_step)
    final_save(global_step)
    accelerator.end_training()
    print("\nStage 3 complete. Final model on HF/bridge_final (safetensors).")


if __name__ == "__main__":
    main()

Overwriting /content/stage3_bridge_256.py


In [ ]:
import os
from google.colab import userdata; os.environ["HF_WRITE_TOKEN"] = userdata.get("HF_TOKEN")
!pip install -q lpips torchmetrics scipy
!python /content/stage3_bridge_256.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading components...
  Adapted VAE loaded from adapted16_final.
Loading weights: 100% 196/196 [00:00<00:00, 1262.32it/s, Materializing param=text_model.final_layer_norm.weight]
CLIPTextM

In [ ]:
%%writefile /content/stage4_evaluate.py
"""
SAR-INTEL Stage 4 — Evaluation. Loads bridge_final (safetensors) + adapted VAE,
runs the full val set and reports the paper's results:
  - PSNR / SSIM / LPIPS overall, per-season, per-terrain
  - FID (headline metric for the paper) via clean-fid / pytorch-fid fallback
  - steps-ablation (1 / 5 / 15 steps): does iterative refinement help or collapse?
  - resolution: 256px (benchmark-comparable) AND 512px
  - visual comparison grids (SAR | predicted | ground-truth) per season
Inference only. No training. ~1-2 hours.
"""
import os, gc, re, json, random, warnings, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from huggingface_hub import hf_hub_download, HfApi
from safetensors.torch import load_file
import lpips as lpips_lib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
from sklearn.model_selection import train_test_split
from collections import defaultdict

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT   = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR  = "/content/stage4_eval"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID  = os.environ["HF_REPO_ID"]
HF_TOKEN    = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG     = "adapted16_final"
BRIDGE_TAG  = "bridge_final"

SEASONS     = ("spring", "summer", "fall", "winter")
TERRAINS    = ("tropical", "temperate", "arctic", "arid", "coastal", "urban")
VAL_FRACTION= 0.05
SEED        = 42
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16

# evaluation scope
EVAL_RES        = (256, 512)        # headline 256, also report 512
N_EVAL          = 1000             # val patches for metrics (set None for full val set)
N_FID           = 1000             # patches for FID (more = more stable)
STEPS_ABLATION  = (1, 5, 15)       # check refinement vs single-step collapse
N_GRID_PER_SEASON = 3
BATCH           = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEASON_DESC = {"spring": "spring", "summer": "summer", "fall": "autumn", "winter": "winter"}

def map_region_to_terrain(region):
    r = (region or "").lower()
    if any(k in r for k in ("trop", "rainforest", "amazon", "jungle")): return "tropical"
    if any(k in r for k in ("arctic", "tundra", "snow", "ice", "polar")): return "arctic"
    if any(k in r for k in ("desert", "arid", "sahara", "dry")): return "arid"
    if any(k in r for k in ("coast", "beach", "shore", "marine", "sea")): return "coastal"
    if any(k in r for k in ("urban", "city", "metro")): return "urban"
    return "temperate"


def collect_pairs(root, seasons):
    root_str = Path(root).as_posix(); pairs = []
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for col in ("s1_fileName", "s2_fileName"):
            df[col] = df[col].astype(str).str.replace("\\", "/", regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            pairs.append({"sar": f"{root_str}/{row['s1_fileName']}",
                          "opt": f"{root_str}/{row['s2_fileName']}",
                          "season": season, "terrain": map_region_to_terrain(row["region"])})
    return pairs


class PairDataset(Dataset):
    def __init__(self, pairs, img_size):
        self.pairs = pairs; self.img_size = img_size
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0
    def __getitem__(self, idx):
        p = self.pairs[idx]
        return {"sar": self._load(p["sar"]), "opt": self._load(p["opt"]),
                "season": p["season"], "terrain": p["terrain"]}


# ── Model defs (must match Stage 3) ───────────────────────────────────────────
class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CH, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),  nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf*2, 3, 2, 1),   nn.GroupNorm(8, nf*2), nn.SiLU(),
            nn.Conv2d(nf*2, nf*4, 3, 2, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, nf*4, 3, 1, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, out_ch, 3, 1, 1))
    def forward(self, x): return self.net(x)


class AdaptedVAE(nn.Module):
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae = base_vae; self.detail_enc = DetailEncoder()
        old = base_vae.decoder.conv_in
        base_vae.decoder.conv_in = nn.Conv2d(TOTAL_CH, old.out_channels, old.kernel_size,
                                             old.stride, old.padding, bias=(old.bias is not None))
    @torch.no_grad()
    def encode_latent(self, x):
        z = self.base_vae.encode(x).latent_dist.mode()
        return torch.cat([self.base_vae.post_quant_conv(z), self.detail_enc(x)], dim=1)
    @torch.no_grad()
    def decode_latent(self, lat):
        return self.base_vae.decoder(lat)


def load_models():
    base_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(base_vae)
    bp = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/base_vae.pth")
    de = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/detail_encoder.pth")
    vae.base_vae.load_state_dict(torch.load(bp, map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(de, map_location="cpu"))
    vae = vae.to(device).eval().float(); vae.requires_grad_(False)

    cfg_path = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{BRIDGE_TAG}/bridge_config.json")
    cfg = json.load(open(cfg_path))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels = 2*TOTAL_CH; unet.config.out_channels = TOTAL_CH
    sd_path = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{BRIDGE_TAG}/unet.safetensors")
    unet.load_state_dict(load_file(sd_path))
    unet = unet.to(device).eval(); unet.requires_grad_(False)
    print(f"  Loaded bridge_final (step {cfg.get('final_step')}), latent_scale={cfg['latent_scale']:.4f}")
    return vae, unet, cfg


def build_embed_cache():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    txt.requires_grad_(False); cache = {}
    @torch.no_grad()
    def embed(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s, ter)] = embed(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = embed(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache


def make_eta(T, e1, eT):
    idx = np.arange(T)
    return torch.tensor(e1 * (eT / e1) ** (idx / (T - 1)), dtype=torch.float32)


def get_cond(seasons, terrains, cache):
    return torch.stack([cache.get((s, t), cache["null"]) for s, t in zip(seasons, terrains)])


@torch.no_grad()
def sample(unet, y, cond, eta_full, kappa, n_steps):
    """ResShift sampling with n_steps (subsampled from the full T schedule)."""
    T = len(eta_full)
    idx = np.linspace(0, T - 1, n_steps).round().astype(int)
    eta = eta_full[idx]
    B = y.shape[0]
    x = y + kappa * torch.sqrt(eta[-1]).to(y.device) * torch.randn_like(y)
    for k in reversed(range(len(eta))):
        tb = torch.full((B,), int(idx[k] * 1000 / T), device=y.device, dtype=torch.long)
        x0 = unet(torch.cat([x, y], 1).to(torch.bfloat16), tb,
                  encoder_hidden_states=cond.to(torch.bfloat16)).sample.float()
        if k > 0:
            ep = eta[k - 1].to(y.device)
            x = x0 + ep * (y - x0) + kappa * torch.sqrt(ep) * torch.randn_like(x)
        else:
            x = x0
    return x


def denorm01(t): return (t.clamp(-1, 1) + 1) / 2


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    vae, unet, cfg = load_models()
    embed_cache = build_embed_cache()
    scale = cfg["latent_scale"]; kappa = cfg["kappa"]
    eta_full = make_eta(cfg["T"], cfg["eta_1"], cfg["eta_T"]).to(device)

    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)

    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    _, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in all_pairs], random_state=SEED)
    eval_pairs = val_pairs if N_EVAL is None else random.Random(SEED).sample(val_pairs, min(N_EVAL, len(val_pairs)))
    print(f"Evaluating on {len(eval_pairs):,} val patches.")

    def enc(imgs):
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            return vae.encode_latent(imgs.float()) * scale

    @torch.no_grad()
    def run_eval(res, n_steps, want_fid=False, want_breakdown=False):
        ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
        fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device) if want_fid else None
        dl = DataLoader(PairDataset(eval_pairs, res), batch_size=BATCH, num_workers=4)
        agg = defaultdict(list)
        by_season = defaultdict(lambda: defaultdict(list))
        by_terrain = defaultdict(lambda: defaultdict(list))
        for b in dl:
            opt = b["opt"].to(device); sar = b["sar"].to(device)
            y = enc(sar)
            cond = get_cond(b["season"], b["terrain"], embed_cache).to(device)
            x0 = sample(unet, y, cond, eta_full, kappa, n_steps)
            with torch.autocast("cuda", enabled=False):
                rec = vae.decode_latent((x0 / scale).float()).clamp(-1, 1)
            for i in range(rec.shape[0]):
                p = 10 * np.log10(4.0 / F.mse_loss(rec[i:i+1], opt[i:i+1]).item())
                s = ssim_fn(rec[i:i+1], opt[i:i+1]).item()
                l = lpips_fn(rec[i:i+1], opt[i:i+1]).item()
                agg["psnr"].append(p); agg["ssim"].append(s); agg["lpips"].append(l)
                if want_breakdown:
                    se, te = b["season"][i], b["terrain"][i]
                    for k, v in (("psnr", p), ("ssim", s), ("lpips", l)):
                        by_season[se][k].append(v); by_terrain[te][k].append(v)
            if want_fid:
                fid.update(denorm01(opt), real=True)
                fid.update(denorm01(rec), real=False)
        out = {k: float(np.mean(v)) for k, v in agg.items()}
        if want_fid: out["fid"] = float(fid.compute().item())
        if want_breakdown:
            out["by_season"] = {s: {k: float(np.mean(v)) for k, v in d.items()} for s, d in by_season.items()}
            out["by_terrain"] = {t: {k: float(np.mean(v)) for k, v in d.items()} for t, d in by_terrain.items()}
        return out

    results = {}

    # 1. Headline @256px, full T steps, with FID + breakdown
    print("\n=== 256px (headline) ===")
    r256 = run_eval(256, cfg["T"], want_fid=True, want_breakdown=True)
    results["256px"] = r256
    print(f"  PSNR {r256['psnr']:.3f} | SSIM {r256['ssim']:.4f} | LPIPS {r256['lpips']:.4f} | FID {r256['fid']:.3f}")
    print("  Per-season:")
    for s, d in r256["by_season"].items():
        print(f"    {s:8s} PSNR {d['psnr']:.2f} | SSIM {d['ssim']:.3f} | LPIPS {d['lpips']:.3f}")
    print("  Per-terrain:")
    for t, d in r256["by_terrain"].items():
        print(f"    {t:10s} PSNR {d['psnr']:.2f} | SSIM {d['ssim']:.3f} | LPIPS {d['lpips']:.3f}")

    # 2. 512px (secondary, comparability)
    print("\n=== 512px (secondary) ===")
    r512 = run_eval(512, cfg["T"], want_fid=True)
    results["512px"] = r512
    print(f"  PSNR {r512['psnr']:.3f} | SSIM {r512['ssim']:.4f} | LPIPS {r512['lpips']:.4f} | FID {r512['fid']:.3f}")

    # 3. Steps-ablation @256px (refinement vs single-step collapse)
    print("\n=== steps-ablation @256px ===")
    abl = {}
    for ns in STEPS_ABLATION:
        r = run_eval(256, ns, want_fid=False)
        abl[ns] = r
        print(f"  {ns:2d} steps: PSNR {r['psnr']:.3f} | SSIM {r['ssim']:.4f} | LPIPS {r['lpips']:.4f}")
    results["steps_ablation"] = abl
    spread = max(abl[ns]["psnr"] for ns in abl) - min(abl[ns]["psnr"] for ns in abl)
    print(f"  PSNR spread across steps: {spread:.3f} dB "
          f"({'refinement helps' if spread > 0.3 else 'collapsed to ~single-step'})")

    # 4. Visual grids per season @256px
    print("\n=== visual grids ===")
    grid_pairs = []
    for s in SEASONS:
        pool = [p for p in eval_pairs if p["season"] == s]
        grid_pairs += random.Random(1).sample(pool, min(N_GRID_PER_SEASON, len(pool)))
    n = len(grid_pairs)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1: axes = axes.reshape(1, -1)
    with torch.no_grad():
        for i, p in enumerate(grid_pairs):
            ds = PairDataset([p], 256)[0]
            sar = ds["sar"].unsqueeze(0).to(device); opt = ds["opt"].unsqueeze(0).to(device)
            y = enc(sar); cond = get_cond([p["season"]], [p["terrain"]], embed_cache).to(device)
            x0 = sample(unet, y, cond, eta_full, kappa, cfg["T"])
            with torch.autocast("cuda", enabled=False):
                rec = vae.decode_latent((x0 / scale).float()).clamp(-1, 1)
            ps = 10 * np.log10(4.0 / F.mse_loss(rec, opt).item())
            for j, (img, ttl) in enumerate([(sar[0], f"SAR ({p['season']})"),
                                            (rec[0], f"Predicted {ps:.1f}dB"),
                                            (opt[0], "Ground truth")]):
                axes[i, j].imshow(denorm01(img).permute(1, 2, 0).cpu().numpy())
                axes[i, j].set_title(ttl, fontsize=8); axes[i, j].axis("off")
    plt.tight_layout()
    gpath = os.path.join(OUTPUT_DIR, "eval_grid.png")
    plt.savefig(gpath, dpi=150, bbox_inches="tight"); display(plt.gcf()); plt.close()

    # save + push results
    with open(os.path.join(OUTPUT_DIR, "results.json"), "w") as f:
        json.dump(results, f, indent=2)
    print("\n=== SUMMARY (paper headline) ===")
    print(f"  256px: PSNR {r256['psnr']:.2f} | SSIM {r256['ssim']:.3f} | "
          f"LPIPS {r256['lpips']:.3f} | FID {r256['fid']:.2f}")
    if HF_TOKEN:
        try:
            HfApi().upload_folder(folder_path=OUTPUT_DIR, path_in_repo="eval_results",
                                  repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)
            print("  results -> HF/eval_results")
        except Exception as e:
            print(f"  push failed: {e}")


if __name__ == "__main__":
    main()

Writing /content/stage4_evaluate.py


In [ ]:
import os
from google.colab import userdata; os.environ["HF_WRITE_TOKEN"] = userdata.get("HF_TOKEN")
!pip install -q lpips torchmetrics safetensors scipy
!python /content/stage4_evaluate.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
bridge_config.json: 100% 144/144 [00:00<00:00, 577kB/s]
bridge_final/unet.safetensors: 100% 3.44G/3.44G [00:35<00:00, 96.1MB/s]
  Loaded bridge_final (step 150000), latent_scale=1.4010
Lo

In [ ]:
%%writefile /content/stage3b_flow_256.py
"""
SAR-INTEL Stage 3b — FLOW-MATCHING bridge on the adapted 16ch latent @ 256px.
Modern rectified-flow formulation to break the ResShift mean-collapse plateau.

Key upgrades vs ResShift:
  - VELOCITY prediction (rectified flow): x_t=(1-t)x0+t*x1, target v=x1-x0.
  - DECODED PIXEL LOSS (anti-collapse): decode predicted x1, L1+LPIPS vs real
    optical image. This is what fights the blur/mean-collapse that capped PSNR.
  - Euler-ODE sampling from SAR latent -> optical latent.

Saving: NO periodic HF push. LOCAL safety checkpoint overwrites a single folder.
Final model saved ONCE to HF in safetensors. Resumable from HF 'flow_resume'.
"""
import os, gc, re, json, random, warnings, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from safetensors.torch import save_file
import lpips as lpips_lib
import pandas as pd
from torchmetrics.image import StructuralSimilarityIndexMeasure
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT    = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR   = "/content/stage3b_flow"
SD_MODEL_ID  = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID   = os.environ["HF_REPO_ID"]
HF_TOKEN     = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG      = "adapted16_final"
RESUME_TAG   = "flow_resume"

IMG_SIZE     = 256
SEASONS      = ("spring", "summer", "fall", "winter")
VAL_FRACTION = 0.05
SEED         = 42
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16

NUM_TRAIN_STEPS = 120_000
BATCH_SIZE   = 4          # smaller: decoded-pixel loss holds a grad graph through the decoder
LR           = 6e-5
LR_WARMUP    = 1_000
COND_DROPOUT = 0.10
PIXEL_START  = 2_000      # warm up velocity loss before adding pixel loss
LAMBDA_LPIPS = 1.0        # decoded LPIPS (anti-collapse) — the key term
LAMBDA_L1    = 0.5        # decoded L1
SAMPLE_STEPS = 10         # Euler steps for validation
VAL_EVERY    = 5_000
LOCAL_SAVE_EVERY = 30_000
N_VAL        = 100
LOG_EVERY    = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEASON_DESC = {"spring": "spring", "summer": "summer", "fall": "autumn", "winter": "winter"}
TERRAINS = ("tropical", "temperate", "arctic", "arid", "coastal", "urban")

def map_region_to_terrain(region):
    r = (region or "").lower()
    if any(k in r for k in ("trop", "rainforest", "amazon", "jungle")): return "tropical"
    if any(k in r for k in ("arctic", "tundra", "snow", "ice", "polar")): return "arctic"
    if any(k in r for k in ("desert", "arid", "sahara", "dry")): return "arid"
    if any(k in r for k in ("coast", "beach", "shore", "marine", "sea")): return "coastal"
    if any(k in r for k in ("urban", "city", "metro")): return "urban"
    return "temperate"


def collect_pairs(root, seasons):
    root_str = Path(root).as_posix(); pairs = []
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for col in ("s1_fileName", "s2_fileName"):
            df[col] = df[col].astype(str).str.replace("\\", "/", regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            pairs.append({"sar": f"{root_str}/{row['s1_fileName']}",
                          "opt": f"{root_str}/{row['s2_fileName']}",
                          "season": season, "terrain": map_region_to_terrain(row["region"])})
    random.shuffle(pairs); return pairs


class PairDataset(Dataset):
    def __init__(self, pairs, img_size=256, augment=True):
        self.pairs = pairs; self.img_size = img_size; self.augment = augment
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0
    def __getitem__(self, idx):
        p = self.pairs[idx]
        sar = self._load(p["sar"]); opt = self._load(p["opt"])
        if self.augment:
            if random.random() > 0.5: sar, opt = torch.flip(sar, [2]), torch.flip(opt, [2])
            if random.random() > 0.5: sar, opt = torch.flip(sar, [1]), torch.flip(opt, [1])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}


class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CH, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),  nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf*2, 3, 2, 1),   nn.GroupNorm(8, nf*2), nn.SiLU(),
            nn.Conv2d(nf*2, nf*4, 3, 2, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, nf*4, 3, 1, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, out_ch, 3, 1, 1))
    def forward(self, x): return self.net(x)


class AdaptedVAE(nn.Module):
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae = base_vae; self.detail_enc = DetailEncoder()
        old = base_vae.decoder.conv_in
        base_vae.decoder.conv_in = nn.Conv2d(TOTAL_CH, old.out_channels, old.kernel_size,
                                             old.stride, old.padding, bias=(old.bias is not None))
    @torch.no_grad()
    def encode_latent(self, x):
        z = self.base_vae.encode(x).latent_dist.mode()
        return torch.cat([self.base_vae.post_quant_conv(z), self.detail_enc(x)], dim=1)
    def decode_grad(self, lat):                 # grad-enabled (params frozen, grad flows to input)
        return self.base_vae.decoder(lat)
    @torch.no_grad()
    def decode_latent(self, lat):
        return self.base_vae.decoder(lat)


def load_adapted_vae():
    base_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(base_vae)
    bp = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/base_vae.pth")
    de = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/detail_encoder.pth")
    vae.base_vae.load_state_dict(torch.load(bp, map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(de, map_location="cpu"))
    vae = vae.to(device).eval().float(); vae.requires_grad_(False)
    print(f"  Adapted VAE loaded from {VAE_TAG}.")
    return vae


def build_embed_cache():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    txt.requires_grad_(False); cache = {}
    @torch.no_grad()
    def embed(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s, ter)] = embed(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = embed(""); del txt; gc.collect(); torch.cuda.empty_cache()
    print(f"  Built {len(cache)} CLIP embeddings.")
    return cache


def build_unet():
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    ni = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    with torch.no_grad():
        ni.weight.zero_()
        for i in range(2*TOTAL_CH): ni.weight[:, i] = oi.weight[:, i % oi.in_channels] / 8.0
        ni.bias.copy_(oi.bias)
    unet.conv_in = ni; unet.config.in_channels = 2*TOTAL_CH
    oo = unet.conv_out
    no = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    with torch.no_grad():
        for i in range(TOTAL_CH):
            no.weight[i] = oo.weight[i % oo.out_channels]; no.bias[i] = oo.bias[i % oo.out_channels]
    unet.conv_out = no; unet.config.out_channels = TOTAL_CH
    print(f"  UNet: conv_in {2*TOTAL_CH}ch, conv_out {TOTAL_CH}ch (flow-matching).")
    return unet


def get_cond(seasons, terrains, cache, drop=0.0):
    out = []
    for s, ter in zip(seasons, terrains):
        out.append(cache["null"] if (drop > 0 and random.random() < drop)
                   else cache.get((s, ter), cache["null"]))
    return torch.stack(out)


@torch.no_grad()
def flow_sample(unet, x0, cond, n_steps, dtype):
    """Euler ODE from SAR latent x0 (t=0) to optical latent (t=1)."""
    x = x0.clone(); dt = 1.0 / n_steps; B = x0.shape[0]
    for i in range(n_steps):
        t = i * dt
        tb = torch.full((B,), int(t * 999), device=x0.device, dtype=torch.long)
        v = unet(torch.cat([x, x0], 1).to(dtype), tb, encoder_hidden_states=cond.to(dtype)).sample.float()
        x = x + dt * v
    return x


def push_hf(folder, tag):
    if not HF_TOKEN: return
    try:
        HfApi().upload_folder(folder_path=folder, path_in_repo=tag, repo_id=HF_REPO_ID,
                              repo_type="model", token=HF_TOKEN)
        print(f"  HF push -> {tag}")
    except Exception as e:
        print(f"  HF push failed: {e}")


def download_resume_from_hf(dest):
    try:
        snapshot_download(repo_id=HF_REPO_ID, allow_patterns=f"{RESUME_TAG}/*",
                          local_dir="/content/_flow_resume_dl", token=HF_TOKEN)
        src = os.path.join("/content/_flow_resume_dl", RESUME_TAG)
        if os.path.exists(os.path.join(src, "meta.pt")):
            if os.path.exists(dest): shutil.rmtree(dest)
            shutil.copytree(src, dest); return True
    except Exception as e:
        print(f"  no HF resume state ({e})")
    return False


def main():
    accelerator = Accelerator(mixed_precision="bf16", project_dir=OUTPUT_DIR)
    set_seed(SEED); is_main = accelerator.is_main_process; dtype = torch.bfloat16
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Loading components...")
    vae = load_adapted_vae(); embed_cache = build_embed_cache(); unet = build_unet()
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    train_pairs, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in all_pairs], random_state=SEED)
    print(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,}")

    train_ds = PairDataset(train_pairs, IMG_SIZE, True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                          pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=2)

    optimizer = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9, 0.999), weight_decay=1e-2)
    lr_sched = get_cosine_schedule_with_warmup(optimizer, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, optimizer, train_dl, lr_sched = accelerator.prepare(unet, optimizer, train_dl, lr_sched)

    latest = os.path.join(OUTPUT_DIR, "latest")
    start_step = 0; latent_scale = None; resume_dir = None
    if is_main and download_resume_from_hf(latest):
        resume_dir = latest; print("  Resume source: HF (flow_resume).")
    accelerator.wait_for_everyone()
    if not resume_dir and os.path.exists(os.path.join(latest, "meta.pt")):
        resume_dir = latest
    if resume_dir and os.path.exists(os.path.join(resume_dir, "meta.pt")):
        meta = torch.load(os.path.join(resume_dir, "meta.pt"), map_location="cpu")
        start_step = meta["step"]; latent_scale = meta["latent_scale"]
        accelerator.load_state(resume_dir)
        print(f"  RESUMED @ step {start_step} (latent_scale={latent_scale:.4f}).")
    else:
        if is_main:
            stds = []
            for b in DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2):
                with torch.autocast("cuda", enabled=False):
                    stds.append(vae.encode_latent(b["opt"].to(device).float()).std().item())
                if len(stds) >= 10: break
            latent_scale = 1.0 / float(np.mean(stds))
            print(f"  Computed latent_scale = {latent_scale:.4f}")
        ls = torch.tensor(latent_scale if latent_scale else 1.0, device=device)
        latent_scale = float(accelerator.gather(ls).mean().item()) if accelerator.num_processes > 1 else float(ls)

    def enc(imgs):
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            return vae.encode_latent(imgs.float()) * latent_scale

    @torch.no_grad()
    def validate(step):
        sub = random.Random(7).sample(val_pairs, min(N_VAL, len(val_pairs)))
        vdl = DataLoader(PairDataset(sub, IMG_SIZE, False), batch_size=8, num_workers=2)
        P, S, L = [], [], []
        m = accelerator.unwrap_model(unet); m.eval()
        for b in vdl:
            opt = b["opt"].to(device)
            x0 = enc(b["sar"].to(device))
            cond = get_cond(b["season"], b["terrain"], embed_cache).to(device)
            x1 = flow_sample(m, x0, cond, SAMPLE_STEPS, dtype)
            with torch.autocast("cuda", enabled=False):
                rec = vae.decode_latent((x1 / latent_scale).float()).clamp(-1, 1)
            P.append(10 * np.log10(4.0 / F.mse_loss(rec, opt).item()))
            S.append(ssim_fn(rec, opt).item()); L.append(lpips_fn(rec, opt).mean().item())
        m.train()
        print(f"\n[VAL step {step}] PSNR {np.mean(P):.3f} | SSIM {np.mean(S):.4f} | LPIPS {np.mean(L):.4f}\n")

    def local_save(step):
        if LOCAL_SAVE_EVERY == 0: return
        accelerator.wait_for_everyone()
        if not is_main: return
        if os.path.exists(latest): shutil.rmtree(latest)        # overwrite, never grows
        os.makedirs(latest, exist_ok=True)
        accelerator.save_state(latest)
        torch.save({"step": step, "latent_scale": latent_scale}, os.path.join(latest, "meta.pt"))
        print(f"  local safety checkpoint (overwritten) @ step {step}")

    def final_save(step):
        accelerator.wait_for_everyone()
        if not is_main: return
        tmp = os.path.join(OUTPUT_DIR, "_final"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k, v in accelerator.unwrap_model(unet).state_dict().items()}
        save_file(sd, os.path.join(tmp, "unet.safetensors"))
        with open(os.path.join(tmp, "bridge_config.json"), "w") as f:
            json.dump({"latent_scale": latent_scale, "method": "flow_matching",
                       "sample_steps": SAMPLE_STEPS, "img_size": IMG_SIZE, "final_step": step}, f, indent=2)
        push_hf(tmp, "flow_final")
        shutil.rmtree(tmp, ignore_errors=True)
        print(f"  FINAL flow model -> HF/flow_final @ step {step}")

    global_step = start_step
    ema_v = ema_p = 0.0; a = 0.98
    spe = len(train_dl); epochs = (NUM_TRAIN_STEPS + spe - 1) // spe
    bar = tqdm(total=NUM_TRAIN_STEPS, initial=start_step, desc="Stage 3b flow",
               disable=not is_main, dynamic_ncols=True)
    unet.train()
    print(f"\n>>> Flow-matching from step {start_step}. Pixel loss kicks in at {PIXEL_START}.\n")

    for epoch in range(epochs):
        for batch in train_dl:
            opt_img = batch["opt"].to(device)
            x0 = enc(batch["sar"].to(device))        # SAR latent (source)
            x1 = enc(opt_img)                         # optical latent (target)
            cond = get_cond(batch["season"], batch["terrain"], embed_cache, COND_DROPOUT).to(device)
            t = torch.rand(x0.shape[0], device=device)            # continuous U(0,1)
            tv = t.view(-1, 1, 1, 1)
            x_t = (1 - tv) * x0 + tv * x1                          # rectified-flow path
            v_target = x1 - x0
            tb = (t * 999).long()

            with accelerator.accumulate(unet):
                v_pred = unet(torch.cat([x_t, x0], 1).to(dtype), tb,
                              encoder_hidden_states=cond.to(dtype)).sample
                loss_v = F.mse_loss(v_pred.float(), v_target.float())
                loss = loss_v
                loss_p_val = 0.0
                if global_step >= PIXEL_START:
                    x1_pred = x_t + (1 - tv) * v_pred.float()      # reconstruct target latent
                    with torch.autocast("cuda", enabled=False):
                        img_pred = vae.decode_grad((x1_pred / latent_scale).float()).clamp(-1, 1)
                    loss_p = LAMBDA_L1 * F.l1_loss(img_pred, opt_img) + \
                             LAMBDA_LPIPS * lpips_fn(img_pred, opt_img).mean()
                    loss = loss + loss_p
                    loss_p_val = float(loss_p.item())
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), 1.0)
                optimizer.step(); lr_sched.step(); optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                global_step += 1
                ema_v = a*ema_v + (1-a)*loss_v.item(); ema_p = a*ema_p + (1-a)*loss_p_val
                if is_main:
                    bar.update(1)
                    if global_step % LOG_EVERY == 0:
                        bar.set_postfix({"v": f"{ema_v:.4f}", "pix": f"{ema_p:.4f}"})
                if global_step % VAL_EVERY == 0 and is_main: validate(global_step)
                if LOCAL_SAVE_EVERY and global_step % LOCAL_SAVE_EVERY == 0: local_save(global_step)
            if global_step >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if global_step >= NUM_TRAIN_STEPS: break

    bar.close()
    if is_main: validate(global_step)
    final_save(global_step)
    accelerator.end_training()
    print("\nStage 3b complete. Flow model on HF/flow_final.")


if __name__ == "__main__":
    main()

Writing /content/stage3b_flow_256.py


In [ ]:
import os
from google.colab import userdata; os.environ["HF_WRITE_TOKEN"] = userdata.get("HF_TOKEN")
!pip install -q lpips torchmetrics safetensors scipy
!python /content/stage3b_flow_256.py

Streaming output truncated to the last 5000 lines.
Stage 3b flow:  43% 52030/120000 [4:53:55<6:16:25,  3.01it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52031/120000 [4:53:56<6:16:03,  3.01it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52032/120000 [4:53:56<6:17:06,  3.00it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52033/120000 [4:53:56<6:16:42,  3.01it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52034/120000 [4:53:56<6:15:58,  3.01it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52035/120000 [4:53:57<6:15:33,  3.02it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52036/120000 [4:53:57<6:15:38,  3.02it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52037/120000 [4:53:57<6:15:19,  3.02it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52038/120000 [4:53:58<6:15:44,  3.01it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52039/120000 [4:53:58<6:16:49,  3.01it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52040/120000 [4:53:58<6:17:02,  3.00it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52041

In [ ]:
%%writefile /content/stage3b_flow_256.py
"""
SAR-INTEL Stage 3b FINAL — flow-matching bridge, adapted 16ch latent @ 256px.
This version adds (vs the run you started):
  - EMA (0.9999): final checkpoint is the smoothed model. Standard, real gain.
  - LPL (latent perceptual loss, Meta 2024): decoder-feature matching. FID weapon.
  - Full augmentation (flips + 90-degree rotations).
  - FID (approx) in every validation, quiet logging (1 line / 500 steps).
Resumes from LOCAL /content/stage3b_flow/latest (your 60k state) automatically.
"""
import os, gc, json, time, random, warnings, shutil
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from safetensors.torch import save_file
import lpips as lpips_lib
import pandas as pd
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def log(*a): print(*a, flush=True)

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT    = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR   = "/content/stage3b_flow"        # SAME dir as your running job -> resumes 60k
SD_MODEL_ID  = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID   = os.environ["HF_REPO_ID"]
HF_TOKEN     = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG      = "adapted16_final"
RESUME_TAG   = "flow_resume"

IMG_SIZE     = 256
SEASONS      = ("spring", "summer", "fall", "winter")
VAL_FRACTION = 0.05
SEED         = 42
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16

NUM_TRAIN_STEPS = 120_000
BATCH_SIZE   = 4          # proven config from your live run; full-batch perceptual
LR           = 6e-5
LR_WARMUP    = 1_000
COND_DROPOUT = 0.10
PIXEL_START  = 2_000
LAMBDA_LPIPS = 1.0
LAMBDA_L1    = 0.5
LAMBDA_LPL   = 1.0        # latent perceptual loss (decoder features) — the FID lever
EMA_DECAY    = 0.9999
SAMPLE_STEPS = 10
VAL_EVERY    = 5_000
N_VAL        = 200
LOCAL_SAVE_EVERY = 40_000
STEP_LOG_EVERY   = 500

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEASON_DESC = {"spring": "spring", "summer": "summer", "fall": "autumn", "winter": "winter"}
TERRAINS = ("tropical", "temperate", "arctic", "arid", "coastal", "urban")

def map_region_to_terrain(region):
    r = (region or "").lower()
    if any(k in r for k in ("trop", "rainforest", "amazon", "jungle")): return "tropical"
    if any(k in r for k in ("arctic", "tundra", "snow", "ice", "polar")): return "arctic"
    if any(k in r for k in ("desert", "arid", "sahara", "dry")): return "arid"
    if any(k in r for k in ("coast", "beach", "shore", "marine", "sea")): return "coastal"
    if any(k in r for k in ("urban", "city", "metro")): return "urban"
    return "temperate"


def collect_pairs(root, seasons):
    root_str = Path(root).as_posix(); pairs = []
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for col in ("s1_fileName", "s2_fileName"):
            df[col] = df[col].astype(str).str.replace("\\", "/", regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            pairs.append({"sar": f"{root_str}/{row['s1_fileName']}",
                          "opt": f"{root_str}/{row['s2_fileName']}",
                          "season": season, "terrain": map_region_to_terrain(row["region"])})
    random.shuffle(pairs); return pairs


class PairDataset(Dataset):
    def __init__(self, pairs, img_size=256, augment=True):
        self.pairs = pairs; self.img_size = img_size; self.augment = augment
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0
    def __getitem__(self, idx):
        p = self.pairs[idx]
        sar = self._load(p["sar"]); opt = self._load(p["opt"])
        if self.augment:
            if random.random() > 0.5: sar, opt = torch.flip(sar, [2]), torch.flip(opt, [2])
            if random.random() > 0.5: sar, opt = torch.flip(sar, [1]), torch.flip(opt, [1])
            k = random.randint(0, 3)                     # satellite imagery: no canonical
            if k: sar, opt = torch.rot90(sar, k, [1, 2]), torch.rot90(opt, k, [1, 2])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}


class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CH, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),  nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf*2, 3, 2, 1),   nn.GroupNorm(8, nf*2), nn.SiLU(),
            nn.Conv2d(nf*2, nf*4, 3, 2, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, nf*4, 3, 1, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, out_ch, 3, 1, 1))
    def forward(self, x): return self.net(x)


class AdaptedVAE(nn.Module):
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae = base_vae; self.detail_enc = DetailEncoder()
        old = base_vae.decoder.conv_in
        base_vae.decoder.conv_in = nn.Conv2d(TOTAL_CH, old.out_channels, old.kernel_size,
                                             old.stride, old.padding, bias=(old.bias is not None))
    @torch.no_grad()
    def encode_latent(self, x):
        z = self.base_vae.encode(x).latent_dist.mode()
        return torch.cat([self.base_vae.post_quant_conv(z), self.detail_enc(x)], dim=1)
    def decode_grad(self, lat):
        return self.base_vae.decoder(lat)
    @torch.no_grad()
    def decode_latent(self, lat):
        return self.base_vae.decoder(lat)


class HookedDecoder:
    """Capture decoder up-block features for LPL (gated so normal decodes are free)."""
    def __init__(self, vae):
        self.feats = []; self.capture = False
        for blk in vae.base_vae.decoder.up_blocks:
            blk.register_forward_hook(
                lambda m, i, o: self.feats.append(o) if self.capture else None)
    def decode(self, vae, lat):
        self.capture = True; self.feats = []
        img = vae.base_vae.decoder(lat)
        feats = list(self.feats); self.capture = False; self.feats = []
        return img, feats

def latent_perceptual_loss(f_pred, f_real):
    loss = 0.0
    for fp, fr in zip(f_pred, f_real):
        loss = loss + F.l1_loss(fp.float(), fr.float().detach())
    return loss / max(len(f_pred), 1)


def load_adapted_vae():
    base_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(base_vae)
    bp = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/base_vae.pth")
    de = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/detail_encoder.pth")
    vae.base_vae.load_state_dict(torch.load(bp, map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(de, map_location="cpu"))
    vae = vae.to(device).eval().float(); vae.requires_grad_(False)
    log("  Adapted VAE loaded from", VAE_TAG)
    return vae


def build_embed_cache():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    txt.requires_grad_(False); cache = {}
    @torch.no_grad()
    def embed(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s, ter)] = embed(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = embed(""); del txt; gc.collect(); torch.cuda.empty_cache()
    log(f"  Built {len(cache)} CLIP embeddings.")
    return cache


def build_unet():
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    ni = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    with torch.no_grad():
        ni.weight.zero_()
        for i in range(2*TOTAL_CH): ni.weight[:, i] = oi.weight[:, i % oi.in_channels] / 8.0
        ni.bias.copy_(oi.bias)
    unet.conv_in = ni; unet.config.in_channels = 2*TOTAL_CH
    oo = unet.conv_out
    no = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    with torch.no_grad():
        for i in range(TOTAL_CH):
            no.weight[i] = oo.weight[i % oo.out_channels]; no.bias[i] = oo.bias[i % oo.out_channels]
    unet.conv_out = no; unet.config.out_channels = TOTAL_CH
    log(f"  UNet: conv_in {2*TOTAL_CH}ch, conv_out {TOTAL_CH}ch (flow-matching).")
    return unet


def get_cond(seasons, terrains, cache, drop=0.0):
    out = []
    for s, ter in zip(seasons, terrains):
        out.append(cache["null"] if (drop > 0 and random.random() < drop)
                   else cache.get((s, ter), cache["null"]))
    return torch.stack(out)


@torch.no_grad()
def flow_sample(unet, x0, cond, n_steps, dtype):
    x = x0.clone(); dt = 1.0 / n_steps; B = x0.shape[0]
    for i in range(n_steps):
        tb = torch.full((B,), int(i * dt * 999), device=x0.device, dtype=torch.long)
        v = unet(torch.cat([x, x0], 1).to(dtype), tb,
                 encoder_hidden_states=cond.to(dtype)).sample.float()
        x = x + dt * v
    return x


def push_hf(folder, tag):
    if not HF_TOKEN: return
    try:
        HfApi().upload_folder(folder_path=folder, path_in_repo=tag, repo_id=HF_REPO_ID,
                              repo_type="model", token=HF_TOKEN)
        log(f"  HF push -> {tag}")
    except Exception as e:
        log(f"  HF push failed: {e}")


def download_resume_from_hf(dest):
    try:
        files = HfApi().list_repo_files(HF_REPO_ID, token=HF_TOKEN)
        if f"{RESUME_TAG}/meta.pt" not in files:
            log("  no resume state on HF"); return False
        snapshot_download(repo_id=HF_REPO_ID, allow_patterns=f"{RESUME_TAG}/*",
                          local_dir="/content/_flow_resume_dl", token=HF_TOKEN)
        src = os.path.join("/content/_flow_resume_dl", RESUME_TAG)
        if os.path.exists(dest): shutil.rmtree(dest)
        shutil.copytree(src, dest); return True
    except Exception as e:
        log(f"  resume check failed ({e})"); return False


def denorm01(t): return (t.clamp(-1, 1) + 1) / 2


def main():
    accelerator = Accelerator(mixed_precision="bf16", project_dir=OUTPUT_DIR)
    set_seed(SEED); is_main = accelerator.is_main_process; dtype = torch.bfloat16
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    log("Loading components...")
    vae = load_adapted_vae(); embed_cache = build_embed_cache(); unet = build_unet()
    hooked = HookedDecoder(vae)
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    train_pairs, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in all_pairs], random_state=SEED)
    log(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,}")

    train_ds = PairDataset(train_pairs, IMG_SIZE, True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                          pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=2)

    optimizer = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9, 0.999), weight_decay=1e-2)
    lr_sched = get_cosine_schedule_with_warmup(optimizer, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, optimizer, train_dl, lr_sched = accelerator.prepare(unet, optimizer, train_dl, lr_sched)

    # ── resume: LOCAL first (your live 60k), then HF ──
    latest = os.path.join(OUTPUT_DIR, "latest")
    start_step = 0; latent_scale = None; resume_dir = None
    if os.path.exists(os.path.join(latest, "meta.pt")):
        resume_dir = latest; log("  Resume source: LOCAL latest.")
    elif is_main and download_resume_from_hf(latest):
        resume_dir = latest; log("  Resume source: HF (flow_resume).")
    accelerator.wait_for_everyone()
    if resume_dir and os.path.exists(os.path.join(resume_dir, "meta.pt")):
        meta = torch.load(os.path.join(resume_dir, "meta.pt"), map_location="cpu")
        start_step = meta["step"]; latent_scale = meta["latent_scale"]
        accelerator.load_state(resume_dir)
        log(f"  RESUMED @ step {start_step} (latent_scale={latent_scale:.4f}).")
    else:
        if is_main:
            stds = []
            for b in DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2):
                with torch.autocast("cuda", enabled=False):
                    stds.append(vae.encode_latent(b["opt"].to(device).float()).std().item())
                if len(stds) >= 100: break
            latent_scale = 1.0 / float(np.mean(stds))
            log(f"  Computed latent_scale = {latent_scale:.4f}")
        ls = torch.tensor(latent_scale if latent_scale else 1.0, device=device)
        latent_scale = float(accelerator.gather(ls).mean().item()) if accelerator.num_processes > 1 else float(ls)

    # EMA of the trained model (initialized from current — i.e. resumed 60k — weights)
    ema_unet = AveragedModel(accelerator.unwrap_model(unet),
                             multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY)).to(device)

    def enc(imgs):
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            return vae.encode_latent(imgs.float()) * latent_scale

    @torch.no_grad()
    def validate(step):
        sub = random.Random(7).sample(val_pairs, min(N_VAL, len(val_pairs)))
        vdl = DataLoader(PairDataset(sub, IMG_SIZE, False), batch_size=8, num_workers=2)
        P, S, L = [], [], []
        fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
        m = ema_unet.module; m.eval()
        for b in vdl:
            opt = b["opt"].to(device)
            x0 = enc(b["sar"].to(device))
            cond = get_cond(b["season"], b["terrain"], embed_cache).to(device)
            x1 = flow_sample(m, x0, cond, SAMPLE_STEPS, dtype)
            with torch.autocast("cuda", enabled=False):
                rec = vae.decode_latent((x1 / latent_scale).float()).clamp(-1, 1)
            P.append(10 * np.log10(4.0 / F.mse_loss(rec, opt).item()))
            S.append(ssim_fn(rec, opt).item()); L.append(lpips_fn(rec, opt).mean().item())
            fid.update(denorm01(opt), real=True); fid.update(denorm01(rec), real=False)
        log(f"\n[VAL step {step} | EMA] PSNR {np.mean(P):.3f} | SSIM {np.mean(S):.4f} | "
            f"LPIPS {np.mean(L):.4f} | FID~ {float(fid.compute()):.2f} (n={len(sub)})\n")

    def local_save(step):
        accelerator.wait_for_everyone()
        if not is_main: return
        if os.path.exists(latest): shutil.rmtree(latest)
        os.makedirs(latest, exist_ok=True)
        accelerator.save_state(latest)
        torch.save({"step": step, "latent_scale": latent_scale}, os.path.join(latest, "meta.pt"))
        log(f"  local safety checkpoint (overwritten) @ step {step}")

    def final_save(step):
        accelerator.wait_for_everyone()
        if not is_main: return
        tmp = os.path.join(OUTPUT_DIR, "_final"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k, v in ema_unet.module.state_dict().items()}
        save_file(sd, os.path.join(tmp, "unet.safetensors"))
        with open(os.path.join(tmp, "bridge_config.json"), "w") as f:
            json.dump({"latent_scale": latent_scale, "method": "flow_matching", "ema": True,
                       "sample_steps": SAMPLE_STEPS, "img_size": IMG_SIZE, "final_step": step},
                      f, indent=2)
        push_hf(tmp, "flow_final")
        shutil.rmtree(tmp, ignore_errors=True)
        log(f"  FINAL EMA model -> HF/flow_final @ step {step}")

    global_step = start_step
    ema_v = ema_p = 0.0; a = 0.98
    t0 = time.time(); last_logged = start_step
    spe = len(train_dl); epochs = (NUM_TRAIN_STEPS + spe - 1) // spe
    unet.train()
    log(f"\n>>> Flow-matching {start_step} -> {NUM_TRAIN_STEPS}. EMA on. "
        f"LPL+pixel loss active from step {PIXEL_START}. One log line per {STEP_LOG_EVERY} steps.\n")

    for epoch in range(epochs):
        for batch in train_dl:
            opt_img = batch["opt"].to(device)
            x0 = enc(batch["sar"].to(device))
            x1 = enc(opt_img)
            cond = get_cond(batch["season"], batch["terrain"], embed_cache, COND_DROPOUT).to(device)
            t = torch.rand(x0.shape[0], device=device)
            tv = t.view(-1, 1, 1, 1)
            x_t = (1 - tv) * x0 + tv * x1
            v_target = x1 - x0
            tb = (t * 999).long()

            with accelerator.accumulate(unet):
                v_pred = unet(torch.cat([x_t, x0], 1).to(dtype), tb,
                              encoder_hidden_states=cond.to(dtype)).sample
                loss_v = F.mse_loss(v_pred.float(), v_target.float())
                loss = loss_v; loss_p_val = 0.0
                if global_step >= PIXEL_START:
                    x1p = (x_t + (1 - tv) * v_pred.float()) / latent_scale
                    with torch.autocast("cuda", enabled=False):
                        img_pred, f_pred = hooked.decode(vae, x1p.float())
                        img_pred = img_pred.clamp(-1, 1)
                        with torch.no_grad():
                            _, f_real = hooked.decode(vae, (x1 / latent_scale).float())
                        l_l1  = F.l1_loss(img_pred, opt_img)
                        l_lp  = lpips_fn(img_pred, opt_img).mean()
                        l_lpl = latent_perceptual_loss(f_pred, f_real)
                    loss_p = LAMBDA_L1*l_l1 + LAMBDA_LPIPS*l_lp + LAMBDA_LPL*l_lpl
                    loss = loss + loss_p; loss_p_val = float(loss_p.item())
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), 1.0)
                optimizer.step(); lr_sched.step(); optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                ema_unet.update_parameters(accelerator.unwrap_model(unet))
                global_step += 1
                ema_v = a*ema_v + (1-a)*loss_v.item(); ema_p = a*ema_p + (1-a)*loss_p_val
                if is_main and global_step % STEP_LOG_EVERY == 0:
                    rate = (global_step - last_logged) / max(time.time() - t0, 1e-9)
                    t0, last_logged = time.time(), global_step
                    log(f"step {global_step}/{NUM_TRAIN_STEPS} | v {ema_v:.4f} | "
                        f"pix+lpl {ema_p:.4f} | {rate:.2f} it/s")
                if global_step % VAL_EVERY == 0 and is_main: validate(global_step)
                if global_step % LOCAL_SAVE_EVERY == 0: local_save(global_step)
            if global_step >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if global_step >= NUM_TRAIN_STEPS: break

    if is_main: validate(global_step)
    final_save(global_step)
    accelerator.end_training()
    log("\nStage 3b complete. EMA model on HF/flow_final.")


if __name__ == "__main__":
    main()

Writing /content/stage3b_flow_256.py


In [ ]:
pip install torch-fidelity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 3.6 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata; os.environ["HF_WRITE_TOKEN"] = userdata.get("HF_TOKEN")
!pip install -q lpips torchmetrics safetensors scipy
!python -u /content/stage3b_flow_256.py

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [ ]:
%%writefile /content/stage4_final_eval.py
"""
SAR-INTEL Stage 4 FINAL — publication evaluation.
  Phase A (n=SWEEP_N): CFG sweep -> steps sweep -> TTA check   (finds best config)
  Phase B (full val): final numbers at best config, per-season/terrain, full-set FID
  Phase C (optional): ResShift (bridge_final) under the identical protocol -> ablation
Metrics: PSNR, SSIM, LPIPS, FID + SAM (deg), ERGAS, CC.
"""
import os, gc, json, math, random, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import load_file
import lpips as lpips_lib
import pandas as pd
import matplotlib.pyplot as plt
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
from sklearn.model_selection import train_test_split
from collections import defaultdict

warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True

def log(*a): print(*a, flush=True)

DATA_ROOT   = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUT         = "/content/stage4_final"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID  = os.environ["HF_REPO_ID"]
HF_TOKEN    = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG     = "adapted16_final"
FLOW_TAG    = "bridge_final"
RESSHIFT_TAG= "bridge_final"
EVAL_RESSHIFT = False          # Phase C ablation row

IMG_SIZE = 256; SEED = 42
SEASONS  = ("spring", "summer", "fall", "winter")
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16
VAL_FRACTION = 0.05
SWEEP_N  = 400                # Phase A subset
FULL_N   = None               # None = entire val set (15,394) for Phase B
TTA_N    = 1000               # subset for the +TTA row (x8 cost)
CFG_GRID   = (1.0, 1.5, 2.0, 3.0)
STEPS_GRID = (15,)
BATCH = 4

device = torch.device("cuda")
SEASON_DESC = {"spring": "spring", "summer": "summer", "fall": "autumn", "winter": "winter"}

def map_region_to_terrain(region):
    r = (region or "").lower()
    if any(k in r for k in ("trop", "rainforest", "amazon", "jungle")): return "tropical"
    if any(k in r for k in ("arctic", "tundra", "snow", "ice", "polar")): return "arctic"
    if any(k in r for k in ("desert", "arid", "sahara", "dry")): return "arid"
    if any(k in r for k in ("coast", "beach", "shore", "marine", "sea")): return "coastal"
    if any(k in r for k in ("urban", "city", "metro")): return "urban"
    return "temperate"

def collect_pairs(root):
    root_str = Path(root).as_posix(); pairs = []
    for season in SEASONS:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for col in ("s1_fileName", "s2_fileName"):
            df[col] = df[col].astype(str).str.replace("\\", "/", regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            pairs.append({"sar": f"{root_str}/{row['s1_fileName']}",
                          "opt": f"{root_str}/{row['s2_fileName']}",
                          "season": season, "terrain": map_region_to_terrain(row["region"])})
    return pairs

class PairDataset(Dataset):
    def __init__(self, pairs, img_size):
        self.pairs = pairs; self.img_size = img_size
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        return torch.from_numpy(np.array(img, dtype=np.float32) / 255.0).permute(2, 0, 1) * 2 - 1
    def __getitem__(self, i):
        p = self.pairs[i]
        return {"sar": self._load(p["sar"]), "opt": self._load(p["opt"]),
                "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CH, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),  nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf*2, 3, 2, 1),   nn.GroupNorm(8, nf*2), nn.SiLU(),
            nn.Conv2d(nf*2, nf*4, 3, 2, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, nf*4, 3, 1, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, out_ch, 3, 1, 1))
    def forward(self, x): return self.net(x)

class AdaptedVAE(nn.Module):
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae = base_vae; self.detail_enc = DetailEncoder()
        old = base_vae.decoder.conv_in
        base_vae.decoder.conv_in = nn.Conv2d(TOTAL_CH, old.out_channels, old.kernel_size,
                                             old.stride, old.padding, bias=(old.bias is not None))
    @torch.no_grad()
    def encode_latent(self, x):
        z = self.base_vae.encode(x).latent_dist.mode()
        return torch.cat([self.base_vae.post_quant_conv(z), self.detail_enc(x)], dim=1)
    @torch.no_grad()
    def decode_latent(self, lat):
        return self.base_vae.decoder(lat)

def load_vae():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(bv)
    bp = hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth")
    de = hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth")
    vae.base_vae.load_state_dict(torch.load(bp, map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(de, map_location="cpu"))
    return vae.to(device).eval().float().requires_grad_(False)

def load_model(tag):
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{tag}/bridge_config.json")))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels = 2*TOTAL_CH; unet.config.out_channels = TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{tag}/unet.safetensors")))
    unet = unet.float()                       # <-- force fp32, matches inputs below
    log(f"  loaded {tag} (step {cfg.get('final_step')}, method {cfg.get('method','resshift')})")
    return unet.to(device).eval().requires_grad_(False), cfg

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in ("tropical","temperate","arctic","arid","coastal","urban"):
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb("")
    del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

# ── samplers with CFG ─────────────────────────────────────────────────────────
@torch.no_grad()
def unet_v(unet, x, x0, tb, cond):
    return unet(torch.cat([x, x0], 1).float(), tb,
                encoder_hidden_states=cond.float()).sample.float()

@torch.no_grad()
def flow_cfg(unet, x0, cond, cond_null, steps, g):
    x = x0.clone(); dt = 1.0/steps; B = x0.shape[0]
    for i in range(steps):
        tb = torch.full((B,), int(i*dt*999), device=device, dtype=torch.long)
        if g == 1.0:
            v = unet_v(unet, x, x0, tb, cond)
        else:
            v_c = unet_v(unet, x, x0, tb, cond); v_n = unet_v(unet, x, x0, tb, cond_null)
            v = v_n + g*(v_c - v_n)
        x = x + dt*v
    return x

def make_eta(T, e1, eT):
    idx = np.arange(T)
    return torch.tensor(e1*(eT/e1)**(idx/(T-1)), dtype=torch.float32).to(device)

@torch.no_grad()
def resshift_cfg(unet, y, cond, cond_null, eta, kappa, g):
    B = y.shape[0]; T = len(eta)
    x = y + kappa*torch.sqrt(eta[-1])*torch.randn_like(y)
    for t in reversed(range(T)):
        tb = torch.full((B,), int(t*1000/T), device=device, dtype=torch.long)
        if g == 1.0:
            x0 = unet_v(unet, x, y, tb, cond)
        else:
            xc = unet_v(unet, x, y, tb, cond); xn = unet_v(unet, x, y, tb, cond_null)
            x0 = xn + g*(xc - xn)
        if t > 0:
            ep = eta[t-1]
            x = x0 + ep*(y - x0) + kappa*torch.sqrt(ep)*torch.randn_like(x)
        else:
            x = x0
    return x

# ── TTA: 8 dihedral transforms of the SAR input, averaged output ──────────────
def d8(img, k, flip):
    out = torch.rot90(img, k, [-2, -1])
    return torch.flip(out, [-1]) if flip else out
def d8_inv(img, k, flip):
    out = torch.flip(img, [-1]) if flip else img
    return torch.rot90(out, -k, [-2, -1])

# ── remote-sensing metrics (on [0,1] images) ──────────────────────────────────
def sam_deg(p, r):
    p = p.flatten(2).transpose(1, 2); r = r.flatten(2).transpose(1, 2)   # B,N,3
    num = (p*r).sum(-1); den = p.norm(dim=-1)*r.norm(dim=-1) + 1e-8
    return torch.rad2deg(torch.acos((num/den).clamp(-1, 1))).mean().item()
def ergas(p, r):
    e = 0.0
    for b in range(3):
        rmse = torch.sqrt(F.mse_loss(p[:, b], r[:, b]))
        e += (rmse/(r[:, b].mean() + 1e-8))**2
    return float(100*torch.sqrt(e/3))
def cc(p, r):
    tot = 0.0
    for b in range(3):
        x = p[:, b].flatten() - p[:, b].mean(); y = r[:, b].flatten() - r[:, b].mean()
        tot += float((x*y).sum()/(x.norm()*y.norm() + 1e-8))
    return tot/3

def dn(t): return (t.clamp(-1, 1) + 1)/2

def main():
    os.makedirs(OUT, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    log(f"HF token set: {bool(HF_TOKEN)}")

    files = HfApi().list_repo_files(HF_REPO_ID)
    if f"{FLOW_TAG}/unet.safetensors" not in files:
        log(f"!! {FLOW_TAG} not on HF yet — run this AFTER Stage 3b finishes."); return

    vae = load_vae(); embeds = build_embeds()
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    all_pairs = collect_pairs(DATA_ROOT)
    _, val_pairs = train_test_split(all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in all_pairs], random_state=SEED)
    log(f"Val set: {len(val_pairs):,}")

    def cond_of(b):
        return (torch.stack([embeds.get((s, t), embeds["null"])
                             for s, t in zip(b["season"], b["terrain"])]).to(device),
                torch.stack([embeds["null"]]*len(b["season"])).to(device))

    def run(unet, cfgd, pairs, steps, g, tta=False, fid_on=True, breakdown=False):
        scale = cfgd["latent_scale"]; method = cfgd.get("method", "resshift")
        eta = make_eta(cfgd["T"], cfgd["eta_1"], cfgd["eta_T"]) if method == "resshift" else None
        dl = DataLoader(PairDataset(pairs, IMG_SIZE), batch_size=BATCH, num_workers=4)
        agg = defaultdict(list); bysn = defaultdict(lambda: defaultdict(list))
        bytr = defaultdict(lambda: defaultdict(list))
        fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device) if fid_on else None
        for b in dl:
            opt = b["opt"].to(device); sar = b["sar"].to(device)
            c, cn = cond_of(b)
            variants = [(0, False)] if not tta else [(k, f) for k in range(4) for f in (False, True)]
            recs = []
            for k, f in variants:
                s_t = d8(sar, k, f)
                x0 = vae.encode_latent(s_t)*scale
                lat = (flow_cfg(unet, x0, c, cn, steps, g) if method == "flow_matching"
                       else resshift_cfg(unet, x0, c, cn, eta, cfgd["kappa"], g))
                r = vae.decode_latent((lat/scale).float()).clamp(-1, 1)
                recs.append(d8_inv(r, k, f))
            rec = torch.stack(recs).mean(0).clamp(-1, 1)
            for i in range(rec.shape[0]):
                pi, ri = rec[i:i+1], opt[i:i+1]
                m = {"psnr": 10*np.log10(4.0/F.mse_loss(pi, ri).item()),
                     "ssim": ssim_fn(pi, ri).item(), "lpips": lpips_fn(pi, ri).item(),
                     "sam": sam_deg(dn(pi), dn(ri)), "ergas": ergas(dn(pi), dn(ri)),
                     "cc": cc(dn(pi), dn(ri))}
                for kk, vv in m.items():
                    agg[kk].append(vv)
                    if breakdown:
                        bysn[b["season"][i]][kk].append(vv); bytr[b["terrain"][i]][kk].append(vv)
            if fid_on:
                fid.update(dn(opt), real=True); fid.update(dn(rec), real=False)
        out = {k: float(np.mean(v)) for k, v in agg.items()}
        if fid_on: out["fid"] = float(fid.compute())
        if breakdown:
            out["by_season"] = {s: {k: float(np.mean(v)) for k, v in d.items()} for s, d in bysn.items()}
            out["by_terrain"] = {t: {k: float(np.mean(v)) for k, v in d.items()} for t, d in bytr.items()}
        return out

    results = {}
    sweep = random.Random(SEED).sample(val_pairs, SWEEP_N)
    flow, fcfg = load_model(FLOW_TAG)

    log("\n== Phase A: CFG sweep (flow, 10 steps, n=%d) ==" % SWEEP_N)
    best_g, best = 1.0, None
    for g in CFG_GRID:
        r = run(flow, fcfg, sweep, 10, g)
        log(f"  g={g}: PSNR {r['psnr']:.3f} | SSIM {r['ssim']:.4f} | LPIPS {r['lpips']:.4f} | FID {r['fid']:.1f}")
        results[f"cfg_{g}"] = r
        if best is None or r["psnr"] > best: best, best_g = r["psnr"], g
    log(f"  -> best guidance (PSNR): g={best_g}")

    log("\n== Phase A: steps sweep (g=%.1f) ==" % best_g)
    best_s, bp = 10, None
    for s in STEPS_GRID:
        r = run(flow, fcfg, sweep, s, best_g)
        log(f"  steps={s}: PSNR {r['psnr']:.3f} | SSIM {r['ssim']:.4f} | LPIPS {r['lpips']:.4f} | FID {r['fid']:.1f}")
        results[f"steps_{s}"] = r
        if bp is None or r["psnr"] > bp: bp, best_s = r["psnr"], s
    log(f"  -> best steps: {best_s}")

    log("\n== Phase A: TTA check (n=%d, x8 cost) ==" % min(TTA_N, SWEEP_N))
    r = run(flow, fcfg, sweep, best_s, best_g, tta=True)
    log(f"  +TTA: PSNR {r['psnr']:.3f} | SSIM {r['ssim']:.4f} | LPIPS {r['lpips']:.4f} | FID {r['fid']:.1f}")
    results["tta"] = r
    use_tta = r["psnr"] > bp + 0.05
    log(f"  -> TTA {'ADOPTED' if use_tta else 'skipped'} for headline")

    log("\n== Phase B: FULL VALIDATION (flow, best config) — this is the long one ==")
    full = val_pairs if FULL_N is None else random.Random(SEED).sample(val_pairs, FULL_N)
    tta_pairs = random.Random(SEED).sample(val_pairs, TTA_N) if use_tta else full
    rB = run(flow, fcfg, tta_pairs if use_tta else full, best_s, best_g,
             tta=use_tta, breakdown=True)
    results["headline"] = {"config": {"g": best_g, "steps": best_s, "tta": use_tta,
                                      "n": len(tta_pairs if use_tta else full)}, **rB}
    log(f"\n  HEADLINE: PSNR {rB['psnr']:.3f} | SSIM {rB['ssim']:.4f} | LPIPS {rB['lpips']:.4f} | "
        f"FID {rB['fid']:.2f} | SAM {rB['sam']:.2f} | ERGAS {rB['ergas']:.1f} | CC {rB['cc']:.3f}")
    for s, d in rB["by_season"].items():
        log(f"    {s:8s} PSNR {d['psnr']:.2f} | SSIM {d['ssim']:.3f} | SAM {d['sam']:.1f}")
    for t, d in rB["by_terrain"].items():
        log(f"    {t:10s} PSNR {d['psnr']:.2f} | SSIM {d['ssim']:.3f} | SAM {d['sam']:.1f}")

    if EVAL_RESSHIFT and f"{RESSHIFT_TAG}/unet.safetensors" in files:
        log("\n== Phase C: ResShift ablation (identical protocol) ==")
        del flow; gc.collect(); torch.cuda.empty_cache()
        rs, rcfg = load_model(RESSHIFT_TAG)
        rC = run(rs, rcfg, sweep, rcfg.get("T", 15), 1.0)
        results["resshift_sweep"] = rC
        log(f"  ResShift n={SWEEP_N}: PSNR {rC['psnr']:.3f} | SSIM {rC['ssim']:.4f} | "
            f"LPIPS {rC['lpips']:.4f} | FID {rC['fid']:.1f}")
        log("\n  ABLATION TABLE (same data, same VAE, same protocol):")
        log(f"    ResShift x0-pred : PSNR {rC['psnr']:.2f} | SSIM {rC['ssim']:.3f} | FID {rC['fid']:.1f}")
        fh = results["headline"]
        log(f"    Flow+LPL+EMA+CFG : PSNR {fh['psnr']:.2f} | SSIM {fh['ssim']:.3f} | FID {fh['fid']:.1f}")

    with open(os.path.join(OUT, "final_results.json"), "w") as f:
        json.dump(results, f, indent=2)
    if HF_TOKEN:
        try:
            HfApi().upload_folder(folder_path=OUT, path_in_repo="eval_results_final",
                                  repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)
            log("  results -> HF/eval_results_final")
        except Exception as e:
            log(f"  push failed: {e}")
    log("\nDONE. final_results.json holds every table for the paper.")

if __name__ == "__main__":
    main()

Overwriting /content/stage4_final_eval.py


In [ ]:

!python -u /content/stage4_final_eval.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
HF token set: True
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
Val se

In [ ]:
print("")

In [ ]:
 %%writefile /content/e1_color_finetune.py
"""E1/E2 — ResShift continuation from bridge_final with optional color supervision.
Loss = latent MSE + LAMBDA_COLOR * L1( blur8(decode(x0_pred)), blur8(gt_optical) ).
blur8 = 8x avg-pool: supervises only low-frequency color/chroma (SOTA's mechanism),
which cooperates with MSE — unlike VGG/SSIM which hurt PSNR in your 186k run.
E1: LAMBDA_COLOR=1.0 -> saves color_final | E2: LAMBDA_COLOR=0.0 -> saves mse_final
"""
import os, gc, json, time, random, warnings, shutil
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import save_file, load_file
import pandas as pd
from torchmetrics.image import StructuralSimilarityIndexMeasure
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR = "/content/e1_color"; SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG, INIT_TAG = "adapted16_final", "bridge_final"

# ---- EXPERIMENT SWITCH ----
LAMBDA_COLOR = 1.0                     # E1=1.0 (color) | E2=0.0 (pure MSE)
FINAL_TAG = "bridge_color_final" if LAMBDA_COLOR > 0 else "bridge_mse_final"

IMG_SIZE = 256; SEED = 42; VAL_FRACTION = 0.05
SEASONS = ("spring","summer","fall","winter")
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16
T_STEPS, KAPPA, ETA_1, ETA_T = 15, 2.0, 0.04, 0.99
NUM_STEPS = 40_000; BATCH = 4; LR = 1e-5; WARMUP = 300; COND_DROP = 0.10
VAL_EVERY = 5_000; SAVE_LOCAL_EVERY = 20_000; N_VAL = 150; LOG_EVERY = 500
BLUR_POOL = 8                          # 8x avg-pool -> 32x32 color field
device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_region_to_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect_pairs(root):
    rs = Path(root).as_posix(); pairs = []
    for season in SEASONS:
        csvs = list((Path(root)/season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            pairs.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                          "season": season, "terrain": map_region_to_terrain(row["region"])})
    random.shuffle(pairs); return pairs

class PairDataset(Dataset):
    def __init__(self, pairs, sz=256, aug=True): self.p, self.sz, self.aug = pairs, sz, aug
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.sz, self.sz): img = img.resize((self.sz, self.sz), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]; sar, opt = self._l(p["sar"]), self._l(p["opt"])
        if self.aug:
            if random.random() > .5: sar, opt = torch.flip(sar,[2]), torch.flip(opt,[2])
            if random.random() > .5: sar, opt = torch.flip(sar,[1]), torch.flip(opt,[1])
            k = random.randint(0,3)
            if k: sar, opt = torch.rot90(sar,k,[1,2]), torch.rot90(opt,k,[1,2])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(self, ic=3, oc=DETAIL_CH, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(self, x): return self.net(x)

class AdaptedVAE(nn.Module):
    def __init__(self, bv):
        super().__init__(); self.base_vae = bv; self.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH, o.out_channels, o.kernel_size, o.stride,
                                       o.padding, bias=(o.bias is not None))
    @torch.no_grad()
    def encode_latent(self, x):
        z = self.base_vae.encode(x).latent_dist.mode()
        return torch.cat([self.base_vae.post_quant_conv(z), self.detail_enc(x)], 1)
    def decode_grad(self, lat): return self.base_vae.decoder(lat)
    @torch.no_grad()
    def decode_latent(self, lat): return self.base_vae.decoder(lat)

def load_vae():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    return v.to(device).eval().float().requires_grad_(False)

def build_unet_from_final():
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/unet.safetensors")))
    log(f"  UNet initialized from {INIT_TAG} (16.7 dB baseline).")
    return unet

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in ("tropical","temperate","arctic","arid","coastal","urban"):
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

def make_eta():
    i = np.arange(T_STEPS)
    return torch.tensor(ETA_1*(ETA_T/ETA_1)**(i/(T_STEPS-1)), dtype=torch.float32).to(device)

def blur8(img): return F.avg_pool2d(img, BLUR_POOL)     # low-frequency color field

def get_cond(ss, tt, cache, drop=0.0):
    return torch.stack([cache["null"] if (drop and random.random() < drop)
                        else cache.get((s,t), cache["null"]) for s,t in zip(ss,tt)])

@torch.no_grad()
def sample(unet, y, cond, eta):
    B = y.shape[0]; x = y + KAPPA*torch.sqrt(eta[-1])*torch.randn_like(y)
    for t in reversed(range(T_STEPS)):
        tb = torch.full((B,), int(t*1000/T_STEPS), device=device, dtype=torch.long)
        x0 = unet(torch.cat([x,y],1).float(), tb, encoder_hidden_states=cond.float()).sample
        if t > 0:
            ep = eta[t-1]; x = x0 + ep*(y-x0) + KAPPA*torch.sqrt(ep)*torch.randn_like(x)
        else: x = x0
    return x

def main():
    acc = Accelerator(mixed_precision="no", project_dir=OUTPUT_DIR)  # fp32: short run, stability first
    set_seed(SEED); is_main = acc.is_main_process
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    log(f"EXPERIMENT: {'E1 color-supervision' if LAMBDA_COLOR>0 else 'E2 pure-MSE'} -> {FINAL_TAG}")

    vae = load_vae(); embeds = build_embeds(); unet = build_unet_from_final()
    eta = make_eta()
    import lpips as lpips_lib
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    pairs = collect_pairs(DATA_ROOT)
    tr, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    log(f"Train {len(tr):,} | Val {len(va):,}")
    dl = DataLoader(PairDataset(tr, IMG_SIZE, True), batch_size=BATCH, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)

    opt = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9,0.999), weight_decay=1e-2)
    sch = get_cosine_schedule_with_warmup(opt, WARMUP, NUM_STEPS)
    unet, opt, dl, sch = acc.prepare(unet, opt, dl, sch)

    latent_scale = 1.4171   # your computed value; consistent with bridge_final training
    def enc(x):
        with torch.no_grad(): return vae.encode_latent(x.float()) * latent_scale

    def sam_deg(p, r):
        p = ((p+1)/2).flatten(2).transpose(1,2); r = ((r+1)/2).flatten(2).transpose(1,2)
        cosv = ((p*r).sum(-1)/(p.norm(dim=-1)*r.norm(dim=-1)+1e-8)).clamp(-1,1)
        return torch.rad2deg(torch.acos(cosv)).mean().item()

    @torch.no_grad()
    def validate(step):
        sub = random.Random(7).sample(va, N_VAL)
        vdl = DataLoader(PairDataset(sub, IMG_SIZE, False), batch_size=8, num_workers=2)
        m = acc.unwrap_model(unet); m.eval(); P,S,L,A = [],[],[],[]
        for b in vdl:
            o = b["opt"].to(device); y = enc(b["sar"].to(device))
            c = get_cond(b["season"], b["terrain"], embeds).to(device)
            rec = vae.decode_latent((sample(m, y, c, eta)/latent_scale).float()).clamp(-1,1)
            P.append(10*np.log10(4.0/F.mse_loss(rec,o).item()))
            S.append(ssim_fn(rec,o).item()); L.append(lpips_fn(rec,o).mean().item())
            A.append(sam_deg(rec,o))
        m.train()
        log(f"\n[VAL {step}] PSNR {np.mean(P):.3f} | SSIM {np.mean(S):.4f} | "
            f"LPIPS {np.mean(L):.4f} | SAM {np.mean(A):.2f}  (baseline: 16.73/0.259/-/7.92)\n")

    gs = 0; ema_m = ema_c = 0.0; a = 0.98; t0 = time.time(); ll = 0
    latest = os.path.join(OUTPUT_DIR, "latest")
    unet.train()
    log(f">>> fine-tune {NUM_STEPS} steps @ LR {LR}, color λ={LAMBDA_COLOR}\n")
    for epoch in range(999):
        for b in dl:
            oimg = b["opt"].to(device)
            x0 = enc(oimg); y = enc(b["sar"].to(device))
            c = get_cond(b["season"], b["terrain"], embeds, COND_DROP).to(device)
            t = torch.randint(0, T_STEPS, (x0.shape[0],), device=device)
            et = eta[t].view(-1,1,1,1)
            x_t = x0 + et*(y-x0) + KAPPA*torch.sqrt(et)*torch.randn_like(x0)
            tb = (t*1000//T_STEPS).long()
            pred = unet(torch.cat([x_t,y],1).float(), tb, encoder_hidden_states=c.float()).sample
            loss_m = F.mse_loss(pred, x0)
            loss = loss_m; cval = 0.0
            if LAMBDA_COLOR > 0:
                img_pred = vae.decode_grad((pred/latent_scale)).clamp(-1,1)
                loss_c = F.l1_loss(blur8(img_pred), blur8(oimg)) * LAMBDA_COLOR
                loss = loss + loss_c; cval = float(loss_c.item())
            acc.backward(loss)
            # spike-guard: never learn from a pathological step
            if gs > 500 and loss_m.item() > 10*max(ema_m, 0.02):
                opt.zero_grad(set_to_none=True); continue
            acc.clip_grad_norm_(unet.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
            gs += 1; ema_m = a*ema_m+(1-a)*loss_m.item(); ema_c = a*ema_c+(1-a)*cval
            if is_main and gs % LOG_EVERY == 0:
                r = (gs-ll)/max(time.time()-t0,1e-9); t0, ll = time.time(), gs
                log(f"step {gs}/{NUM_STEPS} | mse {ema_m:.4f} | color {ema_c:.4f} | {r:.2f} it/s")
            if gs % VAL_EVERY == 0 and is_main: validate(gs)
            if gs % SAVE_LOCAL_EVERY == 0 and is_main:
                if os.path.exists(latest): shutil.rmtree(latest)
                os.makedirs(latest); acc.save_state(latest)
                torch.save({"step": gs}, os.path.join(latest, "meta.pt"))
                log(f"  local checkpoint @ {gs}")
            if gs >= NUM_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if gs >= NUM_STEPS: break

    if is_main:
        validate(gs)
        tmp = os.path.join(OUTPUT_DIR, "_f"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k, v in acc.unwrap_model(unet).state_dict().items()}
        save_file(sd, os.path.join(tmp, "unet.safetensors"))
        json.dump({"latent_scale": latent_scale, "T": T_STEPS, "kappa": KAPPA,
                   "eta_1": ETA_1, "eta_T": ETA_T, "img_size": IMG_SIZE,
                   "method": "resshift", "final_step": gs,
                   "lambda_color": LAMBDA_COLOR}, open(os.path.join(tmp, "bridge_config.json"), "w"), indent=2)
        try:
            HfApi().upload_folder(folder_path=tmp, path_in_repo=FINAL_TAG,
                                  repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)
            log(f"  FINAL -> HF/{FINAL_TAG}")
        except Exception as e: log(f"  push failed: {e} — model still in {tmp}")
    acc.end_training(); log("DONE.")

if __name__ == "__main__":
    main()

Writing /content/e1_color_finetune.py


In [ ]:

!python -u /content/e1_color_finetune.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
EXPERIMENT: E1 color-supervision -> bridge_color_final
  UNet initialized from bridge_final (16.7 dB baseline).
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
L

In [ ]:
 %%writefile /content/e2_color_finetune.py
"""E1/E2 — ResShift continuation from bridge_final with optional color supervision.
Loss = latent MSE + LAMBDA_COLOR * L1( blur8(decode(x0_pred)), blur8(gt_optical) ).
blur8 = 8x avg-pool: supervises only low-frequency color/chroma (SOTA's mechanism),
which cooperates with MSE — unlike VGG/SSIM which hurt PSNR in your 186k run.
E1: LAMBDA_COLOR=1.0 -> saves color_final | E2: LAMBDA_COLOR=0.0 -> saves mse_final
"""
import os, gc, json, time, random, warnings, shutil
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import save_file, load_file
import pandas as pd
from torchmetrics.image import StructuralSimilarityIndexMeasure
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR = "/content/e1_color"; SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG, INIT_TAG = "adapted16_final", "bridge_final"

# ---- EXPERIMENT SWITCH ----
LAMBDA_COLOR = 0                    # E1=1.0 (color) | E2=0.0 (pure MSE)
FINAL_TAG = "bridge_color_final" if LAMBDA_COLOR > 0 else "bridge_mse_final"

IMG_SIZE = 256; SEED = 42; VAL_FRACTION = 0.05
SEASONS = ("spring","summer","fall","winter")
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16
T_STEPS, KAPPA, ETA_1, ETA_T = 15, 2.0, 0.04, 0.99
NUM_STEPS = 40_000; BATCH = 4; LR = 1e-5; WARMUP = 300; COND_DROP = 0.10
VAL_EVERY = 5_000; SAVE_LOCAL_EVERY = 20_000; N_VAL = 150; LOG_EVERY = 500
BLUR_POOL = 8                          # 8x avg-pool -> 32x32 color field
device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_region_to_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect_pairs(root):
    rs = Path(root).as_posix(); pairs = []
    for season in SEASONS:
        csvs = list((Path(root)/season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            pairs.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                          "season": season, "terrain": map_region_to_terrain(row["region"])})
    random.shuffle(pairs); return pairs

class PairDataset(Dataset):
    def __init__(self, pairs, sz=256, aug=True): self.p, self.sz, self.aug = pairs, sz, aug
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.sz, self.sz): img = img.resize((self.sz, self.sz), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]; sar, opt = self._l(p["sar"]), self._l(p["opt"])
        if self.aug:
            if random.random() > .5: sar, opt = torch.flip(sar,[2]), torch.flip(opt,[2])
            if random.random() > .5: sar, opt = torch.flip(sar,[1]), torch.flip(opt,[1])
            k = random.randint(0,3)
            if k: sar, opt = torch.rot90(sar,k,[1,2]), torch.rot90(opt,k,[1,2])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(self, ic=3, oc=DETAIL_CH, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(self, x): return self.net(x)

class AdaptedVAE(nn.Module):
    def __init__(self, bv):
        super().__init__(); self.base_vae = bv; self.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH, o.out_channels, o.kernel_size, o.stride,
                                       o.padding, bias=(o.bias is not None))
    @torch.no_grad()
    def encode_latent(self, x):
        z = self.base_vae.encode(x).latent_dist.mode()
        return torch.cat([self.base_vae.post_quant_conv(z), self.detail_enc(x)], 1)
    def decode_grad(self, lat): return self.base_vae.decoder(lat)
    @torch.no_grad()
    def decode_latent(self, lat): return self.base_vae.decoder(lat)

def load_vae():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    return v.to(device).eval().float().requires_grad_(False)

def build_unet_from_final():
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/unet.safetensors")))
    log(f"  UNet initialized from {INIT_TAG} (16.7 dB baseline).")
    return unet

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in ("tropical","temperate","arctic","arid","coastal","urban"):
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

def make_eta():
    i = np.arange(T_STEPS)
    return torch.tensor(ETA_1*(ETA_T/ETA_1)**(i/(T_STEPS-1)), dtype=torch.float32).to(device)

def blur8(img): return F.avg_pool2d(img, BLUR_POOL)     # low-frequency color field

def get_cond(ss, tt, cache, drop=0.0):
    return torch.stack([cache["null"] if (drop and random.random() < drop)
                        else cache.get((s,t), cache["null"]) for s,t in zip(ss,tt)])

@torch.no_grad()
def sample(unet, y, cond, eta):
    B = y.shape[0]; x = y + KAPPA*torch.sqrt(eta[-1])*torch.randn_like(y)
    for t in reversed(range(T_STEPS)):
        tb = torch.full((B,), int(t*1000/T_STEPS), device=device, dtype=torch.long)
        x0 = unet(torch.cat([x,y],1).float(), tb, encoder_hidden_states=cond.float()).sample
        if t > 0:
            ep = eta[t-1]; x = x0 + ep*(y-x0) + KAPPA*torch.sqrt(ep)*torch.randn_like(x)
        else: x = x0
    return x

def main():
    acc = Accelerator(mixed_precision="no", project_dir=OUTPUT_DIR)  # fp32: short run, stability first
    set_seed(SEED); is_main = acc.is_main_process
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    log(f"EXPERIMENT: {'E1 color-supervision' if LAMBDA_COLOR>0 else 'E2 pure-MSE'} -> {FINAL_TAG}")

    vae = load_vae(); embeds = build_embeds(); unet = build_unet_from_final()
    eta = make_eta()
    import lpips as lpips_lib
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    pairs = collect_pairs(DATA_ROOT)
    tr, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    log(f"Train {len(tr):,} | Val {len(va):,}")
    dl = DataLoader(PairDataset(tr, IMG_SIZE, True), batch_size=BATCH, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)

    opt = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9,0.999), weight_decay=1e-2)
    sch = get_cosine_schedule_with_warmup(opt, WARMUP, NUM_STEPS)
    unet, opt, dl, sch = acc.prepare(unet, opt, dl, sch)

    latent_scale = 1.4171   # your computed value; consistent with bridge_final training
    def enc(x):
        with torch.no_grad(): return vae.encode_latent(x.float()) * latent_scale

    def sam_deg(p, r):
        p = ((p+1)/2).flatten(2).transpose(1,2); r = ((r+1)/2).flatten(2).transpose(1,2)
        cosv = ((p*r).sum(-1)/(p.norm(dim=-1)*r.norm(dim=-1)+1e-8)).clamp(-1,1)
        return torch.rad2deg(torch.acos(cosv)).mean().item()

    @torch.no_grad()
    def validate(step):
        sub = random.Random(7).sample(va, N_VAL)
        vdl = DataLoader(PairDataset(sub, IMG_SIZE, False), batch_size=8, num_workers=2)
        m = acc.unwrap_model(unet); m.eval(); P,S,L,A = [],[],[],[]
        for b in vdl:
            o = b["opt"].to(device); y = enc(b["sar"].to(device))
            c = get_cond(b["season"], b["terrain"], embeds).to(device)
            rec = vae.decode_latent((sample(m, y, c, eta)/latent_scale).float()).clamp(-1,1)
            P.append(10*np.log10(4.0/F.mse_loss(rec,o).item()))
            S.append(ssim_fn(rec,o).item()); L.append(lpips_fn(rec,o).mean().item())
            A.append(sam_deg(rec,o))
        m.train()
        log(f"\n[VAL {step}] PSNR {np.mean(P):.3f} | SSIM {np.mean(S):.4f} | "
            f"LPIPS {np.mean(L):.4f} | SAM {np.mean(A):.2f}  (baseline: 16.73/0.259/-/7.92)\n")

    gs = 0; ema_m = ema_c = 0.0; a = 0.98; t0 = time.time(); ll = 0
    latest = os.path.join(OUTPUT_DIR, "latest")
    unet.train()
    log(f">>> fine-tune {NUM_STEPS} steps @ LR {LR}, color λ={LAMBDA_COLOR}\n")
    for epoch in range(999):
        for b in dl:
            oimg = b["opt"].to(device)
            x0 = enc(oimg); y = enc(b["sar"].to(device))
            c = get_cond(b["season"], b["terrain"], embeds, COND_DROP).to(device)
            t = torch.randint(0, T_STEPS, (x0.shape[0],), device=device)
            et = eta[t].view(-1,1,1,1)
            x_t = x0 + et*(y-x0) + KAPPA*torch.sqrt(et)*torch.randn_like(x0)
            tb = (t*1000//T_STEPS).long()
            pred = unet(torch.cat([x_t,y],1).float(), tb, encoder_hidden_states=c.float()).sample
            loss_m = F.mse_loss(pred, x0)
            loss = loss_m; cval = 0.0
            if LAMBDA_COLOR > 0:
                img_pred = vae.decode_grad((pred/latent_scale)).clamp(-1,1)
                loss_c = F.l1_loss(blur8(img_pred), blur8(oimg)) * LAMBDA_COLOR
                loss = loss + loss_c; cval = float(loss_c.item())
            acc.backward(loss)
            # spike-guard: never learn from a pathological step
            if gs > 500 and loss_m.item() > 10*max(ema_m, 0.02):
                opt.zero_grad(set_to_none=True); continue
            acc.clip_grad_norm_(unet.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
            gs += 1; ema_m = a*ema_m+(1-a)*loss_m.item(); ema_c = a*ema_c+(1-a)*cval
            if is_main and gs % LOG_EVERY == 0:
                r = (gs-ll)/max(time.time()-t0,1e-9); t0, ll = time.time(), gs
                log(f"step {gs}/{NUM_STEPS} | mse {ema_m:.4f} | color {ema_c:.4f} | {r:.2f} it/s")
            if gs % VAL_EVERY == 0 and is_main: validate(gs)
            if gs % SAVE_LOCAL_EVERY == 0 and is_main:
                if os.path.exists(latest): shutil.rmtree(latest)
                os.makedirs(latest); acc.save_state(latest)
                torch.save({"step": gs}, os.path.join(latest, "meta.pt"))
                log(f"  local checkpoint @ {gs}")
            if gs >= NUM_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if gs >= NUM_STEPS: break

    if is_main:
        validate(gs)
        tmp = os.path.join(OUTPUT_DIR, "_f"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k, v in acc.unwrap_model(unet).state_dict().items()}
        save_file(sd, os.path.join(tmp, "unet.safetensors"))
        json.dump({"latent_scale": latent_scale, "T": T_STEPS, "kappa": KAPPA,
                   "eta_1": ETA_1, "eta_T": ETA_T, "img_size": IMG_SIZE,
                   "method": "resshift", "final_step": gs,
                   "lambda_color": LAMBDA_COLOR}, open(os.path.join(tmp, "bridge_config.json"), "w"), indent=2)
        try:
            HfApi().upload_folder(folder_path=tmp, path_in_repo=FINAL_TAG,
                                  repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)
            log(f"  FINAL -> HF/{FINAL_TAG}")
        except Exception as e: log(f"  push failed: {e} — model still in {tmp}")
    acc.end_training(); log("DONE.")

if __name__ == "__main__":
    main()

Writing /content/e2_color_finetune.py


In [ ]:

!python -u /content/e2_color_finetune.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
EXPERIMENT: E2 pure-MSE -> bridge_mse_final
  UNet initialized from bridge_final (16.7 dB baseline).
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading mode

In [ ]:
%%writefile /content/x1_calibration_probe.py
"""X1/X2 — training-free probes on bridge_final (champion, 16.73/125.5):
  X1: Reinhard(2001)-style color calibration toward per-(season,terrain) priors
      built from TRAIN ground truth, blended by alpha in {0.25, 0.5, 1.0}.
  X2: despeckle-at-inference probe (median-3 on SAR input).
Step-0 sanity gate: raw condition must reproduce ~16.3±0.4 on n=400 or ABORT.
"""
import os, gc, json, random, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image, ImageFilter
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import lpips as lpips_lib, pandas as pd
from torchmetrics.image import StructuralSimilarityIndexMeasure
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
VAE_TAG, BRIDGE_TAG = "adapted16_final", "bridge_final"
IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16
N_EVAL, N_PRIOR, BATCH = 400, 2000, 8
ALPHAS = (0.25, 0.5, 1.0)
device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    return ps

class DS(Dataset):
    def __init__(self, pairs, despeckle=False): self.p, self.d = pairs, despeckle
    def __len__(self): return len(self.p)
    def _l(self, path, med=False):
        img = Image.open(path).convert("RGB")
        if med: img = img.filter(ImageFilter.MedianFilter(3))
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        return {"sar": self._l(p["sar"], self.d), "opt": self._l(p["opt"]),
                "season": p["season"], "terrain": p["terrain"]}

# ---- Reinhard lab space ----
_RGB2LMS = torch.tensor([[0.3811,0.5783,0.0402],[0.1967,0.7244,0.0782],[0.0241,0.1288,0.8444]])
_LMS2LAB1 = torch.diag(torch.tensor([1/np.sqrt(3),1/np.sqrt(6),1/np.sqrt(2)])).float()
_LAB_M = torch.tensor([[1.,1.,1.],[1.,1.,-2.],[1.,-1.,0.]]).float()
def rgb_to_lab(img01):
    B,C,H,W = img01.shape
    flat = img01.clamp(1e-4,1).permute(0,2,3,1).reshape(-1,3)
    lms = torch.log10(flat @ _RGB2LMS.T.to(img01.device) + 1e-6)
    lab = lms @ (_LMS2LAB1 @ _LAB_M).T.to(img01.device)
    return lab.reshape(B,H,W,3).permute(0,3,1,2)
def lab_to_rgb(lab):
    B,C,H,W = lab.shape
    flat = lab.permute(0,2,3,1).reshape(-1,3)
    lms = flat @ torch.inverse((_LMS2LAB1 @ _LAB_M).T).to(lab.device)
    rgb = torch.pow(10.0, lms) @ torch.inverse(_RGB2LMS.T).to(lab.device)
    return rgb.reshape(B,H,W,3).permute(0,3,1,2).clamp(0,1)
def calibrate(pred01, mu_p, sd_p, alpha):
    lab = rgb_to_lab(pred01)
    mu = lab.mean(dim=(2,3), keepdim=True); sd = lab.std(dim=(2,3), keepdim=True)+1e-6
    mu_t = (1-alpha)*mu + alpha*mu_p.view(1,3,1,1).to(lab.device)
    sd_t = (1-alpha)*sd + alpha*sd_p.view(1,3,1,1).to(lab.device)
    return lab_to_rgb((lab-mu)/sd*sd_t+mu_t)

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=DETAIL_CH, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)
class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def main():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(bv)
    vae.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    vae = vae.to(device).eval().float().requires_grad_(False)

    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{BRIDGE_TAG}/bridge_config.json")))
    SCALE = cfg["latent_scale"]                  # read from config — never hardcoded again
    log(f"latent_scale from config: {SCALE:.4f} | T={cfg['T']} kappa={cfg['kappa']}")
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{BRIDGE_TAG}/unet.safetensors")))
    unet = unet.to(device).eval().float().requires_grad_(False)

    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    emb = {}
    @torch.no_grad()
    def E(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in ("tropical","temperate","arctic","arid","coastal","urban"):
            emb[(s,ter)] = E(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    emb["null"] = E(""); del txt; gc.collect(); torch.cuda.empty_cache()

    i = np.arange(cfg["T"]); eta = torch.tensor(cfg["eta_1"]*(cfg["eta_T"]/cfg["eta_1"])**(i/(cfg["T"]-1)),
                                                dtype=torch.float32).to(device)
    @torch.no_grad()
    def sample(y, cond):
        B = y.shape[0]; x = y + cfg["kappa"]*torch.sqrt(eta[-1])*torch.randn_like(y)
        for t in reversed(range(cfg["T"])):
            tb = torch.full((B,), int(t*1000/cfg["T"]), device=device, dtype=torch.long)
            x0 = unet(torch.cat([x,y],1), tb, encoder_hidden_states=cond).sample
            if t > 0:
                ep = eta[t-1]; x = x0 + ep*(y-x0) + cfg["kappa"]*torch.sqrt(ep)*torch.randn_like(x)
            else: x = x0
        return x

    pairs = collect(DATA_ROOT)
    tr, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)

    # ---- build per-class lab priors from TRAIN ground truth ----
    log(f"Building color priors from {N_PRIOR} train images...")
    stats = {}
    for p in random.Random(1).sample(tr, N_PRIOR):
        img = DS([p])[0]["opt"].unsqueeze(0)
        lab = rgb_to_lab((img+1)/2)
        k = (p["season"], p["terrain"])
        s = stats.setdefault(k, {"n":0, "mu":torch.zeros(3), "m2":torch.zeros(3)})
        mu = lab.mean(dim=(0,2,3)); s["mu"] += mu; s["m2"] += lab.var(dim=(0,2,3))+mu**2; s["n"] += 1
    priors = {k: ((v["mu"]/v["n"]), torch.sqrt((v["m2"]/v["n"] - (v["mu"]/v["n"])**2).clamp(min=1e-6)))
              for k, v in stats.items()}
    log(f"  priors for {len(priors)} (season,terrain) classes")

    lp = lpips_lib.LPIPS(net="alex").to(device)
    ssim = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
    def sam_deg(p, r):
        p = p.flatten(2).transpose(1,2); r = r.flatten(2).transpose(1,2)
        c = ((p*r).sum(-1)/(p.norm(dim=-1)*r.norm(dim=-1)+1e-8)).clamp(-1,1)
        return torch.rad2deg(torch.acos(c)).mean().item()
    def cc(p, r):
        t = 0.0
        for b in range(3):
            x = p[:,b].flatten()-p[:,b].mean(); y = r[:,b].flatten()-r[:,b].mean()
            t += float((x*y).sum()/(x.norm()*y.norm()+1e-8))
        return t/3

    sub = random.Random(SEED).sample(va, N_EVAL)
    conds = [("raw", False, None)] + [(f"calib_a{a}", False, a) for a in ALPHAS] + \
            [("despeckle", True, None), ("despeckle+calib0.5", True, 0.5)]
    for name, desp, alpha in conds:
        dl = DataLoader(DS(sub, despeckle=desp), batch_size=BATCH, num_workers=4)
        M = {k: [] for k in ("psnr","ssim","lpips","sam","cc")}
        for b in dl:
            o = b["opt"].to(device)
            y = vae.enc(b["sar"].to(device).float())*SCALE
            c = torch.stack([emb.get((s,t), emb["null"]) for s,t in zip(b["season"],b["terrain"])]).to(device)
            rec = vae.dec((sample(y, c)/SCALE)).clamp(-1,1)
            if alpha is not None:
                r01 = (rec+1)/2
                out = []
                for j in range(rec.shape[0]):
                    mu_p, sd_p = priors.get((b["season"][j], b["terrain"][j]),
                                            (torch.zeros(3), torch.ones(3)))
                    out.append(calibrate(r01[j:j+1], mu_p, sd_p, alpha))
                rec = torch.cat(out)*2-1
            for j in range(rec.shape[0]):
                pi, ri = rec[j:j+1], o[j:j+1]
                M["psnr"].append(10*np.log10(4.0/F.mse_loss(pi,ri).item()))
                M["ssim"].append(ssim(pi,ri).item()); M["lpips"].append(lp(pi,ri).item())
                M["sam"].append(sam_deg((pi+1)/2,(ri+1)/2)); M["cc"].append(cc((pi+1)/2,(ri+1)/2))
        r = {k: float(np.mean(v)) for k,v in M.items()}
        log(f"[{name:>18s}] PSNR {r['psnr']:.3f} | SSIM {r['ssim']:.4f} | LPIPS {r['lpips']:.4f} | "
            f"SAM {r['sam']:.2f} | CC {r['cc']:.3f}")
        if name == "raw" and not (15.9 <= r["psnr"] <= 17.1):
            log("!! step-0 gate FAILED: raw does not reproduce baseline — harness bug, aborting."); return

if __name__ == "__main__":
    main()

Writing /content/x1_calibration_probe.py


In [ ]:

!python -u /content/x1_calibration_probe.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
latent_scale from config: 1.4010 | T=15 kappa=2.0
Building color priors from 2000 train images...
  priors for 10 (season,terrain) classes
Setting up [LPIPS] perceptual loss: trunk [ale

In [ ]:
%%writefile /content/stage3b_flow_256.py
"""
SAR-INTEL Stage 3b FINAL (guarded) — flow-matching bridge, adapted 16ch latent @ 256px.

Lessons folded in from the crashed run + E1/E2 harness bugs:
  - LR halved (3e-5): stability over speed.
  - Perceptual losses (L1+LPIPS+LPL) engage at step 5000 and RAMP IN over 5000
    steps (0->1 weight) instead of switching on abruptly — this is what caused
    the step-10k loss spike in the previous run.
  - Loss weights halved (LPIPS 0.5, L1 0.25, LPL 0.5): the full-strength stack
    was too aggressive for this optimization.
  - SPIKE-GUARD before backward(): any step with v_loss > 10x its EMA is
    skipped entirely (zero grad, continue) instead of being learned from.
  - EMA (0.9999): final model is the smoothed weights.
  - Rotations + flips from step 0 (fresh model, no distribution-shift issue).
  - Quiet logging: 1 line / 500 steps. No periodic HF push (local safety net
    only, overwritten each time). Final EMA model -> HF as safetensors, once.

PRE-COMMITTED GATE: at step 30000 VAL, continue ONLY IF v <= 0.10 AND PSNR >= 12.5.
Otherwise kill the run — the champion (bridge_final, 16.73/125.5 FID) stands.
"""
import os, gc, json, time, random, warnings, shutil
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from safetensors.torch import save_file, load_file
import lpips as lpips_lib
import pandas as pd
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def log(*a): print(*a, flush=True)

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT    = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR   = "/content/stage3b_flow_guarded"
SD_MODEL_ID  = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID   = os.environ["HF_REPO_ID"]
HF_TOKEN     = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG      = "adapted16_final"
INIT_TAG     = "bridge_final"          # init from the 150k champion, not from scratch
RESUME_TAG   = "flow_guarded_resume"

IMG_SIZE     = 256
SEASONS      = ("spring", "summer", "fall", "winter")
VAL_FRACTION = 0.05
SEED         = 42
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16

NUM_TRAIN_STEPS = 120_000
BATCH_SIZE   = 4
LR           = 3e-5              # halved
LR_WARMUP    = 1_000
COND_DROPOUT = 0.10

PIXEL_START  = 5_000              # later engage
PIXEL_RAMP   = 5_000              # ...ramped 0->1 over this many steps, no shock
LAMBDA_LPIPS = 0.5                # halved
LAMBDA_L1    = 0.25               # halved
LAMBDA_LPL   = 0.5                # halved
SPIKE_GUARD_MULT = 10.0           # skip any step where v_loss > 10x its EMA
GUARD_WARMUP = 1_000              # don't guard before the loss has stabilized once

EMA_DECAY    = 0.9999
SAMPLE_STEPS = 10
VAL_EVERY    = 5_000
N_VAL        = 200
LOCAL_SAVE_EVERY = 20_000
STEP_LOG_EVERY   = 500

# pre-committed gate (informational print only — YOU decide to kill based on this)
GATE_STEP, GATE_MAX_V, GATE_MIN_PSNR = 30_000, 0.10, 12.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEASON_DESC = {"spring": "spring", "summer": "summer", "fall": "autumn", "winter": "winter"}
TERRAINS = ("tropical", "temperate", "arctic", "arid", "coastal", "urban")

def map_region_to_terrain(region):
    r = (region or "").lower()
    if any(k in r for k in ("trop", "rainforest", "amazon", "jungle")): return "tropical"
    if any(k in r for k in ("arctic", "tundra", "snow", "ice", "polar")): return "arctic"
    if any(k in r for k in ("desert", "arid", "sahara", "dry")): return "arid"
    if any(k in r for k in ("coast", "beach", "shore", "marine", "sea")): return "coastal"
    if any(k in r for k in ("urban", "city", "metro")): return "urban"
    return "temperate"


def collect_pairs(root, seasons):
    root_str = Path(root).as_posix(); pairs = []
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for col in ("s1_fileName", "s2_fileName"):
            df[col] = df[col].astype(str).str.replace("\\", "/", regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            pairs.append({"sar": f"{root_str}/{row['s1_fileName']}",
                          "opt": f"{root_str}/{row['s2_fileName']}",
                          "season": season, "terrain": map_region_to_terrain(row["region"])})
    random.shuffle(pairs); return pairs


class PairDataset(Dataset):
    def __init__(self, pairs, img_size=256, augment=True):
        self.pairs = pairs; self.img_size = img_size; self.augment = augment
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0
    def __getitem__(self, idx):
        p = self.pairs[idx]
        sar = self._load(p["sar"]); opt = self._load(p["opt"])
        if self.augment:
            if random.random() > 0.5: sar, opt = torch.flip(sar, [2]), torch.flip(opt, [2])
            if random.random() > 0.5: sar, opt = torch.flip(sar, [1]), torch.flip(opt, [1])
            k = random.randint(0, 3)
            if k: sar, opt = torch.rot90(sar, k, [1, 2]), torch.rot90(opt, k, [1, 2])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}


class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CH, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),  nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf*2, 3, 2, 1),   nn.GroupNorm(8, nf*2), nn.SiLU(),
            nn.Conv2d(nf*2, nf*4, 3, 2, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, nf*4, 3, 1, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, out_ch, 3, 1, 1))
    def forward(self, x): return self.net(x)


class AdaptedVAE(nn.Module):
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae = base_vae; self.detail_enc = DetailEncoder()
        old = base_vae.decoder.conv_in
        base_vae.decoder.conv_in = nn.Conv2d(TOTAL_CH, old.out_channels, old.kernel_size,
                                             old.stride, old.padding, bias=(old.bias is not None))
    @torch.no_grad()
    def encode_latent(self, x):
        z = self.base_vae.encode(x).latent_dist.mode()
        return torch.cat([self.base_vae.post_quant_conv(z), self.detail_enc(x)], dim=1)
    def decode_grad(self, lat):
        return self.base_vae.decoder(lat)
    @torch.no_grad()
    def decode_latent(self, lat):
        return self.base_vae.decoder(lat)


class HookedDecoder:
    """Gated capture of decoder up-block features, for LPL. Free when not capturing."""
    def __init__(self, vae):
        self.feats = []; self.capture = False
        for blk in vae.base_vae.decoder.up_blocks:
            blk.register_forward_hook(
                lambda m, i, o: self.feats.append(o) if self.capture else None)
    def decode(self, vae, lat):
        self.capture = True; self.feats = []
        img = vae.base_vae.decoder(lat)
        feats = list(self.feats); self.capture = False; self.feats = []
        return img, feats

def latent_perceptual_loss(f_pred, f_real):
    loss = 0.0
    for fp, fr in zip(f_pred, f_real):
        loss = loss + F.l1_loss(fp.float(), fr.float().detach())
    return loss / max(len(f_pred), 1)


def load_adapted_vae():
    base_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(base_vae)
    bp = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/base_vae.pth")
    de = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/detail_encoder.pth")
    vae.base_vae.load_state_dict(torch.load(bp, map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(de, map_location="cpu"))
    vae = vae.to(device).eval().float(); vae.requires_grad_(False)
    log("  Adapted VAE loaded from", VAE_TAG)
    return vae


def build_embed_cache():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    txt.requires_grad_(False); cache = {}
    @torch.no_grad()
    def embed(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s, ter)] = embed(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = embed(""); del txt; gc.collect(); torch.cuda.empty_cache()
    log(f"  Built {len(cache)} CLIP embeddings.")
    return cache


def build_unet_from_champion():
    """Init from bridge_final (150k, 16.73 dB) — NOT from scratch. Reuses everything
    ResShift learned; flow-matching only has to learn the new velocity objective."""
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    sd_path = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{INIT_TAG}/unet.safetensors")
    unet.load_state_dict(load_file(sd_path))
    log(f"  UNet initialized from {INIT_TAG} (champion, 16.73 dB baseline).")
    return unet


def get_cond(seasons, terrains, cache, drop=0.0):
    out = []
    for s, ter in zip(seasons, terrains):
        out.append(cache["null"] if (drop > 0 and random.random() < drop)
                   else cache.get((s, ter), cache["null"]))
    return torch.stack(out)


@torch.no_grad()
def flow_sample(unet, x0, cond, n_steps, dtype):
    x = x0.clone(); dt = 1.0 / n_steps; B = x0.shape[0]
    for i in range(n_steps):
        tb = torch.full((B,), int(i * dt * 999), device=x0.device, dtype=torch.long)
        v = unet(torch.cat([x, x0], 1).to(dtype), tb,
                 encoder_hidden_states=cond.to(dtype)).sample.float()
        x = x + dt * v
    return x


def push_hf(folder, tag):
    if not HF_TOKEN: return
    try:
        HfApi().upload_folder(folder_path=folder, path_in_repo=tag, repo_id=HF_REPO_ID,
                              repo_type="model", token=HF_TOKEN)
        log(f"  HF push -> {tag}")
    except Exception as e:
        log(f"  HF push failed: {e}")


def download_resume_from_hf(dest):
    try:
        files = HfApi().list_repo_files(HF_REPO_ID, token=HF_TOKEN)
        if f"{RESUME_TAG}/meta.pt" not in files:
            log("  no resume state on HF (fresh run)"); return False
        snapshot_download(repo_id=HF_REPO_ID, allow_patterns=f"{RESUME_TAG}/*",
                          local_dir="/content/_flow_guard_dl", token=HF_TOKEN)
        src = os.path.join("/content/_flow_guard_dl", RESUME_TAG)
        if os.path.exists(dest): shutil.rmtree(dest)
        shutil.copytree(src, dest); return True
    except Exception as e:
        log(f"  resume check failed ({e})"); return False


def denorm01(t): return (t.clamp(-1, 1) + 1) / 2


def main():
    accelerator = Accelerator(mixed_precision="bf16", project_dir=OUTPUT_DIR)
    set_seed(SEED); is_main = accelerator.is_main_process; dtype = torch.bfloat16
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    log("Loading components...")
    vae = load_adapted_vae(); embed_cache = build_embed_cache()
    unet = build_unet_from_champion()
    hooked = HookedDecoder(vae)
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    train_pairs, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in all_pairs], random_state=SEED)
    log(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,}")

    train_ds = PairDataset(train_pairs, IMG_SIZE, True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                          pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=2)

    optimizer = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9, 0.999), weight_decay=1e-2)
    lr_sched = get_cosine_schedule_with_warmup(optimizer, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, optimizer, train_dl, lr_sched = accelerator.prepare(unet, optimizer, train_dl, lr_sched)

    # ── resume: HF (guarded run's own state) only — this is a fresh experiment ──
    latest = os.path.join(OUTPUT_DIR, "latest")
    start_step = 0; latent_scale = None; resume_dir = None
    if is_main and download_resume_from_hf(latest):
        resume_dir = latest; log("  Resume source: HF (flow_guarded_resume).")
    accelerator.wait_for_everyone()
    if resume_dir and os.path.exists(os.path.join(resume_dir, "meta.pt")):
        meta = torch.load(os.path.join(resume_dir, "meta.pt"), map_location="cpu")
        start_step = meta["step"]; latent_scale = meta["latent_scale"]
        accelerator.load_state(resume_dir)
        log(f"  RESUMED @ step {start_step} (latent_scale={latent_scale:.4f}).")
    else:
        if is_main:
            # read from champion's own config rather than recomputing/hardcoding
            champ_cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/bridge_config.json")))
            latent_scale = float(champ_cfg["latent_scale"])
            log(f"  latent_scale from {INIT_TAG} config: {latent_scale:.4f}")
        ls = torch.tensor(latent_scale if latent_scale else 1.0, device=device)
        latent_scale = float(accelerator.gather(ls).mean().item()) if accelerator.num_processes > 1 else float(ls)

    ema_unet = AveragedModel(accelerator.unwrap_model(unet),
                             multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY)).to(device)

    def enc(imgs):
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            return vae.encode_latent(imgs.float()) * latent_scale

    @torch.no_grad()
    def validate(step):
        sub = random.Random(7).sample(val_pairs, min(N_VAL, len(val_pairs)))
        vdl = DataLoader(PairDataset(sub, IMG_SIZE, False), batch_size=8, num_workers=2)
        P, S, L = [], [], []
        fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
        m = ema_unet.module; m.eval()
        for b in vdl:
            opt = b["opt"].to(device)
            x0 = enc(b["sar"].to(device))
            cond = get_cond(b["season"], b["terrain"], embed_cache).to(device)
            x1 = flow_sample(m, x0, cond, SAMPLE_STEPS, dtype)
            with torch.autocast("cuda", enabled=False):
                rec = vae.decode_latent((x1 / latent_scale).float()).clamp(-1, 1)
            P.append(10 * np.log10(4.0 / F.mse_loss(rec, opt).item()))
            S.append(ssim_fn(rec, opt).item()); L.append(lpips_fn(rec, opt).mean().item())
            fid.update(denorm01(opt), real=True); fid.update(denorm01(rec), real=False)
        fidv = float(fid.compute())
        m.train()
        log(f"\n[VAL step {step} | EMA] PSNR {np.mean(P):.3f} | SSIM {np.mean(S):.4f} | "
            f"LPIPS {np.mean(L):.4f} | FID~ {fidv:.2f} (n={len(sub)})")
        if step == GATE_STEP:
            ok_v = ema_v <= GATE_MAX_V; ok_p = np.mean(P) >= GATE_MIN_PSNR
            log(f"  GATE @ {GATE_STEP}: v={ema_v:.4f} (need <= {GATE_MAX_V}) | "
                f"PSNR={np.mean(P):.2f} (need >= {GATE_MIN_PSNR}) -> "
                f"{'PASS, continue to 120k' if (ok_v and ok_p) else 'FAIL — recommend stopping'}")
        log("")

    def local_save(step):
        accelerator.wait_for_everyone()
        if not is_main: return
        if os.path.exists(latest): shutil.rmtree(latest)
        os.makedirs(latest, exist_ok=True)
        accelerator.save_state(latest)
        torch.save({"step": step, "latent_scale": latent_scale}, os.path.join(latest, "meta.pt"))
        log(f"  local safety checkpoint (overwritten) @ step {step}")

    def final_save(step):
        accelerator.wait_for_everyone()
        if not is_main: return
        tmp = os.path.join(OUTPUT_DIR, "_final"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k, v in ema_unet.module.state_dict().items()}
        save_file(sd, os.path.join(tmp, "unet.safetensors"))
        with open(os.path.join(tmp, "bridge_config.json"), "w") as f:
            json.dump({"latent_scale": latent_scale, "method": "flow_matching", "ema": True,
                       "guarded": True, "sample_steps": SAMPLE_STEPS, "img_size": IMG_SIZE,
                       "final_step": step}, f, indent=2)
        push_hf(tmp, "flow_guarded_final")
        shutil.rmtree(tmp, ignore_errors=True)
        log(f"  FINAL EMA model -> HF/flow_guarded_final @ step {step}")

    global_step = start_step
    ema_v = ema_p = 0.0; a = 0.98
    t0 = time.time(); last_logged = start_step
    n_guarded = 0
    spe = len(train_dl); epochs = (NUM_TRAIN_STEPS + spe - 1) // spe
    unet.train()
    log(f"\n>>> Guarded flow-matching {start_step} -> {NUM_TRAIN_STEPS}. Init from champion. "
        f"LR={LR}. Perceptual ramps in {PIXEL_START}->{PIXEL_START+PIXEL_RAMP}. "
        f"GATE at {GATE_STEP}: v<={GATE_MAX_V}, PSNR>={GATE_MIN_PSNR}.\n")

    for epoch in range(epochs):
        for batch in train_dl:
            opt_img = batch["opt"].to(device)
            x0 = enc(batch["sar"].to(device))
            x1 = enc(opt_img)
            cond = get_cond(batch["season"], batch["terrain"], embed_cache, COND_DROPOUT).to(device)
            t = torch.rand(x0.shape[0], device=device)
            tv = t.view(-1, 1, 1, 1)
            x_t = (1 - tv) * x0 + tv * x1
            v_target = x1 - x0
            tb = (t * 999).long()

            with accelerator.accumulate(unet):
                v_pred = unet(torch.cat([x_t, x0], 1).to(dtype), tb,
                              encoder_hidden_states=cond.to(dtype)).sample
                loss_v = F.mse_loss(v_pred.float(), v_target.float())

                # ---- SPIKE GUARD: skip pathological steps entirely ----
                if global_step > GUARD_WARMUP and ema_v > 0 and \
                   loss_v.item() > SPIKE_GUARD_MULT * max(ema_v, 0.02):
                    optimizer.zero_grad(set_to_none=True)
                    n_guarded += 1
                    if is_main and n_guarded % 10 == 1:
                        log(f"  [guard] skipped step {global_step} (v={loss_v.item():.3f}, "
                            f"ema={ema_v:.3f}, total guarded={n_guarded})")
                    continue

                loss = loss_v; loss_p_val = 0.0
                if global_step >= PIXEL_START:
                    w = min(1.0, (global_step - PIXEL_START) / PIXEL_RAMP)   # ramped weight
                    x1p = (x_t + (1 - tv) * v_pred.float()) / latent_scale
                    with torch.autocast("cuda", enabled=False):
                        img_pred, f_pred = hooked.decode(vae, x1p.float())
                        img_pred = img_pred.clamp(-1, 1)
                        with torch.no_grad():
                            _, f_real = hooked.decode(vae, (x1 / latent_scale).float())
                        l_l1  = F.l1_loss(img_pred, opt_img)
                        l_lp  = lpips_fn(img_pred, opt_img).mean()
                        l_lpl = latent_perceptual_loss(f_pred, f_real)
                    loss_p = w * (LAMBDA_L1*l_l1 + LAMBDA_LPIPS*l_lp + LAMBDA_LPL*l_lpl)
                    loss = loss + loss_p; loss_p_val = float(loss_p.item())

                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), 1.0)
                optimizer.step(); lr_sched.step(); optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                ema_unet.update_parameters(accelerator.unwrap_model(unet))
                global_step += 1
                ema_v = a*ema_v + (1-a)*loss_v.item(); ema_p = a*ema_p + (1-a)*loss_p_val
                if is_main and global_step % STEP_LOG_EVERY == 0:
                    rate = (global_step - last_logged) / max(time.time() - t0, 1e-9)
                    t0, last_logged = time.time(), global_step
                    log(f"step {global_step}/{NUM_TRAIN_STEPS} | v {ema_v:.4f} | "
                        f"pix+lpl {ema_p:.4f} | guarded={n_guarded} | {rate:.2f} it/s")
                if global_step % VAL_EVERY == 0 and is_main: validate(global_step)
                if global_step % LOCAL_SAVE_EVERY == 0: local_save(global_step)
            if global_step >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if global_step >= NUM_TRAIN_STEPS: break

    if is_main: validate(global_step)
    final_save(global_step)
    accelerator.end_training()
    log(f"\nStage 3b (guarded) complete. {n_guarded} steps skipped by guard. "
        f"EMA model on HF/flow_guarded_final.")


if __name__ == "__main__":
    main()

Overwriting /content/stage3b_flow_256.py


In [ ]:

!grep -n "SPIKE GUARD" /content/stage3b_flow_256.py   # sanity: confirm guard is in the file about to run
!python -u /content/stage3b_flow_256.py

415:                # ---- SPIKE GUARD: skip pathological steps entirely ----
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading components...
  Adapted VAE loaded from adapted16_final
  Built 25 CLIP embeddings.
  UNet initiali

In [ ]:
%%writefile /content/stage3b_flow_256.py
"""
SAR-INTEL Stage 3b FINAL (guarded, EMA-fixed, local-resume-first) — flow-matching
bridge, 16ch latent @ 256px.

Fixes in this version:
  1. EMA bias-correction warmup (from previous fix) — validate() reports BOTH
     raw and EMA, gate decided on RAW.
  2. RESUME ORDER: checks LOCAL disk first (cheap, no download) before ever
     touching HF. A script crash (not a full runtime restart) leaves local
     /content state intact — this was previously skipped, causing an
     unnecessary restart from step 0.
  3. NaN/Inf guard immediately before backward() — catches non-finite loss
     (from a drifted decode inside the perceptual/LPL block) that the
     magnitude-based spike-guard alone doesn't catch, which is the likely
     cause of the crash at step ~22700.
  4. local_save() remains LOCAL-ONLY — no periodic HF push, as requested.
     Only the final EMA model is pushed to HF once, at the very end.
"""
import os, gc, json, time, random, warnings, shutil
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from safetensors.torch import save_file, load_file
import lpips as lpips_lib
import pandas as pd
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def log(*a): print(*a, flush=True)

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT    = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR   = "/content/stage3b_flow_guarded"
SD_MODEL_ID  = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID   = os.environ["HF_REPO_ID"]
HF_TOKEN     = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG      = "adapted16_final"
INIT_TAG     = "bridge_final"
RESUME_TAG   = "flow_guarded_resume"

IMG_SIZE     = 256
SEASONS      = ("spring", "summer", "fall", "winter")
VAL_FRACTION = 0.05
SEED         = 42
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16

NUM_TRAIN_STEPS = 120_000
BATCH_SIZE   = 4
LR           = 3e-5
LR_WARMUP    = 1_000
COND_DROPOUT = 0.10

PIXEL_START  = 5_000
PIXEL_RAMP   = 5_000
LAMBDA_LPIPS = 0.5
LAMBDA_L1    = 0.25
LAMBDA_LPL   = 0.5
SPIKE_GUARD_MULT = 10.0
GUARD_WARMUP = 1_000

EMA_DECAY    = 0.9999
SAMPLE_STEPS = 10
VAL_EVERY    = 5_000
N_VAL        = 200
LOCAL_SAVE_EVERY = 20_000
STEP_LOG_EVERY   = 500

GATE_STEP, GATE_MAX_V, GATE_MIN_PSNR = 30_000, 0.10, 12.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEASON_DESC = {"spring": "spring", "summer": "summer", "fall": "autumn", "winter": "winter"}
TERRAINS = ("tropical", "temperate", "arctic", "arid", "coastal", "urban")

def map_region_to_terrain(region):
    r = (region or "").lower()
    if any(k in r for k in ("trop", "rainforest", "amazon", "jungle")): return "tropical"
    if any(k in r for k in ("arctic", "tundra", "snow", "ice", "polar")): return "arctic"
    if any(k in r for k in ("desert", "arid", "sahara", "dry")): return "arid"
    if any(k in r for k in ("coast", "beach", "shore", "marine", "sea")): return "coastal"
    if any(k in r for k in ("urban", "city", "metro")): return "urban"
    return "temperate"


def collect_pairs(root, seasons):
    root_str = Path(root).as_posix(); pairs = []
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for col in ("s1_fileName", "s2_fileName"):
            df[col] = df[col].astype(str).str.replace("\\", "/", regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            pairs.append({"sar": f"{root_str}/{row['s1_fileName']}",
                          "opt": f"{root_str}/{row['s2_fileName']}",
                          "season": season, "terrain": map_region_to_terrain(row["region"])})
    random.shuffle(pairs); return pairs


class PairDataset(Dataset):
    def __init__(self, pairs, img_size=256, augment=True):
        self.pairs = pairs; self.img_size = img_size; self.augment = augment
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0
    def __getitem__(self, idx):
        p = self.pairs[idx]
        sar = self._load(p["sar"]); opt = self._load(p["opt"])
        if self.augment:
            if random.random() > 0.5: sar, opt = torch.flip(sar, [2]), torch.flip(opt, [2])
            if random.random() > 0.5: sar, opt = torch.flip(sar, [1]), torch.flip(opt, [1])
            k = random.randint(0, 3)
            if k: sar, opt = torch.rot90(sar, k, [1, 2]), torch.rot90(opt, k, [1, 2])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}


class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CH, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),  nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf*2, 3, 2, 1),   nn.GroupNorm(8, nf*2), nn.SiLU(),
            nn.Conv2d(nf*2, nf*4, 3, 2, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, nf*4, 3, 1, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, out_ch, 3, 1, 1))
    def forward(self, x): return self.net(x)


class AdaptedVAE(nn.Module):
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae = base_vae; self.detail_enc = DetailEncoder()
        old = base_vae.decoder.conv_in
        base_vae.decoder.conv_in = nn.Conv2d(TOTAL_CH, old.out_channels, old.kernel_size,
                                             old.stride, old.padding, bias=(old.bias is not None))
    @torch.no_grad()
    def encode_latent(self, x):
        z = self.base_vae.encode(x).latent_dist.mode()
        return torch.cat([self.base_vae.post_quant_conv(z), self.detail_enc(x)], dim=1)
    def decode_grad(self, lat):
        return self.base_vae.decoder(lat)
    @torch.no_grad()
    def decode_latent(self, lat):
        return self.base_vae.decoder(lat)


class HookedDecoder:
    def __init__(self, vae):
        self.feats = []; self.capture = False
        for blk in vae.base_vae.decoder.up_blocks:
            blk.register_forward_hook(
                lambda m, i, o: self.feats.append(o) if self.capture else None)
    def decode(self, vae, lat):
        self.capture = True; self.feats = []
        img = vae.base_vae.decoder(lat)
        feats = list(self.feats); self.capture = False; self.feats = []
        return img, feats

def latent_perceptual_loss(f_pred, f_real):
    loss = 0.0
    for fp, fr in zip(f_pred, f_real):
        loss = loss + F.l1_loss(fp.float(), fr.float().detach())
    return loss / max(len(f_pred), 1)


def load_adapted_vae():
    base_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(base_vae)
    bp = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/base_vae.pth")
    de = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/detail_encoder.pth")
    vae.base_vae.load_state_dict(torch.load(bp, map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(de, map_location="cpu"))
    vae = vae.to(device).eval().float(); vae.requires_grad_(False)
    log("  Adapted VAE loaded from", VAE_TAG)
    return vae


def build_embed_cache():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    txt.requires_grad_(False); cache = {}
    @torch.no_grad()
    def embed(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s, ter)] = embed(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = embed(""); del txt; gc.collect(); torch.cuda.empty_cache()
    log(f"  Built {len(cache)} CLIP embeddings.")
    return cache


def build_unet_from_champion():
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    sd_path = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{INIT_TAG}/unet.safetensors")
    unet.load_state_dict(load_file(sd_path))
    log(f"  UNet initialized from {INIT_TAG} (champion, 16.73 dB baseline).")
    return unet


def get_cond(seasons, terrains, cache, drop=0.0):
    out = []
    for s, ter in zip(seasons, terrains):
        out.append(cache["null"] if (drop > 0 and random.random() < drop)
                   else cache.get((s, ter), cache["null"]))
    return torch.stack(out)


@torch.no_grad()
def flow_sample(unet, x0, cond, n_steps, dtype):
    x = x0.clone(); dt = 1.0 / n_steps; B = x0.shape[0]
    for i in range(n_steps):
        tb = torch.full((B,), int(i * dt * 999), device=x0.device, dtype=torch.long)
        v = unet(torch.cat([x, x0], 1).to(dtype), tb,
                 encoder_hidden_states=cond.to(dtype)).sample.float()
        x = x + dt * v
    return x


def push_hf(folder, tag):
    if not HF_TOKEN: return
    try:
        HfApi().upload_folder(folder_path=folder, path_in_repo=tag, repo_id=HF_REPO_ID,
                              repo_type="model", token=HF_TOKEN)
        log(f"  HF push -> {tag}")
    except Exception as e:
        log(f"  HF push failed: {e}")


def download_resume_from_hf(dest):
    try:
        files = HfApi().list_repo_files(HF_REPO_ID, token=HF_TOKEN)
        if f"{RESUME_TAG}/meta.pt" not in files:
            log("  no resume state on HF"); return False
        snapshot_download(repo_id=HF_REPO_ID, allow_patterns=f"{RESUME_TAG}/*",
                          local_dir="/content/_flow_guard_dl", token=HF_TOKEN)
        src = os.path.join("/content/_flow_guard_dl", RESUME_TAG)
        if os.path.exists(dest): shutil.rmtree(dest)
        shutil.copytree(src, dest); return True
    except Exception as e:
        log(f"  resume check failed ({e})"); return False


def denorm01(t): return (t.clamp(-1, 1) + 1) / 2


def main():
    accelerator = Accelerator(mixed_precision="bf16", project_dir=OUTPUT_DIR)
    set_seed(SEED); is_main = accelerator.is_main_process; dtype = torch.bfloat16
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    log("Loading components...")
    vae = load_adapted_vae(); embed_cache = build_embed_cache()
    unet = build_unet_from_champion()
    hooked = HookedDecoder(vae)
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    train_pairs, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in all_pairs], random_state=SEED)
    log(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,}")

    train_ds = PairDataset(train_pairs, IMG_SIZE, True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                          pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=2)

    optimizer = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9, 0.999), weight_decay=1e-2)
    lr_sched = get_cosine_schedule_with_warmup(optimizer, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, optimizer, train_dl, lr_sched = accelerator.prepare(unet, optimizer, train_dl, lr_sched)

    # ── FIX: resume check LOCAL disk FIRST (cheap, survives a plain process
    #    crash), fall back to HF only if local is genuinely absent ──────────
    latest = os.path.join(OUTPUT_DIR, "latest")
    start_step = 0; latent_scale = None; resume_dir = None

    if os.path.exists(os.path.join(latest, "meta.pt")):
        resume_dir = latest
        log("  Resume source: LOCAL disk (found existing checkpoint).")
    elif is_main and download_resume_from_hf(latest):
        resume_dir = latest
        log("  Resume source: HF (flow_guarded_resume).")
    else:
        log("  No local or HF checkpoint found — starting fresh.")

    accelerator.wait_for_everyone()
    if resume_dir and os.path.exists(os.path.join(resume_dir, "meta.pt")):
        meta = torch.load(os.path.join(resume_dir, "meta.pt"), map_location="cpu")
        start_step = meta["step"]; latent_scale = meta["latent_scale"]
        accelerator.load_state(resume_dir)
        log(f"  RESUMED @ step {start_step} (latent_scale={latent_scale:.4f}).")
    else:
        if is_main:
            champ_cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/bridge_config.json")))
            latent_scale = float(champ_cfg["latent_scale"])
            log(f"  latent_scale from {INIT_TAG} config: {latent_scale:.4f}")
        ls = torch.tensor(latent_scale if latent_scale else 1.0, device=device)
        latent_scale = float(accelerator.gather(ls).mean().item()) if accelerator.num_processes > 1 else float(ls)

    ema_unet = AveragedModel(accelerator.unwrap_model(unet),
                             multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY)).to(device)
    def ema_decay_at(step):
        return min(EMA_DECAY, (step + 1) / (step + 10))

    def enc(imgs):
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            return vae.encode_latent(imgs.float()) * latent_scale

    @torch.no_grad()
    def validate(step):
        sub = random.Random(7).sample(val_pairs, min(N_VAL, len(val_pairs)))
        vdl = DataLoader(PairDataset(sub, IMG_SIZE, False), batch_size=8, num_workers=2)
        raw_psnr = None
        for tag, m in (("RAW", accelerator.unwrap_model(unet)), ("EMA", ema_unet.module)):
            P, S, L = [], [], []
            fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
            was_training = m.training; m.eval()
            for b in vdl:
                opt = b["opt"].to(device)
                x0 = enc(b["sar"].to(device))
                cond = get_cond(b["season"], b["terrain"], embed_cache).to(device)
                x1 = flow_sample(m, x0, cond, SAMPLE_STEPS, dtype)
                with torch.autocast("cuda", enabled=False):
                    rec = vae.decode_latent((x1 / latent_scale).float()).clamp(-1, 1)
                P.append(10 * np.log10(4.0 / F.mse_loss(rec, opt).item()))
                S.append(ssim_fn(rec, opt).item()); L.append(lpips_fn(rec, opt).mean().item())
                fid.update(denorm01(opt), real=True); fid.update(denorm01(rec), real=False)
            if was_training: m.train()
            fidv = float(fid.compute())
            log(f"[VAL step {step} | {tag}] PSNR {np.mean(P):.3f} | SSIM {np.mean(S):.4f} | "
                f"LPIPS {np.mean(L):.4f} | FID~ {fidv:.2f} (n={len(sub)})")
            if tag == "RAW": raw_psnr = np.mean(P)
        if step == GATE_STEP:
            ok_v = ema_v <= GATE_MAX_V; ok_p = raw_psnr >= GATE_MIN_PSNR
            log(f"  GATE @ {GATE_STEP} (decided on RAW): v={ema_v:.4f} (need <= {GATE_MAX_V}) | "
                f"PSNR={raw_psnr:.2f} (need >= {GATE_MIN_PSNR}) -> "
                f"{'PASS, continue to 120k' if (ok_v and ok_p) else 'FAIL — recommend stopping'}")
        log("")

    def local_save(step):
        # LOCAL ONLY — no HF push, per explicit instruction.
        accelerator.wait_for_everyone()
        if not is_main: return
        if os.path.exists(latest): shutil.rmtree(latest)
        os.makedirs(latest, exist_ok=True)
        accelerator.save_state(latest)
        torch.save({"step": step, "latent_scale": latent_scale}, os.path.join(latest, "meta.pt"))
        log(f"  local checkpoint (overwritten, LOCAL ONLY) @ step {step}")

    def final_save(step):
        accelerator.wait_for_everyone()
        if not is_main: return
        tmp = os.path.join(OUTPUT_DIR, "_final"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k, v in ema_unet.module.state_dict().items()}
        save_file(sd, os.path.join(tmp, "unet.safetensors"))
        with open(os.path.join(tmp, "bridge_config.json"), "w") as f:
            json.dump({"latent_scale": latent_scale, "method": "flow_matching", "ema": True,
                       "guarded": True, "sample_steps": SAMPLE_STEPS, "img_size": IMG_SIZE,
                       "final_step": step}, f, indent=2)
        push_hf(tmp, "flow_guarded_final")     # only push at the very end
        shutil.rmtree(tmp, ignore_errors=True)
        log(f"  FINAL EMA model -> HF/flow_guarded_final @ step {step}")

    global_step = start_step
    ema_v = ema_p = 0.0; a = 0.98
    t0 = time.time(); last_logged = start_step
    n_guarded = 0; n_nan = 0
    spe = len(train_dl); epochs = (NUM_TRAIN_STEPS + spe - 1) // spe
    unet.train()
    log(f"\n>>> Guarded flow-matching {start_step} -> {NUM_TRAIN_STEPS}. Init from champion. "
        f"LR={LR}. Perceptual ramps in {PIXEL_START}->{PIXEL_START+PIXEL_RAMP}. "
        f"EMA bias-corrected. NaN-guard active. GATE at {GATE_STEP}: "
        f"v<={GATE_MAX_V}, PSNR(RAW)>={GATE_MIN_PSNR}.\n")

    for epoch in range(epochs):
        for batch in train_dl:
            opt_img = batch["opt"].to(device)
            x0 = enc(batch["sar"].to(device))
            x1 = enc(opt_img)
            cond = get_cond(batch["season"], batch["terrain"], embed_cache, COND_DROPOUT).to(device)
            t = torch.rand(x0.shape[0], device=device)
            tv = t.view(-1, 1, 1, 1)
            x_t = (1 - tv) * x0 + tv * x1
            v_target = x1 - x0
            tb = (t * 999).long()

            with accelerator.accumulate(unet):
                v_pred = unet(torch.cat([x_t, x0], 1).to(dtype), tb,
                              encoder_hidden_states=cond.to(dtype)).sample
                loss_v = F.mse_loss(v_pred.float(), v_target.float())

                # ---- magnitude spike-guard ----
                if global_step > GUARD_WARMUP and ema_v > 0 and \
                   loss_v.item() > SPIKE_GUARD_MULT * max(ema_v, 0.02):
                    optimizer.zero_grad(set_to_none=True)
                    n_guarded += 1
                    if is_main and n_guarded % 10 == 1:
                        log(f"  [guard] skipped step {global_step} (v={loss_v.item():.3f}, "
                            f"ema={ema_v:.3f}, total guarded={n_guarded})")
                    continue

                loss = loss_v; loss_p_val = 0.0
                if global_step >= PIXEL_START:
                    w = min(1.0, (global_step - PIXEL_START) / PIXEL_RAMP)
                    x1p = (x_t + (1 - tv) * v_pred.float()) / latent_scale
                    with torch.autocast("cuda", enabled=False):
                        img_pred, f_pred = hooked.decode(vae, x1p.float())
                        img_pred = img_pred.clamp(-1, 1)
                        with torch.no_grad():
                            _, f_real = hooked.decode(vae, (x1 / latent_scale).float())
                        l_l1  = F.l1_loss(img_pred, opt_img)
                        l_lp  = lpips_fn(img_pred, opt_img).mean()
                        l_lpl = latent_perceptual_loss(f_pred, f_real)
                    loss_p = w * (LAMBDA_L1*l_l1 + LAMBDA_LPIPS*l_lp + LAMBDA_LPL*l_lpl)
                    loss = loss + loss_p; loss_p_val = float(loss_p.item())

                # ---- FIX: NaN/Inf guard, right before backward ----
                if not torch.isfinite(loss):
                    optimizer.zero_grad(set_to_none=True)
                    n_nan += 1
                    log(f"  [nan-guard] skipped step {global_step} (non-finite loss, total={n_nan})")
                    continue

                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), 1.0)
                optimizer.step(); lr_sched.step(); optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                ema_unet.avg_fn = get_ema_multi_avg_fn(ema_decay_at(global_step))
                ema_unet.update_parameters(accelerator.unwrap_model(unet))
                global_step += 1
                ema_v = a*ema_v + (1-a)*loss_v.item(); ema_p = a*ema_p + (1-a)*loss_p_val
                if is_main and global_step % STEP_LOG_EVERY == 0:
                    rate = (global_step - last_logged) / max(time.time() - t0, 1e-9)
                    t0, last_logged = time.time(), global_step
                    log(f"step {global_step}/{NUM_TRAIN_STEPS} | v {ema_v:.4f} | "
                        f"pix+lpl {ema_p:.4f} | guarded={n_guarded} | nan={n_nan} | {rate:.2f} it/s")
                if global_step % VAL_EVERY == 0 and is_main: validate(global_step)
                if global_step % LOCAL_SAVE_EVERY == 0: local_save(global_step)
            if global_step >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if global_step >= NUM_TRAIN_STEPS: break

    if is_main: validate(global_step)
    final_save(global_step)
    accelerator.end_training()
    log(f"\nStage 3b (guarded) complete. {n_guarded} magnitude-guarded, {n_nan} nan-guarded. "
        f"EMA model on HF/flow_guarded_final.")


if __name__ == "__main__":
    main()

Overwriting /content/stage3b_flow_256.py


In [ ]:
!python -u /content/stage3b_flow_256.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading components...
  Adapted VAE loaded from adapted16_final
  Built 25 CLIP embeddings.
  UNet initialized from bridge_final (champion, 16.73 dB baseline).
Setting up [LPIPS] percep

In [ ]:
%%writefile /content/phy_s2o_eval.py
"""
Phy-S2O evaluation on the champion (bridge_final):
  DOIM  — Downstream Operational Invariance: frozen SAM (Segment Anything,
          optical-trained) generates masks on GT optical, translated optical,
          and RAW SAR. Mask-matched IoU retention:
            DOIM(x) = mean best-match IoU of masks(x) against masks(GT).
          KEY COMPARISON: DOIM(SAR) vs DOIM(translated) = utility added by
          translation. This is the paper's motivation, measured.
  SEFS  — Spectral-Edge Fidelity: fraction of strong edges in the output that
          have local backscatter-variance support in the SAR input. Computed on
          GT as well (calibration: GT itself has unsupported edges — documents
          the metric's known blind spot honestly).
Eval-only. No training risk. ~45-90 min for N_EVAL=200 on A100.
"""
import os, gc, json, random, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer, pipeline
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
VAE_TAG, BRIDGE_TAG = "adapted16_final", "bridge_final"
SAM_MODEL = "facebook/sam-vit-base"          # frozen, optical-trained
IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16
N_EVAL = 200
EDGE_PCT = 90        # "strong edge" = top-10% gradient magnitude
SUPPORT_PCT = 60     # SAR support = local std above its 60th percentile
device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    return ps

class DS(Dataset):
    def __init__(self, pairs): self.p = pairs
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        return {"sar": self._l(p["sar"]), "opt": self._l(p["opt"]),
                "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=DETAIL_CH, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def to_pil(t):  # (-1,1) CHW -> PIL
    a = ((t.clamp(-1,1)+1)/2*255).byte().permute(1,2,0).cpu().numpy()
    return Image.fromarray(a)

def masks_to_arrays(sam_out, min_area=64):
    return [np.asarray(m, dtype=bool) for m in sam_out["masks"]
            if np.asarray(m).sum() >= min_area]

def doim_score(masks_ref, masks_test):
    """Mean best-match IoU of reference (GT) masks against test masks."""
    if not masks_ref: return None
    if not masks_test: return 0.0
    ious = []
    for mr in masks_ref:
        best = 0.0
        for mt in masks_test:
            inter = np.logical_and(mr, mt).sum()
            if inter == 0: continue
            union = np.logical_or(mr, mt).sum()
            best = max(best, inter/union)
        ious.append(best)
    return float(np.mean(ious))

def grad_mag(gray):  # HxW float
    gx = np.abs(np.diff(gray, axis=1, append=gray[:, -1:]))
    gy = np.abs(np.diff(gray, axis=0, append=gray[-1:, :]))
    return np.sqrt(gx**2 + gy**2)

def local_std(gray, k=7):
    t = torch.from_numpy(gray)[None, None]
    mean = F.avg_pool2d(t, k, 1, k//2)
    sq = F.avg_pool2d(t**2, k, 1, k//2)
    return (sq - mean**2).clamp(min=0).sqrt()[0, 0].numpy()

def sefs(out_img01, sar_gray01):
    """Fraction of strong output edges with SAR backscatter-variance support."""
    lum = out_img01.mean(axis=2)
    g = grad_mag(lum)
    edges = g >= np.percentile(g, EDGE_PCT)
    support = local_std(sar_gray01) >= np.percentile(local_std(sar_gray01), SUPPORT_PCT)
    n_edges = edges.sum()
    return float((edges & support).sum() / n_edges) if n_edges else None

def main():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    # ── champion + VAE (known-good loading pattern) ──
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(bv)
    vae.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    vae = vae.to(device).eval().float().requires_grad_(False)

    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{BRIDGE_TAG}/bridge_config.json")))
    SCALE = cfg["latent_scale"]
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{BRIDGE_TAG}/unet.safetensors")))
    unet = unet.to(device).eval().float().requires_grad_(False)
    log(f"Champion loaded (step {cfg.get('final_step')}, scale {SCALE:.4f}).")

    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    emb = {}
    @torch.no_grad()
    def E(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in ("tropical","temperate","arctic","arid","coastal","urban"):
            emb[(s,ter)] = E(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    emb["null"] = E(""); del txt; gc.collect(); torch.cuda.empty_cache()

    i = np.arange(cfg["T"])
    eta = torch.tensor(cfg["eta_1"]*(cfg["eta_T"]/cfg["eta_1"])**(i/(cfg["T"]-1)),
                       dtype=torch.float32).to(device)
    @torch.no_grad()
    def sample(y, cond):
        B = y.shape[0]; x = y + cfg["kappa"]*torch.sqrt(eta[-1])*torch.randn_like(y)
        for t in reversed(range(cfg["T"])):
            tb = torch.full((B,), int(t*1000/cfg["T"]), device=device, dtype=torch.long)
            x0 = unet(torch.cat([x,y],1), tb, encoder_hidden_states=cond).sample
            if t > 0:
                ep = eta[t-1]; x = x0 + ep*(y-x0) + cfg["kappa"]*torch.sqrt(ep)*torch.randn_like(x)
            else: x = x0
        return x

    # ── frozen SAM (the downstream "operational" model) ──
    log("Loading frozen SAM mask-generation pipeline...")
    sam = pipeline("mask-generation", model=SAM_MODEL, device=0)

    pairs = collect(DATA_ROOT)
    _, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    sub = random.Random(SEED).sample(va, N_EVAL)
    log(f"Evaluating Phy-S2O on {len(sub)} val images.\n")

    per_terrain = {}
    D_trans, D_sar, S_trans, S_gt, kept = [], [], [], [], 0
    dl = DataLoader(DS(sub), batch_size=4, num_workers=2)
    idx = 0
    for b in dl:
        opt = b["opt"].to(device); sar = b["sar"].to(device)
        y = vae.enc(sar.float())*SCALE
        c = torch.stack([emb.get((s,t), emb["null"]) for s,t in zip(b["season"],b["terrain"])]).to(device)
        rec = vae.dec((sample(y, c)/SCALE)).clamp(-1,1)
        for j in range(rec.shape[0]):
            gt_pil, tr_pil, sar_pil = to_pil(opt[j]), to_pil(rec[j]), to_pil(sar[j])
            try:
                m_gt = masks_to_arrays(sam(gt_pil, points_per_batch=32))
                m_tr = masks_to_arrays(sam(tr_pil, points_per_batch=32))
                m_sr = masks_to_arrays(sam(sar_pil, points_per_batch=32))
            except Exception as e:
                log(f"  [skip img {idx}] SAM error: {e}"); idx += 1; continue
            d_tr = doim_score(m_gt, m_tr); d_sr = doim_score(m_gt, m_sr)
            gt01 = np.array(gt_pil, np.float32)/255.
            tr01 = np.array(tr_pil, np.float32)/255.
            sg = np.array(sar_pil.convert("L"), np.float32)/255.
            s_tr = sefs(tr01, sg); s_gt = sefs(gt01, sg)
            if d_tr is not None:
                D_trans.append(d_tr); D_sar.append(d_sr if d_sr is not None else 0.0)
                ter = b["terrain"][j]
                per_terrain.setdefault(ter, []).append((d_tr, d_sr or 0.0))
            if s_tr is not None: S_trans.append(s_tr)
            if s_gt is not None: S_gt.append(s_gt)
            kept += 1; idx += 1
        if kept and kept % 40 == 0:
            log(f"  [{kept}/{N_EVAL}] DOIM(trans) {np.mean(D_trans):.3f} | "
                f"DOIM(SAR) {np.mean(D_sar):.3f} | SEFS(trans) {np.mean(S_trans):.3f}")

    log("\n" + "="*64)
    log("PHY-S2O RESULTS  (frozen SAM = downstream optical model)")
    log(f"  DOIM — GT-mask IoU retention:")
    log(f"    on TRANSLATED optical : {np.mean(D_trans):.4f}")
    log(f"    on RAW SAR (floor)    : {np.mean(D_sar):.4f}")
    log(f"    >>> UTILITY ADDED BY TRANSLATION: +{np.mean(D_trans)-np.mean(D_sar):.4f} "
        f"({(np.mean(D_trans)/max(np.mean(D_sar),1e-6)-1)*100:.0f}% relative)")
    log(f"  SEFS — edge-support fraction:")
    log(f"    translated : {np.mean(S_trans):.4f}")
    log(f"    GT optical : {np.mean(S_gt):.4f}   <- calibration: GT's own unsupported-edge")
    log(f"      rate documents the metric's blind spot (real optical edges without SAR support)")
    log(f"  Per-terrain DOIM (translated | SAR floor):")
    for t, v in sorted(per_terrain.items()):
        a = np.array(v)
        log(f"    {t:10s} {a[:,0].mean():.3f} | {a[:,1].mean():.3f}   (n={len(v)})")
    log("="*64)
    json.dump({"doim_translated": float(np.mean(D_trans)), "doim_sar": float(np.mean(D_sar)),
               "sefs_translated": float(np.mean(S_trans)), "sefs_gt": float(np.mean(S_gt)),
               "per_terrain": {t: [float(np.array(v)[:,0].mean()), float(np.array(v)[:,1].mean())]
                               for t, v in per_terrain.items()},
               "n": kept},
              open("/content/phy_s2o_results.json", "w"), indent=2)
    log("saved -> /content/phy_s2o_results.json")

if __name__ == "__main__":
    main()

Writing /content/phy_s2o_eval.py


In [ ]:
!python -u /content/phy_s2o_eval.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Champion loaded (step 150000, scale 1.4010).
Loading frozen SAM mask-generation pipeline...
Evaluating Phy-S2O on 200 val images.

[transformers] You seem to be using the pipelines sequ

In [ ]:
%%writefile /content/doim_ceiling_check.py
"""Self-consistency ceiling: DOIM(GT, GT) via two independent SAM runs on the
SAME ground-truth image. If this ceiling is near the floor already measured
(~0.10), DOIM has no dynamic range on this data and the translated≈SAR result
is uninterpretable. If it's much higher, the near-tie is real."""
import os, random, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np
from pathlib import Path
from PIL import Image
from transformers import pipeline
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
SAM_MODEL = "facebook/sam-vit-base"
IMG_SIZE, SEED, VAL_FRACTION, N = 256, 42, 0.05, 60
SEASONS = ("spring","summer","fall","winter")

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        df["s2_fileName"] = df["s2_fileName"].astype(str).str.replace("\\","/",regex=False)
        for _, row in df.iterrows():
            ps.append({"opt": f"{rs}/{row['s2_fileName']}", "terrain": map_terrain(row["region"])})
    return ps

def masks_to_arrays(sam_out, min_area=64):
    return [np.asarray(m, dtype=bool) for m in sam_out["masks"] if np.asarray(m).sum() >= min_area]

def doim_score(masks_ref, masks_test):
    if not masks_ref: return None
    if not masks_test: return 0.0
    ious = []
    for mr in masks_ref:
        best = 0.0
        for mt in masks_test:
            inter = np.logical_and(mr, mt).sum()
            if inter == 0: continue
            best = max(best, inter/np.logical_or(mr, mt).sum())
        ious.append(best)
    return float(np.mean(ious))

def main():
    random.seed(SEED); np.random.seed(SEED)
    sam = pipeline("mask-generation", model=SAM_MODEL, device=0)
    pairs = collect(DATA_ROOT)
    _, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[p["terrain"] for p in pairs], random_state=SEED)
    sub = random.Random(SEED).sample(va, N)

    ceiling, counts_a, counts_b = [], [], []
    for k, p in enumerate(sub):
        img = Image.open(p["opt"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        try:
            m_a = masks_to_arrays(sam(img, points_per_batch=32))   # run 1
            m_b = masks_to_arrays(sam(img, points_per_batch=32))   # run 2, same image
        except Exception as e:
            print(f"  [skip {k}] {e}", flush=True); continue
        d = doim_score(m_a, m_b)
        if d is not None:
            ceiling.append(d); counts_a.append(len(m_a)); counts_b.append(len(m_b))
        if (k+1) % 15 == 0:
            print(f"  [{k+1}/{N}] ceiling so far: {np.mean(ceiling):.3f} | "
                  f"avg masks/img: {np.mean(counts_a):.1f}", flush=True)

    print("\n" + "="*60)
    print(f"DOIM SELF-CONSISTENCY CEILING (GT vs GT, n={len(ceiling)}): {np.mean(ceiling):.4f}")
    print(f"Avg masks per image: run A {np.mean(counts_a):.1f} | run B {np.mean(counts_b):.1f}")
    print(f"For reference: today's translated={0.1053:.4f}, SAR={0.1029:.4f}")
    print("="*60)
    if np.mean(ceiling) < 0.20:
        print("VERDICT: ceiling is near the floor -> DOIM has little dynamic range on this")
        print("data/model; the translated≈SAR near-tie is likely a metric-sensitivity")
        print("artifact, NOT evidence that translation adds no downstream value.")
    else:
        print("VERDICT: ceiling is well above the floor -> DOIM has real range, and")
        print("today's translated≈SAR result is a genuine, reportable finding.")

if __name__ == "__main__":
    main()

Writing /content/doim_ceiling_check.py


In [ ]:
!python -u /content/doim_ceiling_check.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  [15/60] ceiling so far: 1.000 | avg masks/img: 33.5
  [30/60] ceiling so far: 1.000 | avg masks/img: 31.0
  [45/60] ceiling so far: 1.000 | avg masks/img: 27.1
  [60/60] ceiling so far: 1.000 | avg masks/img: 27.3

DOIM SELF-CONSISTENCY CEILING (GT vs GT, n=60): 1.0000
Avg masks per image: run A 27.3 | run B 27.3
For reference: today's translated=0.1053, SAR=0.1029
VERDICT: ceiling is well above the floor -> DOIM has real range, and
today's translated≈SAR resu

In [ ]:
%%writefile /content/ensemble_diversity_gate.py
"""
Step-0 gate for SEN12-Prob: does the champion's stochastic ResShift sampler
produce a genuinely diverse ensemble, or does it collapse to near-identical
samples? Decides whether the probabilistic-benchmark direction is viable.

Measures, per SAR input, across K seeded samples:
  - pairwise LPIPS diversity (mean pairwise distance among K decoded samples)
  - per-pixel ensemble std (in image space)
  - decoder-noise floor (decode the SAME latent twice -> should be ~0,
    confirming any measured spread comes from the diffusion sampler, not the VAE)
  - baseline single-sample PSNR sanity (K=1 should read ~16.7, matching the
    already-established champion number)
GATE: ensemble std must clear the decoder floor by a clear margin.
"""
import os, json, random, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import lpips as lpips_lib
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
VAE_TAG, BRIDGE_TAG = "adapted16_final", "bridge_final"
IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16
N_IMAGES, K_ENSEMBLE = 30, 16     # cheap: 30 images x 16 seeds x 15 steps
device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    return ps

class DS(Dataset):
    def __init__(self, pairs): self.p = pairs
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        return {"sar": self._l(p["sar"]), "opt": self._l(p["opt"]),
                "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=DETAIL_CH, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def main():
    random.seed(SEED); np.random.seed(SEED)
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(bv)
    vae.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    vae = vae.to(device).eval().float().requires_grad_(False)

    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{BRIDGE_TAG}/bridge_config.json")))
    SCALE = cfg["latent_scale"]
    log(f"latent_scale={SCALE:.4f} T={cfg['T']} kappa={cfg['kappa']}")
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{BRIDGE_TAG}/unet.safetensors")))
    unet = unet.to(device).eval().float().requires_grad_(False)
    log("Champion loaded.")

    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    emb = {}
    @torch.no_grad()
    def E(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in ("tropical","temperate","arctic","arid","coastal","urban"):
            emb[(s,ter)] = E(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    emb["null"] = E("")
    del txt; torch.cuda.empty_cache()

    i = np.arange(cfg["T"])
    eta = torch.tensor(cfg["eta_1"]*(cfg["eta_T"]/cfg["eta_1"])**(i/(cfg["T"]-1)),
                       dtype=torch.float32).to(device)

    @torch.no_grad()
    def sample_seeded(y, cond, seed):
        g = torch.Generator(device=device).manual_seed(seed)
        B = y.shape[0]
        x = y + cfg["kappa"]*torch.sqrt(eta[-1])*torch.randn(y.shape, generator=g, device=device)
        for t in reversed(range(cfg["T"])):
            tb = torch.full((B,), int(t*1000/cfg["T"]), device=device, dtype=torch.long)
            x0 = unet(torch.cat([x,y],1), tb, encoder_hidden_states=cond).sample
            if t > 0:
                ep = eta[t-1]
                x = x0 + ep*(y-x0) + cfg["kappa"]*torch.sqrt(ep)*torch.randn(y.shape, generator=g, device=device)
            else:
                x = x0
        return x

    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)

    pairs = collect(DATA_ROOT)
    _, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    sub = random.Random(SEED).sample(va, N_IMAGES)
    log(f"Gate: {N_IMAGES} images x K={K_ENSEMBLE} seeds x T={cfg['T']} steps.\n")

    # ---- decoder-noise floor: decode the SAME latent twice, should be ~0 ----
    d0 = DS(sub[:1])[0]
    opt0 = d0["opt"].unsqueeze(0).to(device)
    with torch.no_grad():
        lat0 = vae.enc(opt0.float())
        dec_a = vae.dec(lat0).clamp(-1,1)
        dec_b = vae.dec(lat0).clamp(-1,1)
    decoder_floor_std = (dec_a - dec_b).std().item()
    log(f"Decoder-noise floor (same latent decoded twice): std={decoder_floor_std:.6f} "
        f"({'as expected, ~0, deterministic decoder' if decoder_floor_std < 1e-4 else 'WARNING: non-deterministic decoder!'})\n")

    all_pixel_std, all_lpips_div, single_psnr = [], [], []
    for idx, p in enumerate(sub):
        d = DS([p])[0]
        opt = d["opt"].unsqueeze(0).to(device)
        sar = d["sar"].unsqueeze(0).to(device)
        y = vae.enc(sar.float()) * SCALE
        cond = torch.stack([emb.get((d["season"], d["terrain"]), emb["null"])]).to(device)

        samples = []
        for k in range(K_ENSEMBLE):
            with torch.no_grad():
                lat = sample_seeded(y, cond, seed=1000 + k)
                img = vae.dec((lat/SCALE).float()).clamp(-1,1)
            samples.append(img)
        stack = torch.cat(samples, dim=0)   # (K,3,H,W)

        pixel_std = stack.std(dim=0).mean().item()
        all_pixel_std.append(pixel_std)

        # mean pairwise LPIPS among the K samples
        with torch.no_grad():
            dists = []
            for a in range(K_ENSEMBLE):
                for b in range(a+1, K_ENSEMBLE):
                    dists.append(lpips_fn(stack[a:a+1], stack[b:b+1]).item())
        all_lpips_div.append(np.mean(dists))

        # K=1 sanity: does a single sample reproduce ~16.7 dB?
        single_psnr.append(10*np.log10(4.0/F.mse_loss(stack[0:1], opt).item()))

        if (idx+1) % 5 == 0:
            log(f"  [{idx+1}/{N_IMAGES}] pixel_std={np.mean(all_pixel_std):.4f} | "
                f"lpips_div={np.mean(all_lpips_div):.4f} | single_psnr={np.mean(single_psnr):.2f}")

    log("\n" + "="*64)
    log("ENSEMBLE DIVERSITY GATE")
    log(f"  Decoder-noise floor (pixel std)     : {decoder_floor_std:.6f}")
    log(f"  Mean per-pixel ensemble std          : {np.mean(all_pixel_std):.4f}")
    log(f"  Mean pairwise LPIPS diversity         : {np.mean(all_lpips_div):.4f}")
    log(f"  Single-sample PSNR (sanity, want ~16.7): {np.mean(single_psnr):.2f}")
    margin = np.mean(all_pixel_std) / max(decoder_floor_std, 1e-6)
    log(f"  Ensemble std / decoder floor ratio    : {margin:.1f}x")
    log("="*64)
    if margin > 50 and np.mean(all_lpips_div) > 0.05:
        log("VERDICT: PASS. Ensemble is genuinely diverse, well above the decoder floor.")
        log("The SEN12-Prob probabilistic-benchmark direction is viable on this model.")
    elif margin > 10:
        log("VERDICT: MARGINAL. Some diversity exists but may be weak for K=16 rank")
        log("histograms to be informative. Consider increasing kappa or initial-noise")
        log("injection before building the full harness.")
    else:
        log("VERDICT: FAIL. Ensemble has collapsed near-identical samples. Do NOT build")
        log("the full CRPS/rank-histogram harness on this model as-is; the T=15/kappa=2.0")
        log("sampler is too weakly stochastic. Report this honestly as a finding if desired,")
        log("or pivot to conformal intervals only (#1 strengthener) which need no ensemble diversity.")

if __name__ == "__main__":
    main()

Writing /content/ensemble_diversity_gate.py


In [ ]:
!python -u /content/ensemble_diversity_gate.py


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
latent_scale=1.4010 T=15 kappa=2.0
Champion loaded.
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Downloading: "https://download.pytorch.org/models/alexnet-owt

In [ ]:
%%writefile /content/stage5_afcrps_256.py
"""
SAR-INTEL Stage 5 — afCRPS finetune of the champion (bridge_final). FP32.

FIX (this version): step-0/validation single-sample PSNR is now computed
PER-IMAGE then averaged (the champion/Stage-4/X1 protocol). The previous
version computed one PSNR over each whole batch of 8 (log of mean-MSE),
which by Jensen's inequality reads ~2 dB lower — this, not bf16 precision,
was the cause of both step-0 gate failures (14.49 on both fp32 and bf16).
The ensemble block was already per-image, which is why ens-mean read 16.7.
"""
import os, gc, json, time, random, warnings, shutil
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import save_file, load_file
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
def log(*a): print(*a, flush=True)

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT   = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR  = "/content/stage5_afcrps"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID  = os.environ["HF_REPO_ID"]
HF_TOKEN    = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG, INIT_TAG = "adapted16_final", "bridge_final"

IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16

NUM_TRAIN_STEPS = 40_000
BATCH_SIZE  = 4            # fp32: effective forward batch = BATCH*M = 16
M_MEMBERS   = 4            # afCRPS ensemble members per training step
ALPHA_FAIR  = 0.95         # afCRPS = a*fairCRPS + (1-a)*CRPS
LR          = 1e-5
LR_WARMUP   = 2_000
COND_DROPOUT= 0.10
SPIKE_MULT, GUARD_WARMUP = 10.0, 500

VAL_EVERY   = 2_500
N_VAL_PSNR  = 100
N_VAL_ENS   = 40
K_VAL       = 8
LOCAL_SAVE_EVERY = 10_000
STEP_LOG_EVERY   = 500
GATE_STEP   = 15_000
PSNR_FLOOR  = 15.9
STEP0_BAND  = (15.6, 17.4)

device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}
TERRAINS = ("tropical","temperate","arctic","arid","coastal","urban")

def map_region_to_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect_pairs(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_region_to_terrain(row["region"])})
    random.shuffle(ps); return ps

class PairDataset(Dataset):
    def __init__(self, pairs, augment=True): self.p, self.aug = pairs, augment
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        sar, opt = self._l(p["sar"]), self._l(p["opt"])
        if self.aug:                                   # flips only — champion's recipe
            if random.random() > .5: sar, opt = torch.flip(sar,[2]), torch.flip(opt,[2])
            if random.random() > .5: sar, opt = torch.flip(sar,[1]), torch.flip(opt,[1])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=DETAIL_CH, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def load_vae():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    return v.to(device).eval().float().requires_grad_(False)

def build_unet_from_champion():
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/unet.safetensors")))
    log(f"  UNet init from {INIT_TAG} (champion, 16.73 dB / FID 125.5 baseline).")
    return unet

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

def get_cond(ss, tt, cache, drop=0.0):
    return torch.stack([cache["null"] if (drop and random.random() < drop)
                        else cache.get((s,t), cache["null"]) for s,t in zip(ss,tt)])

# ── afCRPS (AIFS-CRPS alpha-blend of fair and standard empirical CRPS) ───────
def afcrps_loss(preds, target, alpha=ALPHA_FAIR):
    """preds: (M,B,C,H,W); target: (B,C,H,W)."""
    M = preds.shape[0]
    term1 = (preds - target.unsqueeze(0)).abs().mean(dim=0)
    pair  = (preds.unsqueeze(0) - preds.unsqueeze(1)).abs().sum(dim=(0,1))
    crps_std  = term1 - pair / (2.0 * M * M)
    crps_fair = term1 - pair / (2.0 * M * (M - 1))
    return (alpha * crps_fair + (1.0 - alpha) * crps_std).mean()

def fair_crps_np(members, truth):
    K = members.shape[0]
    t1 = np.abs(members - truth[None]).mean(axis=0)
    s = 0.0
    for i in range(K):
        for j in range(i+1, K):
            s = s + np.abs(members[i] - members[j])
    t2 = s * 2.0 / (K * (K - 1)) / 2.0
    return float((t1 - t2).mean())

def main():
    accelerator = Accelerator(mixed_precision="no", project_dir=OUTPUT_DIR)   # FP32
    set_seed(SEED); is_main = accelerator.is_main_process
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    log("Loading components...")
    vae = load_vae(); embeds = build_embeds(); unet = build_unet_from_champion()
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/bridge_config.json")))
    SCALE, T, KAPPA = float(cfg["latent_scale"]), int(cfg["T"]), float(cfg["kappa"])
    i = np.arange(T)
    eta = torch.tensor(cfg["eta_1"]*(cfg["eta_T"]/cfg["eta_1"])**(i/(T-1)),
                      dtype=torch.float32).to(device)
    log(f"  scale={SCALE:.4f} T={T} kappa={KAPPA} (from {INIT_TAG} config)")

    pairs = collect_pairs(DATA_ROOT)
    tr, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    log(f"Train {len(tr):,} | Val {len(va):,}")
    dl = DataLoader(PairDataset(tr, True), batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)

    optimizer = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9,0.999), weight_decay=1e-2)
    lr_sched  = get_cosine_schedule_with_warmup(optimizer, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, optimizer, dl, lr_sched = accelerator.prepare(unet, optimizer, dl, lr_sched)

    def enc(x):
        with torch.no_grad(): return vae.enc(x.float()) * SCALE

    @torch.no_grad()
    def sample_seeded(m, y, cond, seed):
        g = torch.Generator(device=device).manual_seed(seed)
        B = y.shape[0]
        x = y + KAPPA*torch.sqrt(eta[-1])*torch.randn(y.shape, generator=g, device=device)
        for t in reversed(range(T)):
            tb = torch.full((B,), int(t*1000/T), device=device, dtype=torch.long)
            x0 = m(torch.cat([x,y],1).float(), tb,
                   encoder_hidden_states=cond.float()).sample.float()
            if t > 0:
                ep = eta[t-1]
                x = x0 + ep*(y-x0) + KAPPA*torch.sqrt(ep)*torch.randn(y.shape, generator=g, device=device)
            else: x = x0
        return x

    @torch.no_grad()
    def validate(step, is_step0=False):
        m = accelerator.unwrap_model(unet); m.eval()
        # (a) single-sample PSNR — PER-IMAGE, champion protocol (THE FIX)
        sub = random.Random(7).sample(va, N_VAL_PSNR)
        P = []
        for b in DataLoader(PairDataset(sub, False), batch_size=8, num_workers=2):
            opt = b["opt"].to(device); y = enc(b["sar"].to(device))
            c = get_cond(b["season"], b["terrain"], embeds).to(device)
            rec = vae.dec((sample_seeded(m, y, c, 1234)/SCALE)).clamp(-1,1)
            for j in range(rec.shape[0]):
                P.append(10*np.log10(4.0/F.mse_loss(rec[j:j+1], opt[j:j+1]).item()))
        psnr1 = float(np.mean(P))
        # (b) ensemble block: K_VAL seeds on n=N_VAL_ENS (already per-image)
        sub2 = random.Random(11).sample(va, N_VAL_ENS)
        mP, crps_list, spreads, errs = [], [], [], []
        for b in DataLoader(PairDataset(sub2, False), batch_size=4, num_workers=2):
            opt = b["opt"].to(device); y = enc(b["sar"].to(device))
            c = get_cond(b["season"], b["terrain"], embeds).to(device)
            mems = []
            for k in range(K_VAL):
                rec = vae.dec((sample_seeded(m, y, c, 500+k)/SCALE)).clamp(-1,1)
                mems.append(((rec+1)/2).cpu().numpy())
            mems = np.stack(mems)                              # (K,B,3,H,W)
            gt = ((opt+1)/2).cpu().numpy()
            ens_mean = mems.mean(axis=0)
            mse = ((ens_mean - gt)**2).mean(axis=(1,2,3))      # per-image
            mP.extend(list(10*np.log10(1.0/np.maximum(mse, 1e-10))))
            for j in range(gt.shape[0]):
                crps_list.append(fair_crps_np(mems[:,j], gt[j]))
            spreads.append(mems.std(axis=0).mean()); errs.append(np.sqrt(mse.mean()))
        ssr = float(np.sqrt((K_VAL+1)/K_VAL) * np.mean(spreads) / max(np.mean(errs), 1e-9))
        out = {"psnr1": psnr1, "psnr_mean": float(np.mean(mP)),
               "faircrps": float(np.mean(crps_list)), "ssr": ssr}
        m.train()
        tag = "STEP-0 BASELINE" if is_step0 else f"VAL step {step}"
        log(f"\n[{tag}] single-PSNR {out['psnr1']:.3f} | ens-mean-PSNR(K={K_VAL}) {out['psnr_mean']:.3f} | "
            f"fair-CRPS {out['faircrps']:.5f} | SSR {out['ssr']:.3f}\n")
        return out

    # ── STEP-0: harness gate + baselines ──
    base = validate(0, is_step0=True)
    if not (STEP0_BAND[0] <= base["psnr1"] <= STEP0_BAND[1]):
        log(f"!! STEP-0 GATE FAILED: single-PSNR {base['psnr1']:.2f} outside {STEP0_BAND}. "
            f"Harness wrong — ABORTING."); return
    log(f">>> Step-0 gate PASSED (fp32, per-image protocol). Ensemble-mean vs single: "
        f"{base['psnr_mean']:.2f} vs {base['psnr1']:.2f} "
        f"({base['psnr_mean']-base['psnr1']:+.2f} dB from seed-averaging).")
    log(f">>> Baselines to beat: fair-CRPS {base['faircrps']:.5f}, SSR {base['ssr']:.3f} (1.0 ideal).\n")

    latest = os.path.join(OUTPUT_DIR, "latest")
    global_step = 0; ema_l = 0.0; a = 0.98; n_guard = n_nan = 0
    t0 = time.time(); last = 0
    unet.train()
    log(f">>> afCRPS finetune 0 -> {NUM_TRAIN_STEPS}. M={M_MEMBERS}, alpha={ALPHA_FAIR}, "
        f"LR={LR}, FP32. GATE at {GATE_STEP}: fair-CRPS < {base['faircrps']:.5f} AND psnr1 >= {PSNR_FLOOR}.\n")

    for epoch in range(999):
        for batch in dl:
            opt_img = batch["opt"].to(device)
            x0 = enc(opt_img); y = enc(batch["sar"].to(device))
            B = x0.shape[0]
            cond = get_cond(batch["season"], batch["terrain"], embeds, COND_DROPOUT).to(device)
            t = torch.randint(0, T, (B,), device=device)
            et = eta[t].view(B,1,1,1)
            tb = (t*1000//T).long()
            noise = torch.randn(M_MEMBERS, *x0.shape, device=device)
            x_t = x0.unsqueeze(0) + et.unsqueeze(0)*(y-x0).unsqueeze(0) \
                  + KAPPA*torch.sqrt(et).unsqueeze(0)*noise               # (M,B,C,H,W)
            inp = torch.cat([x_t, y.unsqueeze(0).expand(M_MEMBERS,-1,-1,-1,-1)], dim=2)
            inp = inp.reshape(M_MEMBERS*B, 2*TOTAL_CH, IMG_SIZE//8, IMG_SIZE//8)
            tb_r = tb.repeat(M_MEMBERS)
            cond_r = cond.repeat(M_MEMBERS, 1, 1)

            with accelerator.accumulate(unet):
                preds = unet(inp, tb_r, encoder_hidden_states=cond_r).sample
                preds = preds.view(M_MEMBERS, B, TOTAL_CH, IMG_SIZE//8, IMG_SIZE//8)
                loss = afcrps_loss(preds.float(), x0.float())

                if global_step > GUARD_WARMUP and ema_l > 0 and loss.item() > SPIKE_MULT*max(ema_l, 1e-3):
                    optimizer.zero_grad(set_to_none=True); n_guard += 1
                    if is_main and n_guard % 10 == 1:
                        log(f"  [guard] skipped {global_step} (loss={loss.item():.4f}, ema={ema_l:.4f})")
                    continue
                if not torch.isfinite(loss):
                    optimizer.zero_grad(set_to_none=True); n_nan += 1
                    log(f"  [nan-guard] skipped {global_step}"); continue

                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), 1.0)
                optimizer.step(); lr_sched.step(); optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                global_step += 1
                ema_l = a*ema_l + (1-a)*loss.item()
                if is_main and global_step % STEP_LOG_EVERY == 0:
                    r = (global_step-last)/max(time.time()-t0,1e-9); t0, last = time.time(), global_step
                    log(f"step {global_step}/{NUM_TRAIN_STEPS} | afCRPS {ema_l:.4f} | "
                        f"guard={n_guard} nan={n_nan} | {r:.2f} it/s")
                if global_step % VAL_EVERY == 0 and is_main:
                    v = validate(global_step)
                    if global_step == GATE_STEP:
                        ok = (v["faircrps"] < base["faircrps"]) and (v["psnr1"] >= PSNR_FLOOR)
                        log(f"  GATE @ {GATE_STEP}: fair-CRPS {v['faircrps']:.5f} vs baseline "
                            f"{base['faircrps']:.5f} | psnr1 {v['psnr1']:.2f} (floor {PSNR_FLOOR}) -> "
                            f"{'PASS, continue' if ok else 'FAIL — recommend killing the run'}")
                if global_step % LOCAL_SAVE_EVERY == 0 and is_main:
                    if os.path.exists(latest): shutil.rmtree(latest)
                    os.makedirs(latest); accelerator.save_state(latest)
                    torch.save({"step": global_step}, os.path.join(latest, "meta.pt"))
                    log(f"  local checkpoint (overwritten, LOCAL ONLY) @ {global_step}")
            if global_step >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if global_step >= NUM_TRAIN_STEPS: break

    if is_main:
        final = validate(global_step)
        tmp = os.path.join(OUTPUT_DIR, "_final"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k, v in accelerator.unwrap_model(unet).state_dict().items()}
        save_file(sd, os.path.join(tmp, "unet.safetensors"))
        json.dump({**cfg, "method": "resshift_afcrps", "alpha_fair": ALPHA_FAIR,
                   "m_members": M_MEMBERS, "final_step": global_step,
                   "step0": base, "final_val": final},
                  open(os.path.join(tmp, "bridge_config.json"), "w"), indent=2)
        pushed = False
        try:
            HfApi().upload_folder(folder_path=tmp, path_in_repo="afcrps_final",
                                  repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)
            log("  FINAL -> HF/afcrps_final"); pushed = True
        except Exception as e:
            log(f"  push failed: {e} — weights KEPT at {tmp}")
        if pushed: shutil.rmtree(tmp, ignore_errors=True)
    accelerator.end_training(); log("Stage 5 complete.")

if __name__ == "__main__":
    main()

Overwriting /content/stage5_afcrps_256.py


In [ ]:
!python -u /content/stage5_afcrps_256.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading components...
  UNet init from bridge_final (champion, 16.73 dB / FID 125.5 baseline).
  scale=1.4010 T=15 kappa=2.0 (from bridge_final config)
Train 292,486 | Val 15,394

[STEP

In [ ]:
%%writefile /content/sen12prob_harness.py
"""
SEN12-Prob core harness: champion (bridge_final) vs afCRPS-tuned (afcrps_final).
  - Split-conformal per-pixel intervals (near-free, strengthener #1)
  - Rank/PIT histogram (K+1 bins, chi-sq flatness)
  - AUSE + Spearman err-unc correlation (strengthener companion)
  - fair-CRPS, SSR (already have baselines; recomputed here at eval N for the table)
Two-model comparison table is the paper's calibration-vs-fidelity result.
"""
import os, gc, json, random, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
VAE_TAG = "adapted16_final"
MODELS = {"champion": "bridge_final", "afcrps": "afcrps_final"}
IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16
N_CAL, N_TEST, K = 150, 150, 16    # split-conformal calibration / test, ensemble size
ALPHA = 0.10                        # 90% coverage target
device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    return ps

class DS(Dataset):
    def __init__(self, pairs): self.p = pairs
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        return {"sar": self._l(p["sar"]), "opt": self._l(p["opt"]),
                "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=DETAIL_CH, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def load_vae():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    return v.to(device).eval().float().requires_grad_(False)

def load_model(tag):
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{tag}/bridge_config.json")))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{tag}/unet.safetensors")))
    return unet.to(device).eval().float().requires_grad_(False), cfg

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in ("tropical","temperate","arctic","arid","coastal","urban"):
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

def get_cond(ss, tt, cache):
    return torch.stack([cache.get((s,t), cache["null"]) for s,t in zip(ss,tt)])

@torch.no_grad()
def sample_seeded(unet, y, cond, eta, kappa, T, seed):
    g = torch.Generator(device=device).manual_seed(seed)
    B = y.shape[0]
    x = y + kappa*torch.sqrt(eta[-1])*torch.randn(y.shape, generator=g, device=device)
    for t in reversed(range(T)):
        tb = torch.full((B,), int(t*1000/T), device=device, dtype=torch.long)
        x0 = unet(torch.cat([x,y],1), tb, encoder_hidden_states=cond).sample
        if t > 0:
            ep = eta[t-1]
            x = x0 + ep*(y-x0) + kappa*torch.sqrt(ep)*torch.randn(y.shape, generator=g, device=device)
        else: x = x0
    return x

def fair_crps_np(members, truth):
    K = members.shape[0]
    t1 = np.abs(members - truth[None]).mean(axis=0)
    s = 0.0
    for i in range(K):
        for j in range(i+1, K):
            s += np.abs(members[i] - members[j])
    return float((t1 - s*2.0/(K*(K-1))/2.0).mean())

def eval_model(unet, cfg, vae, embeds, pairs):
    SCALE, T, KAPPA = float(cfg["latent_scale"]), int(cfg["T"]), float(cfg["kappa"])
    i = np.arange(T)
    eta = torch.tensor(cfg["eta_1"]*(cfg["eta_T"]/cfg["eta_1"])**(i/(T-1)), dtype=torch.float32).to(device)
    def enc(x):
        with torch.no_grad(): return vae.enc(x.float()) * SCALE

    all_gt, all_mems, all_mean_err, all_std = [], [], [], []
    for p in pairs:
        d = DS([p])[0]
        opt = d["opt"].unsqueeze(0).to(device); sar = d["sar"].unsqueeze(0).to(device)
        y = enc(sar); c = get_cond([d["season"]], [d["terrain"]], embeds).to(device)
        mems = np.stack([((vae.dec((sample_seeded(unet, y, c, eta, KAPPA, T, 700+k)/SCALE))
                           .clamp(-1,1)+1)/2).cpu().numpy()[0] for k in range(K)])   # (K,3,H,W)
        gt = ((opt[0]+1)/2).cpu().numpy()
        all_gt.append(gt); all_mems.append(mems)
        ens_mean = mems.mean(axis=0)
        all_mean_err.append(np.abs(ens_mean - gt))          # per-pixel abs error
        all_std.append(mems.std(axis=0))                     # per-pixel ensemble std
    return all_gt, all_mems, all_mean_err, all_std, SCALE, T, KAPPA

def rank_histogram(mems_list, gt_list):
    """Fraction of GT pixels falling in each of K+1 rank bins (subsampled for speed)."""
    counts = np.zeros(K+1)
    for mems, gt in zip(mems_list, gt_list):
        sub_idx = np.random.RandomState(0).choice(gt.size, size=min(2000, gt.size), replace=False)
        flat_mems = mems.reshape(K, -1)[:, sub_idx]; flat_gt = gt.reshape(-1)[sub_idx]
        ranks = (flat_mems < flat_gt[None]).sum(axis=0)
        for r in ranks: counts[r] += 1
    p = counts / counts.sum()
    chi2 = float(((counts - counts.mean())**2 / max(counts.mean(),1e-9)).sum())
    return p, chi2

def ause_and_spearman(err_list, std_list):
    err = np.concatenate([e.reshape(-1) for e in err_list])
    std = np.concatenate([s.reshape(-1) for s in std_list])
    idx = np.random.RandomState(0).choice(len(err), size=min(50000, len(err)), replace=False)
    err, std = err[idx], std[idx]
    order_unc = np.argsort(-std)          # remove highest-uncertainty first
    order_oracle = np.argsort(-err)       # oracle: remove highest-error first
    fracs = np.linspace(0, 0.9, 20)
    def sparsify(order):
        curve = []
        for f in fracs:
            keep = order[int(f*len(order)):]
            curve.append(err[keep].mean())
        return np.array(curve)
    c_unc, c_oracle = sparsify(order_unc), sparsify(order_oracle)
    ause = float(np.trapz(c_unc - c_oracle, fracs))
    rho, _ = stats.spearmanr(std, err)
    return ause, float(rho)

def conformal_intervals(cal_err_list, test_err_list, test_std_list, alpha=ALPHA):
    """Split-conformal: q = (1-alpha) quantile of |err|/std on calibration set;
    interval = std * q on test set. Report empirical coverage."""
    cal_scores = np.concatenate([(e/np.maximum(s,1e-6)).reshape(-1)
                                  for e, s in zip(cal_err_list["err"], cal_err_list["std"])])
    n = len(cal_scores)
    q = np.quantile(cal_scores, min(1.0, np.ceil((n+1)*(1-alpha))/n))
    covered, total = 0, 0
    for e, s in zip(test_err_list, test_std_list):
        halfwidth = q * np.maximum(s, 1e-6)
        covered += (e <= halfwidth).sum(); total += e.size
    return float(q), float(covered/total)

def main():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    vae = load_vae(); embeds = build_embeds()
    pairs = collect(DATA_ROOT)
    _, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    cal_pairs = random.Random(SEED).sample(va, N_CAL)
    test_pairs = random.Random(SEED+1).sample([p for p in va if p not in cal_pairs], N_TEST)

    results = {}
    for name, tag in MODELS.items():
        log(f"\n{'='*60}\nEvaluating {name} ({tag})\n{'='*60}")
        unet, cfg = load_model(tag)

        log(f"  Calibration pass (n={N_CAL})...")
        gt_c, mems_c, err_c, std_c, *_ = eval_model(unet, cfg, vae, embeds, cal_pairs)
        log(f"  Test pass (n={N_TEST})...")
        gt_t, mems_t, err_t, std_t, *_ = eval_model(unet, cfg, vae, embeds, test_pairs)

        crps = float(np.mean([fair_crps_np(m, g) for m, g in zip(mems_t, gt_t)]))
        spread = np.mean([s.mean() for s in std_t])
        rmse = np.sqrt(np.mean([((m.mean(0)-g)**2).mean() for m, g in zip(mems_t, gt_t)]))
        ssr = float(np.sqrt((K+1)/K) * spread / max(rmse, 1e-9))
        rank_p, chi2 = rank_histogram(mems_t, gt_t)
        ause, rho = ause_and_spearman(err_t, std_t)
        q, coverage = conformal_intervals({"err": err_c, "std": std_c}, err_t, std_t)

        results[name] = {"fair_crps": crps, "ssr": ssr, "rank_chi2": chi2,
                          "rank_hist": rank_p.tolist(), "ause": ause, "spearman": rho,
                          "conformal_q": q, "conformal_coverage": coverage}
        log(f"\n  fair-CRPS {crps:.5f} | SSR {ssr:.3f} | rank-chi2 {chi2:.1f} | "
            f"AUSE {ause:.5f} | Spearman(err,std) {rho:.3f}")
        log(f"  Conformal: q={q:.3f}, target coverage {(1-ALPHA)*100:.0f}%, "
            f"empirical coverage {coverage*100:.1f}%")
        del unet; gc.collect(); torch.cuda.empty_cache()

    log("\n" + "="*60)
    log("SEN12-Prob SUMMARY TABLE")
    log("="*60)
    for name, r in results.items():
        log(f"  {name:10s} fair-CRPS {r['fair_crps']:.5f} | SSR {r['ssr']:.3f} | "
            f"AUSE {r['ause']:.5f} | Spearman {r['spearman']:.3f} | "
            f"conformal cov {r['conformal_coverage']*100:.1f}% (target {(1-ALPHA)*100:.0f}%)")
    json.dump(results, open("/content/sen12prob_results.json", "w"), indent=2)
    log("\nsaved -> /content/sen12prob_results.json")

if __name__ == "__main__":
    main()

Writing /content/sen12prob_harness.py


In [ ]:
!python -u /content/sen12prob_harness.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.

Evaluating champion (bridge_final)
  Calibration pass (n=150)...
  Test pass (n=150)...

  fair-CRPS 0.10285 | SSR 0.291 | rank-chi2 565442.4 | AUSE 0.03988 | Spearman(err,std) 0.369
 

In [ ]:
%%writefile /content/stageR_regressor.py
"""
Script A — deterministic latent regressor R: z_sar -> z_optical (MMSE estimator).
Doc refs: 3.D (regression baseline), 1.B step 1-2 (regressor for two-stage).
Predicts the posterior MEAN directly -> maximises PSNR/SSIM by construction.
Expect blurry outputs / poor FID: that is the regression-to-the-mean effect and
is the POINT (it anchors the perception-distortion table).

Init from bridge_final. Input = concat[y, y] (32ch), fixed timestep = last
(closest to the champion's training distribution, where x_t ~= y + noise).
Discipline: step-0 sanity, per-image PSNR/SSIM, spike+NaN guards, local ckpt.
"""
import os, gc, json, time, random, warnings, shutil
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import save_file, load_file
from torchmetrics.image import StructuralSimilarityIndexMeasure
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR = "/content/stageR_regressor"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG, INIT_TAG, OUT_TAG = "adapted16_final", "bridge_final", "regressor_final"

IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
TERRAINS = ("tropical","temperate","arctic","arid","coastal","urban")
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16

NUM_TRAIN_STEPS = 30_000
BATCH_SIZE = 4
LR = 5e-5
LR_WARMUP = 1_000
COND_DROPOUT = 0.10
LAMBDA_PIXEL = 0.5          # decoded-L1 term (0 = pure latent L2)
PIXEL_START = 2_000
VAL_EVERY = 2_500
N_VAL = 150
LOCAL_SAVE_EVERY = 10_000
STEP_LOG_EVERY = 500
SPIKE_MULT, GUARD_WARMUP = 10.0, 500
GATE_STEP, GATE_MIN_PSNR = 10_000, 16.5   # must beat champion by 10k or kill

device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    random.shuffle(ps); return ps

class PairDataset(Dataset):
    def __init__(self, pairs, augment=True): self.p, self.aug = pairs, augment
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        sar, opt = self._l(p["sar"]), self._l(p["opt"])
        if self.aug:
            if random.random() > .5: sar, opt = torch.flip(sar,[2]), torch.flip(opt,[2])
            if random.random() > .5: sar, opt = torch.flip(sar,[1]), torch.flip(opt,[1])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=DETAIL_CH, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    def dec_grad(s,l): return s.base_vae.decoder(l)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def load_vae():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    return v.to(device).eval().float().requires_grad_(False)

def build_unet_from_champion():
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/unet.safetensors")))
    log(f"  UNet init from {INIT_TAG} (champion features reused).")
    return unet

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

def get_cond(ss, tt, cache, drop=0.0):
    return torch.stack([cache["null"] if (drop and random.random() < drop)
                        else cache.get((s,t), cache["null"]) for s,t in zip(ss,tt)])

def main():
    acc = Accelerator(mixed_precision="no", project_dir=OUTPUT_DIR)
    set_seed(SEED); is_main = acc.is_main_process
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    vae = load_vae(); embeds = build_embeds(); unet = build_unet_from_champion()
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/bridge_config.json")))
    SCALE, T = float(cfg["latent_scale"]), int(cfg["T"])
    FIXED_TB = int((T-1)*1000/T)        # last timestep: x_t ~= y + noise, closest to feeding y
    log(f"  scale={SCALE:.4f} | fixed timestep={FIXED_TB} | regressor mode (1 forward pass)")
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    pairs = collect(DATA_ROOT)
    tr, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    log(f"Train {len(tr):,} | Val {len(va):,}")
    dl = DataLoader(PairDataset(tr, True), batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)

    opt_ = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9,0.999), weight_decay=1e-2)
    sch = get_cosine_schedule_with_warmup(opt_, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, opt_, dl, sch = acc.prepare(unet, opt_, dl, sch)

    def enc(x):
        with torch.no_grad(): return vae.enc(x.float()) * SCALE

    def regress(m, y, cond):
        B = y.shape[0]
        tb = torch.full((B,), FIXED_TB, device=device, dtype=torch.long)
        return m(torch.cat([y, y], 1), tb, encoder_hidden_states=cond).sample

    @torch.no_grad()
    def validate(step, tag=None):
        m = acc.unwrap_model(unet); m.eval()
        sub = random.Random(7).sample(va, N_VAL)
        P, S = [], []
        for b in DataLoader(PairDataset(sub, False), batch_size=8, num_workers=2):
            o = b["opt"].to(device); y = enc(b["sar"].to(device))
            c = get_cond(b["season"], b["terrain"], embeds).to(device)
            rec = vae.dec((regress(m, y, c)/SCALE)).clamp(-1,1)
            for j in range(rec.shape[0]):                      # PER-IMAGE protocol
                P.append(10*np.log10(4.0/F.mse_loss(rec[j:j+1], o[j:j+1]).item()))
                S.append(ssim_fn(rec[j:j+1], o[j:j+1]).item())
        m.train()
        lbl = tag or f"VAL step {step}"
        log(f"\n[{lbl}] PSNR {np.mean(P):.3f} | SSIM {np.mean(S):.4f}   "
            f"(champion: 16.73 / 0.259 | SOTA: 19.72 / 0.312)\n")
        return float(np.mean(P)), float(np.mean(S))

    p0, s0 = validate(0, "STEP-0 (untrained regressor head)")
    log(">>> Step-0 is expected to be LOW (champion weights used out-of-mode). "
        "This run's gate is at step 10k, not step 0.\n")

    latest = os.path.join(OUTPUT_DIR, "latest")
    gs = 0; ema = 0.0; a = 0.98; ng = nn_ = 0; t0 = time.time(); last = 0
    unet.train()
    log(f">>> Regressor training 0 -> {NUM_TRAIN_STEPS}. GATE @ {GATE_STEP}: PSNR >= {GATE_MIN_PSNR}\n")

    for epoch in range(999):
        for batch in dl:
            oimg = batch["opt"].to(device)
            x0 = enc(oimg); y = enc(batch["sar"].to(device))
            c = get_cond(batch["season"], batch["terrain"], embeds, COND_DROPOUT).to(device)
            with acc.accumulate(unet):
                pred = regress(unet, y, c)
                loss_l = F.mse_loss(pred, x0)
                loss = loss_l
                if gs >= PIXEL_START and LAMBDA_PIXEL > 0:
                    img = vae.dec_grad(pred/SCALE).clamp(-1,1)
                    loss = loss + LAMBDA_PIXEL * F.l1_loss(img, oimg)
                if gs > GUARD_WARMUP and ema > 0 and loss_l.item() > SPIKE_MULT*max(ema,1e-4):
                    opt_.zero_grad(set_to_none=True); ng += 1; continue
                if not torch.isfinite(loss):
                    opt_.zero_grad(set_to_none=True); nn_ += 1; continue
                acc.backward(loss)
                if acc.sync_gradients: acc.clip_grad_norm_(unet.parameters(), 1.0)
                opt_.step(); sch.step(); opt_.zero_grad(set_to_none=True)
            if acc.sync_gradients:
                gs += 1; ema = a*ema + (1-a)*loss_l.item()
                if is_main and gs % STEP_LOG_EVERY == 0:
                    r = (gs-last)/max(time.time()-t0,1e-9); t0, last = time.time(), gs
                    log(f"step {gs}/{NUM_TRAIN_STEPS} | latentL2 {ema:.5f} | g={ng} n={nn_} | {r:.2f} it/s")
                if gs % VAL_EVERY == 0 and is_main:
                    p, s = validate(gs)
                    if gs == GATE_STEP:
                        log(f"  GATE @ {GATE_STEP}: PSNR {p:.2f} (need >= {GATE_MIN_PSNR}) -> "
                            f"{'PASS' if p >= GATE_MIN_PSNR else 'FAIL — recommend killing'}")
                if gs % LOCAL_SAVE_EVERY == 0 and is_main:
                    if os.path.exists(latest): shutil.rmtree(latest)
                    os.makedirs(latest); acc.save_state(latest)
                    torch.save({"step": gs}, os.path.join(latest,"meta.pt"))
                    log(f"  local checkpoint @ {gs}")
            if gs >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if gs >= NUM_TRAIN_STEPS: break

    if is_main:
        pf, sf = validate(gs, "FINAL")
        tmp = os.path.join(OUTPUT_DIR,"_final"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k,v in acc.unwrap_model(unet).state_dict().items()}
        save_file(sd, os.path.join(tmp,"unet.safetensors"))
        json.dump({"latent_scale": SCALE, "fixed_tb": FIXED_TB, "method": "deterministic_regressor",
                   "final_step": gs, "final_psnr": pf, "final_ssim": sf},
                  open(os.path.join(tmp,"bridge_config.json"),"w"), indent=2)
        pushed = False
        try:
            HfApi().upload_folder(folder_path=tmp, path_in_repo=OUT_TAG, repo_id=HF_REPO_ID,
                                  repo_type="model", token=HF_TOKEN)
            log(f"  FINAL -> HF/{OUT_TAG}"); pushed = True
        except Exception as e: log(f"  push failed: {e} — weights KEPT at {tmp}")
        if pushed: shutil.rmtree(tmp, ignore_errors=True)
    acc.end_training(); log("Script A complete.")

if __name__ == "__main__":
    main()

Overwriting /content/stageR_regressor.py


In [ ]:
!python -u /content/stageR_regressor.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
  UNet init from bridge_final (champion features reused).
  scale=1.4010 | fixed timestep=933 | regressor mode (1 forward pass)
Train 292,486 | Val 15,394

[STEP-0 (untrained regressor 

In [ ]:
%%writefile /content/stageRes_residual_bridge.py
"""
Script B — ResShift bridge retargeted to the RESIDUAL (z_gt - R(z_sar)).
Doc ref: 1.B steps 3-4. Inference: z = R(z_sar) + alpha * residual_sample.
alpha=0 -> pure regression (PSNR-max); alpha=1 -> full stochastic (FID-best).
Survives the falsified list: not a loss tweak, not a conditioning swap — it
changes WHAT the diffusion represents. E3's degeneracy cannot apply because the
residual genuinely is the unpredictable component.
GATE @30k: PSNR(alpha=0) >= 18.0 AND PSNR(alpha=0.5) >= 16.73 (champion).
"""
import os, gc, json, time, random, warnings, shutil
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import save_file, load_file
from torchmetrics.image import StructuralSimilarityIndexMeasure
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR = "/content/stageRes_residual"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG, REG_TAG, INIT_TAG, OUT_TAG = "adapted16_final", "regressor_final", "bridge_final", "residual_final"

IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
TERRAINS = ("tropical","temperate","arctic","arid","coastal","urban")
TOTAL_CH = 16
NUM_TRAIN_STEPS = 50_000
BATCH_SIZE = 4
LR = 3e-5
LR_WARMUP = 1_000
COND_DROPOUT = 0.10
VAL_EVERY = 5_000
N_VAL = 120
ALPHAS = (0.0, 0.25, 0.5, 1.0)
LOCAL_SAVE_EVERY = 10_000
STEP_LOG_EVERY = 500
SPIKE_MULT, GUARD_WARMUP = 10.0, 500
GATE_STEP, GATE_A0, GATE_A05 = 30_000, 18.0, 16.73

device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    random.shuffle(ps); return ps

class PairDataset(Dataset):
    def __init__(self, pairs, augment=True): self.p, self.aug = pairs, augment
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        sar, opt = self._l(p["sar"]), self._l(p["opt"])
        if self.aug:
            if random.random() > .5: sar, opt = torch.flip(sar,[2]), torch.flip(opt,[2])
            if random.random() > .5: sar, opt = torch.flip(sar,[1]), torch.flip(opt,[1])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=12, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(16,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def load_vae():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    return v.to(device).eval().float().requires_grad_(False)

def build_unet(tag):
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{tag}/bridge_config.json")))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{tag}/unet.safetensors")))
    return unet, cfg

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

def get_cond(ss, tt, cache, drop=0.0):
    return torch.stack([cache["null"] if (drop and random.random() < drop)
                        else cache.get((s,t), cache["null"]) for s,t in zip(ss,tt)])

def main():
    acc = Accelerator(mixed_precision="no", project_dir=OUTPUT_DIR)
    set_seed(SEED); is_main = acc.is_main_process
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    vae = load_vae(); embeds = build_embeds()
    reg, rcfg = build_unet(REG_TAG)          # frozen regressor
    reg = reg.to(device).eval().requires_grad_(False)
    FIXED_TB = int(rcfg["fixed_tb"])
    unet, cfg = build_unet(INIT_TAG)         # residual bridge, init from champion
    SCALE, T, KAPPA = float(cfg["latent_scale"]), int(cfg["T"]), float(cfg["kappa"])
    i = np.arange(T)
    eta = torch.tensor(cfg["eta_1"]*(cfg["eta_T"]/cfg["eta_1"])**(i/(T-1)), dtype=torch.float32).to(device)
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
    log(f"  regressor {REG_TAG} (tb={FIXED_TB}, reported PSNR {rcfg.get('final_psnr','?')}) | "
        f"bridge init {INIT_TAG} | scale={SCALE:.4f}")

    pairs = collect(DATA_ROOT)
    tr, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    dl = DataLoader(PairDataset(tr, True), batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)

    opt_ = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9,0.999), weight_decay=1e-2)
    sch = get_cosine_schedule_with_warmup(opt_, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, opt_, dl, sch = acc.prepare(unet, opt_, dl, sch)

    def enc(x):
        with torch.no_grad(): return vae.enc(x.float()) * SCALE

    @torch.no_grad()
    def regress(y, cond):
        B = y.shape[0]
        tb = torch.full((B,), FIXED_TB, device=device, dtype=torch.long)
        return reg(torch.cat([y, y], 1), tb, encoder_hidden_states=cond).sample

    @torch.no_grad()
    def sample_residual(m, y, cond, seed=1234):
        g = torch.Generator(device=device).manual_seed(seed)
        B = y.shape[0]
        x = torch.zeros_like(y) + KAPPA*torch.sqrt(eta[-1])*torch.randn(y.shape, generator=g, device=device)
        for t in reversed(range(T)):
            tb = torch.full((B,), int(t*1000/T), device=device, dtype=torch.long)
            r0 = m(torch.cat([x, y], 1), tb, encoder_hidden_states=cond).sample
            if t > 0:
                ep = eta[t-1]
                x = r0 + ep*(torch.zeros_like(y) - r0) + KAPPA*torch.sqrt(ep)*torch.randn(y.shape, generator=g, device=device)
            else: x = r0
        return x

    @torch.no_grad()
    def validate(step, tag=None):
        m = acc.unwrap_model(unet); m.eval()
        sub = random.Random(7).sample(va, N_VAL)
        res = {a: {"psnr": [], "ssim": []} for a in ALPHAS}
        for b in DataLoader(PairDataset(sub, False), batch_size=8, num_workers=2):
            o = b["opt"].to(device); y = enc(b["sar"].to(device))
            c = get_cond(b["season"], b["terrain"], embeds).to(device)
            z_mean = regress(y, c)
            resid = sample_residual(m, y, c)
            for al in ALPHAS:
                rec = vae.dec(((z_mean + al*resid)/SCALE)).clamp(-1,1)
                for j in range(rec.shape[0]):
                    res[al]["psnr"].append(10*np.log10(4.0/F.mse_loss(rec[j:j+1], o[j:j+1]).item()))
                    res[al]["ssim"].append(ssim_fn(rec[j:j+1], o[j:j+1]).item())
        m.train()
        lbl = tag or f"VAL step {step}"
        log(f"\n[{lbl}]")
        out = {}
        for al in ALPHAS:
            p, s = float(np.mean(res[al]["psnr"])), float(np.mean(res[al]["ssim"]))
            out[al] = (p, s)
            log(f"    alpha={al:<4} PSNR {p:.3f} | SSIM {s:.4f}")
        log(f"    (champion 16.73/0.259 | SOTA 19.72/0.312)\n")
        return out

    base = validate(0, "STEP-0 (regressor + untrained residual)")
    if base[0.0][0] < 15.0:
        log(f"!! STEP-0 WARNING: alpha=0 PSNR {base[0.0][0]:.2f} < 15.0 — the regressor "
            f"may not have loaded correctly. Inspect before trusting this run.\n")

    latest = os.path.join(OUTPUT_DIR,"latest")
    gs = 0; ema = 0.0; a_ = 0.98; ng = nn_ = 0; t0 = time.time(); last = 0
    unet.train()
    log(f">>> Residual bridge 0 -> {NUM_TRAIN_STEPS}. GATE @ {GATE_STEP}: "
        f"PSNR(a=0) >= {GATE_A0} AND PSNR(a=0.5) >= {GATE_A05}\n")

    for epoch in range(999):
        for batch in dl:
            oimg = batch["opt"].to(device)
            x0 = enc(oimg); y = enc(batch["sar"].to(device))
            c = get_cond(batch["season"], batch["terrain"], embeds, COND_DROPOUT).to(device)
            with torch.no_grad():
                z_mean = regress(y, get_cond(batch["season"], batch["terrain"], embeds).to(device))
            target = x0 - z_mean                       # the residual: the ill-posed part
            B = x0.shape[0]
            t = torch.randint(0, T, (B,), device=device)
            et = eta[t].view(B,1,1,1)
            x_t = target + et*(torch.zeros_like(target) - target) + KAPPA*torch.sqrt(et)*torch.randn_like(target)
            tb = (t*1000//T).long()
            with acc.accumulate(unet):
                pred = unet(torch.cat([x_t, y],1), tb, encoder_hidden_states=c).sample
                loss = F.mse_loss(pred, target)
                if gs > GUARD_WARMUP and ema > 0 and loss.item() > SPIKE_MULT*max(ema,1e-4):
                    opt_.zero_grad(set_to_none=True); ng += 1; continue
                if not torch.isfinite(loss):
                    opt_.zero_grad(set_to_none=True); nn_ += 1; continue
                acc.backward(loss)
                if acc.sync_gradients: acc.clip_grad_norm_(unet.parameters(), 1.0)
                opt_.step(); sch.step(); opt_.zero_grad(set_to_none=True)
            if acc.sync_gradients:
                gs += 1; ema = a_*ema + (1-a_)*loss.item()
                if is_main and gs % STEP_LOG_EVERY == 0:
                    r = (gs-last)/max(time.time()-t0,1e-9); t0, last = time.time(), gs
                    log(f"step {gs}/{NUM_TRAIN_STEPS} | residualMSE {ema:.5f} | g={ng} n={nn_} | {r:.2f} it/s")
                if gs % VAL_EVERY == 0 and is_main:
                    v = validate(gs)
                    if gs == GATE_STEP:
                        ok = v[0.0][0] >= GATE_A0 and v[0.5][0] >= GATE_A05
                        log(f"  GATE @ {GATE_STEP}: a=0 {v[0.0][0]:.2f} (>= {GATE_A0}) | "
                            f"a=0.5 {v[0.5][0]:.2f} (>= {GATE_A05}) -> "
                            f"{'PASS' if ok else 'FAIL — recommend killing'}")
                if gs % LOCAL_SAVE_EVERY == 0 and is_main:
                    if os.path.exists(latest): shutil.rmtree(latest)
                    os.makedirs(latest); acc.save_state(latest)
                    torch.save({"step": gs}, os.path.join(latest,"meta.pt"))
                    log(f"  local checkpoint @ {gs}")
            if gs >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if gs >= NUM_TRAIN_STEPS: break

    if is_main:
        fin = validate(gs, "FINAL")
        tmp = os.path.join(OUTPUT_DIR,"_final"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k,v in acc.unwrap_model(unet).state_dict().items()}
        save_file(sd, os.path.join(tmp,"unet.safetensors"))
        json.dump({**cfg, "method":"residual_resshift", "regressor_tag": REG_TAG,
                   "fixed_tb": FIXED_TB, "final_step": gs,
                   "alpha_curve": {str(k): v for k,v in fin.items()}},
                  open(os.path.join(tmp,"bridge_config.json"),"w"), indent=2)
        pushed = False
        try:
            HfApi().upload_folder(folder_path=tmp, path_in_repo=OUT_TAG, repo_id=HF_REPO_ID,
                                  repo_type="model", token=HF_TOKEN)
            log(f"  FINAL -> HF/{OUT_TAG}"); pushed = True
        except Exception as e: log(f"  push failed: {e} — weights KEPT at {tmp}")
        if pushed: shutil.rmtree(tmp, ignore_errors=True)
    acc.end_training(); log("Script B complete.")

if __name__ == "__main__":
    main()

Overwriting /content/stageRes_residual_bridge.py


In [ ]:
!python -u /content/stageRes_residual_bridge.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
  regressor regressor_final (tb=933, reported PSNR 17.062663894809436) | bridge init bridge_final | scale=1.4010

[STEP-0 (regressor + untrained residual)]
    alpha=0.0  PSNR 17.023 | 

In [ ]:
%%writefile /content/stageR2_regressor_v2.py
"""
Script C — Regressor v2. Init from regressor_final (17.06 dB / 0.285 SSIM).
Stacks the document's remaining fidelity levers onto the deterministic model:
  §1.A shift-tolerant latent loss  (min over +/-1 latent-cell shifts of target)
        -> absorbs SEN1-2 misregistration, which hits SSIM hardest
  §2.C MIND structure loss          (modality-invariant, pred vs SAR input)
        -> targets SSIM, the weaker metric vs SOTA (0.285 vs 0.312)
  §2.D spectral-angle (SAM) loss    (hue-only, scale-invariant; distinct from
        the failed E1 low-freq colour L1 which was brightness-dominated)
  §0.A EMA weights (0.9995)         -> +0.1-0.4 dB and stabler validation
Each loss has its own flag so ablations are one-line changes.
GATE @15k: PSNR >= 17.20 AND SSIM >= 0.288 (must beat regressor v1), else kill.
"""
import os, gc, json, time, random, warnings, shutil
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import save_file, load_file
from torchmetrics.image import StructuralSimilarityIndexMeasure
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR = "/content/stageR2_regressor_v2"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG, INIT_TAG, OUT_TAG = "adapted16_final", "regressor_final", "regressor_v2_final"

IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
TERRAINS = ("tropical","temperate","arctic","arid","coastal","urban")
TOTAL_CH = 16

NUM_TRAIN_STEPS = 40_000
BATCH_SIZE   = 4
LR           = 2e-5          # lower than v1: refining an already-good model
LR_WARMUP    = 1_500
COND_DROPOUT = 0.10

# ---- loss switches (set any lambda to 0 to ablate that component) ----
USE_SHIFT_TOLERANT = True
SHIFT_RADIUS       = 1        # +/-1 latent cell == +/-8 px image; doc §1.A
LAMBDA_PIXEL_L1    = 0.5      # decoded L1 (kept from v1)
LAMBDA_MIND        = 0.08     # doc §2.C suggests 0.05-0.1
MIND_RADIUS        = 3        # larger radius averages speckle out (X2 lesson)
LAMBDA_SAM         = 0.05     # doc §2.D
AUX_START          = 2_000    # warm up on latent loss before aux losses engage
AUX_RAMP           = 3_000    # ramp aux weights 0->1 (E3 lesson: no abrupt switch)

EMA_DECAY   = 0.9995
VAL_EVERY   = 2_500
N_VAL       = 150
LOCAL_SAVE_EVERY = 10_000
STEP_LOG_EVERY   = 500
SPIKE_MULT, GUARD_WARMUP = 10.0, 500
GATE_STEP, GATE_PSNR, GATE_SSIM = 15_000, 17.20, 0.288
STEP0_BAND = (16.6, 17.5)     # must reproduce regressor v1 (~17.06) or abort

device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    random.shuffle(ps); return ps

class PairDataset(Dataset):
    def __init__(self, pairs, augment=True): self.p, self.aug = pairs, augment
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        sar, opt = self._l(p["sar"]), self._l(p["opt"])
        if self.aug:
            if random.random() > .5: sar, opt = torch.flip(sar,[2]), torch.flip(opt,[2])
            if random.random() > .5: sar, opt = torch.flip(sar,[1]), torch.flip(opt,[1])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=12, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    def dec_grad(s,l): return s.base_vae.decoder(l)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def load_vae():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    return v.to(device).eval().float().requires_grad_(False)

def build_unet_from_v1():
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/bridge_config.json")))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/unet.safetensors")))
    log(f"  init from {INIT_TAG} (PSNR {cfg.get('final_psnr','?')}, SSIM {cfg.get('final_ssim','?')})")
    return unet, cfg

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

def get_cond(ss, tt, cache, drop=0.0):
    return torch.stack([cache["null"] if (drop and random.random() < drop)
                        else cache.get((s,t), cache["null"]) for s,t in zip(ss,tt)])

# ── §1.A shift-tolerant latent loss ─────────────────────────────────────────
def shift_tolerant_l2(pred, target, radius=SHIFT_RADIUS):
    best = None
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            t = torch.roll(target, shifts=(dx, dy), dims=(2,3))
            l = ((pred - t)**2).mean(dim=(1,2,3))          # per-sample
            best = l if best is None else torch.minimum(best, l)
    return best.mean()

# ── §2.C MIND descriptor (Heinrich et al. 2012), modality-invariant ─────────
def mind_descriptor(img, radius=MIND_RADIUS, patch=3):
    gray = img.mean(1, keepdim=True)
    offs = [(radius,0), (-radius,0), (0,radius), (0,-radius)]
    D = []
    for dx, dy in offs:
        sh = torch.roll(gray, shifts=(dx,dy), dims=(2,3))
        D.append(F.avg_pool2d((gray - sh)**2, patch, stride=1, padding=patch//2))
    D = torch.cat(D, 1)
    V = D.mean(dim=1, keepdim=True).clamp(min=1e-6)
    m = torch.exp(-D / V)
    return m / m.max(dim=1, keepdim=True)[0].clamp(min=1e-8)

def mind_loss(pred_img, sar_img):
    return F.l1_loss(mind_descriptor(pred_img), mind_descriptor(sar_img))

# ── §2.D spectral-angle loss (hue-only, brightness-invariant) ───────────────
def sam_loss(pred_img, gt_img):
    p = ((pred_img+1)/2).clamp(1e-4,1).flatten(2).transpose(1,2)
    g = ((gt_img+1)/2).clamp(1e-4,1).flatten(2).transpose(1,2)
    cos = (p*g).sum(-1) / (p.norm(dim=-1)*g.norm(dim=-1) + 1e-8)
    return torch.acos(cos.clamp(-1+1e-6, 1-1e-6)).mean()

def main():
    acc = Accelerator(mixed_precision="no", project_dir=OUTPUT_DIR)
    set_seed(SEED); is_main = acc.is_main_process
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    vae = load_vae(); embeds = build_embeds()
    unet, cfg = build_unet_from_v1()
    SCALE, FIXED_TB = float(cfg["latent_scale"]), int(cfg["fixed_tb"])
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
    log(f"  scale={SCALE:.4f} fixed_tb={FIXED_TB} | shift={USE_SHIFT_TOLERANT}(r={SHIFT_RADIUS}) "
        f"MIND={LAMBDA_MIND} SAM={LAMBDA_SAM} pixL1={LAMBDA_PIXEL_L1} EMA={EMA_DECAY}")

    pairs = collect(DATA_ROOT)
    tr, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    log(f"Train {len(tr):,} | Val {len(va):,}")
    dl = DataLoader(PairDataset(tr, True), batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)

    opt_ = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9,0.999), weight_decay=1e-2)
    sch = get_cosine_schedule_with_warmup(opt_, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, opt_, dl, sch = acc.prepare(unet, opt_, dl, sch)

    ema_unet = AveragedModel(acc.unwrap_model(unet),
                             multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY)).to(device)
    def ema_decay_at(s): return min(EMA_DECAY, (s+1)/(s+10))    # bias-corrected warmup

    def enc(x):
        with torch.no_grad(): return vae.enc(x.float()) * SCALE

    def regress(m, y, cond):
        B = y.shape[0]
        tb = torch.full((B,), FIXED_TB, device=device, dtype=torch.long)
        return m(torch.cat([y, y], 1), tb, encoder_hidden_states=cond).sample

    @torch.no_grad()
    def validate(step, tag=None):
        sub = random.Random(7).sample(va, N_VAL)
        res = {}
        for name, m in (("RAW", acc.unwrap_model(unet)), ("EMA", ema_unet.module)):
            was = m.training; m.eval(); P, S = [], []
            for b in DataLoader(PairDataset(sub, False), batch_size=8, num_workers=2):
                o = b["opt"].to(device); y = enc(b["sar"].to(device))
                c = get_cond(b["season"], b["terrain"], embeds).to(device)
                rec = vae.dec((regress(m, y, c)/SCALE)).clamp(-1,1)
                for j in range(rec.shape[0]):                  # PER-IMAGE protocol
                    P.append(10*np.log10(4.0/F.mse_loss(rec[j:j+1], o[j:j+1]).item()))
                    S.append(ssim_fn(rec[j:j+1], o[j:j+1]).item())
            if was: m.train()
            res[name] = (float(np.mean(P)), float(np.mean(S)))
        lbl = tag or f"VAL step {step}"
        log(f"\n[{lbl}] RAW PSNR {res['RAW'][0]:.3f} SSIM {res['RAW'][1]:.4f} | "
            f"EMA PSNR {res['EMA'][0]:.3f} SSIM {res['EMA'][1]:.4f}")
        log(f"    (v1 regressor 17.06/0.285 | champion 16.73/0.259 | SOTA 19.72/0.312)\n")
        return res

    base = validate(0, "STEP-0 (must reproduce v1)")
    if not (STEP0_BAND[0] <= base["RAW"][0] <= STEP0_BAND[1]):
        log(f"!! STEP-0 GATE FAILED: {base['RAW'][0]:.2f} outside {STEP0_BAND} — "
            f"harness/checkpoint mismatch. ABORTING."); return
    log(">>> Step-0 PASSED. Aux losses engage at "
        f"{AUX_START} and ramp over {AUX_RAMP} steps.\n")

    latest = os.path.join(OUTPUT_DIR,"latest")
    gs = 0; ema_l = 0.0; a_ = 0.98; ng = nn_ = 0; t0 = time.time(); last = 0
    unet.train()
    log(f">>> Regressor v2: 0 -> {NUM_TRAIN_STEPS}. "
        f"GATE @ {GATE_STEP}: PSNR >= {GATE_PSNR} AND SSIM >= {GATE_SSIM}\n")

    for epoch in range(999):
        for batch in dl:
            oimg = batch["opt"].to(device); simg = batch["sar"].to(device)
            x0 = enc(oimg); y = enc(simg)
            c = get_cond(batch["season"], batch["terrain"], embeds, COND_DROPOUT).to(device)
            with acc.accumulate(unet):
                pred = regress(unet, y, c)
                loss_lat = (shift_tolerant_l2(pred, x0) if USE_SHIFT_TOLERANT
                            else F.mse_loss(pred, x0))
                loss = loss_lat
                l_pix = l_mind = l_sam = 0.0
                if gs >= AUX_START:
                    w = min(1.0, (gs - AUX_START) / max(AUX_RAMP, 1))
                    img = vae.dec_grad(pred/SCALE).clamp(-1,1)
                    if LAMBDA_PIXEL_L1 > 0:
                        t_pix = F.l1_loss(img, oimg); l_pix = float(t_pix)
                        loss = loss + w*LAMBDA_PIXEL_L1*t_pix
                    if LAMBDA_MIND > 0:
                        t_mind = mind_loss(img, simg); l_mind = float(t_mind)
                        loss = loss + w*LAMBDA_MIND*t_mind
                    if LAMBDA_SAM > 0:
                        t_sam = sam_loss(img, oimg); l_sam = float(t_sam)
                        loss = loss + w*LAMBDA_SAM*t_sam
                if gs > GUARD_WARMUP and ema_l > 0 and loss_lat.item() > SPIKE_MULT*max(ema_l,1e-4):
                    opt_.zero_grad(set_to_none=True); ng += 1; continue
                if not torch.isfinite(loss):
                    opt_.zero_grad(set_to_none=True); nn_ += 1; continue
                acc.backward(loss)
                if acc.sync_gradients: acc.clip_grad_norm_(unet.parameters(), 1.0)
                opt_.step(); sch.step(); opt_.zero_grad(set_to_none=True)
            if acc.sync_gradients:
                ema_unet.avg_fn = get_ema_multi_avg_fn(ema_decay_at(gs))
                ema_unet.update_parameters(acc.unwrap_model(unet))
                gs += 1; ema_l = a_*ema_l + (1-a_)*loss_lat.item()
                if is_main and gs % STEP_LOG_EVERY == 0:
                    r = (gs-last)/max(time.time()-t0,1e-9); t0, last = time.time(), gs
                    log(f"step {gs}/{NUM_TRAIN_STEPS} | lat {ema_l:.5f} | pix {l_pix:.4f} "
                        f"mind {l_mind:.4f} sam {l_sam:.4f} | g={ng} n={nn_} | {r:.2f} it/s")
                if gs % VAL_EVERY == 0 and is_main:
                    v = validate(gs)
                    if gs == GATE_STEP:
                        best = max(v["RAW"], v["EMA"], key=lambda t: t[0])
                        ok = best[0] >= GATE_PSNR and best[1] >= GATE_SSIM
                        log(f"  GATE @ {GATE_STEP}: best PSNR {best[0]:.2f} (>= {GATE_PSNR}) | "
                            f"SSIM {best[1]:.4f} (>= {GATE_SSIM}) -> "
                            f"{'PASS, continue' if ok else 'FAIL — recommend killing'}")
                if gs % LOCAL_SAVE_EVERY == 0 and is_main:
                    if os.path.exists(latest): shutil.rmtree(latest)
                    os.makedirs(latest); acc.save_state(latest)
                    torch.save({"step": gs}, os.path.join(latest,"meta.pt"))
                    log(f"  local checkpoint @ {gs}")
            if gs >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if gs >= NUM_TRAIN_STEPS: break

    if is_main:
        fin = validate(gs, "FINAL")
        which = "EMA" if fin["EMA"][0] >= fin["RAW"][0] else "RAW"
        model = ema_unet.module if which == "EMA" else acc.unwrap_model(unet)
        log(f"  saving {which} weights (PSNR {fin[which][0]:.3f}, SSIM {fin[which][1]:.4f})")
        tmp = os.path.join(OUTPUT_DIR,"_final"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k,v in model.state_dict().items()}
        save_file(sd, os.path.join(tmp,"unet.safetensors"))
        json.dump({"latent_scale": SCALE, "fixed_tb": FIXED_TB,
                   "method": "deterministic_regressor_v2", "saved_weights": which,
                   "shift_tolerant": USE_SHIFT_TOLERANT, "shift_radius": SHIFT_RADIUS,
                   "lambda_mind": LAMBDA_MIND, "lambda_sam": LAMBDA_SAM,
                   "lambda_pixel_l1": LAMBDA_PIXEL_L1, "ema_decay": EMA_DECAY,
                   "final_step": gs, "final_psnr": fin[which][0], "final_ssim": fin[which][1],
                   "step0": {"psnr": base["RAW"][0], "ssim": base["RAW"][1]}},
                  open(os.path.join(tmp,"bridge_config.json"),"w"), indent=2)
        pushed = False
        try:
            HfApi().upload_folder(folder_path=tmp, path_in_repo=OUT_TAG, repo_id=HF_REPO_ID,
                                  repo_type="model", token=HF_TOKEN)
            log(f"  FINAL -> HF/{OUT_TAG}"); pushed = True
        except Exception as e: log(f"  push failed: {e} — weights KEPT at {tmp}")
        if pushed: shutil.rmtree(tmp, ignore_errors=True)
    acc.end_training(); log("Script C complete.")

if __name__ == "__main__":
    main()

Writing /content/stageR2_regressor_v2.py


In [ ]:
!python -u /content/stageR2_regressor_v2.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
  init from regressor_final (PSNR 17.062663894809436, SSIM 0.28508463614154606)
  scale=1.4010 fixed_tb=933 | shift=True(r=1) MIND=0.08 SAM=0.05 pixL1=0.5 EMA=0.9995
Train 292,486 | Val

In [ ]:
%%writefile /content/stageD_affine_head.py
"""
Script D — §1.C scene-adaptive colour affine head, with the oracle go/no-go.

X1 falsified CLASS-prior colour calibration and concluded the residual colour
error is scene-specific. That conclusion is an argument FOR a scene-specific
correction — X1 tested the wrong granularity, not the wrong idea.

Head H: (z_sar, CLIP season x terrain) -> (gain, bias) in R^6, applied as a
final per-RGB affine on the decoded output. Regressor stays FROZEN.

THE CRITICAL TEST (runs at step 0, before training):
  baseline  : no correction
  ORACLE    : per-image per-channel affine fitted with least squares USING GT
              -> the maximum any affine correction could possibly achieve
  head      : what the model actually predicts from SAR alone
Go/no-go: head must capture >= 25% of (oracle - baseline) PSNR gain by 5k steps.
If the oracle gain itself is < 0.15 dB, the whole direction is dead -> abort
immediately (costs ~10 minutes, not 5k steps).
"""
import os, gc, json, time, random, warnings, shutil
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import save_file, load_file
from torchmetrics.image import StructuralSimilarityIndexMeasure
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR = "/content/stageD_affine"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG = "adapted16_final"
INIT_TAG = "regressor_v2_final"     # <- fall back to "regressor_final" if C failed
OUT_TAG  = "affine_head_final"

IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
TERRAINS = ("tropical","temperate","arctic","arid","coastal","urban")
TOTAL_CH = 16

NUM_TRAIN_STEPS = 6_000
BATCH_SIZE = 4
LR = 1e-4                 # tiny head, can afford a higher LR
LR_WARMUP = 200
VAL_EVERY = 1_000
N_VAL = 150
N_ORACLE = 150
STEP_LOG_EVERY = 250
GATE_STEP = 5_000
GATE_ORACLE_FRACTION = 0.25       # head must capture >=25% of oracle gain
MIN_ORACLE_GAIN = 0.15            # if oracle gain below this, abort at step 0

device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    random.shuffle(ps); return ps

class PairDataset(Dataset):
    def __init__(self, pairs, augment=True): self.p, self.aug = pairs, augment
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        sar, opt = self._l(p["sar"]), self._l(p["opt"])
        if self.aug:
            if random.random() > .5: sar, opt = torch.flip(sar,[2]), torch.flip(opt,[2])
            if random.random() > .5: sar, opt = torch.flip(sar,[1]), torch.flip(opt,[1])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=12, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def load_vae():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    return v.to(device).eval().float().requires_grad_(False)

def load_regressor():
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/bridge_config.json")))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/unet.safetensors")))
    log(f"  frozen regressor {INIT_TAG} (PSNR {cfg.get('final_psnr','?')}, SSIM {cfg.get('final_ssim','?')})")
    return unet.to(device).eval().requires_grad_(False), cfg

# ── the tiny head: (z_sar, cond) -> per-RGB gain & bias ────────────────────
class AffineHead(nn.Module):
    def __init__(self, in_ch=TOTAL_CH, cond_dim=768, nf=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1), nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf*2, 3, 2, 1),  nn.GroupNorm(8, nf*2), nn.SiLU(),
            nn.Conv2d(nf*2, nf*2, 3, 2, 1), nn.GroupNorm(8, nf*2), nn.SiLU())
        self.cond_proj = nn.Linear(cond_dim, nf*2)
        self.mlp = nn.Sequential(nn.Linear(nf*4, 128), nn.SiLU(), nn.Linear(128, 6))
        nn.init.zeros_(self.mlp[-1].weight); nn.init.zeros_(self.mlp[-1].bias)  # identity at init
    def forward(self, z_sar, cond):
        h = self.conv(z_sar).mean(dim=(2,3))                 # (B, nf*2)
        c = self.cond_proj(cond.mean(dim=1))                 # pool 77 tokens -> (B, nf*2)
        out = self.mlp(torch.cat([h, c], dim=1))             # (B, 6)
        gain = 1.0 + out[:, :3].view(-1,3,1,1)               # identity when out==0
        bias = out[:, 3:].view(-1,3,1,1)
        return gain, bias

def apply_affine(img, gain, bias):
    """img in [-1,1] -> operate in [0,1] then map back."""
    x = (img + 1) / 2
    x = (x * gain + bias).clamp(0, 1)
    return x * 2 - 1

@torch.no_grad()
def oracle_affine(pred, gt):
    """Per-image, per-channel least-squares affine fitted USING ground truth.
    Upper bound on any affine correction. pred/gt in [-1,1], shape (1,3,H,W)."""
    p = ((pred+1)/2).flatten(2)[0]      # (3, HW)
    g = ((gt+1)/2).flatten(2)[0]
    pm, gm = p.mean(1, keepdim=True), g.mean(1, keepdim=True)
    var = ((p-pm)**2).mean(1, keepdim=True).clamp(min=1e-8)
    cov = ((p-pm)*(g-gm)).mean(1, keepdim=True)
    a = (cov/var).view(1,3,1,1); b = (gm - (cov/var)*pm).view(1,3,1,1)
    return a, b

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

def get_cond(ss, tt, cache):
    return torch.stack([cache.get((s,t), cache["null"]) for s,t in zip(ss,tt)])

def sam_deg(p, g):
    p = ((p+1)/2).clamp(1e-4,1).flatten(2).transpose(1,2)
    g = ((g+1)/2).clamp(1e-4,1).flatten(2).transpose(1,2)
    c = ((p*g).sum(-1)/(p.norm(dim=-1)*g.norm(dim=-1)+1e-8)).clamp(-1,1)
    return float(torch.rad2deg(torch.acos(c)).mean())

def main():
    acc = Accelerator(mixed_precision="no", project_dir=OUTPUT_DIR)
    set_seed(SEED); is_main = acc.is_main_process
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    vae = load_vae(); embeds = build_embeds()
    reg, rcfg = load_regressor()
    SCALE, FIXED_TB = float(rcfg["latent_scale"]), int(rcfg["fixed_tb"])
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
    head = AffineHead()

    pairs = collect(DATA_ROOT)
    tr, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    log(f"Train {len(tr):,} | Val {len(va):,}")
    dl = DataLoader(PairDataset(tr, True), batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)

    opt_ = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
    sch = get_cosine_schedule_with_warmup(opt_, LR_WARMUP, NUM_TRAIN_STEPS)
    head, opt_, dl, sch = acc.prepare(head, opt_, dl, sch)

    def enc(x):
        with torch.no_grad(): return vae.enc(x.float()) * SCALE

    @torch.no_grad()
    def predict(y, cond):
        B = y.shape[0]
        tb = torch.full((B,), FIXED_TB, device=device, dtype=torch.long)
        z = reg(torch.cat([y, y], 1), tb, encoder_hidden_states=cond).sample
        return vae.dec(z/SCALE).clamp(-1,1)

    @torch.no_grad()
    def evaluate(step, with_oracle=False, tag=None):
        h = acc.unwrap_model(head); h.eval()
        sub = random.Random(7).sample(va, N_VAL)
        base_P, base_S, base_A = [], [], []
        head_P, head_S, head_A = [], [], []
        orc_P,  orc_S,  orc_A  = [], [], []
        for b in DataLoader(PairDataset(sub, False), batch_size=8, num_workers=2):
            o = b["opt"].to(device); y = enc(b["sar"].to(device))
            c = get_cond(b["season"], b["terrain"], embeds).to(device)
            base = predict(y, c)
            g_, b_ = h(y, c)
            hed = apply_affine(base, g_, b_)
            for j in range(base.shape[0]):
                bj, hj, oj = base[j:j+1], hed[j:j+1], o[j:j+1]
                base_P.append(10*np.log10(4.0/F.mse_loss(bj, oj).item()))
                base_S.append(ssim_fn(bj, oj).item()); base_A.append(sam_deg(bj, oj))
                head_P.append(10*np.log10(4.0/F.mse_loss(hj, oj).item()))
                head_S.append(ssim_fn(hj, oj).item()); head_A.append(sam_deg(hj, oj))
                if with_oracle:
                    ga, ba = oracle_affine(bj, oj)
                    orj = apply_affine(bj, ga, ba)
                    orc_P.append(10*np.log10(4.0/F.mse_loss(orj, oj).item()))
                    orc_S.append(ssim_fn(orj, oj).item()); orc_A.append(sam_deg(orj, oj))
        h.train()
        lbl = tag or f"step {step}"
        log(f"\n[{lbl}]")
        log(f"    baseline   PSNR {np.mean(base_P):.3f} | SSIM {np.mean(base_S):.4f} | SAM {np.mean(base_A):.2f}")
        log(f"    +head      PSNR {np.mean(head_P):.3f} | SSIM {np.mean(head_S):.4f} | SAM {np.mean(head_A):.2f}")
        res = {"base": (float(np.mean(base_P)), float(np.mean(base_S))),
               "head": (float(np.mean(head_P)), float(np.mean(head_S)))}
        if with_oracle:
            log(f"    ORACLE     PSNR {np.mean(orc_P):.3f} | SSIM {np.mean(orc_S):.4f} | SAM {np.mean(orc_A):.2f}"
                f"   <- upper bound for ANY affine")
            res["oracle"] = (float(np.mean(orc_P)), float(np.mean(orc_S)))
        log("")
        return res

    # ── STEP 0: the decisive oracle test, BEFORE training ──
    s0 = evaluate(0, with_oracle=True, tag="STEP-0 + ORACLE (decides whether to train at all)")
    oracle_gain = s0["oracle"][0] - s0["base"][0]
    log(f">>> ORACLE GAIN: {oracle_gain:+.3f} dB (max any per-image affine could give)")
    if oracle_gain < MIN_ORACLE_GAIN:
        log(f"!! Oracle gain {oracle_gain:.3f} < {MIN_ORACLE_GAIN} dB — there is essentially no "
            f"affine correction to be found. Direction is DEAD. Aborting (cost: ~10 min).")
        return
    log(f">>> Worth training. Head must capture >= {GATE_ORACLE_FRACTION*100:.0f}% of it "
        f"({GATE_ORACLE_FRACTION*oracle_gain:+.3f} dB) by step {GATE_STEP}.\n")

    gs = 0; ema = 0.0; a_ = 0.98; t0 = time.time(); last = 0
    head.train()
    for epoch in range(999):
        for batch in dl:
            oimg = batch["opt"].to(device)
            y = enc(batch["sar"].to(device))
            c = get_cond(batch["season"], batch["terrain"], embeds).to(device)
            base = predict(y, c)                       # frozen regressor, no grad
            with acc.accumulate(head):
                g_, b_ = head(y, c)
                corrected = apply_affine(base, g_, b_)
                loss = F.mse_loss(corrected, oimg) + 0.1*F.l1_loss(corrected, oimg)
                if not torch.isfinite(loss):
                    opt_.zero_grad(set_to_none=True); continue
                acc.backward(loss)
                if acc.sync_gradients: acc.clip_grad_norm_(head.parameters(), 1.0)
                opt_.step(); sch.step(); opt_.zero_grad(set_to_none=True)
            if acc.sync_gradients:
                gs += 1; ema = a_*ema + (1-a_)*loss.item()
                if is_main and gs % STEP_LOG_EVERY == 0:
                    r = (gs-last)/max(time.time()-t0,1e-9); t0, last = time.time(), gs
                    log(f"step {gs}/{NUM_TRAIN_STEPS} | loss {ema:.5f} | {r:.2f} it/s")
                if gs % VAL_EVERY == 0 and is_main:
                    v = evaluate(gs, with_oracle=(gs == GATE_STEP))
                    if gs == GATE_STEP:
                        captured = v["head"][0] - v["base"][0]
                        frac = captured / max(oracle_gain, 1e-6)
                        log(f"  GATE @ {GATE_STEP}: head gain {captured:+.3f} dB = "
                            f"{frac*100:.1f}% of oracle ({oracle_gain:+.3f}) -> "
                            f"{'PASS — real signal' if frac >= GATE_ORACLE_FRACTION else 'FAIL — affine unpredictable from SAR, kill'}")
            if gs >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if gs >= NUM_TRAIN_STEPS: break

    if is_main:
        fin = evaluate(gs, with_oracle=True, tag="FINAL")
        tmp = os.path.join(OUTPUT_DIR,"_final"); os.makedirs(tmp, exist_ok=True)
        save_file({k: v.contiguous().cpu() for k,v in acc.unwrap_model(head).state_dict().items()},
                  os.path.join(tmp,"affine_head.safetensors"))
        json.dump({"regressor_tag": INIT_TAG, "latent_scale": SCALE, "fixed_tb": FIXED_TB,
                   "baseline_psnr": fin["base"][0], "head_psnr": fin["head"][0],
                   "oracle_psnr": fin["oracle"][0], "final_step": gs},
                  open(os.path.join(tmp,"head_config.json"),"w"), indent=2)
        pushed = False
        try:
            HfApi().upload_folder(folder_path=tmp, path_in_repo=OUT_TAG, repo_id=HF_REPO_ID,
                                  repo_type="model", token=HF_TOKEN)
            log(f"  FINAL -> HF/{OUT_TAG}"); pushed = True
        except Exception as e: log(f"  push failed: {e} — weights KEPT at {tmp}")
        if pushed: shutil.rmtree(tmp, ignore_errors=True)
    acc.end_training(); log("Script D complete.")

if __name__ == "__main__":
    main()

Writing /content/stageD_affine_head.py


In [ ]:
!python -u /content/stageD_affine_head.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
  frozen regressor regressor_v2_final (PSNR 17.448642927105023, SSIM 0.2904802869229267)
Train 292,486 | Val 15,394

[STEP-0 + ORACLE (decides whether to train at all)]
    baseline   P

In [ ]:
%%writefile /content/stageE_heteroscedastic.py
"""
Script E — §2.B confidence-guided (heteroscedastic) regressor.
L = (pred - target)^2 / (2*sigma^2) + log sigma,  sigma from extra output channels.
Zero-init the new 16 logvar channels -> sigma=1 at step 0 -> loss is 0.5*MSE,
so step-0 prediction is IDENTICAL to the regressor (verified by the step-0 gate).
Doc caveat honoured: logvar is CLAMPED and mean sigma is logged every 500 steps —
if mean sigma drifts monotonically upward the run has degenerated, kill it.
Bonus deliverable: the confidence map is a publishable figure localising the
information ceiling (should light up on water, fields, dense urban).
"""
import os, gc, json, time, random, warnings, shutil
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import save_file, load_file
from torchmetrics.image import StructuralSimilarityIndexMeasure
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR = "/content/stageE_hetero"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG = "adapted16_final"
INIT_TAG = "regressor_v2_final"      # fall back to "regressor_final" if C failed
OUT_TAG  = "hetero_final"

IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
TERRAINS = ("tropical","temperate","arctic","arid","coastal","urban")
TOTAL_CH = 16

NUM_TRAIN_STEPS = 40_000
BATCH_SIZE = 4
LR = 2e-5
LR_WARMUP = 1_500
COND_DROPOUT = 0.10
LOGVAR_MIN, LOGVAR_MAX = -6.0, 2.0   # clamp: prevents the sigma-explodes degeneracy
LAMBDA_PIXEL_L1 = 0.5
PIXEL_START = 2_000
VAL_EVERY = 2_500
N_VAL = 150
LOCAL_SAVE_EVERY = 10_000
STEP_LOG_EVERY = 500
SPIKE_MULT, GUARD_WARMUP = 10.0, 500
GATE_STEP, GATE_PSNR = 15_000, 17.20
STEP0_BAND = (16.6, 17.6)

device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    random.shuffle(ps); return ps

class PairDataset(Dataset):
    def __init__(self, pairs, augment=True): self.p, self.aug = pairs, augment
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        sar, opt = self._l(p["sar"]), self._l(p["opt"])
        if self.aug:
            if random.random() > .5: sar, opt = torch.flip(sar,[2]), torch.flip(opt,[2])
            if random.random() > .5: sar, opt = torch.flip(sar,[1]), torch.flip(opt,[1])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=12, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    def dec_grad(s,l): return s.base_vae.decoder(l)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def load_vae():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    return v.to(device).eval().float().requires_grad_(False)

def build_hetero_unet():
    """Load regressor, then widen conv_out 16 -> 32ch. New 16 (logvar) zero-init."""
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/bridge_config.json")))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/unet.safetensors")))
    # widen output head
    old = unet.conv_out
    new = nn.Conv2d(old.in_channels, 2*TOTAL_CH, old.kernel_size, old.stride, old.padding)
    with torch.no_grad():
        new.weight.zero_(); new.bias.zero_()
        new.weight[:TOTAL_CH] = old.weight; new.bias[:TOTAL_CH] = old.bias   # pred = identical
        # logvar channels stay zero -> sigma = 1 at step 0
    unet.conv_out = new; unet.config.out_channels = 2*TOTAL_CH
    log(f"  init {INIT_TAG} (PSNR {cfg.get('final_psnr','?')}), conv_out widened 16->32 "
        f"(logvar zero-init, sigma=1 at step 0)")
    return unet, cfg

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

def get_cond(ss, tt, cache, drop=0.0):
    return torch.stack([cache["null"] if (drop and random.random() < drop)
                        else cache.get((s,t), cache["null"]) for s,t in zip(ss,tt)])

def main():
    acc = Accelerator(mixed_precision="no", project_dir=OUTPUT_DIR)
    set_seed(SEED); is_main = acc.is_main_process
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    vae = load_vae(); embeds = build_embeds()
    unet, cfg = build_hetero_unet()
    SCALE, FIXED_TB = float(cfg["latent_scale"]), int(cfg["fixed_tb"])
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    pairs = collect(DATA_ROOT)
    tr, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    log(f"Train {len(tr):,} | Val {len(va):,}")
    dl = DataLoader(PairDataset(tr, True), batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)

    opt_ = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9,0.999), weight_decay=1e-2)
    sch = get_cosine_schedule_with_warmup(opt_, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, opt_, dl, sch = acc.prepare(unet, opt_, dl, sch)

    def enc(x):
        with torch.no_grad(): return vae.enc(x.float()) * SCALE

    def forward(m, y, cond):
        B = y.shape[0]
        tb = torch.full((B,), FIXED_TB, device=device, dtype=torch.long)
        out = m(torch.cat([y, y], 1), tb, encoder_hidden_states=cond).sample
        pred, logvar = out[:, :TOTAL_CH], out[:, TOTAL_CH:].clamp(LOGVAR_MIN, LOGVAR_MAX)
        return pred, logvar

    @torch.no_grad()
    def validate(step, tag=None):
        m = acc.unwrap_model(unet); m.eval()
        sub = random.Random(7).sample(va, N_VAL)
        P, S, SIG = [], [], []
        for b in DataLoader(PairDataset(sub, False), batch_size=8, num_workers=2):
            o = b["opt"].to(device); y = enc(b["sar"].to(device))
            c = get_cond(b["season"], b["terrain"], embeds).to(device)
            pred, logvar = forward(m, y, c)
            SIG.append(float(torch.exp(0.5*logvar).mean()))
            rec = vae.dec(pred/SCALE).clamp(-1,1)
            for j in range(rec.shape[0]):
                P.append(10*np.log10(4.0/F.mse_loss(rec[j:j+1], o[j:j+1]).item()))
                S.append(ssim_fn(rec[j:j+1], o[j:j+1]).item())
        m.train()
        lbl = tag or f"VAL step {step}"
        log(f"\n[{lbl}] PSNR {np.mean(P):.3f} | SSIM {np.mean(S):.4f} | mean sigma {np.mean(SIG):.4f}")
        log(f"    (regressor v1 17.06/0.285 | champion 16.73/0.259 | SOTA 19.72/0.312)\n")
        return float(np.mean(P)), float(np.mean(S)), float(np.mean(SIG))

    p0, s0, sig0 = validate(0, "STEP-0 (must reproduce the regressor)")
    if not (STEP0_BAND[0] <= p0 <= STEP0_BAND[1]):
        log(f"!! STEP-0 GATE FAILED: {p0:.2f} outside {STEP0_BAND} — conv_out surgery or "
            f"checkpoint mismatch. ABORTING."); return
    log(f">>> Step-0 PASSED (sigma={sig0:.3f}, expected ~1.0). Watch mean sigma: monotonic "
        f"upward drift = degenerate run, kill it.\n")

    latest = os.path.join(OUTPUT_DIR,"latest")
    gs = 0; ema = 0.0; a_ = 0.98; ng = nn_ = 0; t0 = time.time(); last = 0
    sig_hist = []
    unet.train()
    log(f">>> Heteroscedastic regressor 0 -> {NUM_TRAIN_STEPS}. GATE @ {GATE_STEP}: PSNR >= {GATE_PSNR}\n")

    for epoch in range(999):
        for batch in dl:
            oimg = batch["opt"].to(device)
            x0 = enc(oimg); y = enc(batch["sar"].to(device))
            c = get_cond(batch["season"], batch["terrain"], embeds, COND_DROPOUT).to(device)
            with acc.accumulate(unet):
                pred, logvar = forward(unet, y, c)
                # heteroscedastic NLL: downweight pixels the model cannot predict
                nll = (0.5*torch.exp(-logvar)*(pred - x0)**2 + 0.5*logvar).mean()
                loss = nll
                if gs >= PIXEL_START and LAMBDA_PIXEL_L1 > 0:
                    img = vae.dec_grad(pred/SCALE).clamp(-1,1)
                    loss = loss + LAMBDA_PIXEL_L1*F.l1_loss(img, oimg)
                if gs > GUARD_WARMUP and ema > 0 and nll.item() > SPIKE_MULT*max(ema,1e-4):
                    opt_.zero_grad(set_to_none=True); ng += 1; continue
                if not torch.isfinite(loss):
                    opt_.zero_grad(set_to_none=True); nn_ += 1; continue
                acc.backward(loss)
                if acc.sync_gradients: acc.clip_grad_norm_(unet.parameters(), 1.0)
                opt_.step(); sch.step(); opt_.zero_grad(set_to_none=True)
            if acc.sync_gradients:
                gs += 1; ema = a_*ema + (1-a_)*nll.item()
                if is_main and gs % STEP_LOG_EVERY == 0:
                    ms = float(torch.exp(0.5*logvar).mean()); sig_hist.append(ms)
                    r = (gs-last)/max(time.time()-t0,1e-9); t0, last = time.time(), gs
                    warn = ""
                    if len(sig_hist) >= 6 and all(sig_hist[-i] > sig_hist[-i-1] for i in range(1,6)):
                        warn = "  <-- WARNING: sigma rising 5 logs straight, possible degeneracy"
                    log(f"step {gs}/{NUM_TRAIN_STEPS} | nll {ema:.5f} | sigma {ms:.4f} | "
                        f"g={ng} n={nn_} | {r:.2f} it/s{warn}")
                if gs % VAL_EVERY == 0 and is_main:
                    p, s, sg = validate(gs)
                    if gs == GATE_STEP:
                        log(f"  GATE @ {GATE_STEP}: PSNR {p:.2f} (need >= {GATE_PSNR}) -> "
                            f"{'PASS' if p >= GATE_PSNR else 'FAIL — recommend killing'}")
                if gs % LOCAL_SAVE_EVERY == 0 and is_main:
                    if os.path.exists(latest): shutil.rmtree(latest)
                    os.makedirs(latest); acc.save_state(latest)
                    torch.save({"step": gs}, os.path.join(latest,"meta.pt"))
                    log(f"  local checkpoint @ {gs}")
            if gs >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if gs >= NUM_TRAIN_STEPS: break

    if is_main:
        pf, sf, sgf = validate(gs, "FINAL")
        tmp = os.path.join(OUTPUT_DIR,"_final"); os.makedirs(tmp, exist_ok=True)
        save_file({k: v.contiguous().cpu() for k,v in acc.unwrap_model(unet).state_dict().items()},
                  os.path.join(tmp,"unet.safetensors"))
        json.dump({"latent_scale": SCALE, "fixed_tb": FIXED_TB, "method": "heteroscedastic_regressor",
                   "out_channels": 2*TOTAL_CH, "logvar_clamp": [LOGVAR_MIN, LOGVAR_MAX],
                   "final_step": gs, "final_psnr": pf, "final_ssim": sf, "final_sigma": sgf,
                   "step0_psnr": p0},
                  open(os.path.join(tmp,"bridge_config.json"),"w"), indent=2)
        pushed = False
        try:
            HfApi().upload_folder(folder_path=tmp, path_in_repo=OUT_TAG, repo_id=HF_REPO_ID,
                                  repo_type="model", token=HF_TOKEN)
            log(f"  FINAL -> HF/{OUT_TAG}"); pushed = True
        except Exception as e: log(f"  push failed: {e} — weights KEPT at {tmp}")
        if pushed: shutil.rmtree(tmp, ignore_errors=True)
    acc.end_training(); log("Script E complete.")

if __name__ == "__main__":
    main()

Writing /content/stageE_heteroscedastic.py


In [ ]:
!python -u /content/stageE_heteroscedastic.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
  init regressor_v2_final (PSNR 17.448642927105023), conv_out widened 16->32 (logvar zero-init, sigma=1 at step 0)
Train 292,486 | Val 15,394

[STEP-0 (must reproduce the regressor)] PS

In [ ]:
%%writefile /content/stageF_regressor_v3.py
"""
Script F — Regressor v3. Init from regressor_v2_final (17.449 / 0.2905).

TWO NEW LEVERS (everything from C is retained: shift-tolerant, MIND, SAM, EMA):

 1. DIFFERENTIABLE SSIM LOSS. SSIM is your closest metric to SOTA (0.2905 vs
    0.312, a 7% gap vs 12% on PSNR) and has never been optimised directly on a
    DETERMINISTIC model. Phase-D's SSIM failure was on an x0-diffusion bridge
    where it fought the stochastic objective; on a regressor there is no such
    tension.

 2. JOINT DECODER FINE-TUNE  <- my addition, not from the document.
    Rationale: the VAE decoder was trained to decode ENCODED latents. It has
    never seen PREDICTED latents, which are systematically smoother and lower-
    variance. That is a real train/test distribution shift sitting between your
    17.4 dB model and its 27.4 dB reconstruction ceiling. Unfreezing the decoder
    at low LR lets it adapt to the actual input distribution it receives.
    Flagged as speculative: set LAMBDA_DECODER_LR=0 to ablate it cleanly.

Plus §0.A CHECKPOINT SOUP: snapshots every 10k are uniformly averaged at the
end and evaluated against the best single checkpoint. Free, either helps or
is discarded.

GATE @20k: PSNR >= 17.55 AND SSIM >= 0.293 (must beat v2), else kill.
"""
import os, gc, json, time, random, warnings, shutil, copy
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import save_file, load_file
from torchmetrics.image import StructuralSimilarityIndexMeasure
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR = "/content/stageF_regressor_v3"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG, INIT_TAG, OUT_TAG = "adapted16_final", "regressor_v2_final", "regressor_v3_final"

IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
TERRAINS = ("tropical","temperate","arctic","arid","coastal","urban")
TOTAL_CH = 16

NUM_TRAIN_STEPS = 80_000
BATCH_SIZE   = 4          # lower: decoder grads add memory
LR           = 1.5e-5
LR_WARMUP    = 1_500
COND_DROPOUT = 0.10

# ---- retained from v2 (proven) ----
USE_SHIFT_TOLERANT = True
SHIFT_RADIUS       = 1
LAMBDA_PIXEL_L1    = 0.5
LAMBDA_MIND        = 0.08
MIND_RADIUS        = 3
LAMBDA_SAM         = 0.05
# ---- NEW ----
LAMBDA_SSIM        = 0.3      # differentiable 1-SSIM on decoded output
DECODER_LR         = 5e-6     # set 0.0 to freeze the decoder (ablation)
# -----------------------------------
AUX_START, AUX_RAMP = 2_000, 3_000
EMA_DECAY = 0.9995

VAL_EVERY = 5_000
N_VAL = 150
SNAPSHOT_EVERY = 10_000        # for checkpoint soup
LOCAL_SAVE_EVERY = 10_000
STEP_LOG_EVERY = 500
SPIKE_MULT, GUARD_WARMUP = 10.0, 500
GATE_STEP, GATE_PSNR, GATE_SSIM = 20_000, 17.55, 0.293
STEP0_BAND = (17.0, 17.9)      # must reproduce v2's 17.449

device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    random.shuffle(ps); return ps

class PairDataset(Dataset):
    def __init__(self, pairs, augment=True): self.p, self.aug = pairs, augment
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        sar, opt = self._l(p["sar"]), self._l(p["opt"])
        if self.aug:
            if random.random() > .5: sar, opt = torch.flip(sar,[2]), torch.flip(opt,[2])
            if random.random() > .5: sar, opt = torch.flip(sar,[1]), torch.flip(opt,[1])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=12, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    def dec_grad(s,l): return s.base_vae.decoder(l)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def load_vae():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    v = v.to(device).float()
    v.requires_grad_(False)                       # encoder + detail always frozen
    return v

def build_unet_from_v2():
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/bridge_config.json")))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{INIT_TAG}/unet.safetensors")))
    log(f"  init {INIT_TAG} (PSNR {cfg.get('final_psnr','?'):.3f}, SSIM {cfg.get('final_ssim','?'):.4f})")
    return unet, cfg

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

def get_cond(ss, tt, cache, drop=0.0):
    return torch.stack([cache["null"] if (drop and random.random() < drop)
                        else cache.get((s,t), cache["null"]) for s,t in zip(ss,tt)])

def shift_tolerant_l2(pred, target, radius=SHIFT_RADIUS):
    best = None
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            t = torch.roll(target, shifts=(dx, dy), dims=(2,3))
            l = ((pred - t)**2).mean(dim=(1,2,3))
            best = l if best is None else torch.minimum(best, l)
    return best.mean()

def mind_descriptor(img, radius=MIND_RADIUS, patch=3):
    gray = img.mean(1, keepdim=True)
    D = []
    for dx, dy in [(radius,0), (-radius,0), (0,radius), (0,-radius)]:
        sh = torch.roll(gray, shifts=(dx,dy), dims=(2,3))
        D.append(F.avg_pool2d((gray - sh)**2, patch, stride=1, padding=patch//2))
    D = torch.cat(D, 1)
    V = D.mean(dim=1, keepdim=True).clamp(min=1e-6)
    m = torch.exp(-D / V)
    return m / m.max(dim=1, keepdim=True)[0].clamp(min=1e-8)

def mind_loss(pred_img, sar_img):
    return F.l1_loss(mind_descriptor(pred_img), mind_descriptor(sar_img))

def sam_loss(pred_img, gt_img):
    p = ((pred_img+1)/2).clamp(1e-4,1).flatten(2).transpose(1,2)
    g = ((gt_img+1)/2).clamp(1e-4,1).flatten(2).transpose(1,2)
    cos = (p*g).sum(-1) / (p.norm(dim=-1)*g.norm(dim=-1) + 1e-8)
    return torch.acos(cos.clamp(-1+1e-6, 1-1e-6)).mean()

def main():
    acc = Accelerator(mixed_precision="no", project_dir=OUTPUT_DIR)
    set_seed(SEED); is_main = acc.is_main_process
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    vae = load_vae(); embeds = build_embeds()
    unet, cfg = build_unet_from_v2()
    SCALE, FIXED_TB = float(cfg["latent_scale"]), int(cfg["fixed_tb"])
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
    ssim_train  = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)  # differentiable

    tune_decoder = DECODER_LR > 0
    if tune_decoder:
        vae.base_vae.decoder.requires_grad_(True)
        log(f"  DECODER FINE-TUNE ENABLED (lr={DECODER_LR}) — speculative lever")
    else:
        log("  decoder frozen (ablation mode)")
    log(f"  scale={SCALE:.4f} tb={FIXED_TB} | SSIM_loss={LAMBDA_SSIM} MIND={LAMBDA_MIND} "
        f"SAM={LAMBDA_SAM} shift={USE_SHIFT_TOLERANT}")

    pairs = collect(DATA_ROOT)
    tr, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    log(f"Train {len(tr):,} | Val {len(va):,}")
    dl = DataLoader(PairDataset(tr, True), batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)

    groups = [{"params": unet.parameters(), "lr": LR}]
    if tune_decoder:
        groups.append({"params": vae.base_vae.decoder.parameters(), "lr": DECODER_LR})
    opt_ = torch.optim.AdamW(groups, betas=(0.9,0.999), weight_decay=1e-2)
    sch = get_cosine_schedule_with_warmup(opt_, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, opt_, dl, sch = acc.prepare(unet, opt_, dl, sch)

    ema_unet = AveragedModel(acc.unwrap_model(unet),
                             multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY)).to(device)
    def ema_decay_at(s): return min(EMA_DECAY, (s+1)/(s+10))

    snapshots = []          # for checkpoint soup

    def enc(x):
        with torch.no_grad(): return vae.enc(x.float()) * SCALE

    def regress(m, y, cond):
        B = y.shape[0]
        tb = torch.full((B,), FIXED_TB, device=device, dtype=torch.long)
        return m(torch.cat([y, y], 1), tb, encoder_hidden_states=cond).sample

    @torch.no_grad()
    def eval_model(m, sub):
        was = m.training; m.eval(); vae.eval(); P, S = [], []
        for b in DataLoader(PairDataset(sub, False), batch_size=8, num_workers=2):
            o = b["opt"].to(device); y = enc(b["sar"].to(device))
            c = get_cond(b["season"], b["terrain"], embeds).to(device)
            rec = vae.dec(regress(m, y, c)/SCALE).clamp(-1,1)
            for j in range(rec.shape[0]):
                P.append(10*np.log10(4.0/F.mse_loss(rec[j:j+1], o[j:j+1]).item()))
                S.append(ssim_metric(rec[j:j+1], o[j:j+1]).item())
        if was: m.train()
        vae.train() if tune_decoder else None
        return float(np.mean(P)), float(np.mean(S))

    def validate(step, tag=None):
        sub = random.Random(7).sample(va, N_VAL)
        r = eval_model(acc.unwrap_model(unet), sub)
        e = eval_model(ema_unet.module, sub)
        lbl = tag or f"VAL step {step}"
        log(f"\n[{lbl}] RAW {r[0]:.3f}/{r[1]:.4f} | EMA {e[0]:.3f}/{e[1]:.4f}")
        log(f"    (v2 17.449/0.2905 | oracle-affine ceiling 18.85/0.3212 | SOTA 19.72/0.312)\n")
        return {"RAW": r, "EMA": e}

    base = validate(0, "STEP-0 (must reproduce v2)")
    if not (STEP0_BAND[0] <= base["RAW"][0] <= STEP0_BAND[1]):
        log(f"!! STEP-0 FAILED: {base['RAW'][0]:.2f} outside {STEP0_BAND}. ABORTING."); return
    log(">>> Step-0 PASSED.\n")

    latest = os.path.join(OUTPUT_DIR,"latest")
    gs = 0; ema_l = 0.0; a_ = 0.98; ng = nn_ = 0; t0 = time.time(); last = 0
    unet.train()
    if tune_decoder: vae.train()
    log(f">>> Regressor v3: 0 -> {NUM_TRAIN_STEPS}. GATE @ {GATE_STEP}: "
        f"PSNR >= {GATE_PSNR} AND SSIM >= {GATE_SSIM}\n")

    for epoch in range(999):
        for batch in dl:
            oimg = batch["opt"].to(device); simg = batch["sar"].to(device)
            x0 = enc(oimg); y = enc(simg)
            c = get_cond(batch["season"], batch["terrain"], embeds, COND_DROPOUT).to(device)
            with acc.accumulate(unet):
                pred = regress(unet, y, c)
                loss_lat = (shift_tolerant_l2(pred, x0) if USE_SHIFT_TOLERANT
                            else F.mse_loss(pred, x0))
                loss = loss_lat
                l_pix = l_mind = l_sam = l_ssim = 0.0
                if gs >= AUX_START:
                    w = min(1.0, (gs - AUX_START) / max(AUX_RAMP,1))
                    img = vae.dec_grad(pred/SCALE).clamp(-1,1)
                    if LAMBDA_PIXEL_L1 > 0:
                        t = F.l1_loss(img, oimg); l_pix = float(t)
                        loss = loss + w*LAMBDA_PIXEL_L1*t
                    if LAMBDA_SSIM > 0:
                        t = 1.0 - ssim_train(img, oimg); l_ssim = float(t)
                        loss = loss + w*LAMBDA_SSIM*t
                    if LAMBDA_MIND > 0:
                        t = mind_loss(img, simg); l_mind = float(t)
                        loss = loss + w*LAMBDA_MIND*t
                    if LAMBDA_SAM > 0:
                        t = sam_loss(img, oimg); l_sam = float(t)
                        loss = loss + w*LAMBDA_SAM*t
                if gs > GUARD_WARMUP and ema_l > 0 and loss_lat.item() > SPIKE_MULT*max(ema_l,1e-4):
                    opt_.zero_grad(set_to_none=True); ng += 1; continue
                if not torch.isfinite(loss):
                    opt_.zero_grad(set_to_none=True); nn_ += 1; continue
                acc.backward(loss)
                if acc.sync_gradients:
                    acc.clip_grad_norm_(unet.parameters(), 1.0)
                    if tune_decoder:
                        torch.nn.utils.clip_grad_norm_(vae.base_vae.decoder.parameters(), 1.0)
                opt_.step(); sch.step(); opt_.zero_grad(set_to_none=True)
            if acc.sync_gradients:
                ema_unet.avg_fn = get_ema_multi_avg_fn(ema_decay_at(gs))
                ema_unet.update_parameters(acc.unwrap_model(unet))
                gs += 1; ema_l = a_*ema_l + (1-a_)*loss_lat.item()
                if is_main and gs % STEP_LOG_EVERY == 0:
                    r = (gs-last)/max(time.time()-t0,1e-9); t0, last = time.time(), gs
                    log(f"step {gs}/{NUM_TRAIN_STEPS} | lat {ema_l:.5f} | pix {l_pix:.4f} "
                        f"ssim {l_ssim:.4f} mind {l_mind:.4f} sam {l_sam:.4f} | "
                        f"g={ng} n={nn_} | {r:.2f} it/s")
                if gs % VAL_EVERY == 0 and is_main:
                    v = validate(gs)
                    if gs == GATE_STEP:
                        best = max(v["RAW"], v["EMA"], key=lambda t: t[0])
                        ok = best[0] >= GATE_PSNR and best[1] >= GATE_SSIM
                        log(f"  GATE @ {GATE_STEP}: {best[0]:.3f}/{best[1]:.4f} vs "
                            f"({GATE_PSNR}/{GATE_SSIM}) -> "
                            f"{'PASS' if ok else 'FAIL — recommend killing'}")
                if gs % SNAPSHOT_EVERY == 0 and is_main:
                    snapshots.append(copy.deepcopy(acc.unwrap_model(unet).state_dict()))
                    log(f"  snapshot {len(snapshots)} taken @ {gs} (for soup)")
                if gs % LOCAL_SAVE_EVERY == 0 and is_main:
                    if os.path.exists(latest): shutil.rmtree(latest)
                    os.makedirs(latest); acc.save_state(latest)
                    torch.save({"step": gs}, os.path.join(latest,"meta.pt"))
            if gs >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if gs >= NUM_TRAIN_STEPS: break

    if is_main:
        fin = validate(gs, "FINAL")
        results = {"RAW": fin["RAW"], "EMA": fin["EMA"]}

        # ── §0.A checkpoint soup ──
        if len(snapshots) >= 2:
            log(f"  building checkpoint soup from {len(snapshots)} snapshots...")
            soup = {k: sum(sd[k].float() for sd in snapshots)/len(snapshots)
                    for k in snapshots[0]}
            soup_model = copy.deepcopy(acc.unwrap_model(unet))
            soup_model.load_state_dict(soup)
            sub = random.Random(7).sample(va, N_VAL)
            sp = eval_model(soup_model, sub)
            results["SOUP"] = sp
            log(f"  SOUP  PSNR {sp[0]:.3f} | SSIM {sp[1]:.4f}")

        which = max(results, key=lambda k: results[k][0])
        log(f"\n  BEST = {which}: PSNR {results[which][0]:.3f} | SSIM {results[which][1]:.4f}")
        model = {"RAW": acc.unwrap_model(unet), "EMA": ema_unet.module,
                 "SOUP": soup_model if "SOUP" in results else None}[which]

        tmp = os.path.join(OUTPUT_DIR,"_final"); os.makedirs(tmp, exist_ok=True)
        save_file({k: v.contiguous().cpu() for k,v in model.state_dict().items()},
                  os.path.join(tmp,"unet.safetensors"))
        if tune_decoder:
            torch.save(vae.base_vae.state_dict(), os.path.join(tmp,"base_vae_tuned.pth"))
            log("  NOTE: decoder was fine-tuned — base_vae_tuned.pth MUST be used with this UNet")
        json.dump({"latent_scale": SCALE, "fixed_tb": FIXED_TB,
                   "method": "deterministic_regressor_v3", "saved_weights": which,
                   "decoder_finetuned": tune_decoder, "decoder_lr": DECODER_LR,
                   "lambda_ssim": LAMBDA_SSIM, "lambda_mind": LAMBDA_MIND,
                   "lambda_sam": LAMBDA_SAM, "lambda_pixel_l1": LAMBDA_PIXEL_L1,
                   "shift_radius": SHIFT_RADIUS if USE_SHIFT_TOLERANT else 0,
                   "final_step": gs, "final_psnr": results[which][0],
                   "final_ssim": results[which][1],
                   "all_variants": {k: list(v) for k,v in results.items()},
                   "step0_psnr": base["RAW"][0]},
                  open(os.path.join(tmp,"bridge_config.json"),"w"), indent=2)
        pushed = False
        try:
            HfApi().upload_folder(folder_path=tmp, path_in_repo=OUT_TAG, repo_id=HF_REPO_ID,
                                  repo_type="model", token=HF_TOKEN)
            log(f"  FINAL -> HF/{OUT_TAG}"); pushed = True
        except Exception as e: log(f"  push failed: {e} — weights KEPT at {tmp}")
        if pushed: shutil.rmtree(tmp, ignore_errors=True)
    acc.end_training(); log("Script F complete.")

if __name__ == "__main__":
    main()

Writing /content/stageF_regressor_v3.py


In [ ]:
!python -u /content/stageF_regressor_v3.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
  init regressor_v2_final (PSNR 17.449, SSIM 0.2905)
  DECODER FINE-TUNE ENABLED (lr=5e-06) — speculative lever
  scale=1.4010 tb=933 | SSIM_loss=0.3 MIND=0.08 SAM=0.05 shift=True
Train

In [ ]:
!python -u /content/stageF_regressor_v3.py

In [12]:
import os
import json
from huggingface_hub import hf_hub_download

cfg = json.load(open(hf_hub_download(
    repo_id=os.environ["HF_REPO_ID"],
    filename="regressor_v3_final/bridge_config.json")))
print(json.dumps(cfg, indent=2))

bridge_config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

{
  "latent_scale": 1.4010374546051025,
  "fixed_tb": 933,
  "method": "deterministic_regressor_v3",
  "saved_weights": "RAW",
  "decoder_finetuned": true,
  "decoder_lr": 5e-06,
  "lambda_ssim": 0.3,
  "lambda_mind": 0.08,
  "lambda_sam": 0.05,
  "lambda_pixel_l1": 0.5,
  "shift_radius": 1,
  "final_step": 80000,
  "final_psnr": 17.668279561633238,
  "final_ssim": 0.3109552379821738,
  "all_variants": {
    "RAW": [
      17.668279561633238,
      0.3109552379821738
    ],
    "EMA": [
      17.667567483130757,
      0.31089191235601904
    ],
    "SOUP": [
      17.56268141737107,
      0.30884140930293746
    ]
  },
  "step0_psnr": 17.44837014873423
}


In [ ]:
%%writefile /content/stageG_modern_vae.py
"""
Script G — §3.A: swap in a modern pretrained 16-channel VAE (SD3 or FLUX),
keep the SD-1.5 UNet. Tests the project's central thesis directly: if the
autoencoder is the ceiling, a better autoencoder should raise the result.

PLUMBING IS UNCHANGED: SD3/FLUX latents are also 16ch at 8x downsampling, so
conv_in stays 32ch ([y,y]) and conv_out stays 16ch.

PHASE 1 (GATE, ~15 min, no training):
  measure reconstruction PSNR of the new VAE on the val set and compare to
  adapted16_final's 27.40 dB. ABORT if it does not clear RECON_GATE.
  This is the whole premise — if the new VAE isn't a better autoencoder for
  Sentinel-2 imagery, nothing downstream can help.

PHASE 2: train a deterministic regressor in the new latent space.
  Init from regressor_v2_final (INIT_FROM_V2=True) — the conv stacks have
  learned SAR->optical structure even though the target space differs — or
  from raw SD-1.5 (INIT_FROM_V2=False) as the clean control.
  Step-0 WILL be poor either way; that is expected, not a bug.

GATES: @20k PSNR >= 16.0 (on trajectory)   @40k PSNR >= 17.45 (beat v2)

NOTE: SD3 / FLUX repos are GATED. Accept the licence on huggingface.co with
the same account as HF_WRITE_TOKEN, or the download will 401.
LICENCE: FLUX.1-schnell is Apache-2.0 (publication-safe). SD3 is under the
Stability Community Licence — check terms before publishing results with it.
"""
import os, gc, json, time, random, warnings, shutil, copy
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import save_file, load_file
from torchmetrics.image import StructuralSimilarityIndexMeasure
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR = "/content/stageG_modern_vae"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")

# ---- which modern VAE (tried in order; first that loads wins) ----
VAE_CANDIDATES = [
    ("flux",  "black-forest-labs/FLUX.1-schnell"),   # Apache-2.0, publication-safe
    ("sd3",   "stabilityai/stable-diffusion-3-medium-diffusers"),
]
INIT_FROM_V2 = True
V2_TAG  = "regressor_v2_final"
OUT_TAG = "regressor_modernvae_final"

RECON_GATE = 28.50        # new VAE recon must clear this (adapted16 = 27.40)
N_RECON    = 200          # images for the ceiling measurement

IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
TERRAINS = ("tropical","temperate","arctic","arid","coastal","urban")
TOTAL_CH = 16

NUM_TRAIN_STEPS = 80_000
BATCH_SIZE   = 6          # modern VAE decoders are heavier; drop to 6 if OOM
LR           = 3e-5       # higher than v3: adapting to a NEW latent space
LR_WARMUP    = 2_000
COND_DROPOUT = 0.10

USE_SHIFT_TOLERANT = True
SHIFT_RADIUS       = 1
LAMBDA_PIXEL_L1    = 0.5
LAMBDA_SSIM        = 0.3
LAMBDA_SAM         = 0.05
AUX_START, AUX_RAMP = 3_000, 3_000
EMA_DECAY = 0.9995

VAL_EVERY = 5_000
N_VAL = 150
SNAPSHOT_EVERY = 20_000
LOCAL_SAVE_EVERY = 10_000
STEP_LOG_EVERY = 500
SPIKE_MULT, GUARD_WARMUP = 10.0, 500
GATE1_STEP, GATE1_PSNR = 20_000, 16.00
GATE2_STEP, GATE2_PSNR = 40_000, 17.45

device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    random.shuffle(ps); return ps

class PairDataset(Dataset):
    def __init__(self, pairs, augment=True): self.p, self.aug = pairs, augment
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        sar, opt = self._l(p["sar"]), self._l(p["opt"])
        if self.aug:
            if random.random() > .5: sar, opt = torch.flip(sar,[2]), torch.flip(opt,[2])
            if random.random() > .5: sar, opt = torch.flip(sar,[1]), torch.flip(opt,[1])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}

# ── adapted16 (baseline, for the ceiling comparison only) ───────────────────
class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=12, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def load_adapted16():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, "adapted16_final/base_vae.pth"), map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, "adapted16_final/detail_encoder.pth"), map_location="cpu"))
    return v.to(device).eval().float().requires_grad_(False)

# ── modern VAE wrapper (SD3 / FLUX): handles shift+scale normalisation ──────
class ModernVAE(nn.Module):
    def __init__(self, vae):
        super().__init__()
        self.vae = vae
        cfgd = vae.config
        self.scaling = float(getattr(cfgd, "scaling_factor", 1.0) or 1.0)
        self.shift   = float(getattr(cfgd, "shift_factor", 0.0) or 0.0)
        self.ch      = int(getattr(cfgd, "latent_channels", 16))
    @torch.no_grad()
    def enc(self, x):
        z = self.vae.encode(x).latent_dist.mode()
        return (z - self.shift) * self.scaling         # normalised latent
    def dec_grad(self, z):
        return self.vae.decode(z / self.scaling + self.shift).sample
    @torch.no_grad()
    def dec(self, z):
        return self.vae.decode(z / self.scaling + self.shift).sample

def load_modern_vae():
    last_err = None
    for name, repo in VAE_CANDIDATES:
        try:
            log(f"  trying {name}: {repo} ...")
            v = AutoencoderKL.from_pretrained(repo, subfolder="vae", token=HF_TOKEN or None)
            mv = ModernVAE(v).to(device).eval()
            mv.vae.requires_grad_(False)
            log(f"  LOADED {name} | latent_channels={mv.ch} scaling={mv.scaling:.4f} shift={mv.shift:.4f}")
            if mv.ch != TOTAL_CH:
                log(f"  !! latent_channels {mv.ch} != {TOTAL_CH}; plumbing assumption broken."); continue
            return name, mv
        except Exception as e:
            last_err = e; log(f"  failed ({type(e).__name__}): {str(e)[:200]}")
    raise RuntimeError(f"No modern VAE could be loaded. Accept the licence on HuggingFace "
                       f"with your token's account. Last error: {last_err}")

def build_unet():
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    if INIT_FROM_V2:
        try:
            sd = load_file(hf_hub_download(HF_REPO_ID, f"{V2_TAG}/unet.safetensors"))
            unet.load_state_dict(sd)
            log(f"  UNet init from {V2_TAG} (transfer; target latent space differs — "
                f"step-0 will be poor, this is expected)")
        except Exception as e:
            log(f"  v2 init failed ({e}); falling back to raw SD-1.5 init")
    else:
        with torch.no_grad():
            unet.conv_in.weight.zero_()
            for i in range(2*TOTAL_CH):
                unet.conv_in.weight[:, i] = oi.weight[:, i % oi.in_channels] / 8.0
            unet.conv_in.bias.copy_(oi.bias)
            for i in range(TOTAL_CH):
                unet.conv_out.weight[i] = oo.weight[i % oo.out_channels]
                unet.conv_out.bias[i]   = oo.bias[i % oo.out_channels]
        log("  UNet init from raw SD-1.5 (clean control)")
    return unet

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

def get_cond(ss, tt, cache, drop=0.0):
    return torch.stack([cache["null"] if (drop and random.random() < drop)
                        else cache.get((s,t), cache["null"]) for s,t in zip(ss,tt)])

def shift_tolerant_l2(pred, target, radius=SHIFT_RADIUS):
    best = None
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            t = torch.roll(target, shifts=(dx,dy), dims=(2,3))
            l = ((pred - t)**2).mean(dim=(1,2,3))
            best = l if best is None else torch.minimum(best, l)
    return best.mean()

def sam_loss(pred_img, gt_img):
    p = ((pred_img+1)/2).clamp(1e-4,1).flatten(2).transpose(1,2)
    g = ((gt_img+1)/2).clamp(1e-4,1).flatten(2).transpose(1,2)
    cos = (p*g).sum(-1) / (p.norm(dim=-1)*g.norm(dim=-1) + 1e-8)
    return torch.acos(cos.clamp(-1+1e-6, 1-1e-6)).mean()

def main():
    acc = Accelerator(mixed_precision="no", project_dir=OUTPUT_DIR)
    set_seed(SEED); is_main = acc.is_main_process
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
    ssim_train  = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    pairs = collect(DATA_ROOT)
    tr, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    log(f"Train {len(tr):,} | Val {len(va):,}")

    # ════════ PHASE 1: RECONSTRUCTION-CEILING GATE ════════
    log("\n" + "="*66)
    log("PHASE 1 — reconstruction ceiling gate (no training)")
    log("="*66)
    vae_name, mvae = load_modern_vae()
    old = load_adapted16()
    sub = random.Random(SEED).sample(va, N_RECON)
    newP, newS, oldP, oldS = [], [], [], []
    with torch.no_grad():
        for b in DataLoader(PairDataset(sub, False), batch_size=8, num_workers=4):
            o = b["opt"].to(device)
            rn = mvae.dec(mvae.enc(o.float())).clamp(-1,1)
            ro = old.dec(old.enc(o.float())).clamp(-1,1)
            for j in range(o.shape[0]):
                newP.append(10*np.log10(4.0/F.mse_loss(rn[j:j+1], o[j:j+1]).item()))
                newS.append(ssim_metric(rn[j:j+1], o[j:j+1]).item())
                oldP.append(10*np.log10(4.0/F.mse_loss(ro[j:j+1], o[j:j+1]).item()))
                oldS.append(ssim_metric(ro[j:j+1], o[j:j+1]).item())
    np_, ns_ = float(np.mean(newP)), float(np.mean(newS))
    op_, os_ = float(np.mean(oldP)), float(np.mean(oldS))
    log(f"\n  adapted16_final : recon PSNR {op_:.3f} | SSIM {os_:.4f}   (project baseline 27.40)")
    log(f"  {vae_name:<15} : recon PSNR {np_:.3f} | SSIM {ns_:.4f}")
    log(f"  DELTA           : {np_-op_:+.3f} dB | {ns_-os_:+.4f} SSIM")
    del old; gc.collect(); torch.cuda.empty_cache()
    if np_ < RECON_GATE:
        log(f"\n!! RECON GATE FAILED: {np_:.2f} < {RECON_GATE}. The premise of §3.A does not")
        log(f"   hold on this data — the modern VAE is not a better Sentinel-2 autoencoder")
        log(f"   than your adapted one. ABORTING before any training (cost: ~15 min).")
        log(f"   This is itself a publishable finding: your hand-built adapted VAE is")
        log(f"   competitive with SD3/FLUX autoencoders on remote-sensing imagery.")
        return
    log(f"\n>>> RECON GATE PASSED ({np_:.2f} >= {RECON_GATE}). Proceeding to train.\n")

    # ════════ PHASE 2: regressor in the new latent space ════════
    embeds = build_embeds(); unet = build_unet()
    # empirical latent scale on top of the VAE's own normalisation (should be ~1)
    stds = []
    with torch.no_grad():
        for b in DataLoader(PairDataset(tr, False), batch_size=16, shuffle=True, num_workers=2):
            stds.append(mvae.enc(b["opt"].to(device).float()).std().item())
            if len(stds) >= 40: break
    EXTRA = 1.0/float(np.mean(stds))
    log(f"  empirical latent std after VAE normalisation: {np.mean(stds):.4f} -> extra scale {EXTRA:.4f}")
    FIXED_TB = int(14*1000/15)      # same convention as the regressor line

    dl = DataLoader(PairDataset(tr, True), batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)
    opt_ = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9,0.999), weight_decay=1e-2)
    sch = get_cosine_schedule_with_warmup(opt_, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, opt_, dl, sch = acc.prepare(unet, opt_, dl, sch)
    ema_unet = AveragedModel(acc.unwrap_model(unet),
                             multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY)).to(device)
    def ema_decay_at(s): return min(EMA_DECAY, (s+1)/(s+10))
    snapshots = []

    def enc(x):
        with torch.no_grad(): return mvae.enc(x.float()) * EXTRA

    def regress(m, y, cond):
        B = y.shape[0]
        tb = torch.full((B,), FIXED_TB, device=device, dtype=torch.long)
        return m(torch.cat([y, y], 1), tb, encoder_hidden_states=cond).sample

    @torch.no_grad()
    def eval_model(m, sub):
        was = m.training; m.eval(); P, S = [], []
        for b in DataLoader(PairDataset(sub, False), batch_size=8, num_workers=2):
            o = b["opt"].to(device); y = enc(b["sar"].to(device))
            c = get_cond(b["season"], b["terrain"], embeds).to(device)
            rec = mvae.dec(regress(m, y, c)/EXTRA).clamp(-1,1)
            for j in range(rec.shape[0]):
                P.append(10*np.log10(4.0/F.mse_loss(rec[j:j+1], o[j:j+1]).item()))
                S.append(ssim_metric(rec[j:j+1], o[j:j+1]).item())
        if was: m.train()
        return float(np.mean(P)), float(np.mean(S))

    def validate(step, tag=None):
        s = random.Random(7).sample(va, N_VAL)
        r = eval_model(acc.unwrap_model(unet), s); e = eval_model(ema_unet.module, s)
        log(f"\n[{tag or f'VAL step {step}'}] RAW {r[0]:.3f}/{r[1]:.4f} | EMA {e[0]:.3f}/{e[1]:.4f}")
        log(f"    (v2 17.449/0.2905 | new-VAE recon ceiling {np_:.2f} | SOTA 19.72/0.312)\n")
        return {"RAW": r, "EMA": e}

    b0 = validate(0, "STEP-0 (new latent space — poor is EXPECTED)")
    latest = os.path.join(OUTPUT_DIR,"latest")
    gs = 0; ema_l = 0.0; a_ = 0.98; ng = nn_ = 0; t0 = time.time(); last = 0
    unet.train()
    log(f">>> Training in {vae_name} latent space: 0 -> {NUM_TRAIN_STEPS}")
    log(f"    GATE @{GATE1_STEP}: PSNR >= {GATE1_PSNR} | GATE @{GATE2_STEP}: PSNR >= {GATE2_PSNR}\n")

    for epoch in range(999):
        for batch in dl:
            oimg = batch["opt"].to(device)
            x0 = enc(oimg); y = enc(batch["sar"].to(device))
            c = get_cond(batch["season"], batch["terrain"], embeds, COND_DROPOUT).to(device)
            with acc.accumulate(unet):
                pred = regress(unet, y, c)
                loss_lat = (shift_tolerant_l2(pred, x0) if USE_SHIFT_TOLERANT
                            else F.mse_loss(pred, x0))
                loss = loss_lat; l_pix = l_ssim = l_sam = 0.0
                if gs >= AUX_START:
                    w = min(1.0, (gs-AUX_START)/max(AUX_RAMP,1))
                    img = mvae.dec_grad(pred/EXTRA).clamp(-1,1)
                    if LAMBDA_PIXEL_L1 > 0:
                        t = F.l1_loss(img, oimg); l_pix = float(t); loss = loss + w*LAMBDA_PIXEL_L1*t
                    if LAMBDA_SSIM > 0:
                        t = 1.0 - ssim_train(img, oimg); l_ssim = float(t); loss = loss + w*LAMBDA_SSIM*t
                    if LAMBDA_SAM > 0:
                        t = sam_loss(img, oimg); l_sam = float(t); loss = loss + w*LAMBDA_SAM*t
                if gs > GUARD_WARMUP and ema_l > 0 and loss_lat.item() > SPIKE_MULT*max(ema_l,1e-4):
                    opt_.zero_grad(set_to_none=True); ng += 1; continue
                if not torch.isfinite(loss):
                    opt_.zero_grad(set_to_none=True); nn_ += 1; continue
                acc.backward(loss)
                if acc.sync_gradients: acc.clip_grad_norm_(unet.parameters(), 1.0)
                opt_.step(); sch.step(); opt_.zero_grad(set_to_none=True)
            if acc.sync_gradients:
                ema_unet.avg_fn = get_ema_multi_avg_fn(ema_decay_at(gs))
                ema_unet.update_parameters(acc.unwrap_model(unet))
                gs += 1; ema_l = a_*ema_l + (1-a_)*loss_lat.item()
                if is_main and gs % STEP_LOG_EVERY == 0:
                    r = (gs-last)/max(time.time()-t0,1e-9); t0, last = time.time(), gs
                    log(f"step {gs}/{NUM_TRAIN_STEPS} | lat {ema_l:.5f} | pix {l_pix:.4f} "
                        f"ssim {l_ssim:.4f} sam {l_sam:.4f} | g={ng} n={nn_} | {r:.2f} it/s")
                if gs % VAL_EVERY == 0 and is_main:
                    v = validate(gs)
                    best = max(v["RAW"], v["EMA"], key=lambda t: t[0])
                    if gs == GATE1_STEP:
                        log(f"  GATE @{GATE1_STEP}: {best[0]:.3f} (need >= {GATE1_PSNR}) -> "
                            f"{'PASS' if best[0] >= GATE1_PSNR else 'FAIL — recommend killing'}")
                    if gs == GATE2_STEP:
                        log(f"  GATE @{GATE2_STEP}: {best[0]:.3f} (need >= {GATE2_PSNR}) -> "
                            f"{'PASS — modern VAE beats adapted16' if best[0] >= GATE2_PSNR else 'FAIL — recommend killing'}")
                if gs % SNAPSHOT_EVERY == 0 and is_main:
                    snapshots.append(copy.deepcopy(acc.unwrap_model(unet).state_dict()))
                    log(f"  snapshot {len(snapshots)} @ {gs}")
                if gs % LOCAL_SAVE_EVERY == 0 and is_main:
                    if os.path.exists(latest): shutil.rmtree(latest)
                    os.makedirs(latest); acc.save_state(latest)
                    torch.save({"step": gs}, os.path.join(latest,"meta.pt"))
            if gs >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if gs >= NUM_TRAIN_STEPS: break

    if is_main:
        fin = validate(gs, "FINAL"); results = dict(fin)
        soup_model = None
        if len(snapshots) >= 2:
            soup = {k: sum(sd[k].float() for sd in snapshots)/len(snapshots) for k in snapshots[0]}
            soup_model = copy.deepcopy(acc.unwrap_model(unet)); soup_model.load_state_dict(soup)
            results["SOUP"] = eval_model(soup_model, random.Random(7).sample(va, N_VAL))
            log(f"  SOUP PSNR {results['SOUP'][0]:.3f} | SSIM {results['SOUP'][1]:.4f}")
        which = max(results, key=lambda k: results[k][0])
        log(f"\n  BEST = {which}: {results[which][0]:.3f} / {results[which][1]:.4f}")
        model = {"RAW": acc.unwrap_model(unet), "EMA": ema_unet.module, "SOUP": soup_model}[which]
        tmp = os.path.join(OUTPUT_DIR,"_final"); os.makedirs(tmp, exist_ok=True)
        save_file({k: v.contiguous().cpu() for k,v in model.state_dict().items()},
                  os.path.join(tmp,"unet.safetensors"))
        json.dump({"vae_backend": vae_name, "vae_repo": dict(VAE_CANDIDATES)[vae_name],
                   "vae_scaling": mvae.scaling, "vae_shift": mvae.shift, "extra_scale": EXTRA,
                   "fixed_tb": FIXED_TB, "method": "regressor_modern_vae",
                   "recon_ceiling_new": np_, "recon_ceiling_adapted16": op_,
                   "init_from_v2": INIT_FROM_V2, "saved_weights": which,
                   "final_step": gs, "final_psnr": results[which][0],
                   "final_ssim": results[which][1],
                   "all_variants": {k: list(v) for k,v in results.items()}},
                  open(os.path.join(tmp,"bridge_config.json"),"w"), indent=2)
        pushed = False
        try:
            HfApi().upload_folder(folder_path=tmp, path_in_repo=OUT_TAG, repo_id=HF_REPO_ID,
                                  repo_type="model", token=HF_TOKEN)
            log(f"  FINAL -> HF/{OUT_TAG}"); pushed = True
        except Exception as e: log(f"  push failed: {e} — weights KEPT at {tmp}")
        if pushed: shutil.rmtree(tmp, ignore_errors=True)
    acc.end_training(); log("Script G complete.")

if __name__ == "__main__":
    main()

In [9]:
!python -u /content/stageG_modern_vae.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Train 292,486 | Val 15,394

PHASE 1 — reconstruction ceiling gate (no training)
  trying flux: black-forest-labs/FLUX.1-schnell ...
  LOADED flux | latent_channels=16 scaling=0.3611 shi

In [6]:
%%writefile /content/final_eval_full.py
"""
Final evaluation — FULL 15,394-image val set, both models:
  - regressor_v3_final (deterministic, PSNR/SSIM champion)
  - bridge_final        (stochastic ResShift, FID champion)
Metrics: PSNR, SSIM, LPIPS, FID, SAM, ERGAS, CC — overall + per-season + per-terrain.
Produces a visual comparison grid (SAR | GT | v3 | champion) across terrain types.
Saves results JSON + grid PNG. Inference only, no training risk.
"""
import os, gc, json, random, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from huggingface_hub import hf_hub_download, HfApi
from safetensors.torch import load_file
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
import lpips as lpips_lib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from collections import defaultdict
warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUT_DIR = "/content/final_eval"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG = "adapted16_final"
IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
TERRAINS = ("tropical","temperate","arctic","arid","coastal","urban")
TOTAL_CH = 16
BATCH = 8
N_GRID_PER_TERRAIN = 1
device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    return ps

class DS(Dataset):
    def __init__(self, pairs): self.p = pairs
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        return {"sar": self._l(p["sar"]), "opt": self._l(p["opt"]),
                "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=12, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def load_vae():
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/base_vae.pth"), map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(HF_REPO_ID, f"{VAE_TAG}/detail_encoder.pth"), map_location="cpu"))
    return v.to(device).eval().float().requires_grad_(False)

def build_regressor():
    tag = "regressor_v3_final"
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{tag}/bridge_config.json")))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{tag}/unet.safetensors")))
    return unet.to(device).eval().float().requires_grad_(False), cfg

def build_champion():
    tag = "bridge_final"
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{tag}/bridge_config.json")))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{tag}/unet.safetensors")))
    return unet.to(device).eval().float().requires_grad_(False), cfg

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

def get_cond(ss, tt, cache):
    return torch.stack([cache.get((s,t), cache["null"]) for s,t in zip(ss,tt)])

@torch.no_grad()
def regress(m, y, cond, tb_fixed):
    B = y.shape[0]
    tb = torch.full((B,), tb_fixed, device=device, dtype=torch.long)
    return m(torch.cat([y, y], 1), tb, encoder_hidden_states=cond).sample

@torch.no_grad()
def resshift_sample(m, y, cond, eta, kappa, T):
    B = y.shape[0]
    x = y + kappa*torch.sqrt(eta[-1])*torch.randn_like(y)
    for t in reversed(range(T)):
        tb = torch.full((B,), int(t*1000/T), device=device, dtype=torch.long)
        x0 = m(torch.cat([x,y],1), tb, encoder_hidden_states=cond).sample
        if t > 0:
            ep = eta[t-1]; x = x0 + ep*(y-x0) + kappa*torch.sqrt(ep)*torch.randn_like(x)
        else: x = x0
    return x

def sam_deg(p, g):
    p = ((p+1)/2).clamp(1e-4,1).flatten(2).transpose(1,2)
    g = ((g+1)/2).clamp(1e-4,1).flatten(2).transpose(1,2)
    c = ((p*g).sum(-1)/(p.norm(dim=-1)*g.norm(dim=-1)+1e-8)).clamp(-1,1)
    return torch.rad2deg(torch.acos(c)).mean().item()

def ergas(p, g):
    e = 0.0
    for b in range(3):
        rmse = torch.sqrt(F.mse_loss(p[:,b], g[:,b]))
        e += (rmse/(((g[:,b]+1)/2).mean()+1e-8))**2
    return float(100*torch.sqrt(e/3))

def cc(p, g):
    t = 0.0
    for b in range(3):
        x = p[:,b].flatten()-p[:,b].mean(); y = g[:,b].flatten()-g[:,b].mean()
        t += float((x*y).sum()/(x.norm()*y.norm()+1e-8))
    return t/3

def dn(t): return (t.clamp(-1,1)+1)/2

def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    vae = load_vae(); embeds = build_embeds()
    reg, rcfg = build_regressor()
    champ, ccfg = build_champion()
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    pairs = collect(DATA_ROOT)
    _, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    log(f"FULL validation set: {len(va):,} images\n")

    T = int(ccfg["T"]); kappa = float(ccfg["kappa"])
    i = np.arange(T)
    eta = torch.tensor(ccfg["eta_1"]*(ccfg["eta_T"]/ccfg["eta_1"])**(i/(T-1)),
                       dtype=torch.float32).to(device)
    reg_scale = float(rcfg["latent_scale"]); reg_tb = int(rcfg["fixed_tb"])
    ch_scale = float(ccfg["latent_scale"])

    def run(name, sampler_fn, scale):
        dl = DataLoader(DS(va), batch_size=BATCH, num_workers=4)
        agg = defaultdict(list)
        by_season = defaultdict(lambda: defaultdict(list))
        by_terrain = defaultdict(lambda: defaultdict(list))
        fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
        n_done = 0
        for b in dl:
            opt = b["opt"].to(device); sar = b["sar"].to(device)
            y = vae.enc(sar.float()) * scale
            c = get_cond(b["season"], b["terrain"], embeds).to(device)
            lat = sampler_fn(y, c)
            rec = vae.dec((lat/scale)).clamp(-1,1)
            for j in range(rec.shape[0]):
                pi, ri = rec[j:j+1], opt[j:j+1]
                m = {"psnr": 10*np.log10(4.0/F.mse_loss(pi,ri).item()),
                     "ssim": ssim_fn(pi,ri).item(), "lpips": lpips_fn(pi,ri).item(),
                     "sam": sam_deg(pi,ri), "ergas": ergas(pi,ri), "cc": cc(pi,ri)}
                for k,v in m.items(): agg[k].append(v)
                se, te = b["season"][j], b["terrain"][j]
                for k,v in m.items():
                    by_season[se][k].append(v); by_terrain[te][k].append(v)
            fid.update(dn(opt), real=True); fid.update(dn(rec), real=False)
            n_done += rec.shape[0]
            if n_done % 2000 < BATCH:
                log(f"  [{name}] {n_done}/{len(va)} | running PSNR {np.mean(agg['psnr']):.3f}")
        out = {k: float(np.mean(v)) for k,v in agg.items()}
        out["fid"] = float(fid.compute())
        out["by_season"] = {s: {k: float(np.mean(v)) for k,v in d.items()} for s,d in by_season.items()}
        out["by_terrain"] = {t: {k: float(np.mean(v)) for k,v in d.items()} for t,d in by_terrain.items()}
        return out

    # ══════ VISUAL GRID FIRST — while both models are still loaded ══════
    log("Building visual comparison grid (SAR | GT | v3 | champion)...")
    picks = []
    for ter in TERRAINS:
        pool = [p for p in va if p["terrain"] == ter]
        if pool: picks.extend(random.Random(SEED).sample(pool, min(N_GRID_PER_TERRAIN, len(pool))))
    n = len(picks)
    fig, axes = plt.subplots(n, 4, figsize=(12, 3*n))
    with torch.no_grad():
        for row, p in enumerate(picks):
            d = DS([p])[0]
            sar = d["sar"].unsqueeze(0).to(device); opt = d["opt"].unsqueeze(0).to(device)
            c = get_cond([p["season"]], [p["terrain"]], embeds).to(device)

            y_r = vae.enc(sar.float()) * reg_scale
            rec_v3 = vae.dec(regress(reg, y_r, c, reg_tb)/reg_scale).clamp(-1,1)

            y_c = vae.enc(sar.float()) * ch_scale
            rec_champ = vae.dec(resshift_sample(champ, y_c, c, eta, kappa, T)/ch_scale).clamp(-1,1)

            axes[row,0].imshow(dn(sar[0]).permute(1,2,0).cpu().numpy())
            axes[row,0].set_ylabel(p["terrain"], fontsize=10)
            axes[row,1].imshow(dn(opt[0]).permute(1,2,0).cpu().numpy())
            axes[row,2].imshow(dn(rec_v3[0]).permute(1,2,0).cpu().numpy())
            axes[row,3].imshow(dn(rec_champ[0]).permute(1,2,0).cpu().numpy())
            for j in range(4):
                axes[row,j].set_xticks([]); axes[row,j].set_yticks([])
    for j, t in enumerate(["SAR", "Ground Truth", "v3 (deterministic)", "champion (stochastic)"]):
        axes[0,j].set_title(t, fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "final_comparison_grid.png"), dpi=140, bbox_inches="tight")
    from IPython.display import display
    display(plt.gcf()); plt.close()
    log(f"saved -> {OUT_DIR}/final_comparison_grid.png\n")

    # ══════ FULL-VAL-SET METRICS ══════
    log("="*64 + "\nEvaluating regressor_v3_final (deterministic) — FULL val set\n" + "="*64)
    r_res = run("v3", lambda y,c: regress(reg, y, c, reg_tb), reg_scale)
    log(f"\nv3: PSNR {r_res['psnr']:.3f} | SSIM {r_res['ssim']:.4f} | LPIPS {r_res['lpips']:.4f} | "
        f"FID {r_res['fid']:.2f} | SAM {r_res['sam']:.2f} | ERGAS {r_res['ergas']:.1f} | CC {r_res['cc']:.3f}\n")
    del reg; gc.collect(); torch.cuda.empty_cache()

    log("="*64 + "\nEvaluating bridge_final (stochastic champion) — FULL val set\n" + "="*64)
    c_res = run("champion", lambda y,c: resshift_sample(champ, y, c, eta, kappa, T), ch_scale)
    log(f"\nchampion: PSNR {c_res['psnr']:.3f} | SSIM {c_res['ssim']:.4f} | LPIPS {c_res['lpips']:.4f} | "
        f"FID {c_res['fid']:.2f} | SAM {c_res['sam']:.2f} | ERGAS {c_res['ergas']:.1f} | CC {c_res['cc']:.3f}\n")
    del champ; gc.collect(); torch.cuda.empty_cache()

    log("="*64 + "\nFINAL COMPARISON TABLE (full val set, n=%d)\n" % len(va) + "="*64)
    log(f"  {'model':<12} {'PSNR':>7} {'SSIM':>7} {'LPIPS':>7} {'FID':>8} {'SAM':>6} {'ERGAS':>7} {'CC':>6}")
    log(f"  {'v3':<12} {r_res['psnr']:7.3f} {r_res['ssim']:7.4f} {r_res['lpips']:7.4f} "
        f"{r_res['fid']:8.2f} {r_res['sam']:6.2f} {r_res['ergas']:7.1f} {r_res['cc']:6.3f}")
    log(f"  {'champion':<12} {c_res['psnr']:7.3f} {c_res['ssim']:7.4f} {c_res['lpips']:7.4f} "
        f"{c_res['fid']:8.2f} {c_res['sam']:6.2f} {c_res['ergas']:7.1f} {c_res['cc']:6.3f}")
    log(f"  {'SOTA':<12} {'19.72':>7} {'0.312':>7} {'-':>7} {'116.9':>8}")

    json.dump({"v3": r_res, "champion": c_res}, open(os.path.join(OUT_DIR,"full_eval_results.json"),"w"), indent=2)
    log(f"\nsaved -> {OUT_DIR}/full_eval_results.json")

    if HF_TOKEN:
        try:
            HfApi().upload_folder(folder_path=OUT_DIR, path_in_repo="final_eval_results",
                                  repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)
            log("results + grid -> HF/final_eval_results")
        except Exception as e:
            log(f"push failed: {e} (results still in {OUT_DIR})")

if __name__ == "__main__":
    main()

Writing /content/final_eval_full.py


In [7]:


!python -u /content/final_eval_full.py



Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints

In [8]:
%%writefile /content/diag_error_budget.py
"""
Phase-2 diagnostics: recompute the affine oracle on v3 (§0) and decompose the
residual into a frequency-resolved error budget (§1). NO TRAINING.

Oracles (each uses GT to fix ONE defect; the PSNR gain = that defect's cost):
  shift        best (dx,dy) in [-4,4]^2                -> misregistration
  affine       per-image per-channel gain+bias         -> global radiometry
  grid-affine  4x4 blended affines                     -> illumination gradient
  blur         best sigma applied to PREDICTION        -> hallucinated high-freq
  sharpen      best unsharp amount                     -> lost recoverable detail
  LP(GT)+HP(pred)  and  LP(pred)+HP(GT) over cutoffs   -> FREQUENCY BUDGET

Decision rule printed at the end:
  low-freq dominant  -> spatially-varying radiometric correction
  high-freq dominant -> SAR-structure injection (doc 3.1 / 3.2)
  flat               -> mapping capacity (doc 3.3)
"""
import os, gc, json, random, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
from diffusers import AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from huggingface_hub import hf_hub_download, HfApi
from safetensors.torch import load_file
from torchmetrics.image import StructuralSimilarityIndexMeasure
import pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUT_DIR = "/content/diag_budget"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")
MODEL_TAG = "regressor_v3_final"
IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
TERRAINS = ("tropical","temperate","arctic","arid","coastal","urban")
TOTAL_CH = 16
N_EVAL = 400
SHIFT_MAX = 4
CROP = 8                       # border cropped for shift oracle (avoids roll wrap)
BLUR_SIGMAS  = (0.0, 0.3, 0.5, 0.8, 1.2, 1.8)
SHARPEN_AMTS = (0.0, 0.2, 0.4, 0.7, 1.0)
FREQ_SIGMAS  = (0.5, 1.0, 1.5, 2.5, 4.0, 6.0)   # low-pass cutoffs
device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    return ps

class DS(Dataset):
    def __init__(self, pairs): self.p = pairs
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        return {"sar": self._l(p["sar"]), "opt": self._l(p["opt"]),
                "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=12, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def load_vae_for(tag):
    """CRITICAL: v3 fine-tuned the decoder — load base_vae_tuned.pth if present."""
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.detail_enc.load_state_dict(torch.load(
        hf_hub_download(HF_REPO_ID, "adapted16_final/detail_encoder.pth"), map_location="cpu"))
    try:
        p = hf_hub_download(HF_REPO_ID, f"{tag}/base_vae_tuned.pth")
        v.base_vae.load_state_dict(torch.load(p, map_location="cpu"))
        log(f"  VAE: using FINE-TUNED decoder from {tag}/base_vae_tuned.pth")
    except Exception as e:
        p = hf_hub_download(HF_REPO_ID, "adapted16_final/base_vae.pth")
        v.base_vae.load_state_dict(torch.load(p, map_location="cpu"))
        log(f"  VAE: tuned decoder NOT found ({type(e).__name__}); using adapted16_final "
            f"— v3 numbers will be UNDERSTATED")
    return v.to(device).eval().float().requires_grad_(False)

def load_model(tag):
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{tag}/bridge_config.json")))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{tag}/unet.safetensors")))
    return unet.to(device).eval().float().requires_grad_(False), cfg

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

@torch.no_grad()
def regress(m, y, cond, tb_fixed):
    tb = torch.full((y.shape[0],), tb_fixed, device=device, dtype=torch.long)
    return m(torch.cat([y, y], 1), tb, encoder_hidden_states=cond).sample

def psnr(a, b): return 10*np.log10(4.0/F.mse_loss(a, b).item())

# ---- gaussian low-pass ----
def gblur(img, sigma):
    if sigma <= 0: return img
    ks = int(2*round(3*sigma)+1)
    x = torch.arange(ks, dtype=torch.float32, device=img.device) - ks//2
    k = torch.exp(-x**2/(2*sigma**2)); k = k/k.sum()
    C = img.shape[1]; pad = ks//2
    kx = k.view(1,1,1,ks).expand(C,1,1,ks); ky = k.view(1,1,ks,1).expand(C,1,ks,1)
    o = F.conv2d(F.pad(img, (pad,pad,0,0), mode="reflect"), kx, groups=C)
    o = F.conv2d(F.pad(o,   (0,0,pad,pad), mode="reflect"), ky, groups=C)
    return o

def affine_oracle(pred, gt):
    p = ((pred+1)/2).flatten(2)[0]; g = ((gt+1)/2).flatten(2)[0]
    pm, gm = p.mean(1, keepdim=True), g.mean(1, keepdim=True)
    var = ((p-pm)**2).mean(1, keepdim=True).clamp(min=1e-8)
    cov = ((p-pm)*(g-gm)).mean(1, keepdim=True)
    a = (cov/var).view(1,3,1,1); b = (gm-(cov/var)*pm).view(1,3,1,1)
    return ((((pred+1)/2)*a + b).clamp(0,1))*2-1

def grid_affine_oracle(pred, gt, grid=4):
    p = (pred+1)/2; g = (gt+1)/2
    H, W = p.shape[-2:]; bh, bw = H//grid, W//grid
    A = torch.zeros(1,3,grid,grid, device=p.device); B = torch.zeros_like(A)
    for i in range(grid):
        for j in range(grid):
            pb = p[:,:,i*bh:(i+1)*bh, j*bw:(j+1)*bw].flatten(2)
            gb = g[:,:,i*bh:(i+1)*bh, j*bw:(j+1)*bw].flatten(2)
            pm, gm = pb.mean(-1), gb.mean(-1)
            var = ((pb-pm[...,None])**2).mean(-1).clamp(min=1e-8)
            cov = ((pb-pm[...,None])*(gb-gm[...,None])).mean(-1)
            a = cov/var; A[:,:,i,j] = a; B[:,:,i,j] = gm - a*pm
    Af = F.interpolate(A, size=(H,W), mode="bilinear", align_corners=False)
    Bf = F.interpolate(B, size=(H,W), mode="bilinear", align_corners=False)
    return ((p*Af + Bf).clamp(0,1))*2-1

def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    vae = load_vae_for(MODEL_TAG); embeds = build_embeds()
    model, cfg = load_model(MODEL_TAG)
    SCALE, TB = float(cfg["latent_scale"]), int(cfg["fixed_tb"])
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
    log(f"  {MODEL_TAG}: scale={SCALE:.4f} tb={TB} "
        f"(reported {cfg.get('final_psnr',0):.3f}/{cfg.get('final_ssim',0):.4f})\n")

    pairs = collect(DATA_ROOT)
    _, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    sub = random.Random(SEED).sample(va, N_EVAL)
    log(f"Error budget on n={N_EVAL} val images\n")

    R = {k: [] for k in ["base","base_ssim","shift","shift_base","affine","affine_ssim",
                         "grid","blur","sharp"]}
    freq_lp_gt = {s: [] for s in FREQ_SIGMAS}     # LP(GT)+HP(pred)
    freq_lp_pr = {s: [] for s in FREQ_SIGMAS}     # LP(pred)+HP(GT)
    best_blur, best_sharp, best_shift = [], [], []

    with torch.no_grad():
        for n, p in enumerate(sub):
            d = DS([p])[0]
            sar = d["sar"].unsqueeze(0).to(device); gt = d["opt"].unsqueeze(0).to(device)
            c = torch.stack([embeds.get((p["season"], p["terrain"]), embeds["null"])]).to(device)
            pred = vae.dec(regress(model, vae.enc(sar.float())*SCALE, c, TB)/SCALE).clamp(-1,1)

            R["base"].append(psnr(pred, gt)); R["base_ssim"].append(ssim_fn(pred, gt).item())

            # shift oracle (central crop both sides, comparable delta)
            pc = pred[:,:,CROP:-CROP,CROP:-CROP]
            R["shift_base"].append(psnr(pc, gt[:,:,CROP:-CROP,CROP:-CROP]))
            bp, bs = -1e9, (0,0)
            for dx in range(-SHIFT_MAX, SHIFT_MAX+1):
                for dy in range(-SHIFT_MAX, SHIFT_MAX+1):
                    gs = torch.roll(gt, shifts=(dx,dy), dims=(2,3))[:,:,CROP:-CROP,CROP:-CROP]
                    v = psnr(pc, gs)
                    if v > bp: bp, bs = v, (dx,dy)
            R["shift"].append(bp); best_shift.append(bs)

            # radiometric oracles
            R["affine"].append(psnr(affine_oracle(pred, gt), gt))
            R["affine_ssim"].append(ssim_fn(affine_oracle(pred, gt), gt).item())
            R["grid"].append(psnr(grid_affine_oracle(pred, gt), gt))

            # blur / sharpen oracles on the PREDICTION
            bb = max(((psnr(gblur(pred, s).clamp(-1,1), gt), s) for s in BLUR_SIGMAS))
            R["blur"].append(bb[0]); best_blur.append(bb[1])
            base_lp = gblur(pred, 1.0)
            bs2 = max(((psnr((pred + a*(pred-base_lp)).clamp(-1,1), gt), a) for a in SHARPEN_AMTS))
            R["sharp"].append(bs2[0]); best_sharp.append(bs2[1])

            # frequency swaps
            for s in FREQ_SIGMAS:
                lp_gt, lp_pr = gblur(gt, s), gblur(pred, s)
                freq_lp_gt[s].append(psnr((lp_gt + (pred-lp_pr)).clamp(-1,1), gt))
                freq_lp_pr[s].append(psnr((lp_pr + (gt-lp_gt)).clamp(-1,1), gt))

            if (n+1) % 100 == 0: log(f"  [{n+1}/{N_EVAL}] base {np.mean(R['base']):.3f}")

    b = np.mean(R["base"])
    log("\n" + "="*70)
    log(f"ERROR BUDGET — {MODEL_TAG}  (n={N_EVAL})")
    log("="*70)
    log(f"  baseline                     PSNR {b:7.3f}  SSIM {np.mean(R['base_ssim']):.4f}")
    log(f"  + best shift (+/-{SHIFT_MAX}px)       {np.mean(R['shift']):7.3f}  "
        f"({np.mean(R['shift'])-np.mean(R['shift_base']):+.3f} vs crop-baseline "
        f"{np.mean(R['shift_base']):.3f})  <- misregistration")
    log(f"  + affine oracle              {np.mean(R['affine']):7.3f}  "
        f"({np.mean(R['affine'])-b:+.3f})  SSIM {np.mean(R['affine_ssim']):.4f}  <- global radiometry")
    log(f"  + 4x4 grid-affine oracle     {np.mean(R['grid']):7.3f}  ({np.mean(R['grid'])-b:+.3f})"
        f"  <- illumination gradient")
    log(f"  + best blur   (mean sig {np.mean(best_blur):.2f})  {np.mean(R['blur']):7.3f}  "
        f"({np.mean(R['blur'])-b:+.3f})")
    log(f"  + best sharpen(mean amt {np.mean(best_sharp):.2f}) {np.mean(R['sharp']):7.3f}  "
        f"({np.mean(R['sharp'])-b:+.3f})")
    sx = np.mean([s[0] for s in best_shift]); sy = np.mean([s[1] for s in best_shift])
    ax = np.mean([abs(s[0]) for s in best_shift]); ay = np.mean([abs(s[1]) for s in best_shift])
    log(f"  shift stats: mean (dx,dy)=({sx:+.2f},{sy:+.2f})  mean |dx|,|dy|=({ax:.2f},{ay:.2f})")

    log("\n  FREQUENCY BUDGET (cutoff sigma -> PSNR of hybrid vs GT)")
    log(f"  {'sigma':>6} {'LP(GT)+HP(pred)':>18} {'LP(pred)+HP(GT)':>18}")
    for s in FREQ_SIGMAS:
        log(f"  {s:6.1f} {np.mean(freq_lp_gt[s]):18.3f} {np.mean(freq_lp_pr[s]):18.3f}")

    lo = np.mean(freq_lp_gt[FREQ_SIGMAS[-1]])   # GT low-freq restored: high-freq err remains
    hi = np.mean(freq_lp_pr[FREQ_SIGMAS[-1]])   # GT high-freq restored: low-freq err remains
    log("\n  " + "-"*66)
    if hi > lo + 0.5:
        verdict = ("LOW-FREQUENCY dominant -> error is radiometric/illumination. "
                   "Pursue spatially-varying correction (grid affine), doc 2.5/2.6.")
    elif lo > hi + 0.5:
        verdict = ("HIGH-FREQUENCY dominant -> structure is being lost. "
                   "Pursue SAR-structure injection: doc 3.1 (dual-path) / 3.2 (pixel refiner).")
    else:
        verdict = ("FLAT across frequencies -> the mapping is weak everywhere. "
                   "Pursue capacity/architecture: doc 3.3 (random-init ablation, NAFNet).")
    log(f"  VERDICT: {verdict}")
    if np.mean(best_blur) > 0.15:
        log(f"  NOTE: blurring helps (mean sigma {np.mean(best_blur):.2f}) -> the model places "
            f"high-freq detail in the WRONG places. Suppress, don't add.")
    log("="*70)

    plt.figure(figsize=(7,4.5))
    plt.plot(FREQ_SIGMAS, [np.mean(freq_lp_gt[s]) for s in FREQ_SIGMAS], "o-", label="LP(GT)+HP(pred)")
    plt.plot(FREQ_SIGMAS, [np.mean(freq_lp_pr[s]) for s in FREQ_SIGMAS], "s-", label="LP(pred)+HP(GT)")
    plt.axhline(b, ls="--", c="k", label=f"baseline {b:.2f}")
    plt.xlabel("low-pass cutoff sigma"); plt.ylabel("PSNR vs GT (dB)")
    plt.title("Frequency-resolved error budget"); plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR,"frequency_budget.png"), dpi=150, bbox_inches="tight")
    from IPython.display import display; display(plt.gcf()); plt.close()

    json.dump({k: float(np.mean(v)) for k,v in R.items()} |
              {"freq_lp_gt": {str(s): float(np.mean(freq_lp_gt[s])) for s in FREQ_SIGMAS},
               "freq_lp_pred": {str(s): float(np.mean(freq_lp_pr[s])) for s in FREQ_SIGMAS},
               "mean_best_blur": float(np.mean(best_blur)),
               "mean_best_sharpen": float(np.mean(best_sharp)),
               "mean_abs_shift": [float(ax), float(ay)], "verdict": verdict, "n": N_EVAL},
              open(os.path.join(OUT_DIR,"error_budget.json"),"w"), indent=2)
    log(f"\nsaved -> {OUT_DIR}/error_budget.json + frequency_budget.png")
    if HF_TOKEN:
        try:
            HfApi().upload_folder(folder_path=OUT_DIR, path_in_repo="diag_error_budget",
                                  repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)
            log("-> HF/diag_error_budget")
        except Exception as e: log(f"push failed: {e}")

if __name__ == "__main__":
    main()

Writing /content/diag_error_budget.py


In [9]:
%%writefile /content/free_gains.py
"""
Phase-2 free gains on the v3 regressor (§2.2, §2.3). NO TRAINING.
  0. D4 unit test: assert inv_t(t(x)) == x for all 8 group elements
  1. timestep sweep — tb=933 was never swept
  2. 8x dihedral TTA (pixel-space averaging, correctly inverted)
  3. multi-timestep ensemble
  4. TTA x multi-tb combined
Each stage reports PSNR/SSIM so you can see exactly what each lever buys.
"""
import os, gc, json, random, warnings, itertools
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
from diffusers import AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from huggingface_hub import hf_hub_download, HfApi
from safetensors.torch import load_file
from torchmetrics.image import StructuralSimilarityIndexMeasure
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
def log(*a): print(*a, flush=True)

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUT_DIR = "/content/free_gains"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID = os.environ["HF_REPO_ID"]
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")
MODEL_TAG = "regressor_v3_final"
IMG_SIZE, SEED, VAL_FRACTION = 256, 42, 0.05
SEASONS = ("spring","summer","fall","winter")
TERRAINS = ("tropical","temperate","arctic","arid","coastal","urban")
TOTAL_CH = 16
N_SWEEP, N_EVAL = 300, 600
TB_GRID = (800, 850, 900, 933, 960, 980)
device = torch.device("cuda")
SEASON_DESC = {"spring":"spring","summer":"summer","fall":"autumn","winter":"winter"}

def map_terrain(r):
    r = (r or "").lower()
    if any(k in r for k in ("trop","rainforest","amazon","jungle")): return "tropical"
    if any(k in r for k in ("arctic","tundra","snow","ice","polar")): return "arctic"
    if any(k in r for k in ("desert","arid","sahara","dry")): return "arid"
    if any(k in r for k in ("coast","beach","shore","marine","sea")): return "coastal"
    if any(k in r for k in ("urban","city","metro")): return "urban"
    return "temperate"

def collect(root):
    rs = Path(root).as_posix(); ps = []
    for season in SEASONS:
        cs = list((Path(root)/season).glob("*.csv"))
        if not cs: continue
        df = pd.concat([pd.read_csv(f) for f in cs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for c in ("s1_fileName","s2_fileName"):
            df[c] = df[c].astype(str).str.replace("\\","/",regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            ps.append({"sar": f"{rs}/{row['s1_fileName']}", "opt": f"{rs}/{row['s2_fileName']}",
                       "season": season, "terrain": map_terrain(row["region"])})
    return ps

class DS(Dataset):
    def __init__(self, pairs): self.p = pairs
    def __len__(self): return len(self.p)
    def _l(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE): img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.array(img, np.float32)/255.).permute(2,0,1)*2-1
    def __getitem__(self, i):
        p = self.p[i]
        return {"sar": self._l(p["sar"]), "opt": self._l(p["opt"]),
                "season": p["season"], "terrain": p["terrain"]}

class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=12, nf=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(ic,nf,3,2,1), nn.GroupNorm(8,nf), nn.SiLU(),
            nn.Conv2d(nf,nf*2,3,2,1), nn.GroupNorm(8,nf*2), nn.SiLU(),
            nn.Conv2d(nf*2,nf*4,3,2,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,nf*4,3,1,1), nn.GroupNorm(8,nf*4), nn.SiLU(),
            nn.Conv2d(nf*4,oc,3,1,1))
    def forward(s,x): return s.net(x)

class AdaptedVAE(nn.Module):
    def __init__(s, bv):
        super().__init__(); s.base_vae = bv; s.detail_enc = DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH,o.out_channels,o.kernel_size,o.stride,o.padding,
                                       bias=(o.bias is not None))
    @torch.no_grad()
    def enc(s,x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)],1)
    @torch.no_grad()
    def dec(s,l): return s.base_vae.decoder(l)

def load_vae_for(tag):
    bv = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    v = AdaptedVAE(bv)
    v.detail_enc.load_state_dict(torch.load(
        hf_hub_download(HF_REPO_ID, "adapted16_final/detail_encoder.pth"), map_location="cpu"))
    try:
        v.base_vae.load_state_dict(torch.load(
            hf_hub_download(HF_REPO_ID, f"{tag}/base_vae_tuned.pth"), map_location="cpu"))
        log(f"  VAE: FINE-TUNED decoder from {tag}")
    except Exception:
        v.base_vae.load_state_dict(torch.load(
            hf_hub_download(HF_REPO_ID, "adapted16_final/base_vae.pth"), map_location="cpu"))
        log("  VAE: tuned decoder not found; using adapted16_final (numbers understated)")
    return v.to(device).eval().float().requires_grad_(False)

def load_model(tag):
    cfg = json.load(open(hf_hub_download(HF_REPO_ID, f"{tag}/bridge_config.json")))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels, unet.config.out_channels = 2*TOTAL_CH, TOTAL_CH
    unet.load_state_dict(load_file(hf_hub_download(HF_REPO_ID, f"{tag}/unet.safetensors")))
    return unet.to(device).eval().float().requires_grad_(False), cfg

def build_embeds():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    cache = {}
    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s,ter)] = emb(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = emb(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache

# ---- D4 group: k rotations then optional horizontal flip ----
def d4_fwd(x, k, flip):
    o = torch.rot90(x, k, [-2,-1])
    return torch.flip(o, [-1]) if flip else o
def d4_inv(x, k, flip):
    o = torch.flip(x, [-1]) if flip else x
    return torch.rot90(o, -k, [-2,-1])
D4 = [(k, f) for k in range(4) for f in (False, True)]

@torch.no_grad()
def regress(m, y, cond, tb):
    t = torch.full((y.shape[0],), tb, device=device, dtype=torch.long)
    return m(torch.cat([y, y], 1), t, encoder_hidden_states=cond).sample

def psnr(a, b): return 10*np.log10(4.0/F.mse_loss(a, b).item())

def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    # ---- 0. D4 unit test ----
    log("D4 unit test:")
    t = torch.randn(1,3,IMG_SIZE,IMG_SIZE, device=device)
    ok = all(torch.allclose(d4_inv(d4_fwd(t,k,f),k,f), t, atol=1e-6) for k,f in D4)
    log(f"  inv(fwd(x)) == x for all 8 elements: {ok}")
    if not ok:
        log("  !! D4 transforms are not self-inverting — ABORTING (this was the suspected "
            "bug in the earlier TTA result)"); return

    vae = load_vae_for(MODEL_TAG); embeds = build_embeds()
    model, cfg = load_model(MODEL_TAG)
    SCALE, TB0 = float(cfg["latent_scale"]), int(cfg["fixed_tb"])
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
    log(f"  {MODEL_TAG}: scale={SCALE:.4f} default tb={TB0}\n")

    pairs = collect(DATA_ROOT)
    _, va = train_test_split(pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in pairs], random_state=SEED)
    sweep_set = random.Random(SEED).sample(va, N_SWEEP)
    eval_set  = random.Random(SEED+1).sample(va, N_EVAL)

    @torch.no_grad()
    def predict(sar, cond, tb, tta=False):
        """Returns pixel-space prediction. TTA averages over D4 AFTER decode."""
        if not tta:
            return vae.dec(regress(model, vae.enc(sar.float())*SCALE, cond, tb)/SCALE).clamp(-1,1)
        acc = None
        for k, f in D4:
            st = d4_fwd(sar, k, f)
            r = vae.dec(regress(model, vae.enc(st.float())*SCALE, cond, tb)/SCALE).clamp(-1,1)
            r = d4_inv(r, k, f)
            acc = r if acc is None else acc + r
        return (acc/len(D4)).clamp(-1,1)

    @torch.no_grad()
    def evaluate(pairs_list, tbs, tta, label):
        P, S = [], []
        for n, p in enumerate(pairs_list):
            d = DS([p])[0]
            sar = d["sar"].unsqueeze(0).to(device); gt = d["opt"].unsqueeze(0).to(device)
            c = torch.stack([embeds.get((p["season"], p["terrain"]), embeds["null"])]).to(device)
            preds = [predict(sar, c, tb, tta) for tb in tbs]
            out = torch.stack(preds).mean(0).clamp(-1,1)
            P.append(psnr(out, gt)); S.append(ssim_fn(out, gt).item())
        log(f"  {label:<34} PSNR {np.mean(P):7.3f} | SSIM {np.mean(S):.4f}")
        return float(np.mean(P)), float(np.mean(S))

    results = {}
    log(f"1. TIMESTEP SWEEP (n={N_SWEEP}, no TTA)")
    tb_scores = {}
    for tb in TB_GRID:
        tb_scores[tb] = evaluate(sweep_set, [tb], False, f"tb={tb}")
    best_tb = max(tb_scores, key=lambda k: tb_scores[k][0])
    log(f"  -> best single tb = {best_tb} ({tb_scores[best_tb][0]:.3f}), default was {TB0}\n")
    results["tb_sweep"] = {str(k): list(v) for k,v in tb_scores.items()}

    top3 = sorted(tb_scores, key=lambda k: -tb_scores[k][0])[:3]
    log(f"2. FINAL COMPARISON (n={N_EVAL}, held-out from sweep)")
    results["baseline"]      = evaluate(eval_set, [TB0],     False, f"baseline (tb={TB0})")
    results["best_tb"]       = evaluate(eval_set, [best_tb], False, f"best tb ({best_tb})")
    results["multi_tb"]      = evaluate(eval_set, top3,      False, f"multi-tb {sorted(top3)}")
    results["tta"]           = evaluate(eval_set, [best_tb], True,  f"8x D4 TTA (tb={best_tb})")
    results["tta_multi_tb"]  = evaluate(eval_set, top3,      True,  f"TTA x multi-tb (24 passes)")

    base = results["baseline"][0]
    log("\n" + "="*66)
    log(f"FREE-GAIN SUMMARY (vs baseline {base:.3f} dB)")
    log("="*66)
    for k in ["best_tb","multi_tb","tta","tta_multi_tb"]:
        log(f"  {k:<16} {results[k][0]:7.3f} ({results[k][0]-base:+.3f} dB) | "
            f"SSIM {results[k][1]:.4f}")
    best = max(["baseline","best_tb","multi_tb","tta","tta_multi_tb"],
               key=lambda k: results[k][0])
    log(f"\n  BEST CONFIG: {best} -> {results[best][0]:.3f} / {results[best][1]:.4f}")
    log(f"  (v3 reported {cfg.get('final_psnr',0):.3f} | SOTA 19.72 / 0.312)")
    if results["tta"][0] < base:
        log("  !! TTA NEGATIVE on a deterministic model — this should be impossible by "
            "construction. Investigate before trusting other ensemble results.")
    log("="*66)

    json.dump({"best_tb": best_tb, "top3_tb": sorted(top3),
               "results": {k: list(v) if isinstance(v, tuple) else v for k,v in results.items()}},
              open(os.path.join(OUT_DIR,"free_gains.json"),"w"), indent=2)
    log(f"saved -> {OUT_DIR}/free_gains.json")
    if HF_TOKEN:
        try:
            HfApi().upload_folder(folder_path=OUT_DIR, path_in_repo="free_gains",
                                  repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)
            log("-> HF/free_gains")
        except Exception as e: log(f"push failed: {e}")

if __name__ == "__main__":
    main()

Writing /content/free_gains.py


In [10]:

!python -u /content/diag_error_budget.py    # ~30-45 min, prints the VERDICT
!python -u /content/free_gains.py           # ~1-1.5 h, banks the free dB
# ---- disconnect the Colab runtime once done ----
from google.colab import runtime
runtime.unassign()

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
  VAE: using FINE-TUNED decoder from regressor_v3_final/base_vae_tuned.pth
  regressor_v3_final: scale=1.4010 tb=933 (reported 17.668/0.3110)

Error budget on n=400 val images

  [100/4